# Phase 3 - Level 2: France as the source, ARG / USA / CHN as targets  (notebook v3)

**Status: DESIGN NOT YET APPROVED.** Nothing trains until the instructor has signed the decisions *and* the approved design hash is pasted into the approval cell (cell 12). The scoring rule for the USA changed in this version (L2-D11), so the design hash is **new**: the old hash no longer matches and the new one must be approved. If results of earlier seeds exist, cell 12b re-scores them from the saved predictions (no retraining) before the run cell will continue.

| | |
|---|---|
| Source | France - 13 regions are the "states" of Level 1 (region of every row from the verified row map; 18 rows that cannot be placed with certainty are excluded and counted) |
| Targets | Argentina (soybeans 20 %) and USA (corn / soybeans / wheat 15 % each) carry the claims; China is descriptive only (rice exists in two regions). Belgium / Netherlands / UK: appendix label check only (cell 16), no claims |
| Training setup | identical to Level 1 (lr 1e-4, batch 256, <= 200 epochs, patience 15, 7-class head, budgets 1000-4000, k in {1,2,4,8}, seeds 0-4) |
| Seeds | 0-1 exploratory, 2-4 confirmatory. A claim passes only if the sign agrees in both groups **and** the confirmatory 95 % interval excludes 0. Otherwise the table says NOT REPLICATED / NOT TESTABLE |
| USA scoring (L2-D11) | macro-F1 over corn, soybeans, wheat, other; TRUE rice / sugarcane / cotton removed from the USA test set (paper Sec. 4.4 does this for sugarcane and cotton in FRA->USA; extending it to rice is the instructor's decision); predictions of rice / sugarcane / cotton count as "other". Every seed is re-scored from saved predictions and logged |
| USA ordering | cell 8b verifies the USA soybean (and corn / other) row order against the raw files, exactly, and prints the status; it is reported again in the closing cell |
| Exploratory (not claims) | closing cell: every region mix vs the mean of its own single regions and vs its best single region; fixed-pool budget effect; per-doubling slope with / without regions 75 and 76 |
| Level 1 | frozen (3,160 runs, SHA-256 `38d1876c...34f891`); this notebook never reads or writes that folder |

**What this notebook cannot do:** it cannot make the results support any claim. It implements the pre-registered claims and tests them honestly. Argentina verdicts stay as the code prints them. Run order: cells 1-11 (setup, tests, data, USA-order check, plan) -> instructor signs -> cell 12 (approval) -> cell 12b (re-score, once) -> cell 13 (run) -> cells 14-15 (summary, closing analysis) -> cell 16 (appendix label check).

## 1. Environment (Colab, GPU)

In [ ]:
# ======================================================================================
# [COLAB] ENVIRONMENT: mount Drive, paths (your existing files), libraries, GPU check.
# ======================================================================================
import os, sys, subprocess, importlib.util, glob, json, time
PLATFORM = 'colab'
assert importlib.util.find_spec('google.colab') is not None, 'This notebook is for Google Colab. Use the Kaggle notebook on Kaggle.'
from google.colab import drive
drive.mount('/content/drive')
_candidates = ['/content/drive/.shortcut-targets-by-id/1HTRsB7bxydWFYCZG8Wrj-8IQZJKj2Z8w/cropglobe_assignment',
               '/content/drive/MyDrive/cropglobe_assignment']
PROJECT_ROOT = next((p for p in _candidates if os.path.isdir(p)), None)
assert PROJECT_ROOT, f'project folder not found. Looked in: {_candidates}. Edit PROJECT_ROOT here.'
DATA_DIR   = os.path.join(PROJECT_ROOT, 'CropGlobe')                     # contains for_CropNet/
REPO_DIR   = os.path.join(PROJECT_ROOT, 'CropNet')
NAMESPACE  = 'phase3_level2_fra_source_v1'
OUT_DIR    = os.path.join(PROJECT_ROOT, 'outputs', NAMESPACE)            # NEW folder: the legacy results (outputs/phase3 + backup) are never touched
os.makedirs(OUT_DIR, exist_ok=True)
PLATFORM_TIME_LIMIT_MIN = None                                           # no artificial limit; re-run the run cell after a disconnect to resume
# Optional: the row-map CSV from the French matching run.  The 64 region blocks derived from it are EMBEDDED in the config cell, so this file is not required; if it is found
# (and its SHA-256 equals the verified one) it is re-read and must reproduce the embedded blocks.
ROW_MAP_PATH = None
import hashlib
for _p in glob.glob(os.path.join(PROJECT_ROOT, '**', 'row_map*.csv'), recursive=True):
    if os.path.getsize(_p) > 1_000_000 and hashlib.sha256(open(_p, 'rb').read()).hexdigest() == '93abeb51af5f68998818b770c8c5d853db5307e06f5aaf3dd1a6123d6ed69d9c': ROW_MAP_PATH = _p; break
print('row map (optional cross-check):', ROW_MAP_PATH or 'not found -> the embedded region blocks are used (nothing is lost)')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'scikit-learn', 'scipy', 'tqdm', 'statsmodels'], check=True)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE -> Runtime > Change runtime type > GPU')
assert torch.cuda.is_available(), 'enable a GPU runtime'
print('project:', PROJECT_ROOT, '\noutputs:', OUT_DIR)

## 2. Configuration (all locked settings, design decisions L2-D1..D12)

In [ ]:
# ======================================================================================
# CONFIGURATION  (pure python; nothing here touches the network or the GPU)
# LEVEL 2 (France as the SOURCE country; French regions play the role the US states played in Level 1).  Same training setup as Level 1; design in LEVEL2_PREREGISTRATION.
# Derived from the audited Level 1 v3.8 code; every difference is listed in the manifest amendments (L2-A*) and in CHANGES_LEVEL2.md.
# ======================================================================================
import os, sys, json, math, time, glob, re, csv, zlib, random, itertools, hashlib, datetime, pickle, warnings, traceback, subprocess, tempfile, io, contextlib
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from scipy import stats as scipy_stats
from scipy import optimize as scipy_opt
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, precision_recall_fscore_support

EXPERIMENT_VERSION = 'phase3_level2_fra_source_v1'          # NEW results folder: v3.7 trains up to 200 epochs, so results from the 100-epoch cap (v3.2-v3.6) must never be mixed with these
SPEC_TAG = 'level2_fra_source_prereg_v1'          # written into the manifest; a folder without it is NOT ours -> the run stops instead of mixing

# ---------------- design (spec section 4) ----------------
BUDGETS      = [1000, 2000, 3000, 4000]       # TOTAL labelled samples per run (train + validation); never shrunk
K_LIST       = [1, 2, 4, 8]
SEEDS        = [0, 1, 2, 3, 4]
AUG          = 'off'                          # Level 1 = augmentation OFF only (spec 2).  ON needs a step-matched control + approval.
MAX_COMBOS   = 10                             # k>=2: up to 10 random combinations; k=1 is a census of EVERY pool state
MIN_COMBOS   = 5                              # a (target,budget,k) cell with fewer combinations is recorded as infeasible
MIN_KPOINTS  = 3                              # a (target,budget) with fewer feasible k-points is infeasible as a whole
MIN_STATE_ROWS = 1500                         # source pool = French regions with >= 1500 (de-duplicated) rows
VAL_FRACTION = 0.10                           # validation = 10% of the budget, stratified by crop, deterministic
TARGETS_ORDER = ['ARG', 'USA', 'CHN']
TARGETS_TO_RUN = ['ARG', 'USA', 'CHN']
TARGET_ROLE  = {'ARG': 'main', 'USA': 'main', 'CHN': 'descriptive', 'AUS': 'appendix', 'FRA': 'source'}
# KEY-CROP share (fixed, identical at every k); everything else is sampled NATURALLY from the state's remaining rows.
KEY_SHARES   = {'ARG': {'soybeans': 0.20}, 'CHN': {'rice': 0.20}, 'USA': {'corn': 0.15, 'soybeans': 0.15, 'wheat': 0.15}, 'AUS': None}     # USA shares = the Level 1 France design (PROPOSED in decision 2; instructor to confirm)
K_MAX        = {'ARG': 8, 'USA': 8, 'CHN': 2, 'AUS': 2, 'FRA': 8}     # CHN: rice exists in only 2 French regions -> k <= 2
FRA_K8_MIN_POOL = 10                          # FRA k=8 only if the pool has >= 10 states
AUS_STATES   = ['LA', 'FL']                   # AUS = appendix census of {LA, FL}, k <= 2, no sweep (exempt from MIN_COMBOS / MIN_KPOINTS)
# ---- composition modes.  'key_plus_natural' = EXISTING baseline (fixed key-crop share + natural remainder).
#      'fixed_full'        = REVIEW-PROPOSED (complete crop composition fixed at every budget and k).  The VALUES are a scientific choice: they are
#                            NOT set here.  Proposals are printed with their capacity checks in the plan cell; nothing runs until you approve one.
COMPOSITION_MODES_TO_RUN = ['key_plus_natural']
FIXED_COMPOSITION_CHOICE = None               # e.g. 'A_key_plus_equal_rest' | 'B_equal_over_target_classes'  (only after approval)
ALIGNMENT_ORDERING_OVERRIDE = None             # V33-06: None = STOP on an ambiguous state alignment; set to a candidate ordering name only after reading cache/alignment_report.json
NOTEBOOK_VERSION = 'L2-v1'                       # display / run_environment only (not part of the locked design)
STATE_SELECTOR = 'random'                     # the baseline arm: random combinations (unchanged)
# ---- PHASE 4 (definition approved by the researcher): phenology-spectral diversity of the USA source states ------------------------------------------------------------------------------
SELECTION_ARMS = ['random', 'aug_on']   # v3.7 adds common / kcenter / aug_on (see below); 'random' is the baseline and is mandatory; ['random'] reproduces the pre-Phase-4 plan exactly
POOL_RULE = 'strict'                          # 'strict' (as before: a pool state must supply the WHOLE budget alone) | 'equal_share' (OPT-IN, not approved yet: a state in a k-combination only supplies its 1/k share; the pool then depends on k)
# ---- v3.7 (supervisor review of the 409-run seed-0 results): reporting rules, new tracks.  Every item is listed in CHANGELOG_v3_7.md ------------------------------------------------------------
# R2 metrics: ONE main metric and ONE secondary metric are reported.  The other score columns are still computed and stored (audit trail) but are not summarised unless REPORT_ALL_TRACKS = True.
REPORT_PRIMARY, REPORT_SECONDARY, REPORT_ALL_TRACKS = 'main_f1', 'paper_f1', False
# R3 France: rice is never in the training mix (rice F1 = 0 in every run), so a rule is stated IN ADVANCE: FRA's main metric is the coverage-conditional macro-F1 (classes with >= COVERAGE_MIN_TRAIN training samples; the paper does the same for France->USA).
#    ARG / CHN main metric = binary key-crop macro-F1 (soybeans / rice vs. other) = the existing primary_f1.   'coverage_conditional' | 'primary' (= no rule; FRA scored on all classes)
FRA_MAIN_RULE = 'coverage_conditional'     # applies to the multi-class targets in FRA_LIKE_TARGETS (Level 2: USA): sugarcane / cotton never occur in the French source, so the main metric is the coverage-conditional macro-F1, as for France in Level 1
# R4 which input features the CropNet runs use (written into every results row)
INPUT_FEATURES = 'Sentinel-2 5-day medians (CropGlobe for_CropNet/median_S2_*_month1-12_day5.npz; 10 bands x 73 steps; official DOY 121-334 window = 43 steps); AlphaEarth/Presto/Harmonic NOT used'
# R5 fixed common pool: the pool of source units shrinks as the budget grows (ARG 16 -> 6), which confounds budget comparisons.  Arm 'common' re-draws the combinations of EVERY budget from the states eligible at ALL budgets of the target.
# R6 sample-level selection: arm 'kcenter' = same state combinations and same per-state/per-crop counts as 'random', but the rows inside each (state, crop) pool are chosen by greedy k-center (farthest-first) in a PCA space of the USA window features.
KC_TARGETS, KC_COMBOS, KC_DIMS, KC_MAX_CAND = [], 5, 16, 20000     # Level 2: no k-center (decision 4)     # k>=2: at most KC_COMBOS of the random-arm combinations (deterministic), k=1: the whole census
# R7 augmentation ON (authors' CropGlobeDataset aug_mode='augm', pinned commit; training set = original + AUG_REPEAT augmented copies; validation/test never augmented).  Arm 'aug_on' = the same combinations as the random arm.
AUG_TARGETS, AUG_BUDGETS, AUG_COMBOS, AUG_REPEAT = ['ARG'], [1000, 2000], 5, 2
# ---- v3.8 (instructor review of the 1,475-run file; seeds 0 and 1 complete).  These keys are deliberately NOT in LOCKED: the locked design of the runs already on disk is unchanged, so the same results.csv resumes.
#      Every item is written into manifest.json as a dated AMENDMENT (V38_AMENDMENTS) the first time the run cell executes, so nothing about the design changes silently.
KC_SEEDS = [0, 1]          # k-center sample selection runs for these seeds only: seeds 0-1 gave a clean null (-0.47 points over 310 paired runs).  Seeds >= 2 skip the k-center cells.  Set KC_SEEDS = SEEDS to restore every seed.
AUG_CONTROL = True         # steps-matched control for augmentation.  Arm 'aug_copy' = the SAME cells as 'aug_on', but the training rows are COPIED (1 + AUG_REPEAT)x WITHOUT any transform: same rows per epoch, same optimizer steps, no augmentation.
USA_SCORING = 'fixed4_clean'                 # L2-D11 (supersedes L2-D9): USA main metric = macro-F1 over the FIXED classes below, with true rice / sugarcane / cotton removed from the test set.  Not in LOCKED (the results file resumes); it IS in the design hash.
USA_FIXED_CLASSES = ['corn', 'soybeans', 'wheat', 'other']
USA_DROP_TRUE_CLASSES = ['rice', 'sugarcane', 'cotton']          # true samples of these classes are REMOVED from the USA test set before scoring (paper sec. 4.4 excludes sugarcane / cotton for FRA -> USA; rice added by the instructor)
USA_REASSIGN_PRED_TO_OTHER = ['rice', 'sugarcane', 'cotton']      # predictions of these classes (on the remaining samples) are reassigned to 'other'
PREREG_TEXT = ("Level 2 (France as source).  Seeds 0-1 exploratory, seeds 2-4 confirmatory.  Claims (main targets ARG and USA): L2-C1 mixed-model effect of log2(k) on main F1 > 0 (random intercept per region combination, budget and seed adjusted); "
    "L2-C2 gain from the fewest to the most regions > 0 at each budget; L2-C3 steps-matched augmentation aug_on - aug_copy (Argentina, budgets 1000/2000) reported with its interval, no direction assumed.  China is DESCRIPTIVE ONLY (rice exists in 2 French regions): no claim.  "
    "A claim passes only if its sign agrees in the exploratory and the confirmatory seeds AND the confirmatory 95% interval excludes 0.  A failed claim is REPORTED AS NOT REPLICATED: no seed or run is dropped, replaced or re-run to change an outcome, except to repair a recorded error.  "
    "Decisions were fixed before any run; they were not chosen from results.")
V38_AMENDMENTS = [      # name kept for compatibility with the manifest code; these are the LEVEL 2 design decisions (see LEVEL2_decisions_for_approval.pdf)
    dict(id='L2-D1', item='source regions', text='France, region = unit. Pool rule unchanged from Level 1 (>= MIN_STATE_ROWS rows; a region must supply the whole budget alone). Region of every row from the verified row map (FRA_join_verification); rows without a certain region are excluded and counted.'),
    dict(id='L2-D2', item='targets', text='Main: ARG (soybeans 20%) and USA (corn, soybeans, wheat 15% each; shares proposed by the analyst, pending instructor confirmation). BEL/NLD/GBR are NOT included (no verified label sets / key shares).'),
    dict(id='L2-D3', item='China', text='CHN descriptive only: k <= 2 (rice exists only in regions 76 and 93); exempt from MIN_COMBOS / MIN_KPOINTS; no claim.'),
    dict(id='L2-D4', item='arms', text='random arm + steps-matched copy control on ARG budgets 1000/2000 (aug_on and aug_copy, 5 combinations); no k-center, diversity or common-pool arms.'),
    dict(id='L2-D5', item='replication rule', text='seeds 0-1 exploratory, 2-4 confirmatory; same pass rule as Level 1.'),
    dict(id='L2-D6', item='reference model', text='statsmodels MixedLM primary (fallback optimisers), own REML alongside; tolerance coef/SE 0.02.'),
    dict(id='L2-D7', item='metric for USA target', text='multi-class macro-F1 with the coverage-conditional rule (classes with >= 30 training samples), as for France in Level 1; no remap (the authors release no USA rule).'),
    dict(id='L2-D8', item='untestable comparisons', text='L2-C2 is judged only where an interval exists. At ARG budget 1000 the largest k (8) enumerates ALL combinations of the 9-region pool and k=1 is a census, so no sampling interval exists by construction: the code prints NOT TESTABLE there (never PASS, never "interval includes 0"). L2-C1 (mixed model over all cells) is the primary test of the effect of k.'),
    dict(id='L2-D9', item='USA scoring (changed AFTER seed 0 was seen)', text='At the instructor\'s review of seed 0 (306 runs): the USA target is scored on a fixed class set (corn, soybeans, wheat, other); rice is not scored and rice predictions are reassigned to other. Reason: rice exists in only 2 French regions, so under the coverage-conditional rule the scored class set depended on the region mix (rice scored in 15% of k=1, 58% of k=4 combinations; mean 60.7 when rice was scored vs 54.0). Seed-0 USA rows were re-scored from their saved predictions (no retraining; original kept). This change was made after seed 0 was seen; it was requested by the instructor on the grounds above, not chosen from the size of any claim.'),
    dict(id='L2-D10', item='process note', text='Seed 0 (306 of 319 planned runs) was run before the instructor\'s written sign-off on the decision sheet. Seeds 0-1 are exploratory under the pre-registered rule, so no claim rests on them. The plan (design hash, now including the scoring rule) is frozen before seeds 2-4. Any further change made after seeing seed 0 must be logged here as such.'),
    dict(id='L2-D11', item='USA scoring v2 (changed AFTER seeds 0-3 were seen)', text='Instructor review of the interim report: true rice, sugarcane and cotton samples are REMOVED from the USA test set and predictions of rice / cotton / sugarcane are reassigned to other; the main metric is macro-F1 over corn, soybeans, wheat, other (supersedes L2-D9, where true labels were not changed). Basis: the paper (sec. 4.4) excludes sugarcane and cotton from evaluation for FRA -> USA because the source lacks them; the instructor extends this to rice, which is absent from most training mixes (2 of 13 French regions). ALL seeds re-scored from the saved predictions (no retraining; originals kept; every earlier value kept in a column). Made after seeds 0-3 were seen, at the instructor\'s request, not chosen from the size of any claim.'),
    dict(id='L2-D12', item='exploratory analyses added after seeing seeds 0-3', text='(i) mix versus the mean of its own single regions and versus its best single region (every k>=2 combination at the same budget); (ii) the budget effect on a FIXED pool of regions (those available at every budget); (iii) subset analyses with / without regions 75 and 76. All are labelled exploratory / post hoc; none replaces a pre-registered claim.'),
    dict(id='L2-P1', item='analysis protocol', text=PREREG_TEXT),
]
# APPROVED['aug_on'] (below) stays False: the aug-ON runs are PENDING (shown in the plan, not run) until you set it True.  They use 3x the optimizer steps per epoch, so compare them with aug-OFF knowing that.
# ---- EXTENSION PROFILE (appendix, opt-in): CHN and FRA at budgets 3000 / 4000.  PROFILE = 'main' leaves the locked study EXACTLY as it was ----------------------------------------
PROFILE = 'main'                              # 'main' | 'extension'.  'extension' runs ONLY CHN and USA at EXT_BUDGETS, in its own sub-folder (<outputs>/extension) with its own manifest; nothing is mixed with the main results
EXT_TARGETS = ['CHN', 'USA']                  # Level 2: dormant (PROFILE='main'); FRA is the SOURCE now, so the Level 1 value ['CHN','FRA'] could not run
EXT_BUDGETS = [3000, 4000]
EXT_MIN_CONTRIB = 0.5                         # every state of a combination must be able to supply at least this fraction of its EQUAL share of every fixed crop and of the remainder (so a 4-state combination cannot be one big state in disguise)
ALLOC_RULE = 'equal'                          # set by PROFILE: 'equal' (main, unchanged) | 'capacity' (extension: as equal as each state's capacity allows; a short state gives its surplus to the others)
EXT_LOCKED = ['PROFILE', 'EXT_TARGETS', 'EXT_BUDGETS', 'EXT_MIN_CONTRIB', 'ALLOC_RULE']
N_ARM = 10                                    # combinations per arm (k>=2); the diverse and clustered arms need C(pool,k) >= 2*N_ARM so that they are disjoint
ENUM_LIMIT = 50000                            # score EVERY combination when C(pool,k) <= this; otherwise score a fixed random sample of SCORE_SAMPLE distinct combinations (seeded by target,budget,k)
SCORE_SAMPLE = 20000
MIN_PROFILE_N = 20                            # a (state, crop) profile needs >= this many USA samples; states without a profile for a fixed crop leave the matched pool of ALL arms for that target
DIVERSITY_SPACE = ('phenology_spectral: per (state, crop) mean of globally standardised Sentinel-2 features (10 bands x all time steps); state distance = RMS difference of the mean profiles; '
                   'set score = mean pairwise state distance, averaged over the fixed (key) crops of the target design; USA source features only, no target data or scores')
CLASS_WEIGHT_SOURCE = 'train_val'             # = the authors' script: compute_class_weight('balanced') over the WHOLE source set they train on (here: the drawn budget sample = train+val). 'train' = training labels only (earlier review request).
FEATURE_SOURCE = 'Sentinel-2 5-day median time series, 10 bands (median_S2_*_month1-12_day5.npz); NOT AlphaEarth embeddings (64-d)'
TARGET_CROPS = {'ARG': ['soybeans'], 'CHN': ['rice'], 'FRA': ['corn', 'soybeans', 'wheat'], 'USA': ['corn', 'soybeans', 'wheat'], 'AUS': ['sugarcane', 'cotton']}   # crops whose count enters the regression
AUS_SPEC_CONFIRMED = False                    # !! the spec gives NO key-crop share for AUS.  AUS stays OUT of the run until you decide (see plan table).
AUS_KEY_SHARES_IF_CONFIRMED = None            # e.g. {'sugarcane': 0.20}; None = fully natural sampling.  Set deliberately.

# ---------------- training (spec 2; do not change) ----------------
NUM_CLASSES  = 7                              # CropNet(data_channel=1, classes=7): ALL 7 outputs trained, never a binary head
LR, WEIGHT_DECAY, BATCH_SIZE = 1e-4, 0.0, 256 # Adam, no scheduler  (== authors' run_CropNet_AUGM.py: --lr 1e-4, --batch_size 256, optim.Adam(model.parameters(), lr) i.e. default weight_decay 0, no scheduler)
# What the authors' script (x-ytong/CropNet run_CropNet_AUGM.py, read in this project) does, and where this notebook DEVIATES on purpose (printed in the plan cell):
AUTHORS_SCRIPT = dict(lr=1e-4, batch_size=256, num_epochs=50, val_ratio=0.2, num_runs=10, base_seed=81, optimizer='Adam(lr), weight_decay default 0', scheduler=None,
                      loss="CrossEntropyLoss(weight=compute_class_weight('balanced', classes=np.unique(labels of the whole source set)))", model='CropNet(data_channel=1, classes=len(train_index))',
                      train_aug='augm (this notebook: none at Level 1)', val_test_aug='none')
DEVIATIONS_FROM_AUTHORS = {'epochs': '100 with early stopping (authors: 50, no early stopping)', 'val_ratio': '0.10 of the budget, stratified (authors: 0.20)', 'seeds': '0-4 (authors: 82-91)',
                           'train_aug': 'none at Level 1 (authors AUGM: augm)', 'class_weights': 'authors-equivalent: drawn sample (train+val); zero-count crops -> weight 0 because the head has 7 outputs'}
MAX_EPOCHS, PATIENCE, MIN_EPOCHS, MIN_DELTA = 200, 15, 20, 0.0    # TOTAL epochs = 100 (your instruction) with early stopping INSIDE them on VALIDATION macro-F1 (strictly greater, best weights restored). The authors' script runs 50 epochs, no early stopping.
UNDERTRAINED_EPOCH = 90                       # ASSUMPTION: 0.9 x the 100-epoch cap (was 45 of 50).  best_epoch (1-based) >= 90 -> flag 'possibly_undertrained'
RESTORE_TOL  = 1e-6                           # restored model must reproduce its best validation score
EXPECTED_CROPNET_COMMIT = 'bdcf63ec1704a412da73c534c579fe08c5d2fa9b'

# ---------------- label space ----------------
CROP_LABELS = {0: 'corn', 1: 'soybeans', 2: 'rice', 3: 'wheat', 4: 'sugarcane', 5: 'cotton', 6: 'other'}
CROP_ID     = {v: k for k, v in CROP_LABELS.items()}
CROP_ORDER  = [CROP_LABELS[i] for i in range(7)]
RAW_TO_CROP = {'corn': 'corn', 'soybeans': 'soybeans', 'rice': 'rice', 'springwheat': 'wheat', 'winterwheat': 'wheat',
               'wheat': 'wheat', 'sugarcane': 'sugarcane', 'cotton': 'cotton', 'other': 'other'}

# ---------------- LEVEL 2 SOURCE (France) ----------------
SOURCE = 'FRA'; SOURCE_UNIT_NAME = 'region'
SOURCE_UNITS = ['11', '24', '27', '28', '32', '44', '52', '53', '75', '76', '84', '93', '94']       # the 13 metropolitan regions found in the raw file names (FRA_join_verification.md)
FRA_LIKE_TARGETS = ['FRA', 'USA']              # multi-class targets scored with the coverage-conditional main metric
EXEMPT_RULES = ['AUS', 'CHN']                  # targets exempt from MIN_COMBOS / MIN_KPOINTS in the plan (AUS: Level 1 appendix census; CHN: descriptive census of the 2 rice regions)
DESCRIPTIVE_TARGETS = ['CHN']                  # run, reported, but NO claim; exempt from MIN_COMBOS / MIN_KPOINTS (census of the 2 rice regions)
CLAIM_TARGETS = ['ARG', 'USA']
# exact row map produced by the student's Colab run of the raw<->npz matcher (cell 9 of Level1_closing_and_Level2_probe.ipynb).  The file is identified by its hash: a different file STOPS the run.
FRA_ROW_MAP_SHA256 = '93abeb51af5f68998818b770c8c5d853db5307e06f5aaf3dd1a6123d6ed69d9c'
FRA_ROW_MAP_ROWS, FRA_ROW_MAP_AMBIGUOUS = 108599, 96
# raw points per (region, crop token) in that row map (before de-duplication) -- checked against the file, so a wrong or edited map cannot pass
FRA_REGION_TOTALS = {'11': 3351, '24': 9546, '27': 12164, '28': 6493, '32': 7043, '44': 12968, '52': 4872, '53': 5639, '75': 17809, '76': 13224, '84': 10639, '93': 4275, '94': 576}
FRA_BLOCKS = [  # (crop token, region, first packaged row, last packaged row) of each of the 64 raw files, derived from the verified row map (sha256 above); rows outside every block are excluded
    ('corn', '11', 0, 755), ('corn', '24', 756, 3150), ('corn', '27', 3151, 5313), ('corn', '28', 5314, 7365), ('corn', '32', 7366, 9536),
    ('corn', '44', 9537, 12674), ('corn', '52', 12675, 14795), ('corn', '53', 14796, 16598), ('corn', '75', 16599, 20759),
    ('corn', '76', 20760, 22637), ('corn', '84', 22638, 24749), ('corn', '93', 24750, 24882), ('corn', '94', 24883, 24901),
    ('soybeans', '11', 24902, 25688), ('soybeans', '24', 25689, 27359), ('soybeans', '27', 27360, 31374), ('soybeans', '28', 31375, 31407),
    ('soybeans', '32', 31408, 31617), ('soybeans', '44', 31636, 33762), ('soybeans', '52', 33763, 34259), ('soybeans', '53', 34260, 34396),
    ('soybeans', '75', 34397, 39693), ('soybeans', '76', 39694, 43573), ('soybeans', '84', 43574, 45740), ('soybeans', '93', 45741, 45903),
    ('rice', '76', 45904, 46755), ('rice', '93', 46756, 48197), ('springwheat', '11', 48198, 48310), ('springwheat', '24', 48311, 48651),
    ('springwheat', '27', 48652, 49009), ('springwheat', '28', 49010, 49161), ('springwheat', '32', 49162, 49367),
    ('springwheat', '44', 49368, 49787), ('springwheat', '52', 49788, 49894), ('springwheat', '53', 49895, 49926),
    ('springwheat', '75', 49927, 50024), ('springwheat', '76', 50025, 50173), ('springwheat', '84', 50174, 50251),
    ('springwheat', '93', 50252, 50349), ('winterwheat', '11', 50350, 51205), ('winterwheat', '24', 51206, 53707),
    ('winterwheat', '27', 53708, 56165), ('winterwheat', '28', 56166, 58284), ('winterwheat', '32', 58285, 60510),
    ('winterwheat', '44', 60511, 63987), ('winterwheat', '52', 63988, 66134), ('winterwheat', '53', 66135, 67910),
    ('winterwheat', '75', 67911, 71112), ('winterwheat', '76', 71113, 73366), ('winterwheat', '84', 73367, 75350),
    ('winterwheat', '93', 75351, 75878), ('winterwheat', '94', 75879, 75883), ('other', '11', 75884, 76722), ('other', '24', 76723, 79359),
    ('other', '27', 79360, 82529), ('other', '28', 82530, 84666), ('other', '32', 84667, 86896), ('other', '44', 86897, 90684),
    ('other', '53', 90685, 92575), ('other', '75', 92576, 97626), ('other', '76', 97627, 101837), ('other', '84', 101838, 106135),
    ('other', '93', 106136, 108046), ('other', '94', 108047, 108598),
]
FRA_ASSIGNED_PER_REGION = {'11': 3351, '24': 9546, '27': 12164, '28': 6493, '32': 7043, '44': 12950, '52': 4872, '53': 5639, '75': 17809, '76': 13224, '84': 10639, '93': 4275, '94': 576}     # packaged rows that lie inside a file block (108,581); the other 18 rows are excluded
# Level 1 training setup, which Level 2 must keep unchanged (reviewer point); asserted again in the tests
LEVEL1_TRAINING_SETUP = dict(NUM_CLASSES=7, LR=1e-4, WEIGHT_DECAY=0.0, BATCH_SIZE=256, MAX_EPOCHS=200, PATIENCE=15, MIN_EPOCHS=20, MIN_DELTA=0.0, VAL_FRACTION=0.10, BUDGETS=[1000, 2000, 3000, 4000], K_LIST=[1, 2, 4, 8], SEEDS=[0, 1, 2, 3, 4], MAX_COMBOS=10, MIN_COMBOS=5, MIN_KPOINTS=3, MIN_STATE_ROWS=1500, COVERAGE_MIN_TRAIN=30, POOL_RULE='strict', CLASS_WEIGHT_SOURCE='train_val')
# INSTRUCTOR APPROVAL GATE.  The plan cell always runs (nothing trains).  The run cell refuses to start until this is True AND the design hash below is the one that was approved.
INSTRUCTOR_APPROVED = False
APPROVED_DESIGN_SHA256 = ''                    # paste the design hash printed by the plan cell once the instructor has signed the decisions
# paper (arXiv 2509.03497 v3) Table 2 -- raw class counts.  Raw counts and label sets are ASSERTED against this (stop on mismatch).
TABLE2 = {
    'ARG': dict(corn=1630,  soybeans=1808,  rice=1319,  wheat=0,     sugarcane=1253, cotton=1443,  other=2425),
    'AUS': dict(corn=0,     soybeans=0,     rice=0,     wheat=0,     sugarcane=5098, cotton=4825,  other=8938),
    'CHN': dict(corn=1344,  soybeans=1200,  rice=1156,  wheat=0,     sugarcane=0,    cotton=0,     other=1502),
    'FRA': dict(corn=24902, soybeans=21002, rice=2294,  wheat=27686, sugarcane=0,    cotton=0,     other=32715),
    'USA': dict(corn=22224, soybeans=17093, rice=17613, wheat=23226, sugarcane=9897, cotton=14799, other=28427),
}
TABLE2_TOTAL = {'ARG': 9878, 'AUS': 18861, 'CHN': 5202, 'FRA': 108599, 'USA': 133279}
assert all(sum(TABLE2[c].values()) == TABLE2_TOTAL[c] for c in TABLE2), 'Table 2 transcription error'
DEFINED_CLASSES = {t: [c for c in CROP_ORDER if TABLE2[t][c] > 0] for t in TABLE2}     # label set per country = classes with count > 0 in Table 2

# ---------------- scoring: the tracks are kept SEPARATE and separately named ----------------
REMAP_RULES = ['existing_release', 'paper_text_v3', 'review_targetmap']
REVIEW_RULE = 'review_targetmap'
# (results column, human label).  Nothing below is ever averaged with, or compared against, another track.
TRACKS = [('primary_f1', 'EXISTING primary (binary: ARG soybeans / CHN rice; AUS & FRA: existing release remap)'),
          ('binary_f1', 'EXISTING binary-collapse view (ARG soybeans, CHN rice)'),
          ('paper_f1', 'EXISTING release remap (AUS keep {sugarcane,cotton}; CHN keep {corn,soybeans,rice}; ARG wheat->other; FRA none)'),
          ('sevenclass_f1', 'EXISTING seven-class raw (no remap; macro over classes present in the target truth)'),
          ('papertext_f1', 'paper_text_v3: literal reading of paper sec.4.4 (AUS/CHN undefined->other; ARG/FRA no alignment)'),
          ('review_f1', 'REVIEW-PROPOSED: target-map remap (every class the target map does not define -> other) + macro-F1 over np.unique(y_true)')]
TRACK_PAIR = [('primary_f1', 'EXISTING'), ('review_f1', 'REVIEW-PROPOSED')]     # the two equally-ranked headline tracks; reported side by side, never pooled or ranked
# ---------------- scoring (spec 3) ----------------
PRIMARY_VIEW = {'ARG': ('binary', 'soybeans'), 'CHN': ('binary', 'rice'), 'AUS': ('paper', None), 'FRA': ('paper', None), 'USA': ('paper', None)}
BINARY_TARGETS = {t: v[1] for t, v in PRIMARY_VIEW.items() if v[0] == 'binary'}
COVERAGE_MIN_TRAIN = 30                        # 'coverage-conditional' macro-F1: classes with >= 30 TRAINING samples (reported next to, never instead of, the main score)
SCORER_TOL = 1e-12                             # sklearn scorer vs from-scratch numpy scorer
# paper Table 3, CropNet_AUGM, FULL USA source, mF1 (%).  CONTEXT ONLY: NOT comparable to this design (budgets of 1-4k, augmentation off). No flag is derived from it.
TABLE3_REF_MF1 = {'ARG': 75.63, 'AUS': 88.38, 'FRA': 90.00}      # (the paper has no USA->CHN experiment)

# ---------------- USA state-table geography guard (STOP if violated) ----------------
GEOGRAPHY_RULES = {'rice': ['76', '93']}              # France: rice exists only in regions 76 and 93 (row map); sugarcane / cotton do not exist in France (checked in the data cell)
GEOGRAPHY_REPORT_ONLY = ['sugarcane', 'cotton']             # observed state set is printed, not asserted
GEOGRAPHY_MAX_OUTSIDE_ROWS = 0                 # strict (as specified).  USDA lists FL/TN as minor rice states, so a legitimate trace could trip this: raise it only deliberately.
# ---------------- gates (spec 6) ----------------
VAL_F1_WARN = 0.60                             # SKILL outcome: warn only, never abort
MAX_CONSEC_INTEGRITY = 5                       # abort only after this many CONSECUTIVE integrity failures
INTEGRITY_CHECKS = ['I1_true_classes_ge2', 'I2_predictions_valid', 'I3_split_exact', 'I4_restore_matches_best', 'I5_finite', 'I6_scorers_agree']

# ---------------- approvals (spec 9): nothing trains until you flip these, in order ----------------
# ---- AUTO MODE: the whole experiment (all seeds, aug OFF and aug ON cells) runs from the run cell with no manual switches.  AUTO_MODE = False restores the manual approvals below.
#      What stays automatic-STOP either way: the test gate, the smoke test, data / geography / state-alignment checks, 5 consecutive integrity failures, and a seed with any error / integrity failure (the next seed does not start).
AUTO_MODE, AUTO_MAX_HOURS = True, None          # AUTO_MAX_HOURS: e.g. 11 on Kaggle (session limit) -> pauses cleanly between runs; run the same cell again to resume
APPROVED = {'plan': False, 'fixed_composition': False, 'seed0_gate_report': False, 'seed1_gate_report': False, 'seed2_gate_report': False, 'seed3_gate_report': False, 'aug_on': False}
RUN_SEED = 0                                   # which seed the run cell executes this time (seed 0 first; seed s>0 needs APPROVED[f'seed{s-1}_gate_report'])

LOCKED = ['BUDGETS', 'K_LIST', 'SEEDS', 'AUG', 'MAX_COMBOS', 'MIN_COMBOS', 'MIN_KPOINTS', 'MIN_STATE_ROWS', 'VAL_FRACTION', 'KEY_SHARES', 'K_MAX', 'FRA_K8_MIN_POOL',
          'AUS_STATES', 'NUM_CLASSES', 'LR', 'WEIGHT_DECAY', 'BATCH_SIZE', 'MAX_EPOCHS', 'PATIENCE', 'MIN_EPOCHS', 'MIN_DELTA', 'PRIMARY_VIEW', 'COVERAGE_MIN_TRAIN',
          'EXPECTED_CROPNET_COMMIT', 'AUS_SPEC_CONFIRMED', 'AUS_KEY_SHARES_IF_CONFIRMED', 'TARGETS_TO_RUN', 'TARGET_CROPS', 'UNDERTRAINED_EPOCH', 'REVIEW_RULE', 'TRACKS', 'GEOGRAPHY_RULES', 'CLASS_WEIGHT_SOURCE', 'COMPOSITION_MODES_TO_RUN', 'FIXED_COMPOSITION_CHOICE', 'STATE_SELECTOR', 'FEATURE_SOURCE', 'REMAP_RULES',
          'SELECTION_ARMS', 'POOL_RULE', 'N_ARM', 'ENUM_LIMIT', 'SCORE_SAMPLE', 'MIN_PROFILE_N', 'DIVERSITY_SPACE',
          'FRA_MAIN_RULE', 'INPUT_FEATURES', 'KC_TARGETS', 'KC_COMBOS', 'KC_DIMS', 'KC_MAX_CAND', 'AUG_TARGETS', 'AUG_BUDGETS', 'AUG_COMBOS', 'AUG_REPEAT',
          'SOURCE', 'SOURCE_UNITS', 'FRA_LIKE_TARGETS', 'DESCRIPTIVE_TARGETS', 'CLAIM_TARGETS', 'FRA_ROW_MAP_SHA256', 'LEVEL1_TRAINING_SETUP']
if AUTO_MODE:
    APPROVED.update(plan=True, aug_on=True); print('*** AUTO_MODE: plan and augmentation-ON runs are pre-approved; seed gate reports are approved automatically when clean. The plan table is still printed. Set AUTO_MODE = False for manual approval. ***')
assert POOL_RULE in ('strict', 'equal_share') and 'random' in SELECTION_ARMS and set(SELECTION_ARMS) <= {'random', 'diverse', 'clustered', 'common', 'kcenter', 'aug_on'}, 'SELECTION_ARMS / POOL_RULE invalid'
assert PROFILE in ('main', 'extension'), "PROFILE must be 'main' or 'extension'"
MAIN_OUT_DIR = globals().get('MAIN_OUT_DIR') or globals().get('OUT_DIR')      # the main results folder (never written by the extension)
MAIN_DEFAULTS = dict(TARGETS_TO_RUN=list(TARGETS_TO_RUN), BUDGETS=list(BUDGETS), SELECTION_ARMS=list(SELECTION_ARMS))      # what the MAIN study uses (the test gate always runs against this)
if PROFILE == 'extension':
    TARGETS_TO_RUN = list(EXT_TARGETS); BUDGETS = list(EXT_BUDGETS); ALLOC_RULE = 'capacity'; SELECTION_ARMS = ['random']; LOCKED = LOCKED + [k_ for k_ in EXT_LOCKED if k_ not in LOCKED]
    if MAIN_OUT_DIR: OUT_DIR = os.path.join(MAIN_OUT_DIR, 'extension'); os.makedirs(OUT_DIR, exist_ok=True)
    print(f'*** PROFILE = extension (APPENDIX): {TARGETS_TO_RUN} at budgets {BUDGETS}; random arm only; outputs in {globals().get("OUT_DIR")}.  These results are NOT part of the main study and must not be pooled with it. ***')
else:
    ALLOC_RULE = 'equal'
    if MAIN_OUT_DIR: OUT_DIR = MAIN_OUT_DIR                                                                       # switching back to the main profile re-points the outputs to the main folder
IN_NOTEBOOK = 'ipykernel' in sys.modules
print('config OK |', EXPERIMENT_VERSION, '| spec', SPEC_TAG)

## 3. Core code

In [ ]:
# ======================================================================================
# CORE LOGIC (pure numpy / pandas / sklearn -- no torch, no network).  Unit-tested in the next cell.
# ======================================================================================
import shutil
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()

class TooFewClasses(Exception): pass
class InfeasibleSplit(Exception): pass
class IntegrityFailure(Exception): pass

def _check(cond, msg):
    if not cond:                      # cannot be disabled with python -O
        raise AssertionError(msg)

def stable_seed(*parts):
    return zlib.crc32('|'.join(str(p) for p in parts).encode()) & 0x7fffffff

def atomic_write(path, writer, mode='wb'):
    # write to a temp file in the same folder, fsync, then os.replace: a reader never sees a half-written file
    d = os.path.dirname(path); os.makedirs(d, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=d, prefix='.tmp_', suffix=os.path.basename(path))
    try:
        with os.fdopen(fd, mode) as f:
            writer(f); f.flush(); os.fsync(f.fileno())
        os.replace(tmp, path)
    except BaseException:
        if os.path.exists(tmp): os.remove(tmp)
        raise

# ------------------------------------------------------------------------------ SCORING (spec 3)
def binary_collapse(y_true, y_pred, positive_name):
    # EVALUATION VIEW ONLY.  Every non-positive class, true OR predicted, is 'other'.  Nothing is dropped, nothing is set to -1.
    pid = CROP_ID[positive_name]
    return (np.asarray(y_true) == pid).astype(int), (np.asarray(y_pred) == pid).astype(int)

def remap_existing_release(y_pred, target):
    # EXISTING rule, PRESERVED (transcribed from the spec's description of the authors' released util/label.py::remap_prediction; Part B compares it with the
    # released function in Colab and is a real gate):  AUS keep {sugarcane, cotton} | CHN keep {corn, soybeans, rice} | ARG only wheat -> other | FRA no remap
    y = np.asarray(y_pred).astype(int); o = CROP_ID['other']
    if target == 'AUS': return np.where(np.isin(y, [CROP_ID['sugarcane'], CROP_ID['cotton']]), y, o)
    if target == 'CHN': return np.where(np.isin(y, [CROP_ID['corn'], CROP_ID['soybeans'], CROP_ID['rice']]), y, o)
    if target == 'ARG': return np.where(y == CROP_ID['wheat'], o, y)
    if target in FRA_LIKE_TARGETS: return y.copy()
    raise ValueError(f'unknown target {target}')

def _keep_defined(y_pred, target):
    keep = [CROP_ID[c] for c in DEFINED_CLASSES[target]]; y = np.asarray(y_pred).astype(int)
    return np.where(np.isin(y, keep), y, CROP_ID['other'])

def remap_paper_text_v3(y_pred, target):
    # VERIFIED against arXiv 2509.03497v3 sec.4.4 "Category Alignment": only when AUS or CHN are the target are predictions of classes outside their label set
    # reassigned to Other; "no additional alignment is required for other transfer scenarios" (ARG, FRA: predictions untouched).
    if target in ('AUS', 'CHN'): return _keep_defined(y_pred, target)
    if target in ('ARG', 'FRA', 'USA'): return np.asarray(y_pred).astype(int).copy()
    raise ValueError(f'unknown target {target}')

def remap_review_targetmap(y_pred, target):
    # REVIEW-PROPOSED: for EVERY target, a prediction of a crop that the target map does not define (paper Table 2 label set) is mapped to Other.
    #   ARG: wheat | CHN: wheat, sugarcane, cotton | AUS: corn, soybeans, rice, wheat | FRA: sugarcane, cotton
    if target not in DEFINED_CLASSES: raise ValueError(f'unknown target {target}')
    return _keep_defined(y_pred, target)

REMAPS = {'existing_release': remap_existing_release, 'paper_text_v3': remap_paper_text_v3, 'review_targetmap': remap_review_targetmap}

def remap_difference_table():
    # which targets / predicted classes the three rules treat differently (printed in the plan cell so the disagreement is visible, not silent)
    rows = []
    for t in ['ARG', 'CHN', 'AUS', 'FRA', 'USA']:
        allc = np.arange(7); maps = {k: f(allc, t) for k, f in REMAPS.items()}
        rows.append(dict(target=t, **{k: ','.join(f'{CROP_LABELS[a]}->{CROP_LABELS[b]}' for a, b in zip(allc, m) if a != b) or '(none)' for k, m in maps.items()},
                         existing_eq_paper_text=bool((maps['existing_release'] == maps['paper_text_v3']).all()), existing_eq_review=bool((maps['existing_release'] == maps['review_targetmap']).all())))
    return pd.DataFrame(rows)

def _np_prf(yt, yp, labels):
    # scorer #2: from scratch, numpy only (no sklearn)
    P, R, F = [], [], []
    for c in labels:
        tp = float(np.sum((yt == c) & (yp == c))); fp = float(np.sum((yt != c) & (yp == c))); fn = float(np.sum((yt == c) & (yp != c)))
        P.append(tp / (tp + fp) if tp + fp > 0 else 0.0); R.append(tp / (tp + fn) if tp + fn > 0 else 0.0)
        d = 2 * tp + fp + fn; F.append(2 * tp / d if d > 0 else 0.0)
    return np.array(P), np.array(R), np.array(F)

def _sk_prf(yt, yp, labels):
    p, r, f, _ = precision_recall_fscore_support(yt, yp, labels=list(labels), zero_division=0)
    return p, r, f

def _expand(vals, labels, n=7):
    out = np.full(n, np.nan); out[np.asarray(labels, dtype=int)] = vals; return out

def macro_labels_present_sklearn(y_true, y_pred_remapped):
    return float(f1_score(y_true, y_pred_remapped, average='macro', labels=np.unique(y_true), zero_division=0))

def fixed4_v1_scores(y_true, y_pred):
    # L2-D9 (superseded by L2-D11; kept ONLY to verify and document the numbers that were stored under it): fixed class set, rice predictions -> other, true labels untouched.
    yt = np.asarray(y_true).astype(int); yp = np.asarray(y_pred).astype(int); lab = [CROP_ID[c_] for c_ in USA_FIXED_CLASSES]
    _check(set(lab) <= set(np.unique(yt).tolist()), f'fixed-class scoring needs every class of {USA_FIXED_CLASSES} in the target truth')
    yq = np.where(yp == CROP_ID['rice'], CROP_ID['other'], yp)
    return dict(f1=float(f1_score(yt, yq, average='macro', labels=lab, zero_division=0)), f1_numpy=float(_np_prf(yt, yq, lab)[2].mean()), baseline_all_other=float(_np_prf(yt, np.full_like(yt, CROP_ID['other']), lab)[2].mean()))

def fixed4_scores(y_true, y_pred):
    # L2-D11 (instructor review of seeds 0-3): the USA target is scored on the SAME four classes in every run -- corn, soybeans, wheat, other.
    #  * TRUE rice / sugarcane / cotton samples are REMOVED from the test set (the source mix does not contain them in every run; the paper excludes crops absent from the source, sec. 4.4: "In the FRA -> USA
    #    setting, sugarcane and cotton are excluded from evaluation because they are absent in the source region"; extending this to rice is the instructor's decision, rice being absent from most training mixes);
    #  * PREDICTIONS of rice / sugarcane / cotton on the remaining samples are REASSIGNED to 'other'.
    yt = np.asarray(y_true).astype(int); yp = np.asarray(y_pred).astype(int); lab = [CROP_ID[c_] for c_ in USA_FIXED_CLASSES]; drop = [CROP_ID[c_] for c_ in USA_DROP_TRUE_CLASSES]; reas = [CROP_ID[c_] for c_ in USA_REASSIGN_PRED_TO_OTHER]
    keep = ~np.isin(yt, drop); yt2, yp2 = yt[keep], yp[keep]
    _check(set(lab) <= set(np.unique(yt2).tolist()), f'fixed-class scoring needs every class of {USA_FIXED_CLASSES} in the target truth')
    yq = np.where(np.isin(yp2, reas), CROP_ID['other'], yp2)
    f_sk = float(f1_score(yt2, yq, average='macro', labels=lab, zero_division=0)); f_np = float(_np_prf(yt2, yq, lab)[2].mean())
    base = float(_np_prf(yt2, np.full_like(yt2, CROP_ID['other']), lab)[2].mean())
    return dict(f1=f_sk, f1_numpy=f_np, baseline_all_other=base, n_dropped=int((~keep).sum()), n_pred_reassigned=int(np.isin(yp2, reas).sum()), n_scored=int(keep.sum()))

def score_run(target, y_true, y_pred, train_counts):
    # y_true / y_pred : integer ids 0..6 of the FULL target set.  train_counts : crop counts (length 7) of the TRAIN split.
    yt = np.asarray(y_true).astype(int); yp = np.asarray(y_pred).astype(int); tc = np.asarray(train_counts).astype(int)
    _check(yt.ndim == 1 and yt.shape == yp.shape and len(yt) > 0, 'y_true / y_pred empty or length mismatch')
    _check(yt.min() >= 0 and yt.max() <= 6 and yp.min() >= 0 and yp.max() <= 6, 'class ids outside 0..6')
    present = np.unique(yt); O = CROP_ID['other']
    outside = [CROP_LABELS[int(c)] for c in present if CROP_LABELS[int(c)] not in DEFINED_CLASSES[target]]          # V33-08: scoring assumes y_true is already in the target label space (paper Table 2)
    _check(not outside, f'{target}: y_true contains classes outside the target label set {DEFINED_CLASSES[target]}: {outside} -- the predictions-only remap would be inconsistent with such a truth; STOP')
    if len(present) < 2:
        raise TooFewClasses(f'{target}: true labels contain {len(present)} class(es) -- need >= 2; run stopped')
    yr = remap_existing_release(yp, target)
    # ---- EXISTING release remap: macro-F1 over the classes present in y_true (labels=np.unique(y_true)), OA over ALL samples
    p_s, r_s, f_s = _sk_prf(yt, yr, present); p_n, r_n, f_n = _np_prf(yt, yr, present)
    paper_f1, paper_oa = macro_labels_present_sklearn(yt, yr), float(accuracy_score(yt, yr))
    paper_f1_n, paper_oa_n = float(f_n.mean()), float((yt == yr).mean())
    diffs = [abs(paper_f1 - paper_f1_n), abs(paper_oa - paper_oa_n), *np.abs(f_s - f_n), *np.abs(p_s - p_n), *np.abs(r_s - r_n)]
    # ---- baselines in the existing-release-remap view (all-'other'; majority class of the true labels), scored with the same remap
    maj = int(np.bincount(yt, minlength=7).argmax())
    base = dict(paper_all_other=float(_np_prf(yt, remap_existing_release(np.full_like(yt, O), target), present)[2].mean()),
                paper_majority=float(_np_prf(yt, remap_existing_release(np.full_like(yt, maj), target), present)[2].mean()))
    # ---- coverage-conditional (labelled; never replaces the main score): classes with >= 30 TRAINING samples
    cov_idx = [i for i, c in enumerate(present) if tc[c] >= COVERAGE_MIN_TRAIN]
    cov_paper = float(np.mean(f_s[cov_idx])) if cov_idx else float('nan')
    absent_train = [CROP_LABELS[int(c)] for c in present if tc[c] == 0]            # stay IN the score at their F1; only flagged
    res = dict(n=int(len(yt)), n_true_classes=int(len(present)), present=[CROP_LABELS[int(c)] for c in present],
               paper_f1=paper_f1, paper_oa=paper_oa, paper_prec=_expand(p_s, present), paper_rec=_expand(r_s, present), paper_f1_cls=_expand(f_s, present),
               cov_cond_paper_f1=cov_paper, cov_cond_paper_classes=[CROP_LABELS[int(present[i])] for i in cov_idx], absent_train_classes=absent_train,
               yp_remapped=yr, cm7=confusion_matrix(yt, yp, labels=list(range(7))), cm_paper=confusion_matrix(yt, yr, labels=list(range(7))),
               bin_f1=float('nan'), bin_oa=float('nan'), cm_binary=None, cov_cond_binary_f1=float('nan'), bin_prec=None, bin_rec=None, bin_f1_cls=None,
               baseline_all_other_primary=base['paper_all_other'], baseline_majority_primary=base['paper_majority'], baseline_all_other_paper=base['paper_all_other'],
               baseline_majority_paper=base['paper_majority'], primary_view='existing_release_remap', positive=None)
    # ---- seven-class RAW track (existing): no remap; macro over the classes present in the target truth (never over crops the target lacks)
    p7_s, r7_s, f7_s = _sk_prf(yt, yp, present); p7_n, r7_n, f7_n = _np_prf(yt, yp, present)
    res.update(sevenclass_f1=float(f1_score(yt, yp, average='macro', labels=present, zero_division=0)), sevenclass_oa=float(accuracy_score(yt, yp)))
    diffs += [abs(res['sevenclass_f1'] - float(f7_n.mean())), abs(res['sevenclass_oa'] - float((yt == yp).mean())), *np.abs(f7_s - f7_n)]
    # ---- PAPER-TEXT and REVIEW rules (new, separately named; each scored by BOTH scorers, with its own baselines and coverage-conditional score)
    for tag, fn in (('papertext', remap_paper_text_v3), ('review', remap_review_targetmap)):
        yrr = fn(yp, target); _p, _r, f_s2 = _sk_prf(yt, yrr, present); _p2, _r2, f_n2 = _np_prf(yt, yrr, present)
        f1_s2 = float(f1_score(yt, yrr, average='macro', labels=present, zero_division=0)); oa_s2 = float(accuracy_score(yt, yrr))
        diffs += [abs(f1_s2 - float(f_n2.mean())), abs(oa_s2 - float((yt == yrr).mean())), *np.abs(f_s2 - f_n2)]
        res[f'{tag}_f1'], res[f'{tag}_oa'], res[f'{tag}_f1_cls'] = f1_s2, oa_s2, _expand(f_s2, present); res[f'cm_{tag}'] = confusion_matrix(yt, yrr, labels=list(range(7)))
        res[f'baseline_all_other_{tag}'] = float(_np_prf(yt, fn(np.full_like(yt, O), target), present)[2].mean()); res[f'baseline_majority_{tag}'] = float(_np_prf(yt, fn(np.full_like(yt, maj), target), present)[2].mean())
        ci = [i for i, c_ in enumerate(present) if tc[c_] >= COVERAGE_MIN_TRAIN]; res[f'cov_cond_{tag}_f1'] = float(np.mean(f_s2[ci])) if ci else float('nan')
        res[f'{tag}_pred_unique'] = int(len(np.unique(yrr)))
    if target in BINARY_TARGETS:
        pos = BINARY_TARGETS[target]; pid = CROP_ID[pos]; bt, bp = binary_collapse(yt, yp, pos); lab = [0, 1]
        if len(np.unique(bt)) < 2:                                   # STOP/FLAG: the binary ground truth must contain both classes
            raise TooFewClasses(f'{target}: the binary ({pos} vs other) ground truth contains only {len(np.unique(bt))} class -- binary view undefined; run stopped')
        pb_s, rb_s, fb_s = _sk_prf(bt, bp, lab); pb_n, rb_n, fb_n = _np_prf(bt, bp, lab)
        bin_f1 = float(f1_score(bt, bp, average='macro', labels=lab, zero_division=0)); bin_oa = float(accuracy_score(bt, bp))
        diffs += [abs(bin_f1 - float(fb_n.mean())), abs(bin_oa - float((bt == bp).mean())), *np.abs(fb_s - fb_n), *np.abs(pb_s - pb_n), *np.abs(rb_s - rb_n)]
        n_pos_tr, n_oth_tr = int(tc[pid]), int(tc.sum() - tc[pid]); cov_b = [i for i, n in enumerate([n_oth_tr, n_pos_tr]) if n >= COVERAGE_MIN_TRAIN]   # index 0 = other, 1 = positive
        maj_b = int(np.bincount(bt, minlength=2).argmax())
        res.update(bin_f1=bin_f1, bin_oa=bin_oa, cm_binary=confusion_matrix(bt, bp, labels=lab), bin_prec=pb_s, bin_rec=rb_s, bin_f1_cls=fb_s,
                   cov_cond_binary_f1=float(np.mean(fb_s[cov_b])) if cov_b else float('nan'), primary_view=f'binary_{pos}', positive=pos,
                   baseline_all_other_primary=float(_np_prf(bt, np.zeros_like(bt), lab)[2].mean()), baseline_majority_primary=float(_np_prf(bt, np.full_like(bt, maj_b), lab)[2].mean()),
                   binary_pred_unique=int(len(np.unique(bp))))
        if n_pos_tr == 0: res['absent_train_classes'] = sorted(set(res['absent_train_classes']) | {pos})
    if target == 'USA':                                                 # L2-D9: USA scored on a FIXED class set, whatever the training mix contained
        f4 = fixed4_scores(yt, yp); f4v1 = fixed4_v1_scores(yt, yp); res.update(fixed4_clean_f1=f4['f1'], fixed4_f1=f4v1['f1'], fixed4_baseline_all_other=f4['baseline_all_other'], fixed4_n_dropped=f4['n_dropped']); diffs += [abs(f4['f1'] - f4['f1_numpy']), abs(f4v1['f1'] - f4v1['f1_numpy'])]
    else: res.update(fixed4_clean_f1=float('nan'), fixed4_f1=float('nan'), fixed4_baseline_all_other=float('nan'), fixed4_n_dropped=0)
    res['primary_f1'] = res['bin_f1'] if target in BINARY_TARGETS else res['paper_f1']
    res['primary_oa'] = res['bin_oa'] if target in BINARY_TARGETS else res['paper_oa']
    res['baseline_best_primary'] = max(res['baseline_all_other_primary'], res['baseline_majority_primary'])
    res['baseline_best_review'] = max(res['baseline_all_other_review'], res['baseline_majority_review'])
    res['scorer_maxdiff'] = float(max(diffs))
    res['primary_pred_unique'] = res.get('binary_pred_unique', int(len(np.unique(yr))))
    return res

# ------------------------------------------------------------------------------ GATES (spec 6)
def evaluate_integrity(sc, preds, split_chk, info):
    # INTEGRITY failures (abort after 5 consecutive).  Skill outcomes are NOT here.
    p = np.asarray(preds); n_test = sc['n']
    finite_vals = [sc['primary_f1'], sc['paper_f1'], sc['paper_oa'], sc['review_f1'], sc['review_oa'], sc['papertext_f1'], sc['sevenclass_f1'], sc['baseline_all_other_primary'], sc['baseline_majority_primary'], info.get('val_f1', np.nan), info.get('restore_delta', np.nan)]
    if sc['cm_binary'] is not None: finite_vals += [sc['bin_f1'], sc['bin_oa']]
    I = {'I1_true_classes_ge2': bool(sc['n_true_classes'] >= 2),
         'I2_predictions_valid': bool(len(p) == n_test and p.min() >= 0 and p.max() <= 6 and int(sc['cm7'].sum()) == n_test and int(sc['cm_paper'].sum()) == n_test and int(sc['cm_review'].sum()) == n_test
                                       and (sc['cm_binary'] is None or int(sc['cm_binary'].sum()) == n_test)),
         'I3_split_exact': bool(split_chk['ok']),
         'I4_restore_matches_best': bool(info.get('restore_delta', np.inf) <= RESTORE_TOL),
         'I5_finite': bool(all(np.isfinite(finite_vals))),
         'I6_scorers_agree': bool(sc['scorer_maxdiff'] <= SCORER_TOL)}
    return I, all(I.values())

def skill_flags(sc, info):
    # SKILL outcomes: recorded and warned, NEVER an abort
    f = []
    if info.get('val_f1', 1.0) < VAL_F1_WARN: f.append(f'WARN_VAL_F1_LT_{VAL_F1_WARN:.2f}')
    if not (sc['primary_f1'] > sc['baseline_best_primary']): f.append('WARN_NOT_BETTER_THAN_BASELINE')
    if not (sc['review_f1'] > sc['baseline_best_review']): f.append('WARN_REVIEW_TRACK_NOT_BETTER_THAN_BASELINE')
    if sc['primary_pred_unique'] < 2: f.append('WARN_COLLAPSED_PREDICTIONS')
    if info.get('best_epoch', 0) >= UNDERTRAINED_EPOCH: f.append('possibly_undertrained')
    if sc['absent_train_classes']: f.append('FLAG_ABSENT_TRAIN_CLASS:' + '+'.join(sc['absent_train_classes']))
    return f

# ------------------------------------------------------------------------------ EARLY STOPPING (pure; the torch loop uses it)
class EarlyStopper:
    # validation macro-F1; strictly greater than best + MIN_DELTA counts as an improvement; never more than max_epochs
    def __init__(self, max_epochs=None, patience=None, min_epochs=None, min_delta=None):
        self.max_epochs = MAX_EPOCHS if max_epochs is None else max_epochs; self.patience = PATIENCE if patience is None else patience
        self.min_epochs = MIN_EPOCHS if min_epochs is None else min_epochs; self.min_delta = MIN_DELTA if min_delta is None else min_delta
        _check(self.max_epochs <= MAX_EPOCHS, f'max_epochs {self.max_epochs} would exceed the hard cap {MAX_EPOCHS}')
        self.best, self.best_epoch, self.bad, self.epoch, self.reason = -np.inf, 0, 0, 0, None
    def step(self, score):
        self.epoch += 1; improved = score > self.best + self.min_delta
        if improved: self.best, self.best_epoch, self.bad = score, self.epoch, 0
        else: self.bad += 1
        if self.epoch >= self.min_epochs and self.bad >= self.patience: self.reason = 'early_stop_patience'
        elif self.epoch >= self.max_epochs: self.reason = 'max_epochs'
        return improved, self.reason is not None

def balanced_class_weights(y, n_classes=7):
    # == sklearn compute_class_weight('balanced') over the classes present; classes with ZERO samples get weight 0 (no division by zero)
    y = np.asarray(y).astype(int); counts = np.bincount(y, minlength=n_classes).astype(float)
    present = counts > 0; w = np.zeros(n_classes)
    w[present] = len(y) / (present.sum() * counts[present])
    return w

def training_class_weights(ytr, yva, n_classes=7):
    # CLASS_WEIGHT_SOURCE = 'train_val' (default): balanced weights over the WHOLE drawn sample = what the authors' script does with its whole source set;
    #                       'train': training labels only.  A crop with ZERO samples gets weight 0 (the head has 7 outputs; the authors' vector is only as long as the classes present).
    _check(CLASS_WEIGHT_SOURCE in ('train_val', 'train'), 'CLASS_WEIGHT_SOURCE must be train_val or train')
    y = np.concatenate([np.asarray(ytr), np.asarray(yva)]) if CLASS_WEIGHT_SOURCE == 'train_val' else np.asarray(ytr)
    return balanced_class_weights(y, n_classes)

def run_training_loop(train_one_epoch, val_score, snapshot, restore, max_epochs=None):
    # The epoch loop, framework-free (unit-tested): <= max_epochs TOTAL epochs (100), early stopping inside them on validation macro-F1
    # (patience 15, min 20 epochs, strictly greater), best weights restored, and the restored model must reproduce its best validation score.
    max_epochs = MAX_EPOCHS if max_epochs is None else max_epochs; es = EarlyStopper(max_epochs=max_epochs); best, hist = None, []
    for ep in range(max_epochs):
        train_one_epoch(ep); f = float(val_score()); hist.append(f); improved, stop = es.step(f)
        if improved: best = snapshot()
        if stop: break
    if best is None: raise FloatingPointError('no finite validation score was ever produced')
    restore(best); delta = abs(float(val_score()) - es.best)
    return dict(val_f1=es.best, best_epoch=es.best_epoch, epochs_run=es.epoch, stop_reason=es.reason, restore_delta=delta, history=hist)

def steps_per_epoch(n_train, batch_size=None): return int(math.ceil(n_train / (BATCH_SIZE if batch_size is None else batch_size)))     # DataLoader(drop_last=False)

import collections
def check_geography(state, crop):
    # STOP if sugarcane / rice rows fall outside the agronomically possible US states; print the observed state sets (with row counts) for sugarcane, rice and cotton.
    st, cr = np.asarray(state).astype(str), np.asarray(crop).astype(str); out = {}; viol = []
    for name in list(GEOGRAPHY_RULES) + [c for c in GEOGRAPHY_REPORT_ONLY if c not in GEOGRAPHY_RULES]:
        vc = pd.Series(st[cr == name]).value_counts(); out[name] = vc.to_dict(); allowed = GEOGRAPHY_RULES.get(name)
        print(f'  {name:10s}: {int(vc.sum())} rows in {len(vc)} states -> ' + ', '.join(f'{s}:{n}' for s, n in vc.items()) + (f'   [allowed: {allowed}]' if allowed else '   [report only]'))
        if allowed:
            bad = {s: int(n) for s, n in vc.items() if s not in allowed}
            if sum(bad.values()) > GEOGRAPHY_MAX_OUTSIDE_ROWS:
                viol.append(f'{name}: {sum(bad.values())} of {int(vc.sum())} rows ({100 * sum(bad.values()) / max(int(vc.sum()), 1):.2f}%) lie outside {allowed}: {bad}  [tolerance GEOGRAPHY_MAX_OUTSIDE_ROWS={GEOGRAPHY_MAX_OUTSIDE_ROWS}]')
    if viol: raise RuntimeError('STOP -- the source region table contradicts crop geography (wrong region assignment or labels). Report, do not resolve silently:\n  - ' + '\n  - '.join(viol) +
                                '\nDecide deliberately: (a) a handful of rows in a NEIGHBOURING state may be a boundary/assignment error (Level 2: regions come from the raw file names through the verified block table) -> set GEOGRAPHY_MAX_OUTSIDE_ROWS to a stated number and record why; '
                                '(b) many rows or non-neighbouring regions -> the region table or labels are wrong; do not run. The observed counts are saved in run_environment_seed*.json.')
    return out

# ------------------------------------------------------------------------------ DESIGN: two composition modes kept side by side
BASE_MODE, FULL_MODE = 'key_plus_natural', 'fixed_full'
def effective_key_shares(target):
    if target == 'AUS': return dict(AUS_KEY_SHARES_IF_CONFIRMED or {})
    return dict(KEY_SHARES[target])

def shares_to_counts(shares, B, exact=False):
    # exact=True: every share x B must be an integer (existing baseline); exact=False: largest-remainder rounding to EXACTLY B (fixed_full proposals)
    raw = {c: s * B for c, s in shares.items() if s > 0}
    _check(exact or abs(sum(shares.values()) - 1.0) < 1e-9, 'a complete composition must sum to 1')
    if exact:
        for c, n in raw.items(): _check(abs(n - round(n)) < 1e-9, f'share {shares[c]} x B={B} = {n} is not an integer')
        return {c: int(round(n)) for c, n in raw.items()}
    fl = {c: int(math.floor(n + 1e-9)) for c, n in raw.items()}; rem = B - sum(fl.values())
    for c in sorted(raw, key=lambda c: (-(raw[c] - fl[c]), CROP_ID[c]))[:rem]: fl[c] += 1
    _check(sum(fl.values()) == B, 'largest-remainder rounding failed'); return fl

# PROPOSALS for the complete fixed composition.  They are NOT approved defaults: no proportion is invented, each is a documented candidate whose capacity is checked
# in the plan cell.  They use only the crop definitions of the target map (paper Table 2 label set) and the existing key shares -- never target sample counts.
def _prop_A(target):                                  # key-crop share as in the baseline; the remainder split EQUALLY over the other classes the target map defines
    ks = effective_key_shares(target); rest = [c for c in DEFINED_CLASSES[target] if c not in ks]
    return {**ks, **{c: (1 - sum(ks.values())) / len(rest) for c in rest}} if rest else dict(ks)
def _prop_B(target):                                  # EQUAL share over all classes the target map defines (key crop not privileged)
    d = DEFINED_CLASSES[target]; return {c: 1.0 / len(d) for c in d}
FIXED_COMPOSITION_PROPOSALS = {'A_key_plus_equal_rest': _prop_A, 'B_equal_over_target_classes': _prop_B}

def fixed_shares(target):
    _check(FIXED_COMPOSITION_CHOICE in FIXED_COMPOSITION_PROPOSALS, 'fixed_full composition values have not been chosen/approved: set FIXED_COMPOSITION_CHOICE (and APPROVED["fixed_composition"]) after reviewing the proposals')
    return FIXED_COMPOSITION_PROPOSALS[FIXED_COMPOSITION_CHOICE](target)

def key_counts(target, B, mode=BASE_MODE):
    # counts that are FIXED in every draw.  baseline: key crop(s) only; fixed_full: every crop of the composition (the natural remainder is then empty)
    if mode == BASE_MODE: return shares_to_counts(effective_key_shares(target), B, exact=True)
    if mode == FULL_MODE: return shares_to_counts(fixed_shares(target), B, exact=False)
    raise ValueError(mode)

def rest_count(target, B, mode=BASE_MODE): return B - sum(key_counts(target, B, mode).values())

def state_eligibility(target, B, cap, mode=BASE_MODE):
    # COMPLETE per-state table: what the (target, budget, mode) needs vs what each state holds, and the REASON for every exclusion.  No state is dropped silently.
    kc = key_counts(target, B, mode); rest = B - sum(kc.values()); rows = []
    for s in cap.index:
        row = dict(state=s, total=int(cap.loc[s, 'total']), need_total=B, rows_rule_min=MIN_STATE_ROWS); why = []
        if cap.loc[s, 'total'] < MIN_STATE_ROWS: why.append(f'<{MIN_STATE_ROWS} rows')
        if cap.loc[s, 'total'] < B: why.append(f'total {int(cap.loc[s, "total"])} < budget {B}')
        for c, n in kc.items():
            row[f'have_{c}'], row[f'need_{c}'] = int(cap.loc[s, c]), n
            if cap.loc[s, c] < n: why.append(f'{c} {int(cap.loc[s, c])}<{n}')
        nonkey = int(cap.loc[s, 'total'] - sum(cap.loc[s, c] for c in kc)); row['have_nonkey'], row['need_nonkey'] = nonkey, rest
        if nonkey < rest: why.append(f'non-key {nonkey}<{rest}')
        if target == 'AUS' and s not in AUS_STATES: why.append('not in AUS census {LA, FL}')
        row['eligible'] = not why; row['reason'] = '; '.join(why); rows.append(row)
    return pd.DataFrame(rows)

def eligible_states(target, B, cap, mode=BASE_MODE):
    # pool = states that can supply the WHOLE budget ALONE (rows rule + every fixed crop count + natural remainder).  Same pool for every k.
    e = state_eligibility(target, B, cap, mode); return sorted(e.loc[e['eligible'], 'state'].tolist())

def combo_capacity_ok(states, target, B, cap, mode=BASE_MODE):
    # actual capacity of a COMBINATION: each state must supply its equal share (ceil of count / k) of every fixed crop and of the remainder
    kc = key_counts(target, B, mode); rest = B - sum(kc.values()); k = len(states)
    for s in states:
        for c, n in kc.items():
            if cap.loc[s, c] < math.ceil(n / k): return False
        if cap.loc[s, 'total'] - sum(cap.loc[s, c] for c in kc) < math.ceil(rest / k): return False
    return True

# ------------------------------------------------------------------------------ EXTENSION: capacity-aware allocation (used only when ALLOC_RULE == 'capacity'; the main study never calls any of this)
import contextlib
@contextlib.contextmanager
def profile_scope(which):
    # temporarily put the notebook into the MAIN or the EXTENSION settings (used by the test gate so that tests of the main design never depend on which profile is selected)
    g = globals(); names = ['PROFILE', 'ALLOC_RULE', 'TARGETS_TO_RUN', 'BUDGETS', 'SELECTION_ARMS']; saved = {n: g[n] for n in names}
    try:
        if which == 'main': g.update(PROFILE='main', ALLOC_RULE='equal', **{k_: list(v) for k_, v in MAIN_DEFAULTS.items()})
        else: g.update(PROFILE='extension', ALLOC_RULE='capacity', TARGETS_TO_RUN=list(EXT_TARGETS), BUDGETS=list(EXT_BUDGETS), SELECTION_ARMS=['random'])
        yield
    finally: g.update(saved)

def _floor(n, k): return int(n) if k == 1 else math.ceil(EXT_MIN_CONTRIB * n / k)      # a one-state combination must supply everything itself

def water_fill(n, caps, rng):
    # split n rows over states as EQUALLY as the capacities allow: a state that cannot reach the common level gives its whole capacity, the surplus is shared by the others; the integer remainder goes to a random subset of the uncapped states
    caps = [int(c) for c in caps]; k = len(caps)
    if sum(caps) < n: raise InfeasibleSplit(f'capacity {sum(caps)} < {n} requested')
    alloc, free, left = [0] * k, list(range(k)), int(n)
    while free:
        share = left // len(free); capped = [i for i in free if caps[i] <= share]
        if not capped: break
        for i in capped: alloc[i] = caps[i]; left -= caps[i]
        free = [i for i in free if i not in capped]
    if free:
        base, rem = divmod(left, len(free)); extra = set(rng.choice(len(free), size=rem, replace=False).tolist()) if rem else set()
        for j, i in enumerate(free): alloc[i] = base + (1 if j in extra else 0)
    _check(sum(alloc) == n and all(0 <= a <= c for a, c in zip(alloc, caps)), 'water_fill produced an invalid allocation')
    return alloc

def _caps_for(states, target, cap, kc):
    # per fixed crop and for the natural remainder: how many rows each state of the combination holds
    out = {c: [int(cap.loc[s, c]) for s in states] for c in kc}; out[None] = [int(cap.loc[s, 'total'] - sum(cap.loc[s, c] for c in kc)) for s in states]; return out

def combo_feasible_pooled(states, target, B, cap, mode=BASE_MODE):
    # a combination is feasible when its states TOGETHER hold the quota of every fixed crop and of the remainder, and EACH state can give at least EXT_MIN_CONTRIB of its equal share
    kc = key_counts(target, B, mode); rest = B - sum(kc.values()); k = len(states); caps = _caps_for(states, target, cap, kc)
    for c, n in list(kc.items()) + [(None, rest)]:
        if n and (sum(caps[c]) < n or min(caps[c]) < _floor(n, k)): return False
    return True

def combo_ok(states, target, B, cap, mode=BASE_MODE):
    return combo_feasible_pooled(states, target, B, cap, mode) if ALLOC_RULE == 'capacity' else combo_capacity_ok(states, target, B, cap, mode)

def make_combos_pooled(pool, k, target, B, cap, mode=BASE_MODE, max_combos=None):
    # every FEASIBLE combination of the pool is counted (n_total); k=1 is the census of the pool; otherwise all of them if <= max_combos, else a fixed seeded sample
    max_combos = MAX_COMBOS if max_combos is None else max_combos; pool = sorted(pool)
    if k > len(pool): return [], 0
    if k == 1: return [(s,) for s in pool], len(pool)
    _check(math.comb(len(pool), k) <= 200000, f'pool of {len(pool)} states is too large to enumerate k={k}')
    allc = [c for c in itertools.combinations(pool, k) if combo_feasible_pooled(c, target, B, cap, mode)]; n_total = len(allc)
    if n_total <= max_combos: return allc, n_total
    idx = sorted(np.random.default_rng(stable_seed('combo', target, B, k, 'pooled')).choice(n_total, size=max_combos, replace=False).tolist()); return [allc[i] for i in idx], n_total

def alloc_max_share(states, target, B, cap, mode=BASE_MODE):
    # planned share of the budget supplied by the largest contributor (fixed crops + remainder, deterministic preview of water_fill)
    kc = key_counts(target, B, mode); rest = B - sum(kc.values()); caps = _caps_for(states, target, cap, kc); rng = np.random.default_rng(stable_seed('preview', target, B, '-'.join(states))); tot = np.zeros(len(states))
    for c, n in list(kc.items()) + [(None, rest)]:
        if n: tot += np.array(water_fill(n, caps[c], rng))
    return float(tot.max() / B)

def make_combos(pool, k, target, B, max_combos=None):
    max_combos = MAX_COMBOS if max_combos is None else max_combos
    pool = sorted(pool)
    if k > len(pool): return []
    if k == 1: return [(s,) for s in pool]                       # CENSUS: every pool state
    total = math.comb(len(pool), k)
    if total <= max_combos: return list(itertools.combinations(pool, k))
    rng = np.random.default_rng(stable_seed('combo', target, B, k))   # deterministic seed from (target, budget, k)
    seen = set()
    while len(seen) < max_combos:
        seen.add(tuple(sorted(rng.choice(len(pool), size=k, replace=False).tolist())))
    return [tuple(pool[i] for i in idx) for idx in sorted(seen)]
ARM_ORDER = ['random', 'diverse', 'clustered', 'common', 'kcenter', 'aug_on', 'aug_copy']
BASE_ARMS = ['random', 'diverse', 'clustered']            # chosen inside the plan grid; 'common' / 'kcenter' / 'aug_on' are added by add_variant_cells
STATE_SELECTORS = {'random': make_combos}              # the baseline arm; the two Phase-4 arms are built from the SAME pool by select_extreme_arms()

# ------------------------------------------------------------------------------ PHASE 4: phenology-spectral diversity of the USA source states (definition: see DIVERSITY_SPACE)
def compute_diversity_distances(X, state, crop, min_n=None):
    # X: (N, ...) USA Sentinel-2 features, state/crop: per-row labels aligned with X.  Features are standardised GLOBALLY (USA mean/std per feature); for every (state, crop) with >= min_n samples the mean
    # profile is taken; dist[crop][s, s'] = RMS difference of the two mean profiles (NaN if either has < min_n samples).  Uses USA source features only.
    min_n = MIN_PROFILE_N if min_n is None else int(min_n); A = np.asarray(X).reshape(len(X), -1); st, cr = np.asarray(state).astype(str), np.asarray(crop).astype(str)
    s1 = np.zeros(A.shape[1]); s2 = np.zeros(A.shape[1])
    for i in range(0, len(A), 20000): blk = A[i:i + 20000].astype(np.float64); s1 += blk.sum(0); s2 += (blk * blk).sum(0)
    gm = s1 / len(A); gs = np.sqrt(np.maximum(s2 / len(A) - gm ** 2, 0)) + 1e-6; states = sorted(set(st)); dist, cnt = {}, {}
    for c in sorted(set(cr)):
        mu, n_ = {}, {}
        for s_ in states:
            idx = np.flatnonzero((st == s_) & (cr == c)); n_[s_] = len(idx)
            if len(idx) >= min_n: mu[s_] = (A[idx].astype(np.float64).mean(0) - gm) / gs
        M = pd.DataFrame(np.nan, index=states, columns=states); ks = list(mu)
        if ks: P = np.vstack([mu[s_] for s_ in ks]); M.loc[ks, ks] = np.sqrt(((P[:, None, :] - P[None, :, :]) ** 2).mean(-1))
        dist[c] = M; cnt[c] = pd.Series(n_)
    return dict(dist=dist, n=cnt, min_n=min_n)

def diversity_matrix(t, B, mode):
    # state-by-state distance for the target design = mean over its FIXED crops; also the set of states that have a profile for every one of them.  Returns (D or None, valid set, note)
    DD = globals().get('DIVERSITY_DIST')
    if DD is None: return None, set(), 'diversity distances not computed'
    crops = list(key_counts(t, B, mode))
    if not crops: return None, set(), 'no fixed crops in this design: diversity undefined'
    miss = [c for c in crops if c not in DD['dist']]
    if miss: return None, set(), f'no USA profiles for {miss}'
    D = sum(DD['dist'][c] for c in crops) / len(crops); valid = set(D.index[D.notna().any(axis=1)]) & set.intersection(*[set(DD['n'][c].index[DD['n'][c] >= DD['min_n']]) for c in crops])
    return D, valid, ''

def pool_for(t, B, k, cap, mode=BASE_MODE):
    # strict (default, unchanged): states that can supply the WHOLE budget alone.  equal_share (opt-in): states that can supply their 1/k share of every fixed crop and of the remainder.
    if ALLOC_RULE == 'capacity':                                                                                  # extension: a state can join a k-combination if it can give its floor share of every quota
        kc = key_counts(t, B, mode); rest = B - sum(kc.values()); out = []
        for s in cap.index:
            if cap.loc[s, 'total'] < MIN_STATE_ROWS or (t == 'AUS' and s not in AUS_STATES): continue
            caps = _caps_for((s,), t, cap, kc)
            if all((not n) or caps[c][0] >= _floor(n, k) for c, n in list(kc.items()) + [(None, rest)]): out.append(s)
        return sorted(out)
    if POOL_RULE == 'strict': return eligible_states(t, B, cap, mode)
    return sorted(s for s in cap.index if cap.loc[s, 'total'] >= MIN_STATE_ROWS and (t != 'AUS' or s in AUS_STATES) and combo_capacity_ok(tuple([s] * k), t, B, cap, mode))

def _score_idx(Dm, cidx):
    k = cidx.shape[1]; tot = np.zeros(len(cidx))
    for i_ in range(k):
        for j_ in range(i_ + 1, k): tot += Dm[cidx[:, i_], cidx[:, j_]]
    return tot / (k * (k - 1) / 2)

def diversity_population(pool, k, Dm, t, B, mode):
    # the scored combinations of one cell: ALL of them when C(P,k) <= ENUM_LIMIT, otherwise a fixed seeded random sample of SCORE_SAMPLE distinct ones
    P = len(pool); total = math.comb(P, k)
    if total <= ENUM_LIMIT: cidx = np.array(list(itertools.combinations(range(P), k))); enumerated = True
    else:
        rng = np.random.default_rng(stable_seed('divscore', t, B, k, mode)); seen = set()
        while len(seen) < SCORE_SAMPLE: seen.add(tuple(sorted(rng.choice(P, size=k, replace=False).tolist())))
        cidx = np.array(sorted(seen)); enumerated = False
    return cidx, _score_idx(Dm, cidx), enumerated

def select_extreme_arms(pool, k, Dm, t, B, mode):
    # diverse = the N_ARM highest-scoring combinations, clustered = the N_ARM lowest (ties broken by the combination itself -> deterministic); the same function also gives every combination its percentile
    cidx, sc, enumerated = diversity_population(pool, k, Dm, t, B, mode); rows = [(round(float(x), 12), tuple(int(v) for v in c)) for x, c in zip(sc, cidx)]
    hi = sorted(rows, key=lambda r: (-r[0], r[1]))[:N_ARM]; lo = sorted(rows, key=lambda r: (r[0], r[1]))[:N_ARM]; sc_sorted = np.sort(sc)
    mk = lambda lst: [tuple(pool[i] for i in c) for _, c in lst]
    return dict(diverse=mk(hi), clustered=mk(lo), scores=sc_sorted, enumerated=enumerated, n_scored=len(sc))

def combo_score_pct(combo, pool, Dm, sc_sorted):
    idx = np.array([[pool.index(s) for s in combo]]); v = float(_score_idx(Dm, idx)[0]); return v, float(np.searchsorted(sc_sorted, v + 1e-12, side='right') / len(sc_sorted))

# ------------------------------------------------------------------------------ v3.7 tracks: common pool, k-center sample selection, augmentation ON
def train_rows_multiplier(aug):
    # rows seen per epoch relative to the drawn training set: aug ON = original + AUG_REPEAT augmented copies; aug 'copy' (the v3.8 steps-matched control) = AUG_REPEAT exact copies, no transform
    return (1 + AUG_REPEAT) if aug in ('on', 'copy') else 1

def _cell_key(c): return (c['composition'], c['target'], c['budget'], c['k'], tuple(c['states']), c.get('sel', 'random'), c.get('aug', 'off'))

def add_variant_cells(cells, cap):
    # arms 'common' / 'kcenter' / 'aug_on' are added AFTER the base plan; the base cells are never altered except that a shared run also lists 'common' in its arms
    want = [a for a in ('common', 'kcenter', 'aug_on') if a in SELECTION_ARMS]
    if not want: return cells
    out = list(cells); index = {_cell_key(c): c for c in out}
    def nxt(mode, t, B, k): return 1 + max([c['combo_id'] for c in out if (c['composition'], c['target'], c['budget'], c['k']) == (mode, t, B, k)] or [-1])
    if 'common' in want:
        for mode in [BASE_MODE]:                                                   # the new tracks exist for the locked baseline composition only
            for t in [x for x in TARGETS_TO_RUN if x != 'AUS']:                    # AUS has no confirmed spec: no extra tracks
                run_B = sorted({c['budget'] for c in cells if c['target'] == t and c['composition'] == mode and 'random' in c['arms']})
                if len(run_B) < 2: continue
                new_by_B = {}
                for B in run_B:
                    for k in sorted({c['k'] for c in cells if c['target'] == t and c['composition'] == mode and c['budget'] == B}):
                        pk = sorted(set.intersection(*[set(pool_for(t, B2, k, cap, mode)) for B2 in run_B]))
                        if k > len(pk) or k > K_MAX[t]: continue
                        combos = [(s,) for s in pk] if k == 1 else make_combos(pk, k, t, B, MAX_COMBOS)
                        combos = [c for c in combos if combo_ok(c, t, B, cap, mode)]
                        if k >= 2 and len(combos) < MIN_COMBOS and t not in EXEMPT_RULES: continue
                        if combos: new_by_B.setdefault(B, {})[k] = (combos, len(pk))
                for B, byk in new_by_B.items():
                    if len(byk) < MIN_KPOINTS and t not in EXEMPT_RULES: continue                      # the k-point rule applies to this arm too
                    for k, (combos, P) in sorted(byk.items()):
                        for c in combos:
                            key = (mode, t, B, k, tuple(c), 'random', 'off')
                            if key in index: index[key]['arms'] = tuple(index[key]['arms']) + (('common',) if 'common' not in index[key]['arms'] else ()); index[key]['common_pool'] = P
                            else:
                                pend = any(x['pending'] for x in cells if x['target'] == t and x['composition'] == mode)
                                cell = dict(composition=mode, target=t, budget=B, k=k, combo_id=nxt(mode, t, B, k), states=tuple(c), pool_size=P, n_total_combos=math.comb(P, k), pending=pend, arms=('common',), div_score='', div_pct='', max_share='', common_pool=P)
                                out.append(cell); index[key] = cell
    def pick(base, n):
        # deterministic subset of the random-arm combinations: k=1 = the whole census, k>=2 = the n combinations with the smallest stable hash
        grp = {}
        for c in base: grp.setdefault((c['composition'], c['target'], c['budget'], c['k']), []).append(c)
        sel = []
        for (mode, t, B, k), lst in grp.items():
            lst = sorted(lst, key=lambda c: stable_seed('variant', t, B, k, '-'.join(c['states'])))
            sel += lst if k == 1 else lst[:n]
        return sel
    base = [c for c in cells if c['target'] != 'AUS' and c['composition'] == BASE_MODE and 'random' in c['arms'] and c.get('sel', 'random') == 'random' and c.get('aug', 'off') == 'off']
    if 'kcenter' in want:
        for c in pick([c for c in base if c['target'] in KC_TARGETS], KC_COMBOS):
            n = dict(c, arms=('kcenter',), sel='kcenter', div_score='', div_pct='', combo_id=c['combo_id']); out.append(n)
    if 'aug_on' in want:
        for c in pick([c for c in base if c['target'] in AUG_TARGETS and c['budget'] in AUG_BUDGETS], AUG_COMBOS):
            n = dict(c, arms=('aug_on',), aug='on', div_score='', div_pct='', pending=bool(c['pending'] or not APPROVED.get('aug_on', False))); out.append(n)
        if AUG_CONTROL:                                                            # v3.8: the steps-matched control, same combinations as aug_on, rows copied without transforms
            for c in pick([c for c in base if c['target'] in AUG_TARGETS and c['budget'] in AUG_BUDGETS], AUG_COMBOS):
                n = dict(c, arms=('aug_copy',), aug='copy', div_score='', div_pct='', pending=bool(c['pending'] or not APPROVED.get('aug_on', False))); out.append(n)
    return out

# ---- k-center sample selection (arm 'kcenter')
KC_EMBED = None
KC_WINDOW = ((121 - 1) // 5, (334 - 1) // 5 + 1)           # official DOY 121-334 at 5-day steps -> indices 24:67 (43 steps), same as the authors' CropGlobeDataset

def _np(a): return a.detach().cpu().numpy() if hasattr(a, 'detach') else np.asarray(a)

def build_kc_embedding(features, dims=None, fit_n=20000, seed=0, chunk=40000):
    # USA rows only: official-window features (10 bands x 43 steps) -> standardise -> PCA(dims) fitted on a seeded subsample.  Returns float32 (N, dims).
    dims = KC_DIMS if dims is None else dims; N = len(features); w0, w1 = KC_WINDOW
    def win(idx): return _np(features[idx])[:, 0, :, w0:w1].reshape(len(idx), -1).astype(np.float32)
    rng = np.random.default_rng(stable_seed('kc-fit', seed)); fit_idx = np.sort(rng.choice(N, size=min(fit_n, N), replace=False)); S = win(fit_idx)
    _check(np.isfinite(S).all(), 'k-center embedding: non-finite USA features')
    mu, sd = S.mean(0), S.std(0) + 1e-6; Z = (S - mu) / sd; U, sing, Vt = np.linalg.svd(Z, full_matrices=False); P = Vt[:dims].T.astype(np.float32)
    out = np.empty((N, dims), dtype=np.float32)
    for i in range(0, N, chunk): idx = np.arange(i, min(N, i + chunk)); out[idx] = ((win(idx) - mu) / sd) @ P
    return out

def kcenter_pick(pool, n, key, embed=None):
    # greedy farthest-first traversal INSIDE one (state, crop) pool; deterministic given the key (start point from the key); exactly n distinct rows of the pool
    embed = KC_EMBED if embed is None else embed; _check(embed is not None, 'k-center needs KC_EMBED (built in the plan cell from the USA features)')
    pool = np.asarray(pool); _check(0 <= n <= len(pool), f'k-center asked for {n} of {len(pool)} rows')
    if n == 0: return pool[:0]
    r = np.random.default_rng(stable_seed('kc', *key)); cand = pool if len(pool) <= KC_MAX_CAND else np.sort(r.choice(pool, size=KC_MAX_CAND, replace=False))
    if n > len(cand): cand = pool
    E = embed[cand].astype(np.float32); first = int(r.integers(len(cand))); sel = [first]; d = ((E - E[first]) ** 2).sum(1); d[first] = -1.0
    for _ in range(n - 1):
        j = int(d.argmax()); sel.append(j); d = np.minimum(d, ((E - E[j]) ** 2).sum(1)); d[sel] = -1.0
    out = cand[np.array(sel)]; _check(len(set(out.tolist())) == n, 'k-center produced duplicate rows'); return out

def build_plan(cap):
    cells, grid = [], []; arms_req = [a for a in BASE_ARMS if a in SELECTION_ARMS]; need_div = len(arms_req) > 1
    for mode in COMPOSITION_MODES_TO_RUN:
        mode_pending = (mode == FULL_MODE and not (APPROVED.get('fixed_composition') and FIXED_COMPOSITION_CHOICE in FIXED_COMPOSITION_PROPOSALS))
        if mode == FULL_MODE and FIXED_COMPOSITION_CHOICE not in FIXED_COMPOSITION_PROPOSALS: continue          # nothing to plan until a composition is chosen
        for t in TARGETS_TO_RUN:
            pending = (t == 'AUS' and not AUS_SPEC_CONFIRMED) or mode_pending
            for B in BUDGETS:
                kc = key_counts(t, B, mode); rest = B - sum(kc.values()); Dt, valid, div_note = diversity_matrix(t, B, mode) if need_div else (None, set(), '')
                rows = []
                for k in K_LIST:
                    pool = pool_for(t, B, k, cap, mode); dropped = [s for s in pool if Dt is not None and s not in valid]; pool = [s for s in pool if Dt is None or s in valid]; P = len(pool)
                    ntot = math.comb(P, k) if k <= P else 0; note, combos = '', []
                    if k > K_MAX[t]: note = f'k > K_MAX({K_MAX[t]})'
                    elif t == 'FRA' and k == 8 and P < FRA_K8_MIN_POOL: note = f'FRA k=8 needs pool >= {FRA_K8_MIN_POOL} (pool={P})'
                    elif k > P: note = 'pool smaller than k'
                    elif ALLOC_RULE == 'capacity':
                        combos, ntot = make_combos_pooled(pool, k, t, B, cap, mode)
                        if not combos: note = 'no combination of the pool can supply the quotas'
                    else: combos = STATE_SELECTORS[STATE_SELECTOR](pool, k, t, B)
                    if t not in EXEMPT_RULES and combos and len(combos) < MIN_COMBOS and not (ALLOC_RULE == 'capacity' and k == 1): note, combos = f'{len(combos)} combinations < {MIN_COMBOS}', []      # extension: the k=1 CENSUS is kept however few states it has
                    arm_c = {'random': (list(combos), note)}; Dm = Dt.loc[pool, pool].values if (Dt is not None and P >= k >= 2) else None; ext = None
                    if Dm is not None: ext = select_extreme_arms(pool, k, Dm, t, B, mode) if ntot >= 2 * N_ARM else None
                    for arm in arms_req[1:]:
                        if combos and k == 1: arm_c[arm] = (list(combos), '')                                          # the one-state census is shared by every arm
                        elif not combos: arm_c[arm] = ([], note)
                        elif Dt is None: arm_c[arm] = ([], div_note or 'diversity undefined')
                        elif ntot < 2 * N_ARM: arm_c[arm] = ([], f'only {ntot} combinations: the {N_ARM} most and {N_ARM} least diverse sets cannot be disjoint')
                        else: arm_c[arm] = (list(ext[arm]), '')
                    rows.append(dict(k=k, pool=pool, P=P, ntot=ntot, arm_c=arm_c, Dm=Dm, ext=ext, dropped=dropped, note=note))
                for arm in arms_req:                                                                                   # the k-point rule is applied to every arm separately
                    if t not in EXEMPT_RULES and sum(1 for r in rows if r['arm_c'][arm][0]) < MIN_KPOINTS:
                        for r in rows:
                            if r['arm_c'][arm][0]: r['arm_c'][arm] = ([], f'fewer than {MIN_KPOINTS} feasible k-points for this budget')
                for r in rows:
                    k, pool, P, ntot = r['k'], r['pool'], r['P'], r['ntot']; union = {}
                    for arm in arms_req:
                        for c in r['arm_c'][arm][0]: union.setdefault(tuple(c), []).append(arm)
                    for c in union: _check(combo_ok(c, t, B, cap, mode), f'{mode} {t} B={B} k={k}: combination {c} cannot supply its share (pool construction bug)')
                    arm_note = '; '.join(f'{a}: {r["arm_c"][a][1]}' for a in arms_req[1:] if r['arm_c'][a][1] and not r['arm_c'][a][0])
                    grid.append(dict(composition=mode, target=t, role=TARGET_ROLE[t], budget=B, k=k, pool_size=P, n_total_combos=ntot, n_combos=len(union), feasible=bool(union),
                                     note=r['arm_c']['random'][1] + (' [PENDING approval/decision]' if pending and union else ''), key_counts=json.dumps(kc), rest=rest, pool=','.join(pool) if P <= 20 else f'{P} states',
                                     **{f'n_{a}': len(r['arm_c'].get(a, ([], ''))[0]) for a in ARM_ORDER}, arm_note=arm_note, max_share=(max([alloc_max_share(c, t, B, cap, mode) for c in union] or [0.0]) if ALLOC_RULE == 'capacity' else ''), dropped_no_profile=','.join(r['dropped']), arms_enumerated=(r['ext']['enumerated'] if r['ext'] else '')))
                    for ci, (c, arms_of) in enumerate(union.items()):
                        sc, pct = (combo_score_pct(c, pool, r['Dm'], r['ext']['scores']) if (r['Dm'] is not None and r['ext'] is not None and k >= 2) else ('', ''))
                        cells.append(dict(composition=mode, target=t, budget=B, k=k, combo_id=ci, states=tuple(c), pool_size=P, n_total_combos=ntot, pending=pending, arms=tuple(arms_of), div_score=sc, div_pct=pct,
                                      max_share=(alloc_max_share(c, t, B, cap, mode) if ALLOC_RULE == 'capacity' else '')))
    cells = add_variant_cells(cells, cap)
    return cells, pd.DataFrame(grid)

def build_pools(usa_state, usa_crop):
    df = pd.DataFrame({'s': np.asarray(usa_state), 'c': np.asarray(usa_crop), 'i': np.arange(len(usa_state))})
    return {(s, c): g['i'].values.astype(np.int64) for (s, c), g in df.groupby(['s', 'c'])}

def make_cap(usa_state, usa_crop):
    cap = pd.crosstab(np.asarray(usa_state), np.asarray(usa_crop)).reindex(columns=CROP_ORDER, fill_value=0)
    cap['total'] = cap[CROP_ORDER].sum(axis=1); return cap

def allocate_val(counts, total, frac):
    # largest-remainder allocation of EXACTLY `total` validation rows across crops (stratified); every crop with >= 2 rows keeps >= 1 row in train
    ideal = {c: frac * n for c, n in counts.items()}
    v = {c: min(int(math.floor(ideal[c] + 1e-9)), max(n - 1, 0)) for c, n in counts.items()}
    for strict in (True, False):
        order = sorted(counts, key=lambda c: (-(ideal[c] - v[c]), CROP_ID[c]))
        while total - sum(v.values()) > 0:
            moved = False
            for c in order:
                if total - sum(v.values()) == 0: break
                if v[c] < counts[c] - 1 and (not strict or v[c] < math.ceil(ideal[c] - 1e-9)): v[c] += 1; moved = True
            if not moved: break
        if sum(v.values()) == total: return v
    raise InfeasibleSplit(f'cannot allocate {total} validation rows over {counts}')

def draw_split(target, B, states, seed, pools, usa_crop, mode=BASE_MODE, sel='random'):
    # EXACTLY B rows.  Fixed crop count(s) split equally over the states; any remainder = natural draw from each state's non-fixed rows (equal count per state).
    states = tuple(sorted(states)); k = len(states); kc = key_counts(target, B, mode); rest = B - sum(kc.values())
    rng = np.random.default_rng(stable_seed('draw', target, B, '-'.join(states), seed) if mode == BASE_MODE else stable_seed('draw', mode, FIXED_COMPOSITION_CHOICE, target, B, '-'.join(states), seed)); chosen = []
    def split_counts(n, caps=None):
        if ALLOC_RULE == 'capacity' and caps is not None: return water_fill(n, caps, rng)                       # extension only
        base, rem = divmod(n, k); extra = set(rng.choice(k, size=rem, replace=False).tolist()) if rem else set()
        return [base + (1 if i in extra else 0) for i in range(k)]
    for c in sorted(kc, key=lambda x: CROP_ID[x]):
        cnt = split_counts(kc[c], [len(pools.get((s, c), ())) for s in states])
        for i, s in enumerate(states):
            pool = pools.get((s, c), np.zeros(0, dtype=np.int64))
            if len(pool) < cnt[i]: raise InfeasibleSplit(f'{target} B={B}: state {s} has {len(pool)} {c} rows, needs {cnt[i]}')
            pick_ = rng.choice(pool, size=cnt[i], replace=False)
            chosen.append(pick_ if sel == 'random' else kcenter_pick(pool, cnt[i], (target, B, s, seed, c)))
    cnt = split_counts(rest, [sum(len(pools[(s, c)]) for c in CROP_ORDER if c not in kc and (s, c) in pools) for s in states])
    for i, s in enumerate(states):
        if cnt[i] == 0: continue
        parts = [pools[(s, c)] for c in CROP_ORDER if c not in kc and (s, c) in pools]; pool = np.concatenate(parts) if parts else np.zeros(0, dtype=np.int64)
        if len(pool) < cnt[i]: raise InfeasibleSplit(f'{target} B={B}: state {s} has {len(pool)} non-key rows, needs {cnt[i]}')
        pick_ = rng.choice(pool, size=cnt[i], replace=False)
        if sel == 'random': chosen.append(pick_)
        else:                                                                                  # same crop mix as the random draw; only WHICH rows of each (state, crop) pool are taken changes
            cc_ = np.asarray(usa_crop)[pick_]
            for c2 in CROP_ORDER:
                n2 = int((cc_ == c2).sum())
                if n2: chosen.append(kcenter_pick(pools[(s, c2)], n2, (target, B, s, seed, 'rest', c2)))
    allp = np.concatenate(chosen); crops = np.asarray(usa_crop)[allp]
    nval = int(round(VAL_FRACTION * B)); _check(abs(VAL_FRACTION * B - nval) < 1e-9, 'validation size must be an integer')
    counts = {c: int((crops == c).sum()) for c in CROP_ORDER if (crops == c).any()}; vq = allocate_val(counts, nval, VAL_FRACTION); tr, va = [], []
    for c in counts:
        pos = allp[crops == c]; perm = np.random.default_rng(stable_seed('val', target, B, '-'.join(states), seed, c)).permutation(len(pos))
        va.append(pos[perm[:vq[c]]]); tr.append(pos[perm[vq[c]:]])
    return dict(train_pos=np.concatenate(tr), val_pos=np.concatenate(va), key_counts=kc, rest=rest, states=states, val_quota=vq, mode=mode, sel=sel)

def _alloc_checks(need, label, counts, caps, n):
    # extension: within capacity, at least the floor, and "as equal as possible" = a state below the others by more than 1 row must be at its capacity
    k = len(counts); fl = _floor(n, k)
    need(all(c <= cp for c, cp in zip(counts, caps)), f'{label}: a state supplies more than it holds {counts} vs {caps}')
    need(all(c >= fl for c in counts), f'{label}: a state supplies less than the minimum contribution {fl}: {counts}')
    need(all(not (counts[i] > counts[j] + 1 and counts[j] < caps[j]) for i in range(k) for j in range(k)), f'{label}: not as equal as capacity allows {counts} (capacities {caps})')

def verify_split(split, target, B, states, usa_state, usa_crop, mode=BASE_MODE):
    msgs = []; tr, va = split['train_pos'], split['val_pos']; allp = np.concatenate([tr, va]); kc = key_counts(target, B, mode); k = len(states)
    st, cr = np.asarray(usa_state), np.asarray(usa_crop)
    def need(c, m):
        if not c: msgs.append(m)
    nval = int(round(VAL_FRACTION * B))
    need(len(allp) == B, f'train+val={len(allp)} != budget {B}')
    need(len(va) == nval and len(tr) == B - nval, f'split sizes train={len(tr)} val={len(va)} != {B - nval}/{nval}')
    need(len(np.unique(allp)) == len(allp), 'duplicate rows inside the drawn sample (unique-id check)')
    need(len(np.intersect1d(tr, va)) == 0, 'train/validation overlap')
    need(allp.min() >= 0 and allp.max() < len(st), 'row id outside the USA table')
    need(set(st[allp].tolist()) <= set(states), 'a drawn row comes from a state outside the combination')
    for c, n in kc.items():
        need(int((cr[allp] == c).sum()) == n, f'fixed crop {c}: {int((cr[allp] == c).sum())} drawn vs fixed {n}')
        counts = [int(((cr[allp] == c) & (st[allp] == s)).sum()) for s in states]
        if ALLOC_RULE == 'capacity': _alloc_checks(need, f'fixed crop {c}', counts, [int(((cr == c) & (st == s)).sum()) for s in states], n)
        else: need(max(counts) - min(counts) <= 1, f'fixed crop {c}: states not equally represented {counts}')
    nonkey = ~np.isin(cr[allp], list(kc)); rest_counts = [int((nonkey & (st[allp] == s)).sum()) for s in states]
    need(int(nonkey.sum()) == B - sum(kc.values()), 'non-fixed count != natural remainder')
    if ALLOC_RULE == 'capacity': _alloc_checks(need, 'non-fixed rows', rest_counts, [int(((~np.isin(cr, list(kc))) & (st == s)).sum()) for s in states], B - sum(kc.values()))
    else: need(max(rest_counts) - min(rest_counts) <= 1, f'non-fixed rows not equally split over states {rest_counts}')
    for c in set(cr[allp].tolist()):
        n_c = int((cr[allp] == c).sum()); v_c = int((cr[va] == c).sum())
        need(abs(v_c - VAL_FRACTION * n_c) <= 1.0 + 1e-9, f'validation not stratified for {c}: {v_c} of {n_c}')
    return dict(ok=not msgs, msgs=msgs)

# ------------------------------------------------------------------------------ RESULTS TABLE (atomic; the metrics file is written LAST)
def make_run_id(cell, seed):
    mode = cell.get('composition', BASE_MODE); tag = '' if mode == BASE_MODE else f'|comp-{mode}-{FIXED_COMPOSITION_CHOICE}'
    if PROFILE != 'main': tag += f'|prof-{PROFILE}'
    if cell.get('sel', 'random') != 'random': tag += f"|sel-{cell['sel']}"
    return f"{cell['target']}|B{cell['budget']}|k{cell['k']}|{'-'.join(cell['states'])}|aug-{cell.get('aug', AUG)}{tag}|s{seed}"

COLUMNS = (['run_id', 'status', 'composition', 'arms', 'aug', 'sample_sel', 'input_features', 'main_f1', 'main_metric', 'fixed4_f1', 'fixed4_clean_f1', 'main_f1_before_fixed4', 'main_f1_before_clean', 'train_n_effective', 'diversity_score', 'diversity_pct', 'profile', 'state_rows', 'max_state_share', 'class_weight_source', 'target', 'role', 'budget', 'k', 'combo_id', 'states', 'pool_size', 'n_total_combos', 'seed', 'n_train', 'n_val', 'epochs_run', 'best_epoch', 'stop_reason',
            'val_f1', 'restore_delta', 'primary_view', 'primary_f1', 'primary_oa', 'paper_f1', 'paper_oa', 'binary_f1', 'binary_oa',
            'baseline_all_other_primary', 'baseline_majority_primary', 'baseline_all_other_paper', 'baseline_majority_paper', 'cov_cond_primary_f1', 'cov_cond_paper_f1',
            'sevenclass_f1', 'sevenclass_oa', 'papertext_f1', 'papertext_oa', 'review_f1', 'review_oa', 'baseline_all_other_review', 'baseline_majority_review', 'baseline_all_other_papertext',
            'baseline_majority_papertext', 'cov_cond_review_f1', 'remap_rule_review',
            'absent_train_classes', 'n_target_classes', 'n_target_classes_train_ge30', 'n_absent_target_classes_train', 'steps_per_epoch', 'total_optimizer_steps',
            'scorer_maxdiff', 'test_n', 'key_n', 'target_crop_n']
           + [f'train_{c}' for c in CROP_ORDER] + [f'val_{c}' for c in CROP_ORDER] + [f'true_{c}' for c in CROP_ORDER] + [f'pred_{c}' for c in CROP_ORDER]
           + [f'f1_{c}' for c in CROP_ORDER] + [f'review_f1_{c}' for c in CROP_ORDER] + INTEGRITY_CHECKS + ['integrity_pass', 'skill_flags', 'seconds', 'error'])

def main_metric_value(target, primary_f1, cov_cond_primary_f1, fixed4_f1=None, fixed4_clean_f1=None):
    # ARG / CHN: binary key-crop macro-F1 (= primary_f1).  FRA: coverage-conditional macro-F1 (rule FRA_MAIN_RULE, stated in advance because rice is never in the FRA training mix)
    if target == 'USA' and USA_SCORING == 'fixed4_clean' and fixed4_clean_f1 is not None: return fixed4_clean_f1, 'fixed4_clean_macro_f1'   # L2-D11
    if target == 'USA' and USA_SCORING == 'fixed4' and fixed4_f1 is not None: return fixed4_f1, 'fixed4_macro_f1'              # L2-D9 (superseded)
    if target in FRA_LIKE_TARGETS and FRA_MAIN_RULE == 'coverage_conditional': return cov_cond_primary_f1, 'coverage_conditional_macro_f1'
    return primary_f1, ('binary_key_crop_macro_f1' if target in BINARY_TARGETS else 'macro_f1_existing_release_remap')

def add_main_metric(d):
    # fills main_f1 / main_metric where they are missing (results written before v3.7); existing values are never changed
    d = d.copy()
    if 'main_f1' not in d.columns: d['main_f1'] = np.nan
    if 'main_metric' not in d.columns: d['main_metric'] = ''
    miss = pd.to_numeric(d['main_f1'], errors='coerce').isna() & d['primary_f1'].notna() if 'primary_f1' in d.columns else pd.Series(False, index=d.index)
    for i in d.index[miss]:
        v, name = main_metric_value(d.at[i, 'target'], pd.to_numeric(d.at[i, 'primary_f1'], errors='coerce'), pd.to_numeric(d.at[i, 'cov_cond_primary_f1'], errors='coerce')); d.at[i, 'main_f1'] = v; d.at[i, 'main_metric'] = name
    return d

def load_results(path):
    if not os.path.exists(path): return pd.DataFrame(columns=COLUMNS)
    return add_main_metric(pd.read_csv(path))

def append_row(path, row):
    df = load_results(path); df = pd.concat([df, pd.DataFrame([{c: row.get(c, '') for c in COLUMNS}])], ignore_index=True)
    atomic_write(path, lambda f: df.to_csv(f, index=False), mode='w')

def plan_for_seed(seed, cells=None):
    # the cells that exist for ONE seed: all of PLAN_CELLS_RUN, except that k-center cells run only for seeds in KC_SEEDS (v3.8 amendment V38-A1)
    cells = PLAN_CELLS_RUN if cells is None else cells
    return [c for c in cells if c.get('sel', 'random') != 'kcenter' or seed in KC_SEEDS]

def auto_gate_ok(seed, results_csv=None):
    # automatic stand-in for "read the gate report and approve": the seed is complete and has NO error / integrity_fail row.  Skill warnings never block (same policy as before).
    df = load_results(results_csv or os.path.join(OUT_DIR, 'results.csv'));
    if not len(df): return False, 'no results'
    d = df[pd.to_numeric(df['seed'], errors='coerce') == seed].drop_duplicates('run_id', keep='last')      # latest row per run: an error that was later re-run OK no longer blocks
    bad = int((~d['status'].isin(['ok'])).sum()); n_ok = int((d['status'] == 'ok').sum())
    if bad: return False, f'{bad} error / integrity_fail row(s) in seed {seed}'
    n_plan = len(plan_for_seed(seed))
    if n_ok != n_plan: return False, f'seed {seed} incomplete ({n_ok}/{n_plan} runs)'
    return True, f'seed {seed}: {n_ok}/{n_plan} runs, 0 errors, 0 integrity failures'

def plan_consistency_check(results_csv=None):
    # STOP-level guard against a plan that changed between seeds: every run id already on disk must be a run id of the CURRENT plan (otherwise earlier and later seeds would not be the same experiment).
    df = load_results(results_csv or os.path.join(OUT_DIR, 'results.csv'))
    expected = {make_run_id(c, s): (c, s) for s in SEEDS for c in plan_for_seed(s)}
    if not len(df): return dict(ok=True, existing=0, orphans=[], new=len(expected), msg='no results yet')
    have = set(df.loc[df['status'].isin(['ok', 'integrity_fail']), 'run_id'].astype(str)); orphans = sorted(have - set(expected))
    return dict(ok=not orphans, existing=len(have), orphans=orphans, new=len(set(expected) - have), msg=(f'{len(have)} existing run ids are all in the current plan; {len(set(expected) - have)} runs still to do' if not orphans else f'{len(orphans)} existing run ids are NOT in the current plan, e.g. {orphans[:3]}'))

CANARY_TOL = 1e-6
def canary_check(results_csv=None, out_dir=None, n=3):
    # re-run up to n FINISHED runs (aug off, aug on, k-center; seed 0 preferred) with the current code and compare with the stored scores.  Same device + any difference => the code or design changed between seeds: STOP.  Different device => report only (GPU arithmetic may differ).
    out_dir = out_dir or OUT_DIR; df = load_results(results_csv or os.path.join(out_dir, 'results.csv'))
    if not len(df): return dict(ok=True, rows=[], note='no finished runs yet')
    ok_rows = df[df['status'] == 'ok'].copy(); ok_rows['seed'] = pd.to_numeric(ok_rows['seed'], errors='coerce'); by_id = {make_run_id(c, s): (c, s) for s in SEEDS for c in plan_for_seed(s)}
    picks = []
    for pred in (lambda r: str(r['aug']) == 'off' and str(r['sample_sel']) in ('random', '', 'nan') and r['target'] == 'ARG', lambda r: str(r['aug']) == 'on', lambda r: str(r['sample_sel']) == 'kcenter'):
        for _, r in ok_rows.sort_values(['seed', 'run_id']).iterrows():
            if pred(r) and r['run_id'] in by_id and r['run_id'] not in [p['run_id'] for p in picks]: picks.append(r.to_dict()); break
    dev_now = None
    try:
        import torch; dev_now = 'cuda: ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
    except Exception: pass
    dev_then = None; p0 = os.path.join(out_dir, 'run_environment_seed0.json')
    if os.path.exists(p0):
        try: dev_then = json.load(open(p0)).get('device')
        except Exception: pass
    same_dev = (dev_now is None) or (dev_then is None) or (dev_now == dev_then); res = []
    for r in picks[:n]:
        cell, s = by_id[r['run_id']]; row, _p, _y, _sc = run_one(cell, int(s), r['run_id'], out_dir, save=False)
        d_f1 = abs(float(row['main_f1']) - float(r['main_f1'])); d_ep = abs(int(row['epochs_run']) - int(r['epochs_run']))
        res.append(dict(run_id=r['run_id'], stored=float(r['main_f1']), rerun=float(row['main_f1']), abs_diff=d_f1, epochs_stored=int(r['epochs_run']), epochs_rerun=int(row['epochs_run']), match=(d_f1 <= CANARY_TOL and d_ep == 0)))
    bad = [x for x in res if not x['match']]; ok = (not bad) or (not same_dev)
    return dict(ok=ok, rows=res, same_device=same_dev, device_now=dev_now, device_then=dev_then, mismatches=len(bad),
                note=('all canaries reproduce the stored scores exactly' if not bad else (f'{len(bad)} canary mismatch(es) on a DIFFERENT device ({dev_then} -> {dev_now}): reported, not a stop' if not same_dev else f'{len(bad)} canary mismatch(es) on the SAME device: the code or design changed since the stored runs')))

def replication_table(df, metric='main_f1', exploratory=(0, 1), confirmatory=(2, 3, 4)):
    # the pre-declared claim C1 (and C2) evaluated SEPARATELY on the exploratory and the confirmatory seeds; PASS = same sign in both groups and the 95% interval of the confirmatory group excludes 0.
    rows = []; d = df.copy(); d['seed'] = pd.to_numeric(d['seed'], errors='coerce')
    for label, seeds in (('exploratory', exploratory), ('confirmatory', confirmatory)):
        g = d[d['seed'].isin(seeds)]
        if not len(g): continue
        present = sorted(int(x) for x in g['seed'].dropna().unique()); mm = mixed_model_report(g, metric)
        for _, r in mm[mm['term'] == 'log2(k)'].iterrows():
            h = float(scipy_stats.t.ppf(0.975, max(int(r['groups']) - 3, 1))) * r['se']; rows.append(dict(claim='C1 gain per doubling of states (points)', target=r['target'], group=label, seeds=present, estimate=float(r['coef']), ci_lo=float(r['coef'] - h), ci_hi=float(r['coef'] + h)))
        gc = cell_stats(_random_arm_runs(g), metric)
        for _, r in gain_kmin_kmax(gc).iterrows(): rows.append(dict(claim=f"C2 gain k{int(r['k_min'])}->k{int(r['k_max'])} at budget {int(r['budget'])} (points)", target=r['target'], group=label, seeds=present, estimate=100 * float(r['gain']), ci_lo=100 * float(r['ci_lo']), ci_hi=100 * float(r['ci_hi'])))
    t = pd.DataFrame(rows)
    if not len(t): return t
    verdict = []
    for (cl, tg), g in t.groupby(['claim', 'target']):
        ex, co = g[g['group'] == 'exploratory'], g[g['group'] == 'confirmatory']
        if not len(co): v = 'confirmatory seeds not finished'
        elif not len(ex): v = 'no exploratory estimate'
        else:
            e, c = ex.iloc[0], co.iloc[0]; same = np.sign(e['estimate']) == np.sign(c['estimate']); excl = np.isfinite(c['ci_lo']) and (c['ci_lo'] > 0 or c['ci_hi'] < 0)
            has_ci = bool(np.isfinite(c['ci_lo']) and np.isfinite(c['ci_hi']))
            v = 'PASS' if (same and excl) else ('NOT REPLICATED (sign differs)' if not same else ('NOT REPLICATED (confirmatory interval includes 0)' if has_ci else 'NOT TESTABLE (no sampling interval exists: census or fully enumerated cells)'))
        if tg in DESCRIPTIVE_TARGETS: v = 'descriptive only (no claim)'
        verdict.append(((cl, tg), v))
    vm = dict(verdict); t['verdict'] = [vm[(a, b)] for a, b in zip(t['claim'], t['target'])]
    return t

def design_hash():
    # ONE hash of everything that defines the experiment: every LOCKED setting + the complete list of planned run ids for all seeds.  The instructor approves THIS value; the run cell refuses any other.
    snap = {k: json.loads(json.dumps(globals()[k], default=str)) for k in sorted(LOCKED)}
    rids = sorted(make_run_id(c, s_) for s_ in SEEDS for c in plan_for_seed(s_))
    scoring = dict(USA_SCORING=USA_SCORING, USA_FIXED_CLASSES=USA_FIXED_CLASSES, USA_DROP_TRUE_CLASSES=USA_DROP_TRUE_CLASSES, USA_REASSIGN_PRED_TO_OTHER=USA_REASSIGN_PRED_TO_OTHER)       # L2-D9 / L2-D11: the scoring rule is part of the approved design
    return hashlib.sha256(json.dumps(dict(locked=snap, run_ids=rids, scoring=scoring), sort_keys=True).encode()).hexdigest()

def instructor_gate():
    # Level 2 only: nothing trains until the instructor has signed the decisions AND the plan that is about to run is byte-for-byte the plan that was approved.
    if not INSTRUCTOR_APPROVED: raise RuntimeError('STOP: INSTRUCTOR_APPROVED is False. Level 2 may only be run after the instructor has signed the decisions (see the plan cell for the design hash to approve).')
    dh = design_hash()
    if APPROVED_DESIGN_SHA256 != dh: raise RuntimeError(f'STOP: the plan about to run (design hash {dh}) is not the approved one (APPROVED_DESIGN_SHA256 = {APPROVED_DESIGN_SHA256!r}). Nothing was run.')
    return dh

_USA_METRIC_NAME = {'fixed4_clean': 'fixed4_clean_macro_f1', 'fixed4': 'fixed4_macro_f1'}
def scoring_consistency_check(results_path):
    # STOP if finished USA rows were scored with a different rule than the one that is about to be used (they must be re-scored first; see the re-scoring cell)
    if USA_SCORING not in _USA_METRIC_NAME or not os.path.exists(results_path): return True
    want = _USA_METRIC_NAME[USA_SCORING]; d = pd.read_csv(results_path); bad = d[(d['target'] == 'USA') & (d['status'] == 'ok') & (d['main_metric'].astype(str) != want)]
    _check(len(bad) == 0, f'STOP: {len(bad)} finished USA run(s) in results.csv are not scored with the current rule ({want}; L2-D11). Run the RE-SCORING cell first (it reads the saved predictions; no retraining) -- mixing scoring rules in one column is not allowed.')
    return True

def _npz_path(out_dir, rid): return os.path.join(out_dir, 'runs', hashlib.md5(rid.encode()).hexdigest()[:16] + '.npz')

def rescore_usa_fixed4(out_dir, results_path=None, write=True):
    # Re-scores EVERY finished USA row that does not yet carry the current rule, from its SAVED predictions (no retraining), whatever its earlier state (coverage-conditional, or fixed-class v1 of L2-D9).
    # All-or-nothing: a missing file, or saved predictions that do not reproduce the number already stored in the row -> STOP and nothing is written.
    # Before writing: results.csv is copied byte-for-byte to results.csv.before_rescore_<time> (SHA-256 logged).  main_f1_before_clean keeps the value main_f1 had just before; main_f1_before_fixed4 keeps the ORIGINAL
    # coverage-conditional score; fixed4_f1 keeps the L2-D9 value; fixed4_clean_f1 is the new value.
    results_path = results_path or os.path.join(out_dir, 'results.csv'); d = pd.read_csv(results_path)
    for c_ in ('fixed4_f1', 'fixed4_clean_f1', 'main_f1_before_fixed4', 'main_f1_before_clean'):
        if c_ not in d.columns: d[c_] = np.nan
    want = _USA_METRIC_NAME[USA_SCORING]; todo = d.index[(d['target'] == 'USA') & (d['status'] == 'ok') & (d['main_metric'].astype(str) != want)]
    rows = []; O = CROP_ID['other']; lab = [CROP_ID[c_] for c_ in USA_FIXED_CLASSES]
    for i in todo:
        rid = d.at[i, 'run_id']; p = _npz_path(out_dir, rid); _check(os.path.exists(p), f'STOP: saved predictions missing for {rid} ({p}); nothing was changed.')
        z = np.load(p, allow_pickle=False); yt, yp, tc = z['y_true'].astype(int), z['y_pred'].astype(int), z['train_counts'].astype(int)
        present = np.unique(yt); sev = float(f1_score(yt, yp, average='macro', labels=present, zero_division=0))
        _check(abs(sev - float(d.at[i, 'sevenclass_f1'])) < 1e-6, f'STOP: {rid}: saved predictions give 7-class F1 {sev:.6f}, results.csv says {d.at[i, "sevenclass_f1"]}; nothing was changed.')
        old_name = str(d.at[i, 'main_metric'])
        if old_name == 'coverage_conditional_macro_f1':
            cov = [k for k, c_ in enumerate(present) if tc[c_] >= COVERAGE_MIN_TRAIN]; _p, _r, f_s = _sk_prf(yt, yp, present); chk = float(np.mean(f_s[cov])) if cov else float('nan')
        elif old_name == 'fixed4_macro_f1': chk = fixed4_v1_scores(yt, yp)['f1']
        else: raise AssertionError(f'STOP: {rid}: unknown earlier scoring rule {old_name!r}; nothing was changed.')
        _check(abs(chk - float(d.at[i, 'main_f1'])) < 1e-6, f'STOP: {rid}: saved predictions give the old main score {chk:.6f} ({old_name}), results.csv says {d.at[i, "main_f1"]}; nothing was changed.')
        f4 = fixed4_scores(yt, yp); _check(abs(f4['f1'] - f4['f1_numpy']) < SCORER_TOL, f'{rid}: the two scorers disagree on the fixed-class score')
        v1 = fixed4_v1_scores(yt, yp)['f1']                                                                                         # L2-D9 value (stored for rows that did not have it)
        rows.append(dict(i=i, run_id=rid, old_rule=old_name, old=float(d.at[i, 'main_f1']), v1=v1, new=f4['f1'], n_true_dropped=f4['n_dropped'], n_pred_reassigned=f4['n_pred_reassigned'], n_scored=f4['n_scored'], states=d.at[i, 'states'], seed=d.at[i, 'seed']))
    rep_ = pd.DataFrame(rows)
    if write and len(rep_):
        stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S'); bak = results_path + '.before_rescore_' + stamp; shutil.copyfile(results_path, bak); sha = sha256_file(bak)
        for r in rows:
            d.at[r['i'], 'main_f1_before_clean'] = r['old']
            if r['old_rule'] == 'coverage_conditional_macro_f1': d.at[r['i'], 'main_f1_before_fixed4'] = r['old']
            d.at[r['i'], 'fixed4_f1'] = r['v1']; d.at[r['i'], 'fixed4_clean_f1'] = r['new']; d.at[r['i'], 'main_f1'] = r['new']; d.at[r['i'], 'main_metric'] = want
        atomic_write(results_path, lambda f: d.to_csv(f, index=False), mode='w')
        log_p = os.path.join(out_dir, 'rescore_log.json'); hist = json.load(open(log_p)) if os.path.exists(log_p) else []
        hist.append(dict(time=stamp, backup=os.path.basename(bak), backup_sha256=sha, rows_rescored=len(rows), seeds=sorted(int(x) for x in rep_['seed'].unique()), rule=dict(classes=USA_FIXED_CLASSES, drop_true=USA_DROP_TRUE_CLASSES, reassign_predictions_to_other=USA_REASSIGN_PRED_TO_OTHER, name=want)))
        atomic_write(log_p, lambda f: f.write(json.dumps(hist, indent=1)), mode='w')
        print(f're-scored {len(rows)} USA rows (seeds {sorted(int(x) for x in rep_["seed"].unique())}); original file kept as {os.path.basename(bak)} (sha256 {sha[:16]}...)')
    return rep_

def run_seed_cell(seed, deadline=None):
    assert TESTS_PASSED and SMOKE_OK and APPROVED['plan'], 'test gate, smoke test and plan approval are required'
    instructor_gate(); scoring_consistency_check(os.path.join(OUT_DIR, 'results.csv'))
    assert seed in SEEDS, f'seed must be one of {SEEDS}'
    results_csv = os.path.join(OUT_DIR, 'results.csv'); log_path = os.path.join(OUT_DIR, 'run_log.txt')
    if seed > min(SEEDS):
        assert APPROVED[f'seed{seed - 1}_gate_report'], f'seed {seed - 1} gate report has not been approved (APPROVED["seed{seed - 1}_gate_report"])'
        prev = load_results(results_csv); n = int(((pd.to_numeric(prev['seed'], errors='coerce') == seed - 1) & prev['status'].isin(['ok', 'integrity_fail'])).sum()) if len(prev) else 0
        assert n == len(plan_for_seed(seed - 1)), f'seed {seed - 1} is incomplete ({n}/{len(plan_for_seed(seed - 1))} runs) -- finish it first'
    check_namespace(write=True)
    os.makedirs(os.path.join(OUT_DIR, 'runs'), exist_ok=True); os.makedirs(os.path.join(OUT_DIR, 'gate_reports'), exist_ok=True)
    write_run_environment(OUT_DIR, seed)
    status = run_seed(seed, plan_for_seed(seed), results_csv, OUT_DIR, log_path, deadline=deadline)
    if status == 'paused': return 'paused'
    df_ = load_results(results_csv); report = gate_report(df_, seed, plan_for_seed(seed))
    atomic_write(os.path.join(OUT_DIR, 'gate_reports', f'seed{seed}.txt'), lambda f: f.write(report), mode='w'); print('\n' + report)
    return 'complete'

def run_guards():
    # the two seed-consistency guards (V38-A4).  True = safe to run.
    pc = plan_consistency_check(); print('  PLAN CHECK:', pc['msg'])
    if not pc['ok']: print('\n!! STOPPED: the plan changed since the earlier seeds were run (see above).  Do not continue: earlier and later seeds would not be the same experiment.'); return False
    cc = canary_check(); print('  CANARY CHECK:', cc['note'])
    for x in cc.get('rows', []): print(f"     {x['run_id']}: stored {x['stored']:.6f} | re-run {x['rerun']:.6f} | epochs {x['epochs_stored']} vs {x['epochs_rerun']} | {'OK' if x['match'] else 'DIFFERENT'}")
    if not cc['ok']: print('\n!! STOPPED: finished runs do not reproduce on the same device, so the code or design changed since they were run.  Report this; do not continue.'); return False
    return True

def run_all_seeds(max_hours=None):
    # THE WHOLE EXPERIMENT: every seed, every planned run (all arms, augmentation ON and OFF cells together).  Resumable: run it again after a disconnect / time limit.
    # Between seeds the gate report is approved AUTOMATICALLY only if the seed is complete with no error / integrity failure; otherwise it stops and says why.
    deadline = time.time() + 3600 * max_hours if max_hours else None; t0 = time.time()
    print(f'AUTO RUN: seeds {SEEDS}, planned runs per seed {[len(plan_for_seed(s_)) for s_ in SEEDS]} ({sum(1 for c in PLAN_CELLS_RUN if c.get("aug", "off") == "on")} augmentation ON + {sum(1 for c in PLAN_CELLS_RUN if c.get("aug", "off") == "copy")} steps-matched copy control per seed; k-center only for seeds {KC_SEEDS}), time limit {max_hours or "none"} h')
    instructor_gate()
    if not run_guards(): return 'stopped'
    for s in SEEDS:
        if s > min(SEEDS):
            ok, why = auto_gate_ok(s - 1)
            if not ok: print(f'\n!! AUTO RUN STOPPED before seed {s}: {why}.  Read the gate report / errors.log, fix the cause, then run this cell again (finished runs are skipped).'); return 'stopped'
            APPROVED[f'seed{s - 1}_gate_report'] = True; print(f'  auto-approved gate report of seed {s - 1} ({why})')
        status = run_seed_cell(s, deadline)
        if status == 'paused': print(f'\n>>> PAUSED at the time limit after {(time.time() - t0) / 3600:.1f} h: run this cell again to continue with seed {s}.'); return 'paused'
    ok, why = auto_gate_ok(SEEDS[-1])
    if not ok: print(f'\n!! LAST SEED NOT CLEAN: {why}'); return 'stopped'
    print(f'\n>>> ALL {len(SEEDS)} SEEDS COMPLETE ({(time.time() - t0) / 3600:.1f} h this session). Run the summary cell next.'); return 'complete'

def build_row(rid, cell, seed, split, sc, I, ok, skills, info, preds, y_true, seconds, usa_crop):
    tr, va = split['train_pos'], split['val_pos']; row = {c: '' for c in COLUMNS}; cr = np.asarray(usa_crop)
    row.update(run_id=rid, status='ok' if ok else 'integrity_fail', target=cell['target'], role=TARGET_ROLE[cell['target']], budget=cell['budget'], k=cell['k'], combo_id=cell['combo_id'],
               states='-'.join(cell['states']), pool_size=cell.get('pool_size', ''), n_total_combos=cell.get('n_total_combos', ''), seed=seed, n_train=len(tr), n_val=len(va),
               epochs_run=info.get('epochs_run', ''), best_epoch=info.get('best_epoch', ''), stop_reason=info.get('stop_reason', ''), val_f1=info['val_f1'], restore_delta=info.get('restore_delta', ''),
               primary_view=sc['primary_view'], primary_f1=sc['primary_f1'], primary_oa=sc['primary_oa'], paper_f1=sc['paper_f1'], paper_oa=sc['paper_oa'], binary_f1=sc['bin_f1'], binary_oa=sc['bin_oa'],
               baseline_all_other_primary=sc['baseline_all_other_primary'], baseline_majority_primary=sc['baseline_majority_primary'],
               baseline_all_other_paper=sc['baseline_all_other_paper'], baseline_majority_paper=sc['baseline_majority_paper'],
               cov_cond_primary_f1=(sc['cov_cond_binary_f1'] if cell['target'] in BINARY_TARGETS else sc['cov_cond_paper_f1']), cov_cond_paper_f1=sc['cov_cond_paper_f1'],
               absent_train_classes='+'.join(sc['absent_train_classes']), scorer_maxdiff=sc['scorer_maxdiff'], test_n=sc['n'], key_n=sum(split['key_counts'].values()),
               profile=PROFILE, state_rows='|'.join(f'{s_}:{int((np.asarray(SRC_UNIT)[np.concatenate([tr, va])] == s_).sum())}' for s_ in cell['states']), max_state_share=round(max(int((np.asarray(SRC_UNIT)[np.concatenate([tr, va])] == s_).sum()) for s_ in cell['states']) / len(np.concatenate([tr, va])), 4),
               fixed4_f1=sc['fixed4_f1'], fixed4_clean_f1=sc['fixed4_clean_f1'], main_f1=main_metric_value(cell['target'], sc['primary_f1'], (sc['cov_cond_binary_f1'] if cell['target'] in BINARY_TARGETS else sc['cov_cond_paper_f1']), sc['fixed4_f1'], sc['fixed4_clean_f1'])[0], main_metric=main_metric_value(cell['target'], 0, 0, 0.0, 0.0)[1],
               aug=cell.get('aug', AUG), sample_sel=cell.get('sel', 'random'), input_features=INPUT_FEATURES, train_n_effective=len(tr) * train_rows_multiplier(cell.get('aug', AUG)),
               composition=cell.get('composition', BASE_MODE), arms='|'.join(cell.get('arms', ('random',))), diversity_score=cell.get('div_score', ''), diversity_pct=cell.get('div_pct', ''), class_weight_source=CLASS_WEIGHT_SOURCE,
               sevenclass_f1=sc['sevenclass_f1'], sevenclass_oa=sc['sevenclass_oa'], papertext_f1=sc['papertext_f1'], papertext_oa=sc['papertext_oa'], review_f1=sc['review_f1'], review_oa=sc['review_oa'],
               baseline_all_other_review=sc['baseline_all_other_review'], baseline_majority_review=sc['baseline_majority_review'], baseline_all_other_papertext=sc['baseline_all_other_papertext'],
               baseline_majority_papertext=sc['baseline_majority_papertext'], cov_cond_review_f1=sc['cov_cond_review_f1'], remap_rule_review=REVIEW_RULE,
               n_target_classes=len(DEFINED_CLASSES[cell['target']]),
               n_target_classes_train_ge30=int(sum(int((cr[tr] == c_).sum()) >= COVERAGE_MIN_TRAIN for c_ in DEFINED_CLASSES[cell['target']])),
               n_absent_target_classes_train=int(sum(int((cr[tr] == c_).sum()) == 0 for c_ in DEFINED_CLASSES[cell['target']])),
               steps_per_epoch=steps_per_epoch(len(tr) * train_rows_multiplier(cell.get('aug', AUG))), total_optimizer_steps=steps_per_epoch(len(tr) * train_rows_multiplier(cell.get('aug', AUG))) * int(info.get('epochs_run', 0) or 0),
               target_crop_n=int(sum(int((cr[np.concatenate([tr, va])] == c_).sum()) for c_ in TARGET_CROPS[cell['target']])),
               integrity_pass=ok, skill_flags='|'.join(skills), seconds=round(seconds, 2), error='')
    for c in CROP_ORDER:
        row[f'train_{c}'] = int((cr[tr] == c).sum()); row[f'val_{c}'] = int((cr[va] == c).sum())
        row[f'true_{c}'] = int((np.asarray(y_true) == CROP_ID[c]).sum()); row[f'pred_{c}'] = int((np.asarray(preds) == CROP_ID[c]).sum())
        row[f'f1_{c}'] = sc['paper_f1_cls'][CROP_ID[c]]; row[f'review_f1_{c}'] = sc['review_f1_cls'][CROP_ID[c]]
    row.update(I); return row

# ------------------------------------------------------------------------------ RUN ONE JOB + RUN LOOP
#   FIT_FN(Xtr, ytr, Xva, yva, seed, epochs_override) -> dict(model=..., info=dict(val_f1, best_epoch, epochs_run, stop_reason, restore_delta, history))
#   PREDICT_FN(model, X) -> integer class ids 0..6        (both injected by the torch cell)
FIT_FN, PREDICT_FN = None, None

def run_one(cell, seed, rid, out_dir, epochs_override=None, save=True):
    t0 = time.time(); target, B, states = cell['target'], cell['budget'], tuple(cell['states'])
    mode = cell.get('composition', BASE_MODE)
    split = draw_split(target, B, states, seed, POOLS, SRC_CROP, mode, sel=cell.get('sel', 'random'))
    chk = verify_split(split, target, B, states, SRC_UNIT, SRC_CROP, mode)
    if not chk['ok']: raise IntegrityFailure('SPLIT NOT EXACT: ' + '; '.join(chk['msgs']))
    tr, va = split['train_pos'], split['val_pos']
    out = FIT_FN(take(FEATURES[SOURCE], tr), SRC_CROP_ID[tr], take(FEATURES[SOURCE], va), SRC_CROP_ID[va], seed, epochs_override, **({'aug': cell['aug']} if cell.get('aug', AUG) in ('on', 'copy') else {}))
    preds = np.asarray(PREDICT_FN(out['model'], EVAL_X[target])).astype(int); y_true = np.asarray(EVAL_Y[target]).astype(int)
    train_counts = np.bincount(SRC_CROP_ID[tr], minlength=7); val_counts = np.bincount(SRC_CROP_ID[va], minlength=7)
    sc = score_run(target, y_true, preds, train_counts)
    I, ok = evaluate_integrity(sc, preds, chk, out['info']); skills = skill_flags(sc, out['info'])
    row = build_row(rid, cell, seed, split, sc, I, ok, skills, out['info'], preds, y_true, time.time() - t0, SRC_CROP)
    if save:                                                                    # 1) per-run arrays (atomic)   2) the metrics row is appended by the caller LAST
        def w(f):
            np.savez_compressed(f, run_id=rid, y_true=y_true.astype(np.uint8), y_pred=preds.astype(np.uint8), y_pred_existing_release=sc['yp_remapped'].astype(np.uint8), y_pred_review_rule=remap_review_targetmap(preds, target).astype(np.uint8), y_pred_papertext_rule=remap_paper_text_v3(preds, target).astype(np.uint8),
                                target_sample_ids=np.asarray(TARGET_IDS[target]).astype(np.int64), src_train_ids=SRC_NPZ_ROW[tr].astype(np.int64), src_val_ids=SRC_NPZ_ROW[va].astype(np.int64),
                                train_counts=train_counts, val_counts=val_counts, cm7=sc['cm7'], cm_existing_release=sc['cm_paper'], cm_review_rule=sc['cm_review'], cm_papertext_rule=sc['cm_papertext'], review_f1_per_class=sc['review_f1_cls'],
                                cm_binary=(sc['cm_binary'] if sc['cm_binary'] is not None else np.zeros((0, 0), dtype=int)),
                                paper_precision=sc['paper_prec'], paper_recall=sc['paper_rec'], paper_f1_per_class=sc['paper_f1_cls'],
                                binary_precision=(sc['bin_prec'] if sc['bin_prec'] is not None else np.zeros(0)), binary_recall=(sc['bin_rec'] if sc['bin_rec'] is not None else np.zeros(0)),
                                binary_f1_per_class=(sc['bin_f1_cls'] if sc['bin_f1_cls'] is not None else np.zeros(0)), val_history=np.asarray(out['info'].get('history', []), dtype=np.float32))
        atomic_write(os.path.join(out_dir, 'runs', hashlib.md5(rid.encode()).hexdigest()[:16] + '.npz'), w)
    return row, preds, y_true, sc

def take(X, idx):
    idx = np.asarray(idx)
    if type(X).__module__.startswith('torch'):
        import torch
        return X[torch.as_tensor(idx, dtype=torch.long)]
    return X[idx]

def _log(msg, path=None):
    print(msg, flush=True)
    if path:
        with open(path, 'a') as f: f.write(msg + '\n')

def _pct(v): return '  n/a' if v is None or not np.isfinite(v) else f'{100 * v:5.1f}'

def seed_line(row, sc):
    sk = row['skill_flags']
    return (f"  [{row['target']} B{row['budget']} k{row['k']} {row['states']} s{row['seed']}] {row['primary_view']} mF1 {_pct(row['primary_f1'])} "
            f"(all-other {_pct(row['baseline_all_other_primary'])}, majority {_pct(row['baseline_majority_primary'])}) | EXISTING release-remap {_pct(row['paper_f1'])} | REVIEW {_pct(row['review_f1'])} (base {_pct(row['baseline_all_other_review'])}) | 7-class raw {_pct(row['sevenclass_f1'])} | cov-cond EXISTING {_pct(row['cov_cond_primary_f1'])} REVIEW {_pct(row['cov_cond_review_f1'])} "
            f"| steps {row['total_optimizer_steps']} | val {_pct(row['val_f1'])} best-ep {row['best_epoch']} ran {row['epochs_run']}/{MAX_EPOCHS} ({row['stop_reason']}) "
            f"| INTEGRITY {'PASS' if row['integrity_pass'] else 'FAIL'}" + (f' | {sk}' if sk else ''))

def run_seed(seed, cells, results_csv, out_dir, log_path=None, epochs_override=None, deadline=None):
    # one seed over ALL planned cells; resumable (finished runs are skipped); aborts only after MAX_CONSEC_INTEGRITY consecutive integrity failures/errors
    prev = load_results(results_csv); done = set(prev.loc[prev['status'].isin(['ok', 'integrity_fail']), 'run_id']) if len(prev) else set()
    todo = [c for c in cells if make_run_id(c, seed) not in done]
    _log(f'SEED {seed}: {len(cells)} planned runs | already done {len(cells) - len(todo)} | pending {len(todo)}', log_path)
    t_start, n_new, consec = time.time(), 0, 0
    for cell in todo:
        if deadline is not None and time.time() > deadline:
            _log(f'SEED {seed} PAUSED at the time limit ({n_new} new runs this session); run the same cell again to resume.', log_path); return 'paused'
        rid = make_run_id(cell, seed)
        try:
            row, preds, y_true, sc = run_one(cell, seed, rid, out_dir, epochs_override=epochs_override)
        except TooFewClasses: raise                                              # target data problem: stop the whole run
        except Exception as e:
            consec += 1; msg = f'{type(e).__name__}: {e}'[:400]; _log(f'  !! INTEGRITY/ERROR in {rid}: {msg}', log_path)
            with open(os.path.join(out_dir, 'errors.log'), 'a') as f: f.write(f'{rid}\n{traceback.format_exc()}\n')
            er = {c: '' for c in COLUMNS}; er.update(run_id=rid, status='integrity_fail' if isinstance(e, IntegrityFailure) else 'error', target=cell['target'], budget=cell['budget'], k=cell['k'],
                                                     combo_id=cell['combo_id'], states='-'.join(cell['states']), seed=seed, error=msg, integrity_pass=False)
            append_row(results_csv, er)
            if consec >= MAX_CONSEC_INTEGRITY: raise RuntimeError(f'{consec} consecutive integrity failures/errors -- aborted. Last: {msg}')
            continue
        append_row(results_csv, row); n_new += 1                                 # the metrics file is written LAST
        _log(seed_line(row, sc), log_path)
        consec = consec + 1 if row['status'] == 'integrity_fail' else 0
        if consec >= MAX_CONSEC_INTEGRITY: raise RuntimeError(f'{consec} consecutive integrity failures -- aborted (see the lines above).')
        if n_new % 10 == 0:
            el = time.time() - t_start; _log(f'  .. {n_new} new runs, {el / n_new:.1f}s/run, ~{(len(todo) - n_new) * el / n_new / 3600:.1f} h left for this seed', log_path)
    _log(f'SEED {seed} COMPLETE: {n_new} new runs this session', log_path)
    return 'complete'

# ------------------------------------------------------------------------------ STATISTICS (spec 5)
def t_interval_fpc(x, N, level=0.95):
    # t-interval over state COMBINATIONS with finite-population correction sqrt((N-n)/(N-1)); N = all C(pool,k) combinations
    x = np.asarray(x, dtype=float); x = x[np.isfinite(x)]; n = len(x); m = float(x.mean()) if n else float('nan')
    if n < 2: return m, float('nan'), float('nan'), float('nan')
    fpc = (1.0 if not np.isfinite(N) else math.sqrt(max(N - n, 0) / (N - 1))) if N > 1 else 0.0           # N = inf: the combinations are NOT a random sample of a finite population -> no correction
    se = float(x.std(ddof=1)) / math.sqrt(n) * fpc; h = float(scipy_stats.t.ppf(0.5 + level / 2, n - 1)) * se
    return m, m - h, m + h, se

def combo_level(df, metric='primary_f1'):
    # unit = state combination; the seeds are averaged FIRST
    d = df[df['status'] == 'ok'].copy()
    for c_ in ('diversity_score', 'diversity_pct'):
        if c_ not in d.columns: d[c_] = np.nan             # results written before Phase 4 have no diversity columns
    for c in [metric, 'budget', 'k', 'seed', 'pool_size', 'n_total_combos', 'key_n', 'target_crop_n', 'n_target_classes_train_ge30', 'diversity_score', 'diversity_pct']: d[c] = pd.to_numeric(d[c], errors='coerce')
    g = d.groupby(['target', 'budget', 'k', 'states']).agg(f1=(metric, 'mean'), seed_sd=(metric, lambda x: float(np.std(x, ddof=1)) if len(x) > 1 else float('nan')),
                                                         n_seeds=(metric, 'count'), pool_size=('pool_size', 'first'), n_total=('n_total_combos', 'first'), key_n=('key_n', 'first'), target_crop_n=('target_crop_n', 'mean'), n_cls_ge30=('n_target_classes_train_ge30', 'mean'), div_score=('diversity_score', 'first'), div_pct=('diversity_pct', 'first')).reset_index()
    return g

def expand_arms(df):
    # one copy of every run per arm it belongs to (a combination can be in several arms, e.g. the k=1 census is in all of them); column 'arm' added
    d = df.copy(); d['arms'] = d['arms'].fillna('random').replace('', 'random').astype(str); out = []
    for a in ARM_ORDER:
        m = d['arms'].str.split('|').apply(lambda x: a in x)
        if m.any(): out.append(d[m].assign(arm=a))
    return pd.concat(out, ignore_index=True) if out else d.assign(arm=[])

def arm_contrast(df, metric='primary_f1', a='diverse', b='random'):
    # per (target,budget,k>=2): mean difference (arm a - arm b) over combinations, Welch t-interval.  Combinations are NOT independent (they share states) -> descriptive, plus the jackknife of cell_stats.
    E = expand_arms(df); rows = []
    if not len(E): return pd.DataFrame()
    cl = pd.concat([combo_level(E[E['arm'] == x], metric).assign(arm=x) for x in (a, b) if (E['arm'] == x).any()], ignore_index=True)
    for (t, B, k), g in cl.groupby(['target', 'budget', 'k']):
        xa, xb = g[g.arm == a]['f1'].values, g[g.arm == b]['f1'].values
        if k == 1 or len(xa) < 2 or len(xb) < 2: continue
        d_ = float(xa.mean() - xb.mean()); va, vb = xa.var(ddof=1) / len(xa), xb.var(ddof=1) / len(xb); se = math.sqrt(va + vb)
        dfw = (va + vb) ** 2 / (va ** 2 / (len(xa) - 1) + vb ** 2 / (len(xb) - 1)) if se > 0 else float('nan'); h = float(scipy_stats.t.ppf(0.975, dfw)) * se if se > 0 else float('nan')
        rows.append(dict(target=t, budget=B, k=k, contrast=f'{a} - {b}', n_a=len(xa), n_b=len(xb), diff=d_, ci_lo=d_ - h, ci_hi=d_ + h, p_welch=float(scipy_stats.ttest_ind(xa, xb, equal_var=False).pvalue)))
    return pd.DataFrame(rows)

def arm_paired_contrast(df, metric, a, b='random'):
    # same target / budget / k / state combination in BOTH arms: difference of the seed-averaged scores, t-interval over combinations (k=1: over states).  Units overlap in states -> DESCRIPTIVE.
    E = expand_arms(df); rows = []
    if not len(E) or not (E['arm'] == a).any() or not (E['arm'] == b).any(): return pd.DataFrame()
    A = combo_level(E[E['arm'] == a], metric)[['target', 'budget', 'k', 'states', 'f1']].rename(columns={'f1': 'fa'}); Bn = combo_level(E[E['arm'] == b], metric)[['target', 'budget', 'k', 'states', 'f1']].rename(columns={'f1': 'fb'})
    M = A.merge(Bn, on=['target', 'budget', 'k', 'states'])
    for (t, B, k), g in M.groupby(['target', 'budget', 'k']):
        dlt = (g['fa'] - g['fb']).values
        if len(dlt) < 3: continue
        m = float(dlt.mean()); se = float(dlt.std(ddof=1)) / math.sqrt(len(dlt)); h = float(scipy_stats.t.ppf(0.975, len(dlt) - 1)) * se
        rows.append(dict(target=t, budget=int(B), k=int(k), contrast=f'{a} - {b} (paired)', n_pairs=len(dlt), diff=m, ci_lo=m - h, ci_hi=m + h, p_paired=float(scipy_stats.ttest_1samp(dlt, 0.0).pvalue) if se > 0 else float('nan'), unit='states' if k == 1 else 'combinations'))
    return pd.DataFrame(rows)

# ------------------------------------------------------------------------------ v3.8 generalisation statistics (instructor review): mixed model, average / worst-case gain, "beats the best single state"
def _random_arm_runs(df):
    # the random arm only (plain draws, aug OFF, random sample selection): the baseline the review's claims are about
    E = expand_arms(df); d = E[(E['arm'] == 'random') & (E['aug'].fillna('off').astype(str) == 'off') & (E['sample_sel'].fillna('random').replace('', 'random').astype(str) == 'random')].copy()
    for c in ['k', 'budget', 'seed']: d[c] = pd.to_numeric(d[c], errors='coerce')
    return d

def reml_random_intercept(y, X, groups):
    # y = X beta + u_g + e,  u_g ~ N(0, s2u),  e ~ N(0, s2e): REML estimate of lambda = s2u / s2e by a 1-D search, then GLS beta and its covariance.  No external dependency (statsmodels is not needed).
    y = np.asarray(y, float); X = np.asarray(X, float); g = pd.factorize(np.asarray(groups))[0]; G = g.max() + 1; N, p = X.shape
    n_g = np.bincount(g, minlength=G).astype(float); Xs = np.zeros((G, p)); ys = np.zeros(G); np.add.at(Xs, g, X); np.add.at(ys, g, y)
    XtX, Xty, yty = X.T @ X, X.T @ y, float(y @ y)
    def parts(lam):
        c = lam / (1.0 + n_g * lam)                                                  # V_g^-1 = I - c_g * 11'
        A = XtX - (Xs * c[:, None]).T @ Xs; bvec = Xty - (Xs * c[:, None]).T @ ys; q = yty - float((c * ys * ys).sum())
        return A, bvec, q
    def negll(loglam):
        lam = float(np.exp(loglam)); A, bvec, q = parts(lam)
        try: beta = np.linalg.solve(A, bvec)
        except np.linalg.LinAlgError: return 1e12
        rss = q - float(bvec @ beta); s2e = max(rss / (N - p), 1e-12); sign, logdetA = np.linalg.slogdet(A)
        return (N - p) * np.log(s2e) + float(np.log1p(n_g * lam).sum()) + logdetA
    grid = np.linspace(-12, 8, 81); vals = [negll(x) for x in grid]; i = int(np.argmin(vals)); lo, hi = grid[max(i - 1, 0)], grid[min(i + 1, len(grid) - 1)]
    res = scipy_opt.minimize_scalar(negll, bounds=(lo, hi), method='bounded', options=dict(xatol=1e-8)); loglam = res.x if res.fun <= vals[i] else grid[i]
    lam = 0.0 if (loglam <= -11.9) else float(np.exp(loglam)); A, bvec, q = parts(lam); beta = np.linalg.solve(A, bvec); s2e = max((q - float(bvec @ beta)) / (N - p), 1e-12)
    cov = s2e * np.linalg.inv(A); return dict(beta=beta, cov=cov, s2e=s2e, s2u=lam * s2e, lam=lam, G=int(G), N=int(N), p=int(p))

def mixed_model_report(df, metric='main_f1'):
    # per target: F1 (points) ~ log2(k) + budget/1000 [+ seed dummies] with a random intercept per STATE COMBINATION (the same combination is scored at several budgets / seeds, and combinations share states).
    # Estimated with reml_random_intercept; p-values use a t distribution with (groups - parameters) degrees of freedom (conservative).  Never raises.
    d = _random_arm_runs(df); rows = []
    if not len(d): return pd.DataFrame()
    d['y'] = 100 * pd.to_numeric(d[metric], errors='coerce'); d['lk'] = np.log2(d['k']); d['B'] = d['budget'] / 1000.0; d['combo'] = d['target'].astype(str) + '|' + d['states'].astype(str)
    for t, g in d.dropna(subset=['y']).groupby('target'):
        if g['k'].nunique() < 2 or g['combo'].nunique() < 3: rows.append(dict(target=t, term='(not estimable)', coef=float('nan'), se=float('nan'), p=float('nan'), n=len(g), groups=g['combo'].nunique(), note='too few k values / combinations')); continue
        names, cols = ['intercept', 'log2(k)'], [np.ones(len(g)), g['lk'].values]
        if g['B'].nunique() > 1: names.append('budget/1000'); cols.append(g['B'].values)
        sds = sorted(g['seed'].dropna().unique())
        for s_ in sds[1:]: names.append(f'seed{int(s_)}'); cols.append((g['seed'].values == s_).astype(float))
        try:
            X = np.column_stack(cols); r = reml_random_intercept(g['y'].values, X, g['combo'].values); se = np.sqrt(np.diag(r['cov'])); dfree = max(r['G'] - r['p'], 1)
            for i_, nm in enumerate(names):
                if nm == 'intercept' or nm.startswith('seed'): continue
                tv = r['beta'][i_] / se[i_]; rows.append(dict(target=t, term=nm, coef=float(r['beta'][i_]), se=float(se[i_]), p=float(2 * scipy_stats.t.sf(abs(tv), dfree)), n=r['N'], groups=r['G'], note=''))
            rows.append(dict(target=t, term='between-combination SD', coef=float(np.sqrt(r['s2u'])), se=float('nan'), p=float('nan'), n=r['N'], groups=r['G'], note=f"residual SD {np.sqrt(r['s2e']):.2f}"))
        except Exception as e: rows.append(dict(target=t, term='(fit failed)', coef=float('nan'), se=float('nan'), p=float('nan'), n=len(g), groups=g['combo'].nunique(), note=f'{type(e).__name__}: {str(e)[:70]}'))
    return pd.DataFrame(rows)

def single_state_report(df, metric='main_f1', n_boot=2000, seed=0):
    # per (target, budget): the single-state census (k=1) versus every k>=2 combination of the random arm.  Scores are seed-averaged per state / combination.
    d = _random_arm_runs(df); out = []; rng = np.random.default_rng(seed)
    if not len(d): return pd.DataFrame()
    d['y'] = 100 * pd.to_numeric(d[metric], errors='coerce')
    for (t, B), g in d.dropna(subset=['y']).groupby(['target', 'budget']):
        s1 = g[g['k'] == 1].groupby('states')['y'].mean()
        if len(s1) < 2: continue
        for k, gk in sorted(g[g['k'] >= 2].groupby('k')):
            cm = gk.groupby('states')['y'].mean(); v = cm.values
            if len(v) < 2: continue
            boots = [float(rng.choice(v, size=len(v), replace=True).mean()) for _ in range(n_boot)]; mean_gain = float(v.mean() - s1.mean())
            out.append(dict(target=t, budget=int(B), k=int(k), n_single=len(s1), n_combos=len(v), single_mean=float(s1.mean()), single_best=float(s1.max()), single_best_state=str(s1.idxmax()), single_worst=float(s1.min()),
                            combo_mean=float(v.mean()), combo_worst=float(v.min()), combo_best=float(v.max()), avg_gain=mean_gain, avg_gain_ci_lo=float(np.percentile(boots, 2.5)) - float(s1.mean()), avg_gain_ci_hi=float(np.percentile(boots, 97.5)) - float(s1.mean()),
                            worst_case_gain=float(v.min() - s1.mean()), share_beating_best_single=float((v > s1.max()).mean()), worst_vs_best_single=float(v.min() - s1.max())))
    return pd.DataFrame(out)

def alignment_evidence(cache_dir, usa_state, usa_crop, raw_counts, log=print):
    # prints the state-alignment evidence (which ordering was chosen, how close the alternatives were, which crops are disputed) and checks that assigned rows add up to the raw class counts.  Evidence, not a stop: the data cell already stops on a weak or ambiguous alignment.
    p = os.path.join(cache_dir, 'alignment_report.json'); res = dict(report_found=os.path.exists(p), counts_match=None, disputed=[])
    if res['report_found']:
        r = json.load(open(p)); res['disputed'] = list(r.get('disputed_crops', []))
        log(f"  alignment: chosen ordering {r.get('chosen')} (lift {r.get('chosen_lift', float('nan')):.1f}); ambiguous with: {r.get('ambiguous_with') or 'none'}; disputed crops: {res['disputed'] or 'none'}")
        for n_, c_ in (r.get('candidates') or {}).items(): log(f"     candidate {n_:28s} lift {c_.get('lift_all_crops', float('nan')):7.1f}" + (f" | differs from chosen on {c_['differs_from_chosen_pct']:.1f}% of rows" if 'differs_from_chosen_pct' in c_ else ' | chosen'))
        if 'wheat' in res['disputed']: log("  CAVEAT: wheat's state assignment is disputed between orderings.  France's key crops include wheat (state-level France results depend on it); Argentina (soybeans) and China (rice) key crops are identical under every candidate ordering.")
    else: log(f'  alignment: {p} not found (older cache); no ordering evidence printed.')
    ass = pd.Series(np.asarray(usa_crop)).value_counts(); bad = {c: (int(ass.get(c, 0)), int(n)) for c, n in raw_counts.items() if int(ass.get(c, 0)) != int(n)}
    res['counts_match'] = not bad; log('  assigned rows per crop equal the raw USA class counts: ' + ('YES (every row has exactly one state)' if not bad else f'NO {bad}'))
    return res

def diversity_assoc(df, metric='primary_f1'):
    # all evaluated combinations (all arms pooled, k>=2): Spearman correlation of F1 with the diversity percentile inside each (target,budget,k) cell.  Exploratory; the percentile is relative to the SAME pool and k.
    cl = combo_level(df, metric); rows = []
    for (t, B, k), g in cl.groupby(['target', 'budget', 'k']):
        g = g.dropna(subset=['div_pct', 'f1'])
        if k == 1 or len(g) < 5 or g['div_pct'].nunique() < 3: continue
        r, p = scipy_stats.spearmanr(g['div_pct'], g['f1']); rows.append(dict(target=t, budget=B, k=k, n=len(g), spearman=float(r), p=float(p)))
    return pd.DataFrame(rows)

def jackknife_states(cl_group):
    # delete-one-STATE jackknife over the states that appear in the evaluated combinations (secondary generalisation interval)
    combos = [(set(s.split('-')), f) for s, f in zip(cl_group['states'], cl_group['f1'])]; states = sorted(set().union(*[c for c, _ in combos])); thetas = []
    for s in states:
        rest = [f for c, f in combos if s not in c]
        if rest: thetas.append(float(np.mean(rest)))
    m = len(thetas)
    if m < 2: return float('nan'), float('nan'), float('nan')
    tb = float(np.mean(thetas)); se = math.sqrt((m - 1) / m * sum((t - tb) ** 2 for t in thetas)); h = float(scipy_stats.t.ppf(0.975, m - 1)) * se
    return se, float(np.mean([f for _, f in combos])) - h, float(np.mean([f for _, f in combos])) + h

def cell_stats(df, metric='primary_f1'):
    cl = combo_level(df, metric); rows = []; arm = df['arm'].iloc[0] if ('arm' in df.columns and len(df) and df['arm'].nunique() == 1) else 'random'; selected = arm in ('diverse', 'clustered')
    for (t, B, k), g in cl.groupby(['target', 'budget', 'k']):
        x = g['f1'].values; N = int(g['n_total'].iloc[0]) if pd.notna(g['n_total'].iloc[0]) else len(x); census = (k == 1)
        if selected and not census: N = float('inf')          # Phase-4 arms are chosen by their score, not drawn at random
        m, lo, hi, se = t_interval_fpc(x, N) if not census else (float(np.mean(x)), float('nan'), float('nan'), float('nan'))
        jse, jlo, jhi = jackknife_states(g)
        sub = df[(df['status'] == 'ok') & (df['target'] == t) & (df['budget'] == B) & (df['k'] == k)]
        per_seed_means = sub.groupby('seed')[metric].mean().values if len(sub) else np.array([])
        exhaustive = bool((not census) and not selected and N <= MAX_COMBOS and len(x) >= N)      # every combination was enumerated -> the t/FPC interval collapses to a point BY CONSTRUCTION
        rows.append(dict(target=t, budget=int(B), k=int(k), n_combos=len(x), N_total_combos=N, n_seeds_min=int(g['n_seeds'].min()), mean=m, ci_lo=lo, ci_hi=hi, se=se, census=census, exhaustive=exhaustive,
                         ci_note=('census: no CI' if census else 'EXTENSION, few combinations (overlapping states): descriptive only' if (PROFILE == 'extension' and len(x) < MAX_COMBOS) else 'selected arm: interval describes variation among the selected combinations only (not a random sample of all combinations)' if selected else 'exhaustive: CI is degenerate by construction' if exhaustive else ''),
                         census_values=';'.join(f'{s}:{f:.4f}' for s, f in zip(g['states'], g['f1'])) if census else '',
                         jackknife_se=jse, jackknife_lo=jlo, jackknife_hi=jhi,
                         seed_sd_within_combo=float(np.nanmean(g['seed_sd'])) if g['seed_sd'].notna().any() else float('nan'),
                         seed_sd_of_cell_means=float(np.std(per_seed_means, ddof=1)) if len(per_seed_means) > 1 else float('nan')))
    return pd.DataFrame(rows).sort_values(['target', 'budget', 'k']).reset_index(drop=True) if rows else pd.DataFrame()

def gain_kmin_kmax(cs):
    # gain from the smallest to the largest feasible k with a CI (Welch-style; a k=1 census contributes zero sampling variance)
    out = []
    for (t, B), g in cs.groupby(['target', 'budget']):
        g = g.sort_values('k')
        if len(g) < 2: continue
        lo, hi = g.iloc[0], g.iloc[-1]; d = float(hi['mean'] - lo['mean'])
        se2 = [s ** 2 for s in (lo['se'], hi['se']) if np.isfinite(s)]; ns = [n for s, n in ((lo['se'], lo['n_combos']), (hi['se'], hi['n_combos'])) if np.isfinite(s)]
        if se2 and sum(se2) > 0:
            se = math.sqrt(sum(se2)); df_w = (sum(se2) ** 2) / sum((s2 ** 2) / (n - 1) for s2, n in zip(se2, ns)); h = float(scipy_stats.t.ppf(0.975, df_w)) * se
        else: se, h = float('nan'), float('nan')
        out.append(dict(target=t, budget=int(B), k_min=int(lo['k']), k_max=int(hi['k']), gain=d, ci_lo=d - h if np.isfinite(h) else float('nan'), ci_hi=d + h if np.isfinite(h) else float('nan')))
    return pd.DataFrame(out)

def regression_report(df, metric='primary_f1'):
    # OLS of combination-level F1 on k, n_target_classes_train_ge30 (target-defined classes with >= 30 TRAINING samples) and budget, per target.
    # The target-crop COUNT is NOT used: it equals fixed share x budget exactly (perfectly aliased with budget).  A regressor with zero variance is dropped and said so.
    cl = combo_level(df, metric); res = []; alias_note = ('target-crop count (= fixed share x budget) is aliased with budget and was replaced by n_target_classes_train_ge30 | EXPLORATORY: combinations overlap in states and are not '
                  'independent replicates, so these standard errors / p-values are descriptive, not confirmatory')
    for t, g in cl.groupby('target'):
        cols = {'intercept': np.ones(len(g)), 'k': g['k'].values.astype(float), 'n_target_classes_train_ge30': g['n_cls_ge30'].values.astype(float), 'budget': g['budget'].values.astype(float)}
        note = alias_note; names = list(cols)
        for nm in [n for n in names if n != 'intercept' and np.nanstd(cols[n]) == 0]: names.remove(nm); note += f' | {nm} has zero variance in this target and was dropped'
        X = np.column_stack([cols[n] for n in names]); y = g['f1'].values.astype(float)
        if len(y) <= X.shape[1] or np.linalg.matrix_rank(X) < X.shape[1]:
            res.append(dict(target=t, term='(not estimable)', coef=np.nan, se=np.nan, t=np.nan, p=np.nan, n=len(y), r2=np.nan, note=note + ' | design matrix rank-deficient or too few rows')); continue
        beta, *_ = np.linalg.lstsq(X, y, rcond=None); resid = y - X @ beta; dfree = len(y) - X.shape[1]; s2 = float(resid @ resid) / dfree
        cov = s2 * np.linalg.inv(X.T @ X); se = np.sqrt(np.diag(cov)); tv = beta / se; pv = 2 * scipy_stats.t.sf(np.abs(tv), dfree)
        r2 = 1 - float(resid @ resid) / float(((y - y.mean()) ** 2).sum()) if ((y - y.mean()) ** 2).sum() > 0 else float('nan')
        for i_, nm in enumerate(names): res.append(dict(target=t, term=nm, coef=float(beta[i_]), se=float(se[i_]), t=float(tv[i_]), p=float(pv[i_]), n=len(y), r2=r2, note=note))
    return pd.DataFrame(res)

def seed_direction_warnings(df, metric='primary_f1'):
    # WARNING only, never a stop: do the seeds disagree on the SIGN of the k_min -> k_max change?
    w = []; d = df[df['status'] == 'ok'].copy()
    for c in [metric, 'budget', 'k', 'seed']: d[c] = pd.to_numeric(d[c], errors='coerce')
    for (t, B), g in d.groupby(['target', 'budget']):
        ks = sorted(g['k'].unique())
        if len(ks) < 2: continue
        signs = {}
        for s, gs in g.groupby('seed'):
            a, b = gs[gs['k'] == ks[0]][metric].mean(), gs[gs['k'] == ks[-1]][metric].mean()
            if np.isfinite(a) and np.isfinite(b): signs[int(s)] = float(np.sign(b - a))
        if len(set(signs.values())) > 1: w.append(f'WARNING seed-direction disagreement [{t} B{int(B)} k{int(ks[0])}->k{int(ks[-1])}]: per-seed sign {signs}')
    return w

def gate_report(df, seed, cells):
    # the per-seed gate report you approve before the next seed.  Both headline tracks are reported side by side; they are never pooled, averaged or ranked.
    d = df[pd.to_numeric(df['seed'], errors='coerce') == seed].copy(); L = []; planned = len(cells)
    L.append(f'================ GATE REPORT  seed {seed}  ({EXPERIMENT_VERSION}) ================')
    L.append(f'planned runs {planned} | recorded {len(d)} | status {d["status"].value_counts().to_dict()}')
    L.append('-- INTEGRITY (abort-level) --')
    for c_ in INTEGRITY_CHECKS:
        bad = int((d[c_].astype(str) != 'True').sum()) if len(d) else 0; L.append(f'   {c_:28s} failures: {bad}')
    rd = pd.to_numeric(d['restore_delta'], errors='coerce'); sd = pd.to_numeric(d['scorer_maxdiff'], errors='coerce')
    L.append(f'   max restore delta {rd.max():.2e} (tolerance {RESTORE_TOL:.0e}) | max scorer disagreement {sd.max():.2e} (tolerance {SCORER_TOL:.0e})')
    for c_ in ['best_epoch', 'epochs_run', 'steps_per_epoch', 'total_optimizer_steps', 'n_target_classes_train_ge30', 'n_absent_target_classes_train', 'val_f1', 'target_crop_n']: d[c_] = pd.to_numeric(d[c_], errors='coerce')
    # V33-01: rows written by the exception path (status error / integrity_fail raised before scoring) have BLANK metric columns.  They stay in the status and INTEGRITY counts above and are
    # listed here one by one; every training / track / coverage section below uses only rows that carry metrics, so a blank first row can no longer crash the report.
    d_all = d; has_m = pd.to_numeric(d_all['primary_f1'], errors='coerce').notna(); d = d_all[has_m].copy(); ok_ids = set(d_all.loc[d_all['status'] == 'ok', 'run_id'])
    L.append(f'-- RUNS WITHOUT METRICS (error / integrity_fail before scoring): {int((~has_m).sum())}  [excluded from the sections below, listed here] --')
    for _, r_ in d_all[~has_m].iterrows(): L.append(f'   {r_["run_id"]} [{r_["status"]}] {"SUPERSEDED by a later ok row (resume)" if r_["run_id"] in ok_ids else "UNRESOLVED"}: {str(r_["error"])[:160]}')
    L.append('-- TRAINING (<= %d total epochs, early stopping inside them) --' % MAX_EPOCHS); L.append(f'   stop reasons: {d["stop_reason"].value_counts().to_dict()}')
    L.append(f'   epochs run: min {d.epochs_run.min():.0f} median {d.epochs_run.median():.0f} max {d.epochs_run.max():.0f} | best_epoch >= {UNDERTRAINED_EPOCH} (possibly_undertrained): {int((d.best_epoch >= UNDERTRAINED_EPOCH).sum())}')
    for B, g in d.groupby(pd.to_numeric(d['budget'], errors='coerce')):
        L.append(f'   budget {int(B)}: steps/epoch {int(g.steps_per_epoch.iloc[0])} | total optimizer steps min {g.total_optimizer_steps.min():.0f} median {g.total_optimizer_steps.median():.0f} max {g.total_optimizer_steps.max():.0f}')
    L.append('-- SKILL outcomes (warnings, never aborts) --')
    _fc = collections.Counter(f.split(':')[0] for s_ in d['skill_flags'].fillna('').astype(str) for f in s_.split('|') if f)       # V33-02: explicit parse (no .str on a possibly empty/float Series)
    L.append(f'   flags: {dict(_fc.most_common())}')
    L.append('-- TRACKS, side by side (each its own definition; never pooled, averaged or ranked; binary is never compared with multiclass) --')
    for t in [x for x in TARGETS_ORDER if x in set(d['target'])]:
        g = d[d['target'] == t].copy()
        for c_ in ['primary_f1', 'baseline_all_other_primary', 'baseline_majority_primary', 'cov_cond_primary_f1', 'review_f1', 'baseline_all_other_review', 'baseline_majority_review', 'cov_cond_review_f1', 'paper_f1', 'papertext_f1', 'sevenclass_f1']: g[c_] = pd.to_numeric(g[c_], errors='coerce')
        L.append(f'   {t}:')
        L.append(f'      EXISTING  [{g.primary_view.iloc[0]}] main {100 * g.primary_f1.mean():.1f} | cov-cond {100 * g.cov_cond_primary_f1.mean():.1f} | all-other {100 * g.baseline_all_other_primary.mean():.1f} | majority {100 * g.baseline_majority_primary.mean():.1f} | '
                 f'runs not better than baseline {int((g.primary_f1 <= g[["baseline_all_other_primary", "baseline_majority_primary"]].max(axis=1)).sum())}/{len(g)}')
        L.append(f'      REVIEW    [target-map remap] main {100 * g.review_f1.mean():.1f} | cov-cond {100 * g.cov_cond_review_f1.mean():.1f} | all-other {100 * g.baseline_all_other_review.mean():.1f} | majority {100 * g.baseline_majority_review.mean():.1f} | '
                 f'runs not better than baseline {int((g.review_f1 <= g[["baseline_all_other_review", "baseline_majority_review"]].max(axis=1)).sum())}/{len(g)}')
        L.append(f'      also reported: existing release-remap {100 * g.paper_f1.mean():.1f} | paper_text_v3 {100 * g.papertext_f1.mean():.1f} | existing seven-class raw {100 * g.sevenclass_f1.mean():.1f} | val mF1 min {100 * g.val_f1.min():.1f}')
    L.append('-- CLASS COVERAGE by target and k (a k effect can be a coverage effect): share of runs with an absent target class | mean absent classes | mean classes with >=%d train samples --' % COVERAGE_MIN_TRAIN)
    for (t, k), g in d.groupby(['target', pd.to_numeric(d['k'], errors='coerce')]):
        L.append(f'   {t} k={int(k)}: runs {len(g)} | absent-class share {100 * (g.n_absent_target_classes_train > 0).mean():.0f}% | mean absent {g.n_absent_target_classes_train.mean():.2f} | mean classes>={COVERAGE_MIN_TRAIN}: {g.n_target_classes_train_ge30.mean():.2f} of {int(pd.to_numeric(g.n_target_classes).iloc[0])}')
    for metric, lab in TRACK_PAIR:
        cs = cell_stats(df[pd.to_numeric(df['seed'], errors='coerce') == seed], metric)
        if len(cs):
            L.append(f'-- {lab}: mean by (target, budget, k) for THIS seed  [E = exhaustive, CI degenerate by construction; C = k=1 census] --')
            for t, gt in cs.groupby('target'): L.append(f'   {t}: ' + ' | '.join(f'B{int(r.budget)}k{int(r.k)}:{100 * r["mean"]:.1f}{"C" if r.census else ("E" if r.exhaustive else "")}' for _, r in gt.iterrows()))
        L += [f'[{lab}] ' + w for w in seed_direction_warnings(df, metric)]
    return '\n'.join(L)

# V33-09: what actually ran -- software, device, data identity, settings, scoring maps.  One file per seed (never overwritten by another seed); the manifest and LOCKED are untouched.
def write_run_environment(out_dir, seed, info=None, geography=None):
    import platform
    env = dict(notebook_version=globals().get('NOTEBOOK_VERSION'), experiment=EXPERIMENT_VERSION, spec=SPEC_TAG, seed=int(seed), written=datetime.datetime.now().isoformat(timespec='seconds'),
               python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, scipy=scipy_stats.__name__ and __import__('scipy').__version__, cropnet_commit=globals().get('GIT_COMMIT'))
    try:
        import sklearn; env['sklearn'] = sklearn.__version__
        import torch; env.update(torch=torch.__version__, cuda_available=bool(torch.cuda.is_available()), device='cuda: ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
    except Exception as e: env['torch_or_sklearn'] = f'not importable here: {type(e).__name__}'
    env['settings'] = dict(profile=PROFILE, alloc_rule=ALLOC_RULE, class_weight_source=CLASS_WEIGHT_SOURCE, max_epochs=MAX_EPOCHS, patience=PATIENCE, min_epochs=MIN_EPOCHS, batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY,
                           val_fraction=VAL_FRACTION, budgets=BUDGETS, k_list=K_LIST, seeds=SEEDS, composition_modes=COMPOSITION_MODES_TO_RUN, state_selector=STATE_SELECTOR,
                           geography_max_outside_rows=GEOGRAPHY_MAX_OUTSIDE_ROWS, alignment_override=globals().get('ALIGNMENT_ORDERING_OVERRIDE'))
    env['label_map'] = {int(i): CROP_LABELS[i] for i in CROP_LABELS}; env['target_label_sets_table2'] = DEFINED_CLASSES; env['remap_rules'] = remap_difference_table().to_dict('records')
    info = info if info is not None else globals().get('INFO')
    if info: env['data_files'] = {c: dict(path=v.get('path'), size_bytes=(os.path.getsize(v['path']) if v.get('path') and os.path.exists(v['path']) else None), shape=list(v.get('shape', [])),
                                          raw_rows=v.get('raw'), kept_rows=v.get('kept'), duplicates_removed=v.get('dup'), raw_class_counts=v.get('raw_by_class')) for c, v in info.items()}
    env['geography_observed'] = geography if geography is not None else globals().get('GEOGRAPHY_OBSERVED')
    path = os.path.join(out_dir, f'run_environment_seed{int(seed)}.json')
    if os.path.exists(path): path = os.path.join(out_dir, f"run_environment_seed{int(seed)}__resume_{datetime.datetime.now().strftime('%Y%m%dT%H%M%S')}.json")     # a resumed seed keeps its ORIGINAL record (device, library versions) and gets a new one beside it
    atomic_write(path, lambda f: f.write(json.dumps(env, indent=1, default=str)), mode='w'); return env

# ------------------------------------------------------------------------------ RESULTS-FILE RECONCILIATION (e.g. the 5,004-row vs 5,590-row files of the review)
def reconcile_results(a, b, key='run_id', value_cols=('primary_f1', 'paper_f1', 'binary_f1', 'val_f1'), tol=1e-9, names=('A', 'B')):
    # generic, read-only comparison of two results tables: duplicates, rows only in one, rows in both with conflicting metrics, column differences
    rep = {}; na, nb = names
    rep['rows'] = {na: len(a), nb: len(b)}; rep['columns_only_in_' + na] = sorted(set(a.columns) - set(b.columns)); rep['columns_only_in_' + nb] = sorted(set(b.columns) - set(a.columns))
    for nm, d in ((na, a), (nb, b)): rep[f'duplicate_{key}_in_{nm}'] = int(d[key].duplicated().sum()) if key in d else 'key missing'
    if key not in a or key not in b: rep['error'] = f'"{key}" must exist in both files'; return rep
    ka, kb = set(a[key]), set(b[key]); rep[f'only_in_{na}'] = len(ka - kb); rep[f'only_in_{nb}'] = len(kb - ka); rep['in_both'] = len(ka & kb)
    m = a.drop_duplicates(key).merge(b.drop_duplicates(key), on=key, suffixes=('_A', '_B')); conflicts = {}
    for col in value_cols:
        if f'{col}_A' in m and f'{col}_B' in m:
            x, y = pd.to_numeric(m[f'{col}_A'], errors='coerce'), pd.to_numeric(m[f'{col}_B'], errors='coerce'); bad = ~(np.isclose(x, y, atol=tol, equal_nan=True))
            if bad.any(): conflicts[col] = m.loc[bad, key].tolist()[:20]
    rep['conflicting_rows_by_metric'] = {k: len(v) for k, v in conflicts.items()}; rep['conflict_examples'] = conflicts
    rep['verdict'] = 'IDENTICAL on shared keys' if not conflicts and not rep[f'only_in_{na}'] and not rep[f'only_in_{nb}'] else 'DIFFERENT -- inspect before merging; do not pool silently'
    return rep

## 4. Test gate, part A (unit tests; training is blocked if any fail)

In [ ]:
# ======================================================================================
# PRE-TRAINING TEST GATE, PART A (spec section 8).  No GPU, no repo, no real data.  Training cells refuse to run unless Part A AND Part B pass.
# ======================================================================================
from sklearn.linear_model import LogisticRegression
TEST_REPORT = []
def _t(name):
    def deco(fn):
        try:
            with profile_scope('main'): fn()
            TEST_REPORT.append((name, 'PASS', ''))
        except Exception as e: TEST_REPORT.append((name, 'FAIL', f'{type(e).__name__}: {e}'))
        return fn
    return deco
def _near(a, b, tol=1e-9): assert abs(a - b) <= tol, f'{a} != {b}'
def _ids(*pairs): return np.concatenate([np.full(n, CROP_ID[c]) for c, n in pairs])
def _labels_from_table2(t): return np.concatenate([np.full(n, CROP_ID[c]) for c, n in TABLE2[t].items() if n > 0])

@_t('Table 2 label sets: ARG/CHN/AUS/FRA exactly as the paper')
def _():
    assert DEFINED_CLASSES['ARG'] == ['corn', 'soybeans', 'rice', 'sugarcane', 'cotton', 'other']
    assert DEFINED_CLASSES['CHN'] == ['corn', 'soybeans', 'rice', 'other'] and DEFINED_CLASSES['AUS'] == ['sugarcane', 'cotton', 'other']
    assert DEFINED_CLASSES['FRA'] == ['corn', 'soybeans', 'rice', 'wheat', 'other']

@_t('binary_collapse on the REAL label sets: other = every non-positive class (true OR predicted); nothing dropped, nothing -1')
def _():
    for t, pos in BINARY_TARGETS.items():
        yt = _labels_from_table2(t); rng = np.random.default_rng(0); yp = rng.integers(0, 7, len(yt))
        bt, bp = binary_collapse(yt, yp, pos)
        assert len(bt) == len(yt) and len(bp) == len(yp) and set(np.unique(bt)) <= {0, 1} and set(np.unique(bp)) <= {0, 1} and bt.min() >= 0 and bp.min() >= 0
        assert (bt == (yt == CROP_ID[pos])).all() and (bp == (yp == CROP_ID[pos])).all()
    # spec example "true corn predicted soybeans":  CHN (positive=rice) -> other/other (a correct 'other');  ARG (positive=soybeans) -> other/POSITIVE (a false positive)
    a, b = binary_collapse([CROP_ID['corn']], [CROP_ID['soybeans']], 'rice'); assert (a[0], b[0]) == (0, 0)
    a, b = binary_collapse([CROP_ID['corn']], [CROP_ID['soybeans']], 'soybeans'); assert (a[0], b[0]) == (0, 1)
    a, b = binary_collapse([CROP_ID['wheat'], CROP_ID['cotton']], [CROP_ID['wheat'], CROP_ID['rice']], 'rice'); assert list(a) == [0, 0] and list(b) == [0, 1]
    # a true crop outside {positive, other} is allowed (never required to be a subset)
    sc = score_run('CHN', _ids(('corn', 5), ('rice', 5), ('other', 5)), _ids(('corn', 5), ('rice', 5), ('other', 5)), np.array([50, 50, 50, 0, 0, 0, 50])); _near(sc['primary_f1'], 1.0)

@_t("all-'other' baseline reproduces 0.4375 (CHN, rice view) and 0.4496 (ARG, soybeans view) from the Table 2 counts")
def _():
    for t, exp in (('CHN', 0.4375), ('ARG', 0.4496)):
        yt = _labels_from_table2(t); sc = score_run(t, yt, np.full(len(yt), CROP_ID['other']), np.array([100] * 7))
        assert round(sc['baseline_all_other_primary'], 4) == exp, (t, sc['baseline_all_other_primary'])
        assert round(sc['primary_f1'], 4) == exp and sc['primary_view'] == f'binary_{BINARY_TARGETS[t]}'
        assert len(yt) == TABLE2_TOTAL[t]

def _authors_remap_reference(y_pred, target):
    # INDEPENDENT re-implementation of the spec text (lookup tables, no np.isin) -- compared with remap_existing_release
    out = []
    for y in np.asarray(y_pred).tolist():
        n = CROP_LABELS[y]
        if target == 'AUS': out.append(CROP_ID[n] if n in ('sugarcane', 'cotton') else CROP_ID['other'])
        elif target == 'CHN': out.append(CROP_ID[n] if n in ('corn', 'soybeans', 'rice') else CROP_ID['other'])
        elif target == 'ARG': out.append(CROP_ID['other'] if n == 'wheat' else CROP_ID[n])
        else: out.append(CROP_ID[n])
    return np.array(out)

@_t('existing-remap scorer == macro f1_score(y_true, remapped, labels=np.unique(y_true)) on random data, all targets (reference remap)')
def _():
    rng = np.random.default_rng(1)
    for t in ['ARG', 'CHN', 'AUS', 'FRA']:
        for _rep in range(5):
            yt = rng.choice([CROP_ID[c] for c in DEFINED_CLASSES[t]], 3000, p=None); yp = rng.integers(0, 7, 3000)
            assert (remap_existing_release(yp, t) == _authors_remap_reference(yp, t)).all()
            sc = score_run(t, yt, yp, np.array([100] * 7)); yr = _authors_remap_reference(yp, t)
            _near(sc['paper_f1'], f1_score(yt, yr, average='macro', labels=np.unique(yt)), 1e-12); _near(sc['paper_oa'], accuracy_score(yt, yr), 1e-12)

@_t('FRA: NO remap (wrong crop stays an error); ARG: only wheat->other; CHN keeps corn/soy/rice; AUS keeps sugarcane/cotton')
def _():
    allc = np.arange(7)
    assert (remap_existing_release(allc, 'FRA') == allc).all()
    assert list(remap_existing_release(allc, 'ARG')) == [0, 1, 2, 6, 4, 5, 6]
    assert list(remap_existing_release(allc, 'CHN')) == [0, 1, 2, 6, 6, 6, 6] and list(remap_existing_release(allc, 'AUS')) == [6, 6, 6, 6, 4, 5, 6]

@_t('two independent scorers (sklearn vs from-scratch numpy) agree to 1e-12 on random data (paper rule + binary view, per-class P/R/F1)')
def _():
    rng = np.random.default_rng(2)
    for t in ['ARG', 'CHN', 'AUS', 'FRA']:
        for _rep in range(8):
            n = int(rng.integers(50, 4000)); yt = rng.choice([CROP_ID[c] for c in DEFINED_CLASSES[t]], n); yp = rng.integers(0, 7, n)
            tc = rng.integers(0, 400, 7); sc = score_run(t, yt, yp, tc); assert sc['scorer_maxdiff'] <= SCORER_TOL, sc['scorer_maxdiff']

@_t('stop rule: fewer than 2 true classes raises TooFewClasses')
def _():
    try: score_run('ARG', _ids(('soybeans', 20)), _ids(('soybeans', 20)), np.ones(7)); raise RuntimeError('no stop')
    except TooFewClasses: pass

@_t('classes absent from training stay IN the score at their F1 (flagged); coverage-conditional macro-F1 is separate and labelled')
def _():
    yt = _ids(('corn', 50), ('soybeans', 50), ('rice', 50), ('sugarcane', 50), ('cotton', 50), ('other', 50)); yp = yt.copy()
    tc = np.array([100, 100, 100, 0, 0, 0, 100])                    # cotton / sugarcane absent from training
    yp[yt == CROP_ID['sugarcane']] = CROP_ID['other']
    sc = score_run('ARG', yt, yp, tc)
    assert 'sugarcane' in sc['absent_train_classes'] and 'cotton' in sc['absent_train_classes']
    assert sc['paper_f1_cls'][CROP_ID['sugarcane']] == 0.0 and not np.isnan(sc['paper_f1_cls'][CROP_ID['cotton']])      # still scored
    assert sc['cov_cond_paper_classes'] == ['corn', 'soybeans', 'rice', 'other'] and sc['cov_cond_paper_f1'] != sc['paper_f1']

@_t('no 7-class macro over crops the target lacks: per-class F1 is NaN for classes absent from y_true; macro only over np.unique(y_true)')
def _():
    yt = _ids(('rice', 30), ('other', 70)); sc = score_run('CHN', yt, yt.copy(), np.array([100] * 7))
    assert sc['present'] == ['rice', 'other'] and np.isnan(sc['paper_f1_cls'][CROP_ID['corn']]) and sc['paper_f1'] == 1.0

@_t('integrity vs skill: a skill failure (low val F1 / no skill / collapsed) is a WARNING; only integrity checks can fail the run')
def _():
    yt = _ids(('soybeans', 50), ('other', 50)); allo = np.full(100, CROP_ID['other']); sc = score_run('ARG', yt, allo, np.array([100] * 7))
    info = dict(val_f1=0.3, restore_delta=0.0, best_epoch=UNDERTRAINED_EPOCH + 5); I, ok = evaluate_integrity(sc, allo, dict(ok=True), info); fl = skill_flags(sc, info)
    assert ok and all(I.values()); assert any(f.startswith('WARN_VAL') for f in fl) and 'WARN_NOT_BETTER_THAN_BASELINE' in fl and 'WARN_COLLAPSED_PREDICTIONS' in fl and 'possibly_undertrained' in fl
    I2, ok2 = evaluate_integrity(sc, allo, dict(ok=False), info); assert not ok2 and not I2['I3_split_exact']
    I3, ok3 = evaluate_integrity(sc, allo, dict(ok=True), dict(val_f1=0.9, restore_delta=1e-3)); assert not ok3 and not I3['I4_restore_matches_best']
    I4, ok4 = evaluate_integrity(sc, allo, dict(ok=True), dict(val_f1=float('nan'), restore_delta=0.0)); assert not ok4 and not I4['I5_finite']
    bad = np.full(100, 9); 
    try: score_run('ARG', yt, bad, np.ones(7)); raise RuntimeError('no stop')
    except AssertionError: pass

@_t('Table 3 reference: only the three published values, no too-good flag, no PAPER_REF_F1')
def _():
    assert TABLE3_REF_MF1 == {'ARG': 75.63, 'AUS': 88.38, 'FRA': 90.00} and 'PAPER_REF_F1' not in globals()
    assert not any('TOO_GOOD' in f for f in skill_flags(score_run('ARG', _ids(('soybeans', 50), ('other', 50)), _ids(('soybeans', 50), ('other', 50)), np.ones(7) * 100), dict(val_f1=1.0, best_epoch=3)))

@_t('early stopping: strictly-greater improvement, patience 15, min 20 epochs, approved 200-epoch cap (v3.7; was 100), reasons logged')
def _():
    es = EarlyStopper(); stops = None
    for i in range(100):                                            # never improves after epoch 1
        _imp, stop = es.step(0.5)
        if stop: stops = es.epoch; break
    assert stops == 20 and es.reason == 'early_stop_patience' and es.best_epoch == 1          # min epochs 20 respected, bad count already >= 15
    es = EarlyStopper(); seq = [0.001 * i for i in range(1, MAX_EPOCHS + 1)]                # improves every epoch -> runs to the cap
    for s in seq:
        _imp, stop = es.step(s)
        if stop: break
    assert MAX_EPOCHS == 200 and es.epoch == MAX_EPOCHS and es.reason == 'max_epochs' and es.best_epoch == MAX_EPOCHS
    es = EarlyStopper()
    for s in [0.5] + [0.5] * 30:                                    # equal score is NOT an improvement (strictly greater)
        _imp, stop = es.step(s)
        if stop: break
    assert es.best_epoch == 1 and es.epoch == 20
    es = EarlyStopper()                                              # best at epoch 10 -> stops at 25
    for i in range(60):
        _imp, stop = es.step(0.9 if i == 9 else 0.5)
        if stop: break
    assert es.epoch == 25 and es.best_epoch == 10
    try: EarlyStopper(max_epochs=MAX_EPOCHS + 1); raise RuntimeError('no stop')
    except AssertionError: pass

@_t('class weights: absent crops get weight 0 (no division by zero) and match sklearn on the present classes')
def _():
    from sklearn.utils.class_weight import compute_class_weight
    y = np.array([1] * 30 + [6] * 10 + [2] * 60); w = balanced_class_weights(y, 7)
    assert w[0] == w[3] == w[4] == w[5] == 0 and np.isfinite(w).all() and np.allclose(w[[1, 2, 6]], compute_class_weight('balanced', classes=np.array([1, 2, 6]), y=y))

def _synthetic_usa(n_states=14, rows=1100, seed=0, floor=0.04):
    rng = np.random.default_rng(seed); st, cr = [], []
    for i in range(n_states):
        p = rng.dirichlet(np.ones(7) * 1.5); p = floor + p; p /= p.sum()
        for _ in range(rows): st.append('ABCDEFGHIJKLMNOPQRSTUVWXYZ'[i]); cr.append(CROP_LABELS[int(rng.choice(7, p=p))])
    return np.array(st), np.array(cr)

@_t('design: key counts exact (ARG soy 20% / CHN rice 20% / USA 15% x3), identical at every k; pool = states that can supply the whole budget alone')
def _():
    for B in BUDGETS:
        assert key_counts('ARG', B) == {'soybeans': int(0.2 * B)} and key_counts('CHN', B) == {'rice': int(0.2 * B)}
        assert key_counts('USA', B) == {'corn': int(0.15 * B), 'soybeans': int(0.15 * B), 'wheat': int(0.15 * B)} and rest_count('USA', B) == B - 3 * int(0.15 * B)
    cap = pd.DataFrame({c: [900, 900, 900] for c in CROP_ORDER}, index=['X', 'Y', 'Z']); cap['total'] = cap[CROP_ORDER].sum(axis=1)
    cap.loc['X', 'soybeans'] = 100; cap.loc['Y', 'total'] = 1400                                    # X: too few soybeans for B=1000 (needs 200) ; Y: below the 1500-row rule
    assert eligible_states('ARG', 1000, cap) == ['Z'] and 'X' not in eligible_states('ARG', 1000, cap)

@_t('plan rules: k=1 census of the pool; k>=2 <=10 deterministic combos; <5 combos or <3 k-points -> infeasible; CHN k<=2 (Level 2: descriptive, exempt from the combination / k-point minimums); never shrinks a budget')
def _():
    st, cr = _synthetic_usa(14, 1800); cap = make_cap(st, cr); cells, grid = build_plan(cap)
    g = grid[(grid.target == 'ARG') & (grid.budget == 1000)].set_index('k'); P = int(g.pool_size.iloc[0]); pool = eligible_states('ARG', 1000, cap); assert P == len(pool)
    assert g.loc[1, 'n_combos'] == P and all(g.loc[k, 'n_combos'] <= MAX_COMBOS for k in g.index if k >= 2)
    assert not grid[(grid.target == 'CHN') & (grid.k >= 4)].feasible.any()
    c1 = make_combos(list('ABCDEFGHIJKL'), 4, 'ARG', 1000); assert len(c1) == len(set(c1)) == MAX_COMBOS and c1 == make_combos(list('ABCDEFGHIJKL'), 4, 'ARG', 1000)
    assert make_combos(list('ABCDEFGHIJKL'), 4, 'ARG', 1000) != make_combos(list('ABCDEFGHIJKL'), 4, 'ARG', 2000)
    small = st != 'Z'; s2, c2 = st[:], cr[:]                                                         # force a tiny pool -> everything infeasible
    cap_small = make_cap(*_synthetic_usa(4, 1800)); _c, gr = build_plan(cap_small)
    assert not gr[(gr.target == 'USA') & (gr.k == 8)].feasible.any()
    only = gr[(gr.target == 'ARG') & (gr.budget == 1000)]; assert (only.feasible.sum() == 0) or (only.feasible.sum() >= MIN_KPOINTS)
    assert all(c['pool_size'] >= 1 for c in cells) and set(c['budget'] for c in cells) <= set(BUDGETS)
    aus = [c for c in cells if c['target'] == 'AUS']; assert all(c['pending'] for c in aus) == (not AUS_SPEC_CONFIRMED)

@_t('sampling: exact budget, SAME key-crop count at k=1,2,4,8, 10% stratified validation, unique ids, no overlap, deterministic, seed-dependent')
def _():
    st, cr = _synthetic_usa(14, 1800); cap = make_cap(st, cr); pools = build_pools(st, cr)
    for t in ['ARG', 'CHN', 'USA']:
        for B in [1000, 4000]:
            pool = eligible_states(t, B, cap)
            if len(pool) < 8: continue
            ref = None
            for k in [1, 2, 4, 8]:
                states = tuple(pool[:k]); sp = draw_split(t, B, states, 0, pools, cr); v = verify_split(sp, t, B, states, st, cr); assert v['ok'], v['msgs']
                allp = np.concatenate([sp['train_pos'], sp['val_pos']]); kc = {c: int((cr[allp] == c).sum()) for c in key_counts(t, B)}
                assert len(sp['val_pos']) == B // 10 and len(allp) == B and len(np.unique(allp)) == B
                ref = ref or kc; assert kc == ref == key_counts(t, B), 'key-crop count changed with k'
    t, B = 'ARG', 1000; pool = eligible_states(t, B, cap); a = draw_split(t, B, tuple(pool[:2]), 0, pools, cr); b = draw_split(t, B, tuple(pool[:2]), 0, pools, cr); c = draw_split(t, B, tuple(pool[:2]), 1, pools, cr)
    assert (a['train_pos'] == b['train_pos']).all() and not np.array_equal(np.sort(a['train_pos']), np.sort(c['train_pos']))
    bad = dict(a); bad['val_pos'] = a['train_pos'][:100]; assert not verify_split(bad, t, B, tuple(pool[:2]), st, cr)['ok']            # overlap is caught
    bad2 = dict(a); bad2['train_pos'] = np.concatenate([a['train_pos'][:-1], a['train_pos'][:1]]); assert not verify_split(bad2, t, B, tuple(pool[:2]), st, cr)['ok']   # duplicate id is caught

@_t('statistics: FPC interval collapses when all combinations are evaluated; k=1 census has no CI; jackknife defined; regression reports the target-crop-count/budget aliasing')
def _():
    x = np.array([0.5, 0.6, 0.7, 0.55, 0.65]); m, lo, hi, se = t_interval_fpc(x, 5); assert se == 0 and lo == hi == m
    m2, lo2, hi2, se2 = t_interval_fpc(x, 100); assert lo2 < m2 < hi2 and se2 > 0
    rows = []
    for t, kc in (('ARG', 200), ('ARG', 400)):
        for k in (1, 2, 4):
            for ci in range(6):
                for s in range(2): rows.append(dict(run_id=f'{t}{kc}{k}{ci}{s}', status='ok', target=t, budget=kc * 5, k=k, states='-'.join(f'S{(ci + j) % 8}' for j in range(k)), seed=s, primary_f1=0.5 + 0.01 * k + 0.0001 * ci + 0.01 * s,
                                                  pool_size=8, n_total_combos=math.comb(8, k), key_n=kc, target_crop_n=kc, n_target_classes_train_ge30=3 + (1 if k > 1 else 0) + (1 if k > 2 else 0) + (ci % 2)))
    df = pd.DataFrame(rows); cs = cell_stats(df); assert (cs[cs.k == 1].ci_lo.isna()).all() and cs[cs.k == 1].census.all() and cs.jackknife_se.notna().all()
    reg = regression_report(df); assert 'aliased with budget' in ' '.join(reg['note'].astype(str)) and 'target_crop_count' not in set(reg.term) and 'n_target_classes_train_ge30' in set(reg.term) and {'k', 'budget'} <= set(reg.term)
    gn = gain_kmin_kmax(cs); assert len(gn) == 2 and (gn.gain > 0).all()

@_t('atomic write: no temp files left behind; content replaced as a whole')
def _():
    with tempfile.TemporaryDirectory() as td:
        p = os.path.join(td, 'x.csv'); atomic_write(p, lambda f: f.write('a,b\n1,2\n'), mode='w'); atomic_write(p, lambda f: f.write('a,b\n3,4\n'), mode='w')
        assert open(p).read() == 'a,b\n3,4\n' and os.listdir(td) == ['x.csv']

def selftest_pipeline(return_report=False, return_df=False):
    with profile_scope('main'): return _selftest_pipeline_impl(return_report, return_df)

def _selftest_pipeline_impl(return_report=False, return_df=False):
    # END-TO-END dry run with a stand-in model (logistic regression): plan -> draw -> verify -> fit -> predict -> score (2 scorers) -> gates -> npz -> CSV (last) -> resume -> stats -> gate report.
    names = ['BUDGETS', 'SEEDS', 'K_LIST', 'MAX_COMBOS', 'MIN_STATE_ROWS', 'AUS_SPEC_CONFIRMED', 'AUS_STATES', 'AUS_KEY_SHARES_IF_CONFIRMED', 'FRA_K8_MIN_POOL', 'AUG_BUDGETS', 'SELECTION_ARMS', 'KC_TARGETS']
    g = globals(); saved_cfg = {n: g[n] for n in names}
    keys = ['KC_EMBED', 'DIVERSITY_DIST', 'FEATURES', 'SRC_UNIT', 'SRC_CROP', 'SRC_CROP_ID', 'SRC_NPZ_ROW', 'POOLS', 'EVAL_X', 'EVAL_Y', 'TARGET_IDS', 'FIT_FN', 'PREDICT_FN']; saved = {n: g.get(n, '<unset>') for n in keys}
    try:
        g.update(BUDGETS=[200, 400], SEEDS=[0, 1], K_LIST=[1, 2, 4, 8], MIN_STATE_ROWS=300, AUS_SPEC_CONFIRMED=True, AUS_STATES=['A', 'B'], AUS_KEY_SHARES_IF_CONFIRMED={'sugarcane': 0.2}, FRA_K8_MIN_POOL=10, AUG_BUDGETS=[200, 400], SELECTION_ARMS=['random', 'diverse', 'clustered', 'common', 'kcenter', 'aug_on'], KC_TARGETS=['ARG', 'CHN', 'USA'])      # the dormant Level 1 arms are switched ON here only so that their code stays tested
        st, cr = _synthetic_usa(14, 700); yid = np.array([CROP_ID[c] for c in cr]); rng = np.random.default_rng(0); mu = rng.normal(0, 1, (7, 730)).astype(np.float32)
        def feats(y): return (mu[y] + rng.normal(0, 1.5, (len(y), 730))).astype(np.float32).reshape(len(y), 1, 10, 73)
        g.update(FEATURES={SOURCE: feats(yid)}, SRC_UNIT=st, SRC_CROP=cr, SRC_CROP_ID=yid, SRC_NPZ_ROW=np.arange(len(st)) * 2, POOLS=build_pools(st, cr))
        g['DIVERSITY_DIST'] = compute_diversity_distances(g['FEATURES'][SOURCE], st, cr)
        EX, EY, TI = {}, {}, {}
        for t in TABLE2:
            if t == SOURCE: continue
            y = np.array([CROP_ID[c] for c in DEFINED_CLASSES[t] for _ in range(60)]); EY[t] = y; EX[t] = feats(y); TI[t] = np.arange(len(y))
        g.update(EVAL_X=EX, EVAL_Y=EY, TARGET_IDS=TI)
        def fake_fit(Xtr, ytr, Xva, yva, seed, ep=None, aug='off'):
            if aug in ('on', 'copy'): Xtr = np.concatenate([Xtr] * (1 + AUG_REPEAT)); ytr = np.concatenate([ytr] * (1 + AUG_REPEAT))             # stand-in for the 3x augmented training set
            m = LogisticRegression(max_iter=100, C=0.05).fit(Xtr.reshape(len(Xtr), -1), ytr); vf = float(f1_score(yva, m.predict(Xva.reshape(len(Xva), -1)), average='macro'))
            return dict(model=m, info=dict(val_f1=vf, best_epoch=3, epochs_run=23, stop_reason='early_stop_patience', restore_delta=0.0, history=[vf]))
        g.update(FIT_FN=fake_fit, PREDICT_FN=lambda m, Xe: m.predict(Xe.reshape(len(Xe), -1)))
        cap = make_cap(st, cr); cells, grid = build_plan(cap); assert len(cells) > 0 and {c['target'] for c in cells} >= {'ARG', 'CHN', 'USA'}
        g['KC_EMBED'] = build_kc_embedding(g['FEATURES'][SOURCE], dims=8, fit_n=3000)
        _variants = [c for c in cells if c.get('sel') == 'kcenter'][:4] + [c for c in cells if c.get('aug') == 'on'][:4]; assert len(_variants) == 8, 'plan must contain kcenter and aug_on cells'
        cells = [c for c in cells if (c['k'] <= 2 or c['target'] == 'USA') and c.get('sel', 'random') == 'random' and c.get('aug', 'off') == 'off'][:40] + _variants
        with tempfile.TemporaryDirectory() as td:
            csvp = os.path.join(td, 'results.csv'); buf = io.StringIO()
            with contextlib.redirect_stdout(buf): r1 = run_seed(0, cells, csvp, td); r1b = run_seed(1, cells, csvp, td)
            df = load_results(csvp); assert (df['sample_sel'] == 'kcenter').sum() == 8 and (df['aug'] == 'on').sum() == 8 and df['run_id'].is_unique and df['main_f1'].notna().all()
            assert (pd.to_numeric(df.loc[df['aug'] == 'on', 'train_n_effective']) == (1 + AUG_REPEAT) * pd.to_numeric(df.loc[df['aug'] == 'on', 'n_train'])).all() and (pd.to_numeric(df.loc[df['aug'] == 'on', 'steps_per_epoch']) == np.ceil((1 + AUG_REPEAT) * pd.to_numeric(df.loc[df['aug'] == 'on', 'n_train']) / BATCH_SIZE)).all()
            assert r1 == 'complete' and len(df) == 2 * len(cells) and (df['status'] == 'ok').all(), df['status'].value_counts().to_dict()
            assert len(os.listdir(os.path.join(td, 'runs'))) == 2 * len(cells) and not [f for f in os.listdir(td) if f.startswith('.tmp_')]
            z = np.load(os.path.join(td, 'runs', hashlib.md5(df.run_id.iloc[0].encode()).hexdigest()[:16] + '.npz'))
            assert {'y_true', 'y_pred', 'target_sample_ids', 'src_train_ids', 'cm7', 'cm_existing_release', 'cm_review_rule', 'paper_f1_per_class', 'train_counts'} <= set(z.files) and len(z['src_train_ids']) + len(z['src_val_ids']) == df.budget.iloc[0]
            assert (pd.to_numeric(df.scorer_maxdiff) <= SCORER_TOL).all()
            with contextlib.redirect_stdout(buf): run_seed(0, cells, csvp, td)
            assert len(load_results(csvp)) == 2 * len(cells), 'resume re-ran finished runs'
            kc = df[df.target == 'ARG'].groupby('budget')['key_n'].nunique().max(); assert kc == 1
            cs = cell_stats(df); assert len(cs) > 0; txt = gate_report(df, 0, cells); assert 'GATE REPORT' in txt and 'INTEGRITY' in txt
            for c_ in ['n_target_classes_train_ge30', 'n_absent_target_classes_train', 'steps_per_epoch', 'total_optimizer_steps']: assert df[c_].notna().all(), c_
            assert (pd.to_numeric(df.total_optimizer_steps) == pd.to_numeric(df.steps_per_epoch) * pd.to_numeric(df.epochs_run)).all()
            _tr = df.iloc[0]; _def = DEFINED_CLASSES[_tr.target]; assert int(_tr.n_target_classes_train_ge30) == sum(int(_tr[f'train_{c_}']) >= COVERAGE_MIN_TRAIN for c_ in _def) and int(_tr.n_absent_target_classes_train) == sum(int(_tr[f'train_{c_}']) == 0 for c_ in _def)
            if return_df: return df.copy(), list(cells)
            if return_report: return txt
            # skill failure never aborts:
            g['FIT_FN'] = lambda *a, **k: dict(model=None, info=dict(val_f1=0.1, best_epoch=47, epochs_run=50, stop_reason='max_epochs', restore_delta=0.0, history=[0.1]))
            g['PREDICT_FN'] = lambda m, Xe: np.full(len(Xe), CROP_ID['other'])
            csv2 = os.path.join(td, 'skill.csv')
            with contextlib.redirect_stdout(buf): run_seed(0, cells[:8], csv2, td)
            d2 = load_results(csv2); assert len(d2) == 8 and (d2.status == 'ok').all() and d2.skill_flags.str.contains('WARN_VAL').all()
            # 5 consecutive integrity failures abort:
            g['FIT_FN'] = lambda *a, **k: dict(model=None, info=dict(val_f1=0.9, best_epoch=3, epochs_run=23, stop_reason='x', restore_delta=0.5, history=[0.9]))
            csv3 = os.path.join(td, 'integ.csv')
            try:
                with contextlib.redirect_stdout(buf): run_seed(0, cells[:12], csv3, td)
                raise RuntimeError('did not abort')
            except RuntimeError as e: assert 'consecutive integrity' in str(e), e
            assert len(load_results(csv3)) == MAX_CONSEC_INTEGRITY
        return True
    finally:
        g.update(saved_cfg)
        for n, v in saved.items():
            if isinstance(v, str) and v == '<unset>': g.pop(n, None)
            else: g[n] = v

@_t('END-TO-END dry run with a stand-in model: plan, run, 2 scorers, gates, npz, CSV-last, resume, stats, gate report, skill warnings never abort, 5 integrity failures abort')
def _():
    assert selftest_pipeline()


# =================================================================== NEW TESTS for the dual-track update
@_t('three remap rules are separately named; existing rule PRESERVED unchanged (ARG wheat->other, FRA none); paper-text and review rules differ exactly where expected')
def _():
    allc = np.arange(7)
    assert list(REMAPS) == REMAP_RULES and 'remap_paper_rule' not in globals()
    ex = {t: list(remap_existing_release(allc, t)) for t in TABLE2_TOTAL if t != 'USA'}; pt = {t: list(remap_paper_text_v3(allc, t)) for t in ex}; rv = {t: list(remap_review_targetmap(allc, t)) for t in ex}
    assert ex['ARG'] == [0, 1, 2, 6, 4, 5, 6] and ex['FRA'] == list(allc) and ex['CHN'] == [0, 1, 2, 6, 6, 6, 6] and ex['AUS'] == [6, 6, 6, 6, 4, 5, 6]     # existing, unchanged
    assert pt['ARG'] == list(allc) and pt['FRA'] == list(allc) and pt['CHN'] == ex['CHN'] and pt['AUS'] == ex['AUS']                                      # paper text sec.4.4
    assert rv['ARG'] == ex['ARG'] and rv['CHN'] == ex['CHN'] and rv['AUS'] == ex['AUS'] and rv['FRA'] == [0, 1, 2, 3, 6, 6, 6]                            # target-map rule: only FRA differs from existing
    d = remap_difference_table().set_index('target'); assert not d.loc['FRA', 'existing_eq_review'] and d.loc['ARG', 'existing_eq_review'] and not d.loc['ARG', 'existing_eq_paper_text']

@_t('review track: FRA predicted sugarcane/cotton counts as Other (review) but stays an error (existing); both tracks reported, never merged')
def _():
    yt = _ids(('corn', 40), ('other', 40), ('wheat', 40)); yp = yt.copy(); yp[yt == CROP_ID['other']] = CROP_ID['sugarcane']                 # model calls every true Other 'sugarcane'
    sc = score_run('FRA', yt, yp, np.array([100] * 7))
    assert sc['paper_f1_cls'][CROP_ID['other']] == 0.0 and sc['review_f1_cls'][CROP_ID['other']] == 1.0 and sc['review_f1'] > sc['paper_f1']
    _near(sc['papertext_f1'], sc['paper_f1']); assert sc['scorer_maxdiff'] <= SCORER_TOL
    assert sc['sevenclass_f1'] == sc['paper_f1'] and sc['primary_f1'] == sc['paper_f1']                                                              # FRA: existing primary = existing paper rule

@_t('seven-class raw track: no remap, macro over classes present in truth only; differs from the paper rule when remapping changes predictions')
def _():
    yt = _ids(('corn', 30), ('rice', 30), ('other', 30)); yp = yt.copy(); yp[yt == CROP_ID['other']] = CROP_ID['cotton']          # CHN: cotton -> other under every remap, raw stays wrong
    sc = score_run('CHN', yt, yp, np.array([100] * 7)); assert sc['sevenclass_f1'] < 1.0 and sc['paper_f1'] == 1.0 and sc['review_f1'] == 1.0 and sc['papertext_f1'] == 1.0
    assert np.isnan(sc['paper_f1_cls'][CROP_ID['sugarcane']]) and sc['present'] == ['corn', 'rice', 'other']

@_t('binary truth with < 2 classes: STOP/flag (ARG truth without soybeans; CHN truth without rice), even though the 7-class truth has 2 classes')
def _():
    for t, pos in (('ARG', 'soybeans'), ('CHN', 'rice')):
        yt = _ids(('corn', 20), ('other', 20)); yp = yt.copy()
        try: score_run(t, yt, yp, np.array([100] * 7)); raise RuntimeError('no stop')
        except TooFewClasses as e: assert 'binary' in str(e)
    yt = _ids(('soybeans', 20)); 
    try: score_run('ARG', yt, yt, np.array([100] * 7)); raise RuntimeError('no stop')
    except TooFewClasses: pass

@_t('class weights: default = authors-equivalent (whole drawn sample, train+val); "train" option = training labels only; zero-count crops weight 0 in both')
def _():
    g = globals(); keep = g['CLASS_WEIGHT_SOURCE']
    try:
        assert keep == 'train_val'
        ytr, yva = np.array([1] * 30 + [2] * 10), np.array([3] * 5 + [1] * 5)
        w = training_class_weights(ytr, yva); assert w[3] > 0 and w[0] == 0 and np.isfinite(w).all()
        from sklearn.utils.class_weight import compute_class_weight
        ref = compute_class_weight('balanced', classes=np.unique(np.concatenate([ytr, yva])), y=np.concatenate([ytr, yva])); assert np.allclose(w[[1, 2, 3]], ref)
        g['CLASS_WEIGHT_SOURCE'] = 'train'; w2 = training_class_weights(ytr, yva); assert w2[3] == 0 and w2[0] == 0 and w2[2] > w2[1] > 0
        assert training_class_weights(np.array([1, 1, 2]), np.array([6]))[6] == 0
    finally: g['CLASS_WEIGHT_SOURCE'] = keep

@_t('eligibility table: EVERY state listed with need/have per fixed crop and an explicit reason for each exclusion; pool == eligible rows; combos verified for capacity')
def _():
    st, cr = _synthetic_usa(10, 1800); cap = make_cap(st, cr); cap.loc['A', 'soybeans'] = 5; cap.loc['B', 'total'] = 1000                      # A lacks soybeans, B below the row rule
    e = state_eligibility('ARG', 1000, cap); assert list(e.state) == list(cap.index) and len(e) == len(cap)
    assert not e.set_index('state').loc['A', 'eligible'] and 'soybeans' in e.set_index('state').loc['A', 'reason'] and '<1500' in e.set_index('state').loc['B', 'reason']
    assert (e.loc[~e.eligible, 'reason'] != '').all() and sorted(e.loc[e.eligible, 'state']) == eligible_states('ARG', 1000, cap)
    pool = eligible_states('ARG', 1000, cap)
    for k in (1, 2, 4): assert all(combo_capacity_ok(c, 'ARG', 1000, cap) for c in make_combos(pool, k, 'ARG', 1000))
    assert not combo_capacity_ok(('A', pool[0]), 'ARG', 1000, cap)                                                                        # a combination containing a short state is rejected

@_t('fixed_full composition (opt-in): proposals sum to B exactly; same counts at k=1,2,4,8; no approval -> pending/STOP; baseline untouched')
def _():
    g = globals(); keep = {n: g[n] for n in ('FIXED_COMPOSITION_CHOICE', 'COMPOSITION_MODES_TO_RUN')}; ap = dict(APPROVED)
    try:
        for ch in FIXED_COMPOSITION_PROPOSALS:
            for t in ('ARG', 'CHN', 'USA'):
                sh = FIXED_COMPOSITION_PROPOSALS[ch](t); _near(sum(sh.values()), 1.0, 1e-9); assert set(sh) <= set(DEFINED_CLASSES[t])        # only classes the target MAP defines
                for B in BUDGETS: assert sum(shares_to_counts(sh, B).values()) == B
        try: key_counts('ARG', 1000, FULL_MODE); raise RuntimeError('no stop')
        except AssertionError as e: assert 'not been chosen' in str(e)
        g['FIXED_COMPOSITION_CHOICE'] = 'A_key_plus_equal_rest'; g['COMPOSITION_MODES_TO_RUN'] = [BASE_MODE, FULL_MODE]
        assert key_counts('ARG', 1000, FULL_MODE)['soybeans'] == 200 and key_counts('ARG', 1000, BASE_MODE) == {'soybeans': 200}                  # baseline unchanged
        st, cr = _synthetic_usa(14, 1800, floor=0.4); cap = make_cap(st, cr); pools = build_pools(st, cr)
        cells, grid = build_plan(cap); full = [c for c in cells if c['composition'] == FULL_MODE]; assert full and all(c['pending'] for c in full)       # not approved -> excluded
        APPROVED['fixed_composition'] = True; cells, grid = build_plan(cap); full = [c for c in cells if c['composition'] == FULL_MODE]; assert full and not any(c['pending'] for c in full if c['target'] != 'AUS')
        pool = eligible_states('ARG', 1000, cap, FULL_MODE); ref = None
        for k in (1, 2, 4, 8):
            if len(pool) < k: continue
            states = tuple(pool[:k]); sp = draw_split('ARG', 1000, states, 0, pools, cr, FULL_MODE); v = verify_split(sp, 'ARG', 1000, states, st, cr, FULL_MODE); assert v['ok'], v['msgs']
            allp = np.concatenate([sp['train_pos'], sp['val_pos']]); cnt = {c: int((cr[allp] == c).sum()) for c in CROP_ORDER}; ref = ref or cnt; assert cnt == ref and sum(cnt.values()) == 1000
            assert all(cnt[c] == 0 for c in CROP_ORDER if c not in DEFINED_CLASSES['ARG'])                                                       # wheat never drawn: absent from training (flagged downstream)
        r_base = make_run_id(dict(target='ARG', budget=1000, k=1, states=('A',)), 0); r_full = make_run_id(dict(target='ARG', budget=1000, k=1, states=('A',), composition=FULL_MODE), 0)
        assert r_base == 'ARG|B1000|k1|A|aug-off|s0' and r_full != r_base and 'comp-fixed_full' in r_full
    finally:
        g.update(keep); APPROVED.clear(); APPROVED.update(ap)

@_t('baseline sampling unchanged by the update: identical draw for a fixed (target, budget, states, seed) and key counts')
def _():
    st, cr = _synthetic_usa(10, 1800); cap = make_cap(st, cr); pools = build_pools(st, cr); el = eligible_states('USA', 1000, cap); assert len(el) >= 2
    a = draw_split('USA', 1000, tuple(el[:2]), 3, pools, cr); b = draw_split('USA', 1000, tuple(el[:2]), 3, pools, cr, BASE_MODE)
    assert (a['train_pos'] == b['train_pos']).all() and a['key_counts'] == {'corn': 150, 'soybeans': 150, 'wheat': 150}

@_t('results reconciliation tool: reports duplicates, rows only in one file, conflicting metrics; refuses to call different files identical')
def _():
    A = pd.DataFrame({'run_id': list('abcd') + ['a'], 'primary_f1': [.5, .6, .7, .8, .5]}); B = pd.DataFrame({'run_id': list('bcdef'), 'primary_f1': [.6, .7, .81, .1, .2]})
    r = reconcile_results(A, B, names=('A', 'B')); assert r['duplicate_run_id_in_A'] == 1 and r['only_in_A'] == 1 and r['only_in_B'] == 2 and r['in_both'] == 3
    assert r['conflicting_rows_by_metric'] == {'primary_f1': 1} and r['verdict'].startswith('DIFFERENT'); assert reconcile_results(A.drop_duplicates('run_id'), A.drop_duplicates('run_id'))['verdict'].startswith('IDENTICAL')

@_t('feature source: S2 10-band guard text present; AlphaEarth (64-d) shape would fail the (N,1,10,T) check')
def _():
    assert 'NOT AlphaEarth' in FEATURE_SOURCE and 'median_S2_' in FEATURE_SOURCE


# =================================================================== NEW TESTS (authors alignment, coverage, safeguards)
@_t('authors-script recipe constants: Adam lr 1e-4 / weight decay 0 / batch 256 / no scheduler; TOTAL 200 epochs (v3.7; supervisor review: 38% of 8-state runs hit the old 100 cap) with early stopping (patience 15, min 20, strictly greater); deviations listed')
def _():
    assert (LR, WEIGHT_DECAY, BATCH_SIZE) == (AUTHORS_SCRIPT['lr'], 0.0, AUTHORS_SCRIPT['batch_size']) and AUTHORS_SCRIPT['scheduler'] is None
    assert (MAX_EPOCHS, PATIENCE, MIN_EPOCHS, MIN_DELTA) == (200, 15, 20, 0.0) and NUM_CLASSES == 7 and AUG == 'off'
    assert {'epochs', 'val_ratio', 'seeds', 'train_aug', 'class_weights'} <= set(DEVIATIONS_FROM_AUTHORS)

@_t('run_training_loop: <=200 total epochs, early stopping inside them, best epoch restored, restore verified (framework-free loop used by the torch cell)')
def _():
    def mk(scores):
        st = {'w': None, 'trained': 0}
        def train(ep): st['trained'] += 1; st['w'] = ('w', st['trained'])
        def val():
            cur = st['w']; return scores[cur[1] - 1] if cur is not None and cur[1] <= len(scores) else 0.0
        return st, train, val, (lambda: st['w']), (lambda s: st.__setitem__('w', s))
    st, tr, va, sn, rs = mk([0.001 * i for i in range(1, MAX_EPOCHS + 1)]); o = run_training_loop(tr, va, sn, rs); assert o['epochs_run'] == MAX_EPOCHS and o['stop_reason'] == 'max_epochs' and o['best_epoch'] == MAX_EPOCHS and o['restore_delta'] == 0 and st['trained'] == MAX_EPOCHS
    sc = [0.5 + 0.01 * i for i in range(30)] + [0.78 - 0.001 * i for i in range(1, 71)]; st, tr, va, sn, rs = mk(sc); o = run_training_loop(tr, va, sn, rs)
    assert o['best_epoch'] == 30 and o['epochs_run'] == 45 and o['stop_reason'] == 'early_stop_patience' and st['w'] == ('w', 30) and o['restore_delta'] == 0 and len(o['history']) == 45
    st, tr, va, sn, rs = mk([0.5] * MAX_EPOCHS); o = run_training_loop(tr, va, sn, rs); assert o['epochs_run'] == 20 and o['best_epoch'] == 1
    st, tr, va, sn, rs = mk([0.1 * (i + 1) / 10 for i in range(30)]); o = run_training_loop(tr, va, sn, lambda s: st.__setitem__('w', ('w', 1))); assert o['restore_delta'] > RESTORE_TOL
    st, tr, va, sn, rs = mk([float('nan')] * MAX_EPOCHS)
    try: run_training_loop(tr, va, sn, rs); raise RuntimeError('no stop')
    except FloatingPointError: pass
    try: run_training_loop(tr, va, sn, rs, max_epochs=MAX_EPOCHS + 1); raise RuntimeError('no stop')
    except AssertionError: pass

@_t('steps: steps_per_epoch = ceil(0.9*B/256) = 4/8/11/15; total_optimizer_steps = steps x epochs_run; both stored per run')
def _():
    assert [steps_per_epoch(int(0.9 * B)) for B in BUDGETS] == [4, 8, 11, 15] and 'steps_per_epoch' in COLUMNS and 'total_optimizer_steps' in COLUMNS

@_t('coverage columns: n_target_classes_train_ge30 / n_absent_target_classes_train in results; locked-design list contains every required name')
def _():
    for c_ in ['n_target_classes', 'n_target_classes_train_ge30', 'n_absent_target_classes_train']: assert c_ in COLUMNS
    for k in ['AUS_SPEC_CONFIRMED', 'AUS_KEY_SHARES_IF_CONFIRMED', 'TARGETS_TO_RUN', 'TARGET_CROPS', 'UNDERTRAINED_EPOCH', 'REVIEW_RULE', 'TRACKS']: assert k in LOCKED and k in globals(), k

@_t('geography STOP (France): rice only in regions 76 / 93; sugarcane / cotton reported; any violating row stops with the offending regions named')
def _():
    ok_st = np.array(['76', '93', '11', '24']); ok_cr = np.array(['rice', 'rice', 'corn', 'other'])
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf): out = check_geography(ok_st, ok_cr)
    assert set(out['rice']) == {'76', '93'} and 'report only' in buf.getvalue()
    try:
        with contextlib.redirect_stdout(io.StringIO()): check_geography(np.array(['11', '76']), np.array(['rice', 'rice']))
        raise RuntimeError('no stop')
    except RuntimeError as e: assert 'STOP' in str(e) and 'rice' in str(e) and "'11'" in str(e)

@_t('exhaustive cells flagged ("CI is degenerate by construction"); census k=1 flagged separately; sampled cells (C(P,k)>10) are neither')
def _():
    rows = []
    for bi, (k, P, ncomb) in enumerate(((1, 5, 5), (2, 5, 10), (2, 8, 10), (4, 12, 10))):
        for ci, cmb in enumerate(list(itertools.combinations(range(P), k))[:ncomb]):
            for s in range(2): rows.append(dict(run_id=f'{k}{P}{ci}{s}', status='ok', target='ARG', budget=1000 + 100 * bi, k=k, states='-'.join(f'S{j}' for j in cmb), seed=s, primary_f1=0.5 + 0.01 * ci + 0.001 * s, pool_size=P, n_total_combos=math.comb(P, k), key_n=200, target_crop_n=200, n_target_classes_train_ge30=4))
    df = pd.DataFrame(rows); cs = cell_stats(df)
    r = {(int(a), int(b)): (bool(c), bool(d), n) for a, b, c, d, n in zip(cs.k, cs.N_total_combos, cs.census, cs.exhaustive, cs.ci_note)}
    assert r[(1, 5)][0] and not r[(1, 5)][1]; assert r[(2, 10)][1] and 'degenerate by construction' in r[(2, 10)][2]; assert not r[(2, 28)][1] and not r[(4, 495)][1]

@_t('gate report: BOTH tracks side by side (EXISTING and REVIEW) with baselines + cov-cond, class coverage per target and k, optimizer steps, no pooled/ranked track')
def _():
    ok_pipeline = selftest_pipeline(return_report=True)
    assert 'EXISTING' in ok_pipeline and 'REVIEW' in ok_pipeline and 'cov-cond' in ok_pipeline and 'CLASS COVERAGE by target and k' in ok_pipeline and 'absent-class share' in ok_pipeline
    assert 'steps/epoch' in ok_pipeline and 'total optimizer steps' in ok_pipeline and 'EXISTING: mean by (target, budget, k)' in ok_pipeline and 'REVIEW-PROPOSED: mean by (target, budget, k)' in ok_pipeline

@_t('no track is declared "the paper\'s rule" in code labels (existing track = "existing release remap"; paper_text_v3 = literal sec.4.4 reading)')
def _():
    import re as _re
    labels = ' '.join(t_[1] for t_ in TRACKS) + ' ' + ' '.join(REMAP_RULES)
    assert not _re.search(r"paper'?s rule|the paper rule|paper-rule", labels, _re.I)
    sc = score_run('FRA', _ids(('corn', 20), ('other', 20)), _ids(('corn', 20), ('other', 20)), np.ones(7) * 100); assert sc['primary_view'] == 'existing_release_remap'


# ======================= v3.3 regression tests (each one belongs to a CHANGE ID in the change log) =======================
_PIPE_MEMO = {}
def _pipe_df():
    if 'v' not in _PIPE_MEMO: _PIPE_MEMO['v'] = selftest_pipeline(return_df=True)
    return _PIPE_MEMO['v'][0].copy(), list(_PIPE_MEMO['v'][1])

def _mk_err_row(template, **kw):
    e = template.copy()
    for c_ in e.index:
        if c_ not in ('run_id', 'status', 'target', 'budget', 'k', 'combo_id', 'states', 'seed', 'error', 'integrity_pass'): e[c_] = ''
    e['status'] = 'error'; e['error'] = 'boom'; e['integrity_pass'] = False
    for k_, v_ in kw.items(): e[k_] = v_
    return e

@_t('V33-01 gate_report: an error / blank-metric row FIRST in a budget group, a target/k group, a budget with only error rows, or zero ok rows never crashes; rows are listed')
def _():
    df0, cells = _pipe_df(); d0 = df0[pd.to_numeric(df0.seed) == 0].reset_index(drop=True); t0 = d0.iloc[0]
    base = gate_report(d0, 0, cells); assert 'RUNS WITHOUT METRICS (error / integrity_fail before scoring): 0' in base
    e1 = _mk_err_row(t0, run_id='E1')                                                                               # first of its budget AND target/k group
    r1 = gate_report(pd.concat([pd.DataFrame([e1]), d0], ignore_index=True), 0, cells); assert 'E1 [error] UNRESOLVED: boom' in r1 and 'RUNS WITHOUT METRICS (error / integrity_fail before scoring): 1' in r1
    e2 = _mk_err_row(t0, run_id='E2', budget=987654)                                                                 # a budget group that holds ONLY an error row
    r2 = gate_report(pd.concat([d0, pd.DataFrame([e2])], ignore_index=True), 0, cells); assert 'budget 987654' not in r2 and 'E2 [error]' in r2
    ok_id = d0.iloc[3]['run_id']; e3 = _mk_err_row(d0.iloc[3], run_id=ok_id)                                         # an error row that was later resolved by a resume
    r3 = gate_report(pd.concat([pd.DataFrame([e3]), d0], ignore_index=True), 0, cells); assert f'{ok_id} [error] SUPERSEDED' in r3
    e4 = _mk_err_row(t0, run_id='E4'); e4['status'] = 'integrity_fail'                                               # integrity_fail raised before scoring (blank metrics)
    r4 = gate_report(pd.concat([pd.DataFrame([e4]), d0], ignore_index=True), 0, cells); assert 'E4 [integrity_fail]' in r4
    r5 = gate_report(pd.DataFrame([e1]), 0, cells); assert 'RUNS WITHOUT METRICS (error / integrity_fail before scoring): 1' in r5        # nothing but errors
    # the abort-level integrity section still counts the error rows as failures (the gate is NOT weakened)
    assert int([l for l in r1.splitlines() if l.strip().startswith('I1_true_classes_ge2')][0].split('failures:')[1]) == 1

@_t('V33-02 gate_report skill flags: empty / NaN / one / several / mixed flag values all give a plain dict (no pandas .str accessor)')
def _():
    df0, cells = _pipe_df(); d0 = df0[pd.to_numeric(df0.seed) == 0].reset_index(drop=True); n = len(d0)
    def flags_line(vals):
        z = d0.copy(); z['skill_flags'] = vals; return [l for l in gate_report(z, 0, cells).splitlines() if l.strip().startswith('flags:')][0].strip()
    assert flags_line([''] * n) == 'flags: {}' and flags_line([np.nan] * n) == 'flags: {}' and flags_line([None] * n) == 'flags: {}'
    assert flags_line(['WARN_VAL:0.1'] + [''] * (n - 1)) == "flags: {'WARN_VAL': 1}"
    assert flags_line(['a:1|b:2'] * n) == "flags: {'a': %d, 'b': %d}" % (n, n)
    mixed = (['a:1|b:2', '', np.nan, 'a:9'] * n)[:n]; txt = flags_line(mixed); assert "'a'" in txt and "'b'" in txt
    z = d0.iloc[0:0]; assert 'GATE REPORT' in gate_report(z, 0, [])                                                   # zero rows

@_t('V33-08 score_run: y_true outside the target label set (paper Table 2) STOPs; valid truth, single-class, empty test set and undefined predictions behave as specified')
def _():
    for t in ['ARG', 'CHN', 'AUS', 'FRA']:
        yt = _labels_from_table2(t); undefined = [c for c in CROP_ORDER if c not in DEFINED_CLASSES[t]]
        for u in undefined:
            bad = np.r_[yt, np.full(30, CROP_ID[u])]
            try: score_run(t, bad, bad, np.array([100] * 7)); raise RuntimeError('no stop')
            except AssertionError as e: assert 'outside the target label set' in str(e), str(e)
        sc = score_run(t, yt, yt, np.array([100] * 7)); assert sc['existing_f1' if False else 'paper_f1'] == 1.0 and sc['review_f1'] == 1.0 and sc['papertext_f1'] == 1.0     # perfect predictions of a valid truth
        for u in undefined:                                                                                         # a prediction of an UNDEFINED class: existing / paper-text / review differ exactly as the rule table says
            yp = yt.copy(); yp[0] = CROP_ID[u]; sc = score_run(t, yt, yp, np.array([100] * 7)); assert 0 <= sc['review_f1'] <= 1 and 0 <= sc['paper_f1'] <= 1
    try: score_run('FRA', np.array([], int), np.array([], int), np.ones(7)); raise RuntimeError('no stop')
    except AssertionError as e: assert 'empty' in str(e)
    try: score_run('FRA', np.full(20, CROP_ID['corn']), np.full(20, CROP_ID['corn']), np.ones(7)); raise RuntimeError('no stop')
    except TooFewClasses: pass
    yt = _labels_from_table2('CHN'); yp = np.where(yt == CROP_ID['rice'], CROP_ID['corn'], yt); sc = score_run('CHN', yt, yp, np.array([100] * 7))      # a defined class never predicted
    assert sc['paper_f1_cls'][CROP_ID['rice']] == 0.0 and sc['review_f1'] < 1

@_t('V33-09 run_environment_seed<N>.json is written (software, device, settings, data identity, remap tables) without touching the manifest / LOCKED design')
def _():
    with tempfile.TemporaryDirectory() as td:
        locked_before = json.dumps({k_: globals()[k_] for k_ in LOCKED}, default=str, sort_keys=True)
        env = write_run_environment(td, 3, info={'ARG': dict(path=os.path.join(td, 'nope.npz'), shape=(10, 1, 10, 73), raw=10, kept=9, dup=1, raw_by_class={'corn': 10})}, geography={'rice': {'AR': 5}})
        p = os.path.join(td, 'run_environment_seed3.json'); assert os.path.exists(p); j = json.load(open(p))
        assert j['seed'] == 3 and j['settings']['class_weight_source'] == CLASS_WEIGHT_SOURCE and j['settings']['max_epochs'] == MAX_EPOCHS and 'ARG' in j['data_files'] and j['geography_observed'] == {'rice': {'AR': 5}}
        assert {'existing_release', 'paper_text_v3', 'review_targetmap'} <= set(j['remap_rules'][0]) and 'python' in j and 'numpy' in j
        assert locked_before == json.dumps({k_: globals()[k_] for k_ in LOCKED}, default=str, sort_keys=True)

@_t('V33-03 geography STOP message (France): counts, percentage, tolerance and the two ways forward are printed; tolerance 0 unchanged; a stated tolerance admits boundary rows')
def _():
    st = np.array(['76'] * 50 + ['11'] * 3 + ['93'] * 40); cr = np.array(['rice'] * 93); assert GEOGRAPHY_MAX_OUTSIDE_ROWS == 0
    try:
        with contextlib.redirect_stdout(io.StringIO()): check_geography(st, cr)
        raise RuntimeError('no stop')
    except RuntimeError as e: m = str(e); assert '3 of 93 rows' in m and "'11'" in m and 'GEOGRAPHY_MAX_OUTSIDE_ROWS=0' in m and 'NEIGHBOURING' in m
    old = globals()['GEOGRAPHY_MAX_OUTSIDE_ROWS']
    try:
        globals()['GEOGRAPHY_MAX_OUTSIDE_ROWS'] = 3
        with contextlib.redirect_stdout(io.StringIO()): out = check_geography(st, cr)
        assert out['rice']['11'] == 3
    finally: globals()['GEOGRAPHY_MAX_OUTSIDE_ROWS'] = old

@_t('V33-10 regression output is labelled EXPLORATORY (overlapping state combinations are not independent replicates)')
def _():
    df0, cells = _pipe_df(); rg = regression_report(df0, 'primary_f1'); assert len(rg) > 0 and rg['note'].str.contains('EXPLORATORY').all()

N_FAIL_A = sum(1 for r in TEST_REPORT if r[1] != 'PASS')
for name, res, msg in TEST_REPORT: print(f'  [{res}] {name}' + (f'\n         -> {msg}' if msg else ''))
TESTS_A_PASSED = (N_FAIL_A == 0)
print('\nTEST GATE PART A:', 'ALL PASSED' if TESTS_A_PASSED else f'{N_FAIL_A} FAILED -- training is BLOCKED')

## 5. France region assignment + data loading code

In [ ]:
# ======================================================================================
# DATA (Level 2): load the Sentinel-2 median time series (official files), de-duplicate like the authors' loader, attach the FRENCH REGION to every source sample from the verified
#       row map, and VERIFY everything before any training.   (Torch-free: arrays stay numpy; the torch cell converts them.)
#       Differences from Level 1: the source is France (units = regions); there is no point-in-polygon step -- the region of a row is read from the raw FILE NAME through the exact
#       row-by-row match of raw points to packaged rows (FRA_join_verification.md); the 96 packaged rows that match two identical raw points are placed only when their position is certain.
# ======================================================================================
def unique_row_index(a):
    # indices of the FIRST occurrence of every distinct feature row == what the authors' data_load() keeps via np.unique(axis=0, return_index=True)
    a = np.ascontiguousarray(np.asarray(a).reshape(len(a), -1)); seen, keep = set(), []
    for i in range(len(a)):
        h = hashlib.blake2b((a[i] + 0.0).tobytes(), digest_size=16).digest()   # +0.0 turns -0.0 into 0.0 (np.unique treats them as equal)
        if h not in seen: seen.add(h); keep.append(i)
    return np.asarray(keep, dtype=np.int64)

def _sha256_file(p, chunk=1 << 24):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()

def find_npz(data_dir, country, log=print):
    # V33-05: several matching files are LISTED.  Byte-identical copies -> the documented rule picks one (path containing for_CropNet first, then the shortest path).
    # Files with DIFFERENT content -> STOP, because the choice would silently change the data.
    name = f'median_S2_{country}_month1-12_day5.npz'; hits = []
    for root, _d, files in os.walk(data_dir):
        if name in files: hits.append(os.path.join(root, name))
    _check(hits, f'{name} not found under {data_dir}')
    hits = sorted(set(hits))
    if len(hits) > 1:
        info = [(p, os.path.getsize(p), _sha256_file(p)) for p in hits]
        log(f'  {country}: {len(hits)} files match {name}:'); [log(f'      {p}  size {n}  sha256 {h[:16]}...') for p, n, h in info]
        if len({h for _p, _n, h in info}) > 1:
            raise RuntimeError(f'STOP: {len(hits)} DIFFERENT files match {name} under {data_dir}. Remove or move the wrong ones (or point DATA_DIR at one folder); no file is chosen silently.')
        log('      byte-identical copies -> deterministic rule: path containing for_CropNet first, then the shortest path')
    hits.sort(key=lambda p: ('for_CropNet' not in p, len(p)))
    return hits[0]

def load_country(path, country):
    z = np.load(path); lbl = z['lbl'].astype(int)
    cand = ['fea'] if 'fea' in z.files else [k for k in z.files if k != 'lbl' and z[k].ndim >= 2 and z[k].shape[0] == len(lbl)]
    _check(cand, f'{country}: no per-sample feature array (keys={z.files})')
    a = z[cand[0]].astype(np.float32); band_axes = [ax for ax in range(1, a.ndim) if a.shape[ax] == 10]
    _check(band_axes, f'{country}: cannot find the 10-band axis in shape {a.shape}')
    a = np.moveaxis(a, band_axes[0], 1).reshape(a.shape[0], 10, -1)
    _check(len(lbl) == len(a), f'{country}: label/feature length mismatch')
    _check(lbl.min() >= 0 and lbl.max() <= 6, f'{country}: label ids outside 0..6')
    nan = int((~np.isfinite(a)).sum()); _check(nan == 0, f'{country}: {nan} non-finite feature values (NaN/inf) -- CropNet would produce NaN')
    return lbl, a


FRA_FILE_RE = re.compile(r'^S2_FRA_(?P<tok>[a-z]+)_LAMB93_R(?P<reg>\d+)\.csv$')

def fra_blocks_from_row_map(row_map_path):
    # Re-derives the 64 file blocks from the verified row map (used to check the embedded FRA_BLOCKS, and in the tests).  STOP on any structural difference from what the verification found.
    h = _sha256_file(row_map_path); _check(h == FRA_ROW_MAP_SHA256, f'STOP: the row map {row_map_path} is not the verified file (sha256 {h[:16]}... != {FRA_ROW_MAP_SHA256[:16]}...).')
    r = pd.read_csv(row_map_path); need = ['file', 'raw_index', 'n_match', 'npz_row', 'region_code', 'crop_token']; miss = [c for c in need if c not in r.columns]
    _check(not miss, f'row map lacks columns {miss}')
    _check(len(r) == FRA_ROW_MAP_ROWS, f'row map has {len(r)} rows, expected {FRA_ROW_MAP_ROWS}')
    amb = r['n_match'] > 1; _check(int(amb.sum()) == FRA_ROW_MAP_AMBIGUOUS and bool((r.loc[amb, 'npz_row'] == -1).all()) and bool((r.loc[~amb, 'npz_row'] >= 0).all()) and bool((r['n_match'] >= 1).all()), 'row-map match structure differs from the verification')
    r['region'] = r['region_code'].astype(int).astype(str); _check(set(r['region']) == set(SOURCE_UNITS), f'regions in the map {sorted(set(r["region"]))} != {SOURCE_UNITS}')
    _check(r.groupby('region').size().to_dict() == FRA_REGION_TOTALS, 'points per region differ from the recorded table')
    pf = r['file'].map(lambda f: FRA_FILE_RE.match(str(f))); _check(pf.notna().all(), 'a file name in the row map does not match S2_FRA_<crop>_LAMB93_R<region>.csv')
    _check(bool((pf.map(lambda m: m.group('tok')) == r['crop_token']).all() and (pf.map(lambda m: str(int(m.group('reg')))) == r['region']).all()), 'file name disagrees with the crop_token / region columns')
    _check(set(r['crop_token']) <= set(RAW_TO_CROP), 'unknown crop token in the row map')
    u = r[~amb]; _check(u['npz_row'].is_unique, 'a packaged row is claimed by two raw points')
    for f_, g in u.groupby('file'): _check(bool((np.diff(g.sort_values('raw_index')['npz_row'].values) > 0).all()), f'{f_}: packaged rows do not increase with the raw position')
    b = u.groupby('file').agg(lo=('npz_row', 'min'), hi=('npz_row', 'max'), tok=('crop_token', 'first'), region=('region', 'first')).sort_values('lo')
    return [(t, g, int(lo), int(hi)) for t, g, lo, hi in zip(b['tok'], b['region'], b['lo'], b['hi'])]

def build_fra_meta(lbl_raw, row_map_path=None, log=print):
    # Returns (meta over ALL N packaged rows, report).  meta columns: npz_row, crop_id, state (region code, str), assigned (bool).
    #   * FRA_BLOCKS (embedded, derived from the verified row map) must be disjoint, ordered as (crop label, token A-Z, region), inside the file, and give the recorded rows per region
    #   * every row inside a block gets the block's region; rows outside every block (cannot be placed with certainty) stay unassigned and are excluded later
    #   * the packaged label of EVERY assigned row must equal the label of its raw crop token (full-row check against the real data)
    #   * if the row-map file is available it is re-read and must give exactly the embedded blocks
    N = len(lbl_raw); B = pd.DataFrame(FRA_BLOCKS, columns=['tok', 'region', 'lo', 'hi']); _check(len(B) == 64 and set(B['region']) == set(SOURCE_UNITS), 'embedded FRA_BLOCKS is not the 64-file table')
    _check(N == FRA_ROW_MAP_ROWS, f'the packaged France file has {N} rows, expected {FRA_ROW_MAP_ROWS}')
    B['label'] = B['tok'].map(lambda t: CROP_ID[RAW_TO_CROP[t]]); _check(bool((B['lo'].values[1:] > B['hi'].values[:-1]).all()) and bool((B['lo'] <= B['hi']).all()) and int(B['hi'].max()) < N, 'FRA_BLOCKS are not disjoint / ordered / inside the file')
    order_ok = B.sort_values(['label', 'tok', 'region'], key=lambda c: c.astype(int) if c.name == 'region' else c).index.tolist() == B.index.tolist()
    _check(order_ok, 'the block order is not (crop label, token A-Z, region): the ordering convention verified for France does not hold')
    region_of = np.full(N, '', dtype=object); lab_blk = np.full(N, -1)
    for b in B.itertuples(): region_of[b.lo:b.hi + 1] = b.region; lab_blk[b.lo:b.hi + 1] = b.label
    assigned = region_of != ''; crop_id = np.asarray(lbl_raw).astype(int)
    _check(bool((lab_blk[assigned] == crop_id[assigned]).all()), 'the packaged label of some row differs from the label of its raw crop token (full-row label check)')
    per = {k: int(v) for k, v in pd.Series(region_of[assigned]).value_counts().sort_index().items()}
    _check(per == FRA_ASSIGNED_PER_REGION, f'rows per region from the blocks {per} != recorded {FRA_ASSIGNED_PER_REGION}')
    n_excl = int((~assigned).sum()); gaps = np.flatnonzero(~assigned)
    if row_map_path is not None:
        _check(fra_blocks_from_row_map(row_map_path) == [tuple(x) for x in FRA_BLOCKS], 'the row-map file does not reproduce the embedded FRA_BLOCKS')
        log('  row-map file re-read: reproduces the embedded blocks exactly')
    meta = pd.DataFrame({'npz_row': np.arange(N), 'crop_id': crop_id, 'state': np.where(assigned, region_of, 'NA'), 'assigned': assigned})
    rep = dict(rows=N, blocks=len(B), block_order_ok=bool(order_ok), assigned=int(assigned.sum()), excluded_outside_blocks=n_excl, excluded_rows=gaps.tolist(), labels_checked=int(assigned.sum()), rows_per_region=per, row_map_sha256=FRA_ROW_MAP_SHA256)
    log(f'  FRA region table: 64 raw-file blocks (from the verified row map, sha256 {FRA_ROW_MAP_SHA256[:12]}...) tile the packaged rows in the order (label, token, region); labels of all {int(assigned.sum())} assigned rows equal their raw crop token;')
    log(f'  {n_excl} rows lie between blocks (matched two identical raw points; region NOT certain) -> excluded')
    return meta, rep

def load_all_data(data_dir, out_dir, row_map_path=None):
    cache_dir = os.path.join(out_dir, 'cache'); os.makedirs(cache_dir, exist_ok=True)
    needed = [SOURCE] + [t for t in TARGETS_TO_RUN]; FEATURES, NPZ_LBL, INFO, meta, report = {}, {}, {}, None, None
    for c in needed:
        path = find_npz(data_dir, c); lbl, a = load_country(path, c); keep = unique_row_index(a); extra = {}
        if c == SOURCE:
            meta_full, report = build_fra_meta(lbl, row_map_path); ok = meta_full['assigned'].values[keep]
            extra = dict(excluded_unassigned=int((~ok).sum())); keep = keep[ok]; meta = meta_full.iloc[keep].reset_index(drop=True)
        FEATURES[c] = np.ascontiguousarray(a[keep][:, None, :, :]); NPZ_LBL[c] = lbl[keep]
        INFO[c] = dict(path=path, raw=len(lbl), kept=len(keep), dup=len(lbl) - len(keep) - extra.get('excluded_unassigned', 0), shape=tuple(FEATURES[c].shape), keep=keep, raw_by_class=pd.Series(lbl).map(CROP_LABELS).value_counts().reindex(CROP_ORDER, fill_value=0).to_dict(),
                       mean=float(a.mean()), std=float(a.std()), vmin=float(a.min()), vmax=float(a.max()), **extra); del a
    meta['crop'] = meta['crop_id'].map(CROP_LABELS); meta = meta[['npz_row', 'crop_id', 'state', 'crop']]
    print(f"SOURCE {SOURCE}: raw {INFO[SOURCE]['raw']} | removed as exact duplicates {INFO[SOURCE]['dup']} | excluded (region not certain) {INFO[SOURCE]['excluded_unassigned']} | kept {INFO[SOURCE]['kept']}")
    return FEATURES, NPZ_LBL, INFO, meta, report

def row_hash_set(F):
    a = np.ascontiguousarray(F.reshape(len(F), -1)); return {hashlib.blake2b((a[i] + 0.0).tobytes(), digest_size=16).digest() for i in range(len(a))}


def verify_data(FEATURES, NPZ_LBL, INFO, meta):
    # EVERY check here is a STOP: raw counts, label sets, class counts vs paper v3 Table 2, no target row inside the source table, region alignment.
    print('\n================ DATA VERIFICATION (paper arXiv 2509.03497 v3, Table 2) ================')
    rows = []
    for c in INFO:
        rows.append(dict(country=c, raw=INFO[c]['raw'], table2_total=TABLE2_TOTAL[c], kept_after_dedup=INFO[c]['kept'], removed_duplicates=INFO[c]['dup'], shape=INFO[c]['shape'], **{f'kept_{k}': v for k, v in
                         pd.Series(NPZ_LBL[c]).map(CROP_LABELS).value_counts().reindex(CROP_ORDER, fill_value=0).to_dict().items()}))
    print(pd.DataFrame(rows).to_string(index=False))
    problems = []
    for c in INFO:
        if INFO[c]['raw'] != TABLE2_TOTAL[c]: problems.append(f'{c}: raw rows {INFO[c]["raw"]} != Table 2 total {TABLE2_TOTAL[c]}')
        if INFO[c]['raw_by_class'] != TABLE2[c]: problems.append(f'{c}: raw class counts {INFO[c]["raw_by_class"]} != Table 2 {TABLE2[c]}')
    for t in [SOURCE] + list(TARGETS_TO_RUN):
        got = sorted([CROP_LABELS[i] for i in np.unique(NPZ_LBL[t])], key=CROP_ORDER.index); exp = sorted(DEFINED_CLASSES[t], key=CROP_ORDER.index)
        rawset = sorted([c for c, n in INFO[t]['raw_by_class'].items() if n > 0], key=CROP_ORDER.index)
        print(f'  label set {t}: after dedup {got} | raw {rawset} | Table 2 {exp} -> {"OK" if got == exp and rawset == exp else "MISMATCH"}')
        if got != exp or rawset != exp: problems.append(f'{t}: label set data(raw {rawset}, dedup {got}) != Table 2 {exp}')
    for c in INFO:
        if 'median_S2_' not in os.path.basename(INFO[c]['path']): problems.append(f'{c}: file {INFO[c]["path"]} is not a Sentinel-2 median file (AlphaEarth embeddings must not be used here)')
        if INFO[c]['shape'][1:3] != (1, 10): problems.append(f'{c}: unexpected tensor shape {INFO[c]["shape"]} (need (N,1,10,T))')
        elif INFO[c]['shape'][3] < 67: problems.append(f'{c}: time axis {INFO[c]["shape"][3]} too short for the official DOY 121-334 window (needs >= 67 five-day steps)')
    if problems: raise RuntimeError('STOP -- data conflicts with the paper (Table 2). Report, do not resolve silently:\n  - ' + '\n  - '.join(problems))
    print('feature source:', FEATURE_SOURCE, '| files:', {c: os.path.basename(INFO[c]['path']) for c in INFO})
    print('\nde-duplication difference (raw -> kept):  ' + ' | '.join(f"{c}: -{INFO[c]['dup']}" for c in INFO))
    print('feature scale per country (mean / std / min / max):')
    for c in INFO: print(f"  {c}: {INFO[c]['mean']:.1f} / {INFO[c]['std']:.1f} / {INFO[c]['vmin']:.1f} / {INFO[c]['vmax']:.1f}")
    means = [INFO[c]['mean'] for c in INFO]
    if max(means) > 10 * max(min(means), 1e-9): print('  !! WARNING: country feature means differ by >10x -- check that all files use the same reflectance scale')
    _check(len(meta) == len(FEATURES[SOURCE]) and (meta['crop_id'].values == NPZ_LBL[SOURCE]).all(), 'source region table is not aligned with the source features')
    _check(set(meta['state']) <= set(SOURCE_UNITS), f'unknown region codes in the source table: {sorted(set(meta["state"]) - set(SOURCE_UNITS))}')
    hs = row_hash_set(FEATURES[SOURCE])                                              # LEAKAGE: no target sample may be identical to any source row (targets never enter training)
    for t in TARGETS_TO_RUN:
        n_dup = len(hs & row_hash_set(FEATURES[t])); print(f'  leakage check {SOURCE} vs {t}: {n_dup} identical rows')
        _check(n_dup == 0, f'{t}: {n_dup} target rows are identical to {SOURCE} rows -- leakage; STOP and report')
    big = meta.groupby('state').size(); print(f"{SOURCE}: {len(meta)} samples, {meta['state'].nunique()} regions ({int((big >= MIN_STATE_ROWS).sum())} with >= {MIN_STATE_ROWS} rows) -- region table aligned row by row")

## 6. Data / Level 2 tests

In [ ]:
# ======================================================================================
# TEST GATE - PART A2 (no GPU, no repo, no real data): the DATA functions on synthetic inputs (v3.3).  Runs after the data-function cell; blocks training like Part A.
# ======================================================================================
_n0 = len(TEST_REPORT)
@_t('V33-05 find_npz: one match -> that file; identical copies -> listed + documented rule; DIFFERENT files -> STOP')
def _():
    with tempfile.TemporaryDirectory() as td:
        name = 'median_S2_ARG_month1-12_day5.npz'
        for sub, payload in (('a', b'same'), ('for_CropNet/b', b'same')): os.makedirs(os.path.join(td, sub), exist_ok=True); open(os.path.join(td, sub, name), 'wb').write(payload)
        out = io.StringIO()
        with contextlib.redirect_stdout(out): p = find_npz(td, 'ARG', log=print)
        assert 'for_CropNet' in p and 'byte-identical' in out.getvalue() and out.getvalue().count('sha256') == 2
        open(os.path.join(td, 'a', name), 'wb').write(b'different')
        try:
            with contextlib.redirect_stdout(io.StringIO()): find_npz(td, 'ARG', log=print)
            raise RuntimeError('no stop')
        except RuntimeError as e: assert 'DIFFERENT files' in str(e)
        try: find_npz(td, 'FRA'); raise RuntimeError('no stop')
        except AssertionError as e: assert 'not found' in str(e)

def _p4_fixture(n_states=10, per=60, seed=0):
    rng = np.random.default_rng(seed); sts = [f'S{i:02d}' for i in range(n_states)]; st, cr, X = [], [], []
    shift = {s: rng.normal(0, 1, 730) * (0.2 + 0.25 * i) for i, s in enumerate(sts)}          # state-specific phenology shift, growing with i
    for s in sts:
        for ci_, c in enumerate(['soybeans', 'corn', 'wheat']):
            m = np.full(730, ci_ - 1.0)
            X.append((m + shift[s] + rng.normal(0, 0.3, (per, 730))).astype(np.float32)); st += [s] * per; cr += [c] * per
    return np.vstack(X).reshape(-1, 1, 10, 73), np.array(st), np.array(cr), sts

@_t('P4-01 diversity distances: symmetric, zero diagonal, USA-only inputs, NaN where a (state,crop) has < MIN_PROFILE_N samples')
def _():
    X, st, cr, sts = _p4_fixture(); DD = compute_diversity_distances(X, st, cr, min_n=20)
    for c in ('soybeans', 'corn', 'wheat'):
        M = DD['dist'][c]; assert M.shape == (10, 10) and np.allclose(M.values, M.values.T) and np.allclose(np.diag(M.values), 0) and M.notna().all().all()
    keep = ~((st == 'S03') & (cr == 'soybeans')) | (np.arange(len(st)) % 60 < 5); DD2 = compute_diversity_distances(X[keep], st[keep], cr[keep], min_n=20)
    assert DD2['dist']['soybeans'].loc['S03'].drop('S03').isna().all() and DD2['dist']['corn'].loc['S03'].notna().all()

@_t('P4-02 select_extreme_arms equals brute force (top / bottom N_ARM), is deterministic, and the two arms are disjoint; scores are mean pairwise distance')
def _():
    X, st, cr, sts = _p4_fixture(); DD = compute_diversity_distances(X, st, cr, min_n=20); D = DD['dist']['soybeans']; Dm = D.loc[sts, sts].values
    e1 = select_extreme_arms(sts, 3, Dm, 'ARG', 1000, BASE_MODE); e2 = select_extreme_arms(sts, 3, Dm, 'ARG', 1000, BASE_MODE); assert e1['diverse'] == e2['diverse'] and e1['clustered'] == e2['clustered']
    allc = list(itertools.combinations(range(10), 3)); sc = {c: np.mean([Dm[a, b] for a, b in itertools.combinations(c, 2)]) for c in allc}; srt = sorted(allc, key=lambda c: (-round(sc[c], 12), c))
    assert [tuple(sts.index(s) for s in c) for c in e1['diverse']] == srt[:N_ARM] and e1['enumerated'] and e1['n_scored'] == 120
    assert not (set(map(tuple, e1['diverse'])) & set(map(tuple, e1['clustered']))) and min(sc[c] for c in srt[:N_ARM]) >= max(sc[c] for c in srt[-N_ARM:])
    v, pct = combo_score_pct(e1['diverse'][0], sts, Dm, e1['scores']); assert abs(v - sc[srt[0]]) < 1e-9 and pct == 1.0

@_t('P4-03 large pools: scoring uses a fixed seeded sample of SCORE_SAMPLE distinct combinations (enumerated=False) and is reproducible')
def _():
    n = 40; rng = np.random.default_rng(1); A = rng.normal(0, 1, (n, 5)); Dm = np.sqrt(((A[:, None] - A[None]) ** 2).sum(-1)); pool = [f'P{i:02d}' for i in range(n)]
    old = (globals()['ENUM_LIMIT'], globals()['SCORE_SAMPLE'])
    try:
        globals()['ENUM_LIMIT'], globals()['SCORE_SAMPLE'] = 1000, 300
        a = select_extreme_arms(pool, 5, Dm, 'USA', 2000, BASE_MODE); b = select_extreme_arms(pool, 5, Dm, 'USA', 2000, BASE_MODE)
        assert not a['enumerated'] and a['n_scored'] == 300 and a['diverse'] == b['diverse'] and len(a['diverse']) == N_ARM and a['scores'].max() >= a['scores'].min()
    finally: globals()['ENUM_LIMIT'], globals()['SCORE_SAMPLE'] = old

@_t('P4-04 build_plan: SELECTION_ARMS=[random] reproduces the pre-Phase-4 random combinations; the arms share ONE pool; every combination passes combo_capacity_ok; k=1 is one shared census')
def _():
    X, st, cr, sts = _p4_fixture(12, 150); cap = make_cap(st, cr); g = globals(); names = ['BUDGETS', 'K_LIST', 'MIN_STATE_ROWS', 'TARGETS_TO_RUN', 'SELECTION_ARMS', 'MIN_KPOINTS', 'MIN_COMBOS', 'DIVERSITY_DIST', 'K_MAX']
    saved = {n_: g.get(n_) for n_ in names}
    try:
        g.update(BUDGETS=[300], K_LIST=[1, 2, 3, 4], MIN_STATE_ROWS=100, TARGETS_TO_RUN=['ARG'], MIN_KPOINTS=3, MIN_COMBOS=5, K_MAX={'ARG': 8, 'USA': 8, 'CHN': 2, 'AUS': 2, 'FRA': 8})
        g['SELECTION_ARMS'] = ['random']; c0, g0 = build_plan(cap); old_keys = {(c['k'], c['states']) for c in c0}
        g['SELECTION_ARMS'] = ['random', 'diverse', 'clustered']; g['DIVERSITY_DIST'] = compute_diversity_distances(X, st, cr, min_n=20); c1, g1 = build_plan(cap)
        assert {(c['k'], c['states']) for c in c1 if 'random' in c['arms']} == old_keys, 'the random arm changed'
        for c in c1: assert combo_capacity_ok(c['states'], c['target'], c['budget'], cap, c['composition'])
        k1 = [c for c in c1 if c['k'] == 1]; assert k1 and all(set(c['arms']) == {'random', 'diverse', 'clustered'} for c in k1)
        k2 = [c for c in c1 if c['k'] == 2]; ad = [c for c in k2 if 'diverse' in c['arms']]; ac = [c for c in k2 if 'clustered' in c['arms']]
        assert len(ad) == N_ARM and len(ac) == N_ARM and not ({c['states'] for c in ad} & {c['states'] for c in ac}) and all(c['div_score'] != '' for c in k2)
        assert min(c['div_score'] for c in ad) >= max(c['div_score'] for c in ac)
        r = g1[g1.k == 2].iloc[0]; assert r.n_diverse == N_ARM and r.n_clustered == N_ARM and r.n_random >= 5
    finally: g.update(saved)

@_t('P4-05 a state without a profile for a fixed crop leaves the pool of ALL arms (matched pool); undefined diversity yields NO diverse/clustered runs, never a silent substitute')
def _():
    X, st, cr, sts = _p4_fixture(12, 150); keep = ~((st == 'S05') & (cr == 'soybeans')); cap = make_cap(st, cr); g = globals(); names = ['BUDGETS', 'K_LIST', 'MIN_STATE_ROWS', 'TARGETS_TO_RUN', 'SELECTION_ARMS', 'DIVERSITY_DIST', 'K_MAX']; saved = {n_: g.get(n_) for n_ in names}
    try:
        g.update(BUDGETS=[300], K_LIST=[2], MIN_STATE_ROWS=100, TARGETS_TO_RUN=['ARG'], K_MAX={'ARG': 8, 'USA': 8, 'CHN': 2, 'AUS': 2, 'FRA': 8}, SELECTION_ARMS=['random', 'diverse', 'clustered'])
        g['DIVERSITY_DIST'] = compute_diversity_distances(X[keep], st[keep], cr[keep], min_n=20); c1, g1 = build_plan(cap)
        assert all('S05' not in c['states'] for c in c1) and 'S05' in g1.iloc[0].dropped_no_profile
        g['DIVERSITY_DIST'] = None; c2, g2 = build_plan(cap); assert all(c['arms'] == ('random',) for c in c2) and g2.iloc[0].n_diverse == 0 and 'not computed' in g2.iloc[0].arm_note
    finally: g.update(saved)

@_t('P4-06 pool rule: strict is the default and unchanged; equal_share is opt-in and only enlarges the pool')
def _():
    assert POOL_RULE == 'strict'; X, st, cr, sts = _p4_fixture(12, 150); cap = make_cap(st, cr); g = globals(); saved = (g['POOL_RULE'], g['MIN_STATE_ROWS'])
    try:
        g['MIN_STATE_ROWS'] = 100; s_ = pool_for('ARG', 2000, 4, cap); g['POOL_RULE'] = 'equal_share'; e_ = pool_for('ARG', 2000, 4, cap); assert set(s_) <= set(e_)
    finally: g['POOL_RULE'], g['MIN_STATE_ROWS'] = saved

@_t('P4-07 statistics: t_interval_fpc handles N=inf (selected arms: no finite-population correction); arm_contrast / expand_arms / diversity_assoc run and are labelled')
def _():
    m, lo, hi, se = t_interval_fpc(np.array([.5, .6, .55, .7]), float('inf')); assert np.isfinite([m, lo, hi, se]).all() and lo < m < hi
    rows = []
    for i, (arms, st_, f) in enumerate([('random', 'A-B', .60), ('random|diverse', 'A-C', .62), ('random', 'B-C', .58), ('diverse', 'C-D', .70), ('diverse', 'D-E', .72), ('clustered', 'A-E', .50), ('clustered', 'B-E', .52), ('random', 'B-D', .61), ('random', 'A-D', .59)]):
        rows.append(dict(run_id=f'r{i}', status='ok', composition=BASE_MODE, arms=arms, target='ARG', budget=1000, k=2, seed=0, states=st_, pool_size=6, n_total_combos=15, key_n=100, target_crop_n=100, n_target_classes_train_ge30=3, primary_f1=f, diversity_score=0.1 * i, diversity_pct=i / 9))
    d = pd.DataFrame(rows); E = expand_arms(d); assert (E.arm == 'diverse').sum() == 3 and (E.arm == 'random').sum() == 5 and (E.arm == 'clustered').sum() == 2
    cs = cell_stats(E[E.arm == 'diverse']); assert 'selected arm' in cs.iloc[0].ci_note and not cs.iloc[0].exhaustive
    ct = arm_contrast(d, 'primary_f1', 'diverse', 'random'); assert len(ct) == 1 and abs(ct.iloc[0]['diff'] - (np.mean([.62, .70, .72]) - np.mean([.60, .62, .58, .61, .59]))) < 1e-9
    assoc = diversity_assoc(d, 'primary_f1'); assert len(assoc) == 1 and assoc.iloc[0].n == 9

@_t('P4-08 results rows carry arms, diversity_score and diversity_pct; Phase-4 settings are in the LOCKED manifest and the run environment')
def _():
    for c in ('arms', 'diversity_score', 'diversity_pct'): assert c in COLUMNS
    for k in ('SELECTION_ARMS', 'POOL_RULE', 'N_ARM', 'ENUM_LIMIT', 'SCORE_SAMPLE', 'MIN_PROFILE_N', 'DIVERSITY_SPACE'): assert k in LOCKED, k
    df0, cells = _pipe_df(); assert df0['arms'].astype(str).str.contains('diverse').any() and df0['diversity_pct'].notna().any()

import hashlib
def _e3_scenario_hash():
    r = np.random.default_rng(7); sts = ['A', 'B', 'C', 'D']; st, cr = [], []
    for s in sts:
        for c, n in [('soybeans', 400), ('corn', 500), ('wheat', 500), ('rice', 300)]: st += [s] * n; cr += [c] * n
    st = np.array(st); cr = np.array(cr); crid = np.array([CROP_ID[c] for c in cr]); pools = {(s, c): np.where((st == s) & (cr == c))[0] for s in sts for c in set(cr)}
    h = hashlib.sha256()
    for tg, B, ss in [('ARG', 1000, ('A', 'B')), ('CHN', 1000, ('A', 'B', 'C'))]:
        for seed in (0, 1):
            sp = draw_split(tg, B, ss, seed, pools, cr)
            for key in sorted(sp) if isinstance(sp, dict) else range(len(sp)):
                v = sp[key]
                if key == 'sel': continue                                                          # key added in v3.7 (value 'random'); the draw itself is what is frozen
                h.update(str(key).encode()); h.update(np.ascontiguousarray(v).tobytes() if isinstance(v, np.ndarray) else repr(v).encode())
    return h.hexdigest()[:20]

_E_FROZEN_HASH = '666787c93d8b407377f2'          # Level 2: sha256 prefix of draw_split output (ARG + CHN, fixed scenario) computed with the FROZEN LEVEL 1 code (v3.8). USA is not in it: the target name is part of the draw seed, so a USA draw can never equal the old FRA draw; USA is checked structurally in L2-01
_E_FROZEN_LOCKED = ['BUDGETS', 'K_LIST', 'SEEDS', 'AUG', 'MAX_COMBOS', 'MIN_COMBOS', 'MIN_KPOINTS', 'MIN_STATE_ROWS', 'VAL_FRACTION', 'KEY_SHARES', 'K_MAX', 'FRA_K8_MIN_POOL']

def _e_cap(rows):
    # rows: {state: {crop: n}}; unlisted crops 0
    cap = pd.DataFrame({s: {c: v.get(c, 0) for c in CROP_ORDER} for s, v in rows.items()}).T; cap['total'] = cap[CROP_ORDER].sum(axis=1); cap.index.name = 'state'; return cap

@_t('E1 water_fill: exact total, never above capacity, as equal as capacity allows, deterministic, infeasible requests STOP')
def _():
    r0 = np.random.default_rng(0)
    for _i in range(300):
        k = int(r0.integers(1, 6)); caps = [int(x) for x in r0.integers(0, 400, k)]; n = int(r0.integers(0, max(1, sum(caps)) + 1))
        a = water_fill(n, caps, np.random.default_rng(_i)); b = water_fill(n, caps, np.random.default_rng(_i))
        assert a == b and sum(a) == n and all(0 <= x <= c for x, c in zip(a, caps))
        assert all(not (a[i] > a[j] + 1 and a[j] < caps[j]) for i in range(k) for j in range(k)), (n, caps, a)
    assert water_fill(10, [100, 100], np.random.default_rng(0)) in ([5, 5],) and sorted(water_fill(11, [100, 100], np.random.default_rng(0))) == [5, 6]
    assert water_fill(100, [10, 500, 500], np.random.default_rng(0)) == [10, 45, 45]
    try: water_fill(101, [50, 50], np.random.default_rng(0)); raise AssertionError('should STOP')
    except InfeasibleSplit: pass

@_t('E2 extension draw_split: a short state gives all it has and the others make up the surplus; rows are disjoint and verify_split accepts them; the minimum contribution is enforced')
def _():
    with profile_scope('extension'):
        rows = {'A': {'rice': 400, 'corn': 3000}, 'B': {'rice': 60, 'corn': 3000}, 'C': {'rice': 600, 'corn': 400}}   # CHN B=1000: rice 200, rest 800
        st = np.concatenate([np.full(sum(v.values()), s) for s, v in rows.items()]); cr = np.concatenate([np.array([c for c, n in v.items() for _ in range(n)]) for v in rows.values()])
        pools = {(s, c): np.where((st == s) & (cr == c))[0] for s in rows for c in set(cr)}
        sp = draw_split('CHN', 1000, ('A', 'B'), 0, pools, cr); v = verify_split(sp, 'CHN', 1000, ('A', 'B'), st, cr); assert v['ok'], v['msgs']
        allp = np.concatenate([sp['train_pos'], sp['val_pos']]); assert len(allp) == 1000 and len(set(allp.tolist())) == 1000
        nr = {s: int(((st[allp] == s) & (cr[allp] == 'rice')).sum()) for s in 'AB'}; assert nr == {'A': 140, 'B': 60}, nr          # B holds only 60 rice -> gives 60, A gives 140
        # tampering: swap B's rice for A's rice (B then gives less than it can while A gives more) must be caught
        bi = [i for i in sp['train_pos'] if st[i] == 'B' and cr[i] == 'rice'][:20]; ai = [i for i in pools[('A', 'rice')] if i not in set(allp.tolist())][:20]
        bad = dict(sp); bad['train_pos'] = np.concatenate([[i for i in sp['train_pos'] if i not in set(bi)], ai]).astype(sp['train_pos'].dtype); assert not verify_split(bad, 'CHN', 1000, ('A', 'B'), st, cr)['ok'], 'not-as-equal-as-possible split accepted'

@_t('E3 MAIN IS UNTOUCHED: draw_split gives byte-identical output to the v3.5 code (frozen hash), the main LOCKED list is unchanged, run ids carry no profile tag')
def _():
    assert _e3_scenario_hash() == _E_FROZEN_HASH, 'ARG/CHN draws differ from the frozen Level 1 code'
    assert LOCKED[:len(_E_FROZEN_LOCKED)] == _E_FROZEN_LOCKED
    assert ALLOC_RULE == 'equal' and PROFILE == 'main'; rid = make_run_id(dict(target='ARG', budget=1000, k=2, states=('A', 'B')), 0); assert 'prof-' not in rid
    with profile_scope('extension'): assert 'prof-extension' in make_run_id(dict(target='CHN', budget=3000, k=2, states=('A', 'B')), 0)

@_t('E4 extension plan: cells exist where the strict pool has none; every combination passes the pooled check and the floor; k=1 census kept; random arm only; MAIN plan unchanged on the same table')
def _():
    rows = {s: {'rice': 700, 'corn': 2500} for s in ('S1', 'S2')}                     # S1, S2 can each supply CHN B=3000 alone (rice 600 + rest 2400)
    rows.update({s: {'rice': 400, 'corn': 1100} for s in ('S3', 'S4', 'S5', 'S6')})   # these cannot alone, but can pool
    cap = _e_cap(rows); g = globals(); saved = {n_: g[n_] for n_ in ['MIN_STATE_ROWS', 'K_LIST', 'MIN_KPOINTS', 'K_MAX']}
    try:
        g.update(MIN_STATE_ROWS=100, K_LIST=[1, 2, 4], MIN_KPOINTS=3, K_MAX={'ARG': 8, 'USA': 8, 'CHN': 4, 'AUS': 2, 'FRA': 8})   # fixture of the DORMANT Level 1 extension; CHN k<=4 here is test-only
        cm, gm = build_plan(cap); main_cells = [c for c in cm if c['target'] == 'CHN' and c['budget'] == 3000]
        with profile_scope('extension'):
            ce, ge = build_plan(cap)
            assert ce and all(c['target'] == 'CHN' and c['budget'] in (3000, 4000) or c['target'] == 'USA' for c in ce) and all(c['arms'] == ('random',) for c in ce)
            c3 = [c for c in ce if c['target'] == 'CHN' and c['budget'] == 3000]; assert {c['k'] for c in c3} == {1, 2, 4}
            for c in ce: assert combo_feasible_pooled(c['states'], c['target'], c['budget'], cap)
            for c in c3:
                if c['k'] == 1: assert c['states'][0] in ('S1', 'S2')
            assert len([c for c in c3 if c['k'] == 1]) == 2 and any(c['k'] == 4 for c in c3)
            assert all(float(c['max_share']) >= 1.0 / c['k'] - 1e-9 for c in c3)
        assert all(set(c['states']) <= {'S1', 'S2'} for c in main_cells), 'strict main plan must contain only states that can supply the budget alone (CHN is exempt from MIN_COMBOS in Level 2, so the pair S1+S2 may appear)'
        cm2, gm2 = build_plan(cap); assert [(c['k'], c['states']) for c in cm2] == [(c['k'], c['states']) for c in cm]          # leaving the extension scope restores the main plan exactly
    finally: g.update(saved)

@_t('E5 extension settings: EXT_* keys are locked only when the extension is selected; extension output folder is a sub-folder of the main folder; results columns exist')
def _():
    for c in ('profile', 'state_rows', 'max_state_share'): assert c in COLUMNS
    has = [k for k in EXT_LOCKED if k in LOCKED]; assert len(has) in (0, len(EXT_LOCKED))
    if has: assert os.path.basename(os.path.normpath(OUT_DIR)) == 'extension' and os.path.dirname(os.path.normpath(OUT_DIR)) == os.path.normpath(MAIN_OUT_DIR)
    else: assert PROFILE == 'main'
    assert EXT_MIN_CONTRIB > 0 and set(EXT_TARGETS) <= {'ARG', 'USA', 'CHN', 'AUS'} and AUG == 'off'


@_t('V37-01 supervisor-review settings: 200-epoch cap, new results folder name, main/secondary metric, FRA coverage rule, input-feature text, new keys locked')
def _():
    assert MAX_EPOCHS == 200 and 'level2' in EXPERIMENT_VERSION and REPORT_PRIMARY == 'main_f1' and REPORT_SECONDARY == 'paper_f1' and FRA_MAIN_RULE == 'coverage_conditional' and 'Sentinel-2' in INPUT_FEATURES
    for k in ('FRA_MAIN_RULE', 'INPUT_FEATURES', 'KC_TARGETS', 'KC_COMBOS', 'AUG_TARGETS', 'AUG_BUDGETS', 'AUG_COMBOS', 'AUG_REPEAT', 'MAX_EPOCHS'): assert k in LOCKED, k
    assert 'aug_on' in APPROVED and AUTO_MODE in (True, False)
    for c in ('aug', 'sample_sel', 'input_features', 'main_f1', 'main_metric', 'train_n_effective'): assert c in COLUMNS

@_t('V37-02 main metric: ARG/CHN = binary key-crop (primary); FRA = coverage-conditional; old results get main_f1 filled in, existing values are never changed')
def _():
    assert main_metric_value('ARG', .7, .8)[0] == .7 and main_metric_value('CHN', .9, .95)[0] == .9 and main_metric_value('USA', .5, .68) == (.68, 'coverage_conditional_macro_f1')
    old = pd.DataFrame(dict(status=['ok'] * 3, target=['ARG', 'USA', 'USA'], primary_f1=[.7, .5, .5], cov_cond_primary_f1=[.8, .68, .68], main_f1=[np.nan, np.nan, .123]))
    new = add_main_metric(old); assert list(new.main_f1.round(3)) == [.7, .68, .123], list(new.main_f1)
    g = globals(); saved = g['FRA_MAIN_RULE']
    try: g['FRA_MAIN_RULE'] = 'primary'; assert main_metric_value('USA', .5, .68)[0] == .5
    finally: g['FRA_MAIN_RULE'] = saved

@_t('V37-03 kcenter_pick: exactly n DISTINCT rows of the pool, deterministic, covers the cloud better than a random pick of the same size')
def _():
    rng = np.random.default_rng(0); E = rng.normal(0, 1, (3000, 6)).astype(np.float32); pool = np.arange(500, 2500); n = 60
    a = kcenter_pick(pool, n, ('t', 1, 'S', 0, 'c'), embed=E); b = kcenter_pick(pool, n, ('t', 1, 'S', 0, 'c'), embed=E); c = kcenter_pick(pool, n, ('t', 1, 'S', 1, 'c'), embed=E)
    assert len(a) == n and len(set(a.tolist())) == n and set(a.tolist()) <= set(pool.tolist()) and (a == b).all() and not (a == c).all()
    def radius(sel): return float(np.max(np.min(((E[pool][:, None, :] - E[sel][None]) ** 2).sum(-1), axis=1)))
    rr = np.mean([radius(np.random.default_rng(i).choice(pool, n, replace=False)) for i in range(5)]); assert radius(a) < rr, (radius(a), rr)       # smaller covering radius than random
    assert set(kcenter_pick(pool[:10], 10, ('x',), embed=E).tolist()) == set(pool[:10].tolist()) and len(kcenter_pick(pool, 0, ('x',), embed=E)) == 0

@_t('V37-04 draw_split sel=kcenter: same per-state / per-crop counts as the random draw, rows come from the same pools, verify_split accepts; random path unchanged')
def _():
    X, st, cr, sts = _p4_fixture(6, 400); pools = build_pools(st, cr); g = globals(); saved = g['KC_EMBED']
    try:
        g['KC_EMBED'] = build_kc_embedding(X, dims=6, fit_n=1500); crops = np.asarray(cr)
        for tg, B, ss in (('ARG', 300, ('S00', 'S02')), ('ARG', 300, ('S01',)), ('ARG', 300, ('S00', 'S03', 'S04', 'S05'))):
            a = draw_split(tg, B, ss, 0, pools, cr); b = draw_split(tg, B, ss, 0, pools, cr, sel='kcenter'); v = verify_split(b, tg, B, ss, st, cr); assert v['ok'], v['msgs']
            ia, ib = np.concatenate([a['train_pos'], a['val_pos']]), np.concatenate([b['train_pos'], b['val_pos']]); assert len(set(ib.tolist())) == B
            for s in ss:
                for c in set(crops): assert int(((st[ia] == s) & (crops[ia] == c)).sum()) == int(((st[ib] == s) & (crops[ib] == c)).sum())
            assert set(ib.tolist()) != set(ia.tolist()) or len(ss) == 0
        assert a['sel'] == 'random' and b['sel'] == 'kcenter'
    finally: g['KC_EMBED'] = saved

@_t('V37-05 plan: common-pool / k-center / aug_on cells (shared runs not duplicated, common combos inside the common pool, aug_on pending until approved, distinct run ids); base plan unchanged')
def _():
    X, st, cr, sts = _p4_fixture(12, 150); cap = make_cap(st, cr); g = globals()
    names = ['BUDGETS', 'K_LIST', 'MIN_STATE_ROWS', 'TARGETS_TO_RUN', 'SELECTION_ARMS', 'MIN_KPOINTS', 'MIN_COMBOS', 'DIVERSITY_DIST', 'K_MAX', 'AUG_BUDGETS', 'AUG_TARGETS', 'KC_TARGETS']; saved = {n_: g.get(n_) for n_ in names}; ap = dict(APPROVED); APPROVED['aug_on'] = False
    try:
        g.update(BUDGETS=[300, 600], K_LIST=[1, 2, 3], MIN_STATE_ROWS=100, TARGETS_TO_RUN=['ARG'], MIN_KPOINTS=3, MIN_COMBOS=5, K_MAX={'ARG': 8, 'USA': 8, 'CHN': 2, 'AUS': 2, 'FRA': 8}, AUG_BUDGETS=[300, 600], AUG_TARGETS=['ARG'], KC_TARGETS=['ARG'], DIVERSITY_DIST=None)
        g['SELECTION_ARMS'] = ['random']; c0, _g0 = build_plan(cap); g['SELECTION_ARMS'] = ['random', 'common', 'kcenter', 'aug_on']; c1, _g1 = build_plan(cap)
        base = [c for c in c1 if c.get('sel', 'random') == 'random' and c.get('aug', 'off') == 'off' and 'random' in c['arms']]; assert {(c['k'], c['budget'], c['states']) for c in base} == {(c['k'], c['budget'], c['states']) for c in c0}
        ids = [make_run_id(c, 0) for c in c1]; assert len(ids) == len(set(ids)), 'duplicate run ids'
        for B in (300, 600):
            commonpool = set.intersection(*[set(pool_for('ARG', B2, 2, cap)) for B2 in (300, 600)])
            for c in c1:
                if 'common' in c['arms'] and c['budget'] == B: assert set(c['states']) <= commonpool and combo_ok(c['states'], 'ARG', B, cap)
        shared = [c for c in c1 if 'common' in c['arms'] and 'random' in c['arms']]; only = [c for c in c1 if c['arms'] == ('common',)]; assert len({_cell_key(c) for c in c1}) == len(c1)
        kc = [c for c in c1 if c.get('sel') == 'kcenter']; assert kc and all(c['arms'] == ('kcenter',) for c in kc) and all(sum(1 for x in kc if (x['budget'], x['k']) == (c['budget'], c['k'])) <= (KC_COMBOS if c['k'] > 1 else 99) for c in kc)
        au = [c for c in c1 if c.get('aug') == 'on']; assert au and all(c['pending'] for c in au)
        APPROVED['aug_on'] = True; c2, _g2 = build_plan(cap); assert not any(c['pending'] for c in c2 if c.get('aug') == 'on' and c['target'] != 'AUS')
        assert 'sel-kcenter' in make_run_id(kc[0], 0) and 'aug-on' in make_run_id(au[0], 0) and 'sel-' not in make_run_id(base[0], 0) and 'aug-off' in make_run_id(base[0], 0)
    finally: g.update(saved); APPROVED.clear(); APPROVED.update(ap)

@_t('V37-06 paired contrast: kcenter - random on the SAME combinations; unpaired combinations are ignored; units overlap -> labelled descriptive')
def _():
    rows = []
    for i, s in enumerate(['A-B', 'A-C', 'B-C', 'C-D', 'D-E']):
        for arm, f in (('random', .60 + .01 * i), ('kcenter', .58 + .01 * i)):
            for sd in (0, 1): rows.append(dict(run_id=f'{arm}{i}{sd}', status='ok', composition=BASE_MODE, arms=arm, target='ARG', budget=1000, k=2, seed=sd, states=s, pool_size=6, n_total_combos=15, key_n=100, target_crop_n=100, n_target_classes_train_ge30=3, diversity_score=np.nan, diversity_pct=np.nan, main_f1=f + .001 * sd))
    rows.append(dict(rows[0], run_id='extra', states='X-Y', arms='kcenter')); d = pd.DataFrame(rows)
    ct = arm_paired_contrast(d, 'main_f1', 'kcenter', 'random'); assert len(ct) == 1 and ct.iloc[0].n_pairs == 5 and abs(ct.iloc[0]['diff'] + .02) < 1e-9 and 'paired' in ct.iloc[0].contrast

def _v38_cells():
    X, st, cr, sts = _p4_fixture(12, 150); cap = make_cap(st, cr); g = globals()
    names = ['BUDGETS', 'K_LIST', 'MIN_STATE_ROWS', 'TARGETS_TO_RUN', 'SELECTION_ARMS', 'MIN_KPOINTS', 'MIN_COMBOS', 'DIVERSITY_DIST', 'K_MAX', 'AUG_BUDGETS', 'AUG_TARGETS', 'KC_TARGETS', 'AUG_CONTROL', 'KC_SEEDS', 'SEEDS', 'PLAN_CELLS_RUN']; saved = {n_: g.get(n_) for n_ in names}; ap = dict(APPROVED); APPROVED['aug_on'] = True
    g.update(BUDGETS=[300, 600], K_LIST=[1, 2, 3], MIN_STATE_ROWS=100, TARGETS_TO_RUN=['ARG'], MIN_KPOINTS=3, MIN_COMBOS=5, K_MAX={'ARG': 8, 'USA': 8, 'CHN': 2, 'AUS': 2, 'FRA': 8}, AUG_BUDGETS=[300, 600], AUG_TARGETS=['ARG'], KC_TARGETS=['ARG'], DIVERSITY_DIST=None, SELECTION_ARMS=['random', 'common', 'kcenter', 'aug_on'])
    return g, saved, ap, cap

@_t('V38-01 plan: aug_copy cells mirror aug_on (same states / budgets / k, aug="copy", distinct run ids, own arm); AUG_CONTROL=False removes them; plan_for_seed drops k-center only for seeds outside KC_SEEDS; the v3.7 run ids are unchanged')
def _():
    g, saved, ap, cap = _v38_cells()
    try:
        g['AUG_CONTROL'] = False; c_off, _x = build_plan(cap); g['AUG_CONTROL'] = True; c_on, _x = build_plan(cap)
        assert not [c for c in c_off if c.get('aug') == 'copy'] and [c for c in c_on if c.get('aug') == 'copy']
        au = [c for c in c_on if c.get('aug') == 'on']; cp = [c for c in c_on if c.get('aug') == 'copy']
        assert sorted((c['target'], c['budget'], c['k'], c['states']) for c in au) == sorted((c['target'], c['budget'], c['k'], c['states']) for c in cp) and all(c['arms'] == ('aug_copy',) for c in cp)
        ids = [make_run_id(c, 0) for c in c_on]; assert len(ids) == len(set(ids)) and all('aug-copy' in make_run_id(c, 0) for c in cp)
        assert [make_run_id(c, s) for c in c_off for s in (0, 1)] == [make_run_id(c, s) for c in c_on if c.get('aug') != 'copy' for s in (0, 1)], 'v3.7 run ids must not change'       # existing results.csv stays valid
        g['KC_SEEDS'] = [0, 1]; kc = [c for c in c_on if c.get('sel') == 'kcenter']; assert kc
        for sd, has in ((0, True), (1, True), (2, False), (3, False), (4, False)):
            pl = plan_for_seed(sd, c_on); assert (any(c.get('sel') == 'kcenter' for c in pl)) == has, sd; assert len(pl) == len(c_on) - (0 if has else len(kc))
            assert len([c for c in pl if c.get('aug') == 'copy']) == len(cp)
    finally: g.update(saved); APPROVED.clear(); APPROVED.update(ap)

def _v38_row(cell, seed, f1=0.7, **kw):
    r = {c: '' for c in COLUMNS}; r.update(run_id=make_run_id(cell, seed), status='ok', seed=seed, target=cell['target'], budget=cell['budget'], k=cell['k'], states='-'.join(cell['states']), arms='+'.join(cell['arms']), aug=cell.get('aug', 'off'), sample_sel=cell.get('sel', 'random'), main_f1=f1, primary_f1=f1, epochs_run=20, composition=cell['composition']); r.update(kw); return r

@_t('V38-02 plan_consistency_check: results of the current plan pass; a run id that is not in the plan (design drift) is reported as an orphan and blocks; k-center rows of seeds 0-1 stay valid')
def _():
    g, saved, ap, cap = _v38_cells()
    try:
        g['AUG_CONTROL'] = True; g['KC_SEEDS'] = [0, 1]; g['SEEDS'] = [0, 1, 2]; cells, _x = build_plan(cap); g['PLAN_CELLS_RUN'] = cells
        with tempfile.TemporaryDirectory() as td:
            p = os.path.join(td, 'results.csv'); assert plan_consistency_check(p)['ok']
            for c in cells[:5] + [c for c in cells if c.get('sel') == 'kcenter'][:2]:
                for sd in (0, 1): append_row(p, _v38_row(c, sd))
            r = plan_consistency_check(p); assert r['ok'] and r['existing'] == 14 and not r['orphans'], r
            kc = [c for c in cells if c.get('sel') == 'kcenter'][0]; append_row(p, _v38_row(kc, 2)); r = plan_consistency_check(p); assert not r['ok'] and len(r['orphans']) == 1 and 'sel-kcenter' in r['orphans'][0], r        # k-center at seed 2 is NOT in the plan
            g['KC_SEEDS'] = [0, 1, 2]; assert plan_consistency_check(p)['ok']
            g['KC_SEEDS'] = [0, 1]; d = dict(cells[0], budget=cells[0]['budget'] + 1); append_row(p, _v38_row(d, 0)); assert not plan_consistency_check(p)['ok']
    finally: g.update(saved); APPROVED.clear(); APPROVED.update(ap)

@_t('V38-03 reml_random_intercept recovers a simulated fixed effect and the variance components (own implementation, no statsmodels); equals ordinary least squares when the group variance is 0; mixed_model_report runs on a random-arm table')
def _():
    rng = np.random.default_rng(1); G, m = 60, 6; grp = np.repeat(np.arange(G), m); x = rng.normal(0, 1, G * m); u = rng.normal(0, 2.0, G)[grp]; y = 3.0 + 1.5 * x + u + rng.normal(0, 1.0, G * m)
    X = np.column_stack([np.ones(G * m), x]); r = reml_random_intercept(y, X, grp); se = np.sqrt(np.diag(r['cov']))
    assert abs(r['beta'][1] - 1.5) < 4 * se[1] and abs(np.sqrt(r['s2u']) - 2.0) < 0.6 and abs(np.sqrt(r['s2e']) - 1.0) < 0.2, (r['beta'], r['s2u'], r['s2e'])
    # exact cross-check: GLS with the estimated variances, built the slow way (dense covariance)
    V = np.zeros((G * m, G * m))
    for gi in range(G): ix = np.where(grp == gi)[0]; V[np.ix_(ix, ix)] = r['s2u']
    V += r['s2e'] * np.eye(G * m); Vi = np.linalg.inv(V); beta = np.linalg.solve(X.T @ Vi @ X, X.T @ Vi @ y); assert np.allclose(beta, r['beta'], atol=1e-6) and np.allclose(r['cov'], np.linalg.inv(X.T @ Vi @ X), atol=1e-8)
    y0 = 1.0 + 2.0 * x + rng.normal(0, 1.0, G * m); r0 = reml_random_intercept(y0, X, grp); ols = np.linalg.lstsq(X, y0, rcond=None)[0]; assert r0['s2u'] < 0.05 and np.allclose(r0['beta'], ols, atol=0.05)
    rows = []
    for ti, tg in enumerate(['ARG', 'CHN']):
        for ci in range(12):
            for k in (1, 2, 4):
                for sd in (0, 1):
                    for B in (1000, 2000):
                        rows.append(dict(run_id=f'{tg}{ci}{k}{sd}{B}', status='ok', arms='random', target=tg, budget=B, k=k, seed=sd, states=f'S{ci}-{k}', aug='off', sample_sel='random', main_f1=(60 + 3 * np.log2(k) + 2 * B / 1000 + (ci % 3)) / 100 + rng.normal(0, .01)))
    mm = mixed_model_report(pd.DataFrame(rows)); b = mm[(mm.term == 'log2(k)')].set_index('target'); assert set(b.index) == {'ARG', 'CHN'} and (abs(b.coef - 3.0) < 1.0).all() and (b.p < 0.05).all(), b

@_t('V38-04 single_state_report: average gain, worst-case gain and "beats the best single state" computed by hand on a small table')
def _():
    rows = []
    def add(states, k, f, sd=0): rows.append(dict(run_id=f'{states}{k}{sd}{f}', status='ok', arms='random', target='ARG', budget=1000, k=k, seed=sd, states=states, aug='off', sample_sel='random', main_f1=f))
    for s_, f in (('A', .50), ('B', .60), ('C', .70)): add(s_, 1, f); add(s_, 1, f + .02, sd=1)                      # seed-averaged singles: 0.51 0.61 0.71
    for s_, f in (('A-B', .65), ('A-C', .80), ('B-C', .55), ('A-D', .72)): add(s_, 2, f)
    r = single_state_report(pd.DataFrame(rows), n_boot=200).iloc[0]
    assert r.n_single == 3 and r.n_combos == 4 and abs(r.single_mean - 61) < 1e-9 and abs(r.single_best - 71) < 1e-9 and r.single_best_state == 'C' and abs(r.combo_mean - 68) < 1e-9
    assert abs(r.avg_gain - 7) < 1e-9 and abs(r.worst_case_gain - (55 - 61)) < 1e-9 and abs(r.share_beating_best_single - .5) < 1e-9 and abs(r.worst_vs_best_single - (55 - 71)) < 1e-9 and r.avg_gain_ci_lo < 7 < r.avg_gain_ci_hi

@_t('V38-05 replication_table: same sign in both seed groups and a confirmatory interval that excludes 0 = PASS; a reversed or null confirmatory group = NOT REPLICATED (never dropped)')
def _():
    rng = np.random.default_rng(2)
    def table(slope_conf):
        rows = []
        for sd in range(5):
            sl = 3.0 if sd < 2 else slope_conf
            for ci in range(14):
                for k in (1, 2, 4, 8):
                    for B in (1000, 2000): rows.append(dict(run_id=f'{sd}{ci}{k}{B}', status='ok', composition=BASE_MODE, pool_size=16, n_total_combos=(16 if k == 1 else 10), key_n=100, target_crop_n=100, n_target_classes_train_ge30=3, diversity_score=np.nan, diversity_pct=np.nan, arms='random', target='ARG', budget=B, k=k, seed=sd, states=f'S{ci}-{k}', aug='off', sample_sel='random', main_f1=(60 + sl * np.log2(k) + ci % 4) / 100 + rng.normal(0, .01)))
        return pd.DataFrame(rows)
    t = replication_table(table(3.0)); c1 = t[t.claim.str.startswith('C1')]; assert set(c1.verdict) == {'PASS'}, c1
    t = replication_table(table(-3.0)); c1 = t[t.claim.str.startswith('C1')]; assert set(c1.verdict) == {'NOT REPLICATED (sign differs)'}, c1
    t = replication_table(table(0.0)); c1 = t[t.claim.str.startswith('C1')]; assert c1.verdict.iloc[0].startswith('NOT REPLICATED'), c1.verdict.tolist()

@_t('V38-06 write_run_environment never overwrites an existing seed file (a resumed run writes a separate, time-stamped file)')
def _():
    with tempfile.TemporaryDirectory() as td:
        write_run_environment(td, 0); p = os.path.join(td, 'run_environment_seed0.json'); assert os.path.exists(p); a = open(p).read(); write_run_environment(td, 0)
        assert open(p).read() == a and len([f for f in os.listdir(td) if f.startswith('run_environment_seed0__resume_')]) == 1

_new = TEST_REPORT[_n0:]
for name, res, msg in _new: print(f'  [{res}] {name}' + (f'\n         -> {msg}' if msg else ''))
N_FAIL_A2 = sum(1 for r in _new if r[1] != 'PASS'); TESTS_A_PASSED = bool(TESTS_A_PASSED and N_FAIL_A2 == 0)
print('\nTEST GATE PART A2 (data functions):', 'ALL PASSED' if N_FAIL_A2 == 0 else f'{N_FAIL_A2} FAILED -- training is BLOCKED')

# ======================================================================================
# LEVEL 2 tests (added): France-source data, instructor gate, USA target, CHN descriptive, Level-1 setup equality
# ======================================================================================
@_t('L2-01 USA target: draw_split / verify_split accept a USA split (3 key crops 15% each + non-key remainder); a tampered split is caught')
def _():
    rows = {s: {'corn': 900, 'soybeans': 900, 'wheat': 900, 'other': 1500} for s in ('R1', 'R2', 'R3')}
    st = np.concatenate([np.full(sum(v.values()), s) for s, v in rows.items()]); cr = np.concatenate([np.array([c for c, n in v.items() for _ in range(n)]) for v in rows.values()])
    pools = {(s, c): np.where((st == s) & (cr == c))[0] for s in rows for c in set(cr)}
    sp = draw_split('USA', 2000, ('R1', 'R2'), 3, pools, cr); v = verify_split(sp, 'USA', 2000, ('R1', 'R2'), st, cr); assert v['ok'], v['msgs']
    allp = np.concatenate([sp['train_pos'], sp['val_pos']]); assert len(allp) == 2000 and len(set(allp.tolist())) == 2000
    for c in ('corn', 'soybeans', 'wheat'): assert int((cr[allp] == c).sum()) == int(0.15 * 2000), c
    bad = dict(sp); bad['train_pos'] = sp['train_pos'][:-1]; assert not verify_split(bad, 'USA', 2000, ('R1', 'R2'), st, cr)['ok']

@_t('L2-02 CHN is descriptive only: k>=4 infeasible, k<=2 feasible with the exemption; CHN is not a claim target; claim targets are exactly ARG and USA')
def _():
    assert 'CHN' in DESCRIPTIVE_TARGETS and 'CHN' in EXEMPT_RULES and sorted(CLAIM_TARGETS) == ['ARG', 'USA'] and 'CHN' not in CLAIM_TARGETS
    assert K_MAX['CHN'] == 2 and SOURCE == 'FRA' and len(SOURCE_UNITS) == 13 and SOURCE not in TARGETS_TO_RUN

@_t('L2-03 Level-1 training setup is unchanged: LEVEL1_TRAINING_SETUP equals the live configuration (lr, batch, epochs, patience, head, val fraction, seeds, budgets, k list)')
def _():
    g = globals(); assert 'LEVEL1_TRAINING_SETUP' in g
    for k, v in LEVEL1_TRAINING_SETUP.items():
        assert k in g, k
        assert g[k] == v, f'{k}: live {g[k]!r} != Level 1 {v!r}'

@_t('L2-04 instructor gate: refuses when not approved, when no hash is pasted, and when the hash differs; accepts only the exact current hash; hash changes when a locked setting changes OR the planned runs change')
def _():
    g = globals(); saved = (g['INSTRUCTOR_APPROVED'], g['APPROVED_DESIGN_SHA256'], g.get('PLAN_CELLS_RUN'))
    def refused():
        try:
            with contextlib.redirect_stdout(io.StringIO()): instructor_gate()
        except RuntimeError as e: return 'STOP' in str(e)
        return False
    try:
        g['PLAN_CELLS_RUN'] = [dict(target='ARG', budget=1000, k=2, states=('11', '24'), composition=BASE_MODE, sel='random', arms=('random',), aug='off', pending=False, combo=0)]
        h = design_hash(); assert len(h) == 64
        for ok_, hh in ((False, ''), (True, ''), (True, 'f' * 64), (False, h)):
            g['INSTRUCTOR_APPROVED'], g['APPROVED_DESIGN_SHA256'] = ok_, hh; assert refused(), (ok_, hh[:6])
        g['INSTRUCTOR_APPROVED'], g['APPROVED_DESIGN_SHA256'] = True, h
        with contextlib.redirect_stdout(io.StringIO()): assert instructor_gate() == h
        old = g['MAX_EPOCHS']
        try: g['MAX_EPOCHS'] = old + 1; assert design_hash() != h, 'hash ignores a locked setting'; assert refused()
        finally: g['MAX_EPOCHS'] = old
        g['PLAN_CELLS_RUN'] = [dict(g['PLAN_CELLS_RUN'][0], states=('11', '27'))]; assert design_hash() != h, 'hash ignores a change of the planned runs'; assert refused()
        g['PLAN_CELLS_RUN'] = [dict(target='ARG', budget=1000, k=2, states=('11', '24'), composition=BASE_MODE, sel='random', arms=('random',), aug='off', pending=False, combo=0)]
        assert design_hash() == h, 'hash not reproducible'
    finally: g['INSTRUCTOR_APPROVED'], g['APPROVED_DESIGN_SHA256'] = saved[0], saved[1]; (g.__setitem__('PLAN_CELLS_RUN', saved[2]) if saved[2] is not None else g.pop('PLAN_CELLS_RUN', None))

@_t('L2-05 France assignment (synthetic 64-block table): rows inside blocks get the block region, rows between blocks are excluded, labels are checked row by row; a swapped block order or a wrong label STOPs')
def _():
    g = globals(); keys = ['FRA_BLOCKS', 'FRA_ASSIGNED_PER_REGION', 'FRA_ROW_MAP_ROWS']; saved = {k: g[k] for k in keys}
    try:
        L = 3; gap_after = {5, 20}; nb = []; pos = 0
        for n_, (t_, r_, _lo, _hi) in enumerate(saved['FRA_BLOCKS']):
            nb.append((t_, r_, pos, pos + L - 1)); pos += L + (2 if n_ in gap_after else 0)
        N = pos; per = {}
        for t_, r_, lo, hi in nb: per[r_] = per.get(r_, 0) + (hi - lo + 1)
        g['FRA_BLOCKS'] = nb; g['FRA_ASSIGNED_PER_REGION'] = dict(sorted(per.items())); g['FRA_ROW_MAP_ROWS'] = N
        lab = np.full(N, CROP_ID['other'])
        for t_, r_, lo, hi in nb: lab[lo:hi + 1] = CROP_ID[RAW_TO_CROP[t_]]
        with contextlib.redirect_stdout(io.StringIO()): meta_, rep_ = build_fra_meta(lab)
        assert len(meta_) == N and int(meta_['assigned'].sum()) == 64 * L and rep_['excluded_outside_blocks'] == 4 and set(meta_.loc[meta_['assigned'], 'state']) == set(SOURCE_UNITS)
        assert (meta_.loc[~meta_['assigned'], 'state'] == 'NA').all()
        bad = lab.copy(); bad[nb[7][2]] = (bad[nb[7][2]] + 1) % 7
        try:
            with contextlib.redirect_stdout(io.StringIO()): build_fra_meta(bad)
            raise AssertionError('wrong label accepted')
        except RuntimeError as e: assert 'label' in str(e)
        except AssertionError as e:
            if 'wrong label accepted' in str(e): raise
            assert 'label' in str(e)
        g['FRA_BLOCKS'] = [nb[1], nb[0]] + nb[2:]
        try:
            with contextlib.redirect_stdout(io.StringIO()): build_fra_meta(lab)
            raise RuntimeError('swapped order accepted')
        except (RuntimeError, AssertionError) as e: assert 'accepted' not in str(e)
    finally: g.update(saved)

@_t('L2-06 USA clean score (L2-D11): same four classes in every run; TRUE rice/sugarcane/cotton removed; their predictions -> other; hand-computed; independent of the training mix; both scorers agree; v1 kept for provenance')
def _():
    I = CROP_ID; yt = np.array([I['corn']] * 4 + [I['soybeans']] * 4 + [I['wheat']] * 4 + [I['other']] * 4 + [I['rice']] * 2 + [I['cotton']])
    yp = np.array([I['corn']] * 4 + [I['soybeans']] * 3 + [I['sugarcane']] + [I['wheat']] * 4 + [I['other']] * 4 + [I['rice']] * 2 + [I['other']])          # one soybean predicted as sugarcane -> counts as 'other'
    r = fixed4_scores(yt, yp)
    # 16 samples scored (3 true rice/cotton dropped): corn F1 1, wheat F1 1, soybeans tp3 fn1 fp0 -> 6/7, other tp4 fp1 (the soybean) fn0 -> 8/9
    exp = (1 + 1 + 6 / 7 + 8 / 9) / 4; assert abs(r['f1'] - exp) < 1e-12 and abs(r['f1_numpy'] - exp) < 1e-12 and r['n_dropped'] == 3 and r['n_scored'] == 16 and r['n_pred_reassigned'] == 1, (r, exp)
    v1 = fixed4_v1_scores(yt, yp)['f1']; assert abs(v1 - exp) > 1e-3, 'clean and v1 must differ when true rice / cotton exist'
    sc = score_run('USA', yt, yp, np.array([0, 100, 100, 0, 0, 0, 100])); sc2 = score_run('USA', yt, yp, np.array([0, 100, 100, 50, 0, 0, 100]))
    assert abs(sc['fixed4_clean_f1'] - exp) < 1e-12 and sc['fixed4_clean_f1'] == sc2['fixed4_clean_f1'] and abs(sc['fixed4_f1'] - v1) < 1e-12
    assert sc['cov_cond_paper_f1'] != sc2['cov_cond_paper_f1'], 'the ORIGINAL rule depended on the training mix (the problem L2-D9 / D11 remove)'
    assert main_metric_value('USA', .5, .68, .6, .7) == (.7, 'fixed4_clean_macro_f1') and main_metric_value('ARG', .7, .8, .1, .2)[0] == .7
    try: fixed4_scores(np.array([I['corn']] * 3 + [I['rice']] * 3), np.array([I['corn']] * 6)); raise RuntimeError('no stop')
    except AssertionError as e: assert 'every class' in str(e)
    assert np.isnan(score_run('ARG', np.array([I['soybeans']] * 5 + [I['other']] * 5), np.array([I['soybeans']] * 5 + [I['other']] * 5), np.array([0, 50, 0, 0, 0, 0, 50]))['fixed4_clean_f1'])

@_t('L2-07 re-scoring ALL seeds from saved predictions: handles rows still on the original rule AND rows on L2-D9; reproduces their stored numbers first; keeps every earlier value and the original file; all-or-nothing; idempotent; the run cell refuses a mixed file')
def _():
    I = CROP_ID; rng = np.random.default_rng(3); cls = [I['corn'], I['soybeans'], I['wheat'], I['other'], I['rice'], I['cotton']]
    with tempfile.TemporaryDirectory() as td:
        os.makedirs(os.path.join(td, 'runs')); rows = []
        for n_, kind in enumerate(['cov', 'v1', 'v1', 'cov']):
            yt = rng.choice(cls, 500); yp = np.where(rng.random(500) < .6, yt, rng.choice(cls + [I['sugarcane']], 500)); tc = np.array([0, 200, 200, 100, 0, 0, 100 if n_ % 2 else 0]); rid = f'USA|t{n_}'; present = np.unique(yt)
            np.savez_compressed(_npz_path(td, rid), y_true=yt.astype(np.uint8), y_pred=yp.astype(np.uint8), train_counts=tc)
            _p, _r, fs = _sk_prf(yt, yp, present); cov = [k for k, c_ in enumerate(present) if tc[c_] >= COVERAGE_MIN_TRAIN]
            row = dict(run_id=rid, status='ok', target='USA', states='11-24', seed=n_, sevenclass_f1=float(f1_score(yt, yp, average='macro', labels=present, zero_division=0)))
            if kind == 'cov': row.update(main_f1=float(np.mean(fs[cov])), main_metric='coverage_conditional_macro_f1')
            else: row.update(main_f1=fixed4_v1_scores(yt, yp)['f1'], fixed4_f1=fixed4_v1_scores(yt, yp)['f1'], main_f1_before_fixed4=.5, main_metric='fixed4_macro_f1')
            rows.append(row)
        rows.append(dict(run_id='ARG|x', status='ok', target='ARG', states='11', seed=0, main_f1=.5, main_metric='binary_key_crop_macro_f1', sevenclass_f1=.1))
        path = os.path.join(td, 'results.csv'); pd.DataFrame(rows).to_csv(path, index=False); orig = sha256_file(path)
        try: scoring_consistency_check(path); raise RuntimeError('mixed file accepted')
        except AssertionError as e: assert 'RE-SCORING' in str(e)
        with contextlib.redirect_stdout(io.StringIO()): rep_ = rescore_usa_fixed4(td)
        assert len(rep_) == 4 and sorted(rep_.old_rule.unique()) == ['coverage_conditional_macro_f1', 'fixed4_macro_f1'] and (rep_.n_true_dropped > 0).all()
        bak = [f for f in os.listdir(td) if f.startswith('results.csv.before_rescore_')]; assert len(bak) == 1 and sha256_file(os.path.join(td, bak[0])) == orig, 'the backup is not the original file'
        d2 = pd.read_csv(path); u = d2[d2.target == 'USA']; assert (u.main_metric == 'fixed4_clean_macro_f1').all() and np.allclose(u.main_f1, u.fixed4_clean_f1) and u.main_f1_before_clean.notna().all() and u.fixed4_f1.notna().all()
        assert u[u.run_id.isin(['USA|t0', 'USA|t3'])].main_f1_before_fixed4.notna().all() and (u[u.run_id.isin(['USA|t1', 'USA|t2'])].main_f1_before_fixed4 == .5).all(), 'an earlier value was lost or overwritten'
        assert d2[d2.target == 'ARG'].main_f1.iloc[0] == .5 and d2[d2.target == 'ARG'].main_metric.iloc[0] == 'binary_key_crop_macro_f1', 'a non-USA row was touched'
        assert scoring_consistency_check(path) is True
        with contextlib.redirect_stdout(io.StringIO()): assert len(rescore_usa_fixed4(td)) == 0                      # idempotent
        assert len(json.load(open(os.path.join(td, 'rescore_log.json')))) == 1
        d3 = pd.read_csv(os.path.join(td, bak[0])); d3.loc[1, 'main_f1'] += 0.01; p3 = os.path.join(td, 'r3'); os.makedirs(os.path.join(p3, 'runs')); [shutil.copy(os.path.join(td, 'runs', f), os.path.join(p3, 'runs', f)) for f in os.listdir(os.path.join(td, 'runs'))]
        d3.to_csv(os.path.join(p3, 'results.csv'), index=False); h3 = sha256_file(os.path.join(p3, 'results.csv'))
        try:
            with contextlib.redirect_stdout(io.StringIO()): rescore_usa_fixed4(p3)
            raise RuntimeError('mismatch accepted')
        except AssertionError as e: assert 'old main score' in str(e)
        assert sha256_file(os.path.join(p3, 'results.csv')) == h3 and not [f for f in os.listdir(p3) if f.startswith('results.csv.before')], 'a failed re-score changed something'

@_t('L2-08 the scoring rule is part of the design hash (changing the dropped classes or the rule changes the hash)')
def _():
    g = globals(); saved = (g.get('PLAN_CELLS_RUN'), list(g['USA_DROP_TRUE_CLASSES']))
    try:
        g['PLAN_CELLS_RUN'] = [dict(target='USA', budget=1000, k=1, states=('11',), composition=BASE_MODE, sel='random', arms=('random',), aug='off', pending=False, combo=0)]
        h = design_hash(); g['USA_DROP_TRUE_CLASSES'] = ['rice']; assert design_hash() != h
    finally: g['USA_DROP_TRUE_CLASSES'] = saved[1]; (g.__setitem__('PLAN_CELLS_RUN', saved[0]) if saved[0] is not None else g.pop('PLAN_CELLS_RUN', None))

## 7. Acquire data + authors' code (pinned commit)

In [ ]:
# ======================================================================================
# [COLAB] DATA + CODE: use the files already in Drive; download only what is missing.  The authors' repo is PINNED to the commit in the spec.
# ======================================================================================
NEEDED = [SOURCE] + list(TARGETS_TO_RUN)
def _have(c): return bool(glob.glob(os.path.join(DATA_DIR, '**', f'median_S2_{c}_month1-12_day5.npz'), recursive=True))
if not all(_have(c) for c in NEEDED):
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='x-ytong/CropGlobe', repo_type='dataset', local_dir=DATA_DIR)
assert all(_have(c) for c in NEEDED), f'missing Sentinel-2 median files for {[c for c in NEEDED if not _have(c)]} in {DATA_DIR}'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/x-ytong/CropNet.git', REPO_DIR], check=True)
def _git(*a): return subprocess.run(['git', '-C', REPO_DIR, *a], capture_output=True, text=True)
if _git('rev-parse', 'HEAD').stdout.strip() != EXPECTED_CROPNET_COMMIT:
    _git('fetch', '--all', '--tags')
    if _git('cat-file', '-e', EXPECTED_CROPNET_COMMIT + '^{commit}').returncode != 0:
        raise RuntimeError(f'STOP -- spec conflict: commit {EXPECTED_CROPNET_COMMIT} does not exist in x-ytong/CropNet. Report; do not substitute another commit silently.')
    r = _git('checkout', '--detach', EXPECTED_CROPNET_COMMIT); assert r.returncode == 0, r.stderr
GIT_COMMIT = _git('rev-parse', 'HEAD').stdout.strip(); assert GIT_COMMIT == EXPECTED_CROPNET_COMMIT, f'repo at {GIT_COMMIT}, spec says {EXPECTED_CROPNET_COMMIT}'
_dirty = _git('status', '--porcelain', '--untracked-files=no').stdout.strip()
assert not _dirty, f'the authors\' code must be imported UNCHANGED, but the repo has local modifications:\n{_dirty}'
assert os.path.isfile(os.path.join(REPO_DIR, 'models', 'CropNet.py')), 'authors\' repo is incomplete'
print('authors\' repo pinned at', GIT_COMMIT, '(clean, unmodified)')

# ---- namespace guard: never mix with, or overwrite, results of an earlier notebook (spec 7)
def check_namespace(write=False):
    mp, rp = os.path.join(OUT_DIR, 'manifest.json'), os.path.join(OUT_DIR, 'results.csv')
    snap = {k: json.loads(json.dumps(globals()[k], default=str)) for k in LOCKED}
    if os.path.exists(mp):
        m = json.load(open(mp))
        if m.get('spec') != SPEC_TAG:
            raise RuntimeError(f'STOP: {OUT_DIR} holds a manifest from an EARLIER notebook (spec={m.get("spec")}). Spec 7 says never touch earlier outputs: rename/move that folder (keep it), then re-run.')
        diff = [k for k in LOCKED if m['locked'].get(k) != snap[k]]
        if diff: raise RuntimeError(f'STOP: locked design differs from the manifest in {diff}. Use a new namespace instead of mixing designs.')
        if write:                                                                  # v3.8: every design amendment is recorded (dated) in the manifest the first time it is seen; nothing changes silently
            _have = {a_.get('id') for a_ in m.get('amendments', [])}; _new = [dict(a_, recorded=datetime.datetime.now().isoformat(timespec='seconds'), notebook=NOTEBOOK_VERSION) for a_ in V38_AMENDMENTS if a_['id'] not in _have]
            if _new: m['amendments'] = m.get('amendments', []) + _new; atomic_write(mp, lambda f: f.write(json.dumps(m, indent=1)), mode='w'); print('manifest: recorded amendments', [a_['id'] for a_ in _new])
    elif os.path.exists(rp):
        raise RuntimeError(f'STOP: {rp} exists without a v3-spec manifest (results of an earlier notebook). Move that folder aside (do not delete) and re-run.')
    elif write:
        man = dict(spec=SPEC_TAG, experiment=EXPERIMENT_VERSION, created=datetime.datetime.now().isoformat(timespec='seconds'), locked=snap, cropnet_commit=GIT_COMMIT,
                   table2=TABLE2, table3_reference_context_only=TABLE3_REF_MF1,
                   amendments=[dict(a_, recorded=datetime.datetime.now().isoformat(timespec='seconds'), notebook=NOTEBOOK_VERSION) for a_ in V38_AMENDMENTS])
        atomic_write(mp, lambda f: f.write(json.dumps(man, indent=1)), mode='w')
    return True
check_namespace(write=False)
print('Colab data + code ready. Sentinel-2 files for', NEEDED, 'found. Namespace OK:', OUT_DIR)

## 8. Test gate, part B (our remap vs the authors' released function)

In [ ]:
# ======================================================================================
# PRE-TRAINING TEST GATE, PART B -- a REAL gate.  Our `existing_release` remap must equal the AUTHORS' OWN util/label.py::remap_prediction on 5000 random predictions
# for ARG, CHN, AUS and FRA, or training stays blocked.  All three named rules are compared with the authors' function and the full table is printed and saved.
# The authors' signature is (test_pred, target, crop_label): called exactly as remap_prediction(list(y_pred), TARGET, CROP_ID), uppercase target, NO fallback spellings.
# ======================================================================================
import inspect, importlib
for _m in ('util', 'util.label'):
    if _m in sys.modules and REPO_DIR not in str(getattr(sys.modules[_m], '__file__', '') or ''): del sys.modules[_m]
if REPO_DIR in sys.path: sys.path.remove(REPO_DIR)
sys.path.insert(0, REPO_DIR)
UL = importlib.import_module('util.label'); assert os.path.abspath(inspect.getfile(UL)).startswith(os.path.abspath(REPO_DIR)), 'util.label is not the authors\' module'
AUTH_REMAP = UL.remap_prediction
_sig = list(inspect.signature(AUTH_REMAP).parameters)
print('authors\' remap_prediction signature:', inspect.signature(AUTH_REMAP)); print(inspect.getsource(AUTH_REMAP))
TESTS_B_PASSED = False; AUTH_REMAP_REPORT = []; PLAN_NOTE_ARG = ''
if _sig != ['test_pred', 'target', 'crop_label']:
    raise RuntimeError(f'STOP -- the authors\' remap_prediction signature is {_sig}, not (test_pred, target, crop_label) as the spec states. Report; nothing is guessed.')

def call_authors_remap(y_pred, target):
    out = AUTH_REMAP(list(np.asarray(y_pred).tolist()), target, CROP_ID)                  # uppercase target, CROP_ID as crop_label, no retries
    out = out.tolist() if hasattr(out, 'tolist') else list(out); arr = np.asarray(out).astype(int)
    if arr.shape != (len(y_pred),): raise RuntimeError(f'authors\' remap_prediction returned shape {arr.shape} for {len(y_pred)} predictions')
    return arr

# ---- 1) the authors' function must REALLY change a probe (so a no-op can never pass for a match)
_O = CROP_ID['other']; _allc = np.arange(7); _probe_expect = {
    'ARG': [c if c != CROP_ID['wheat'] else _O for c in _allc],                                         # only wheat -> other
    'CHN': [c if c in (CROP_ID['corn'], CROP_ID['soybeans'], CROP_ID['rice']) else _O for c in _allc], # keeps ONLY corn / soybeans / rice
    'AUS': [c if c in (CROP_ID['sugarcane'], CROP_ID['cotton']) else _O for c in _allc]}                # keeps ONLY sugarcane / cotton
_probe_bad = []
for _t_, _exp in _probe_expect.items():
    _got = call_authors_remap(_allc, _t_).tolist()
    print(f'  probe {_t_}: authors map {dict(zip(CROP_ORDER, [CROP_LABELS[g] for g in _got]))}' + ('' if _got == _exp else f'   <-- expected {[CROP_LABELS[e] for e in _exp]}'))
    if _got == list(_allc): _probe_bad.append(f'{_t_}: the authors\' function returned the probe UNCHANGED (a no-op)')
    elif _got != _exp: _probe_bad.append(f'{_t_}: authors map {[CROP_LABELS[g] for g in _got]} but the spec says {[CROP_LABELS[e] for e in _exp]}')
if _probe_bad: raise RuntimeError('STOP -- the authors\' remap_prediction does not behave as the spec states. Report, do not resolve silently:\n  - ' + '\n  - '.join(_probe_bad))
print('  probes OK: the authors\' function really changes predictions (ARG wheat->other; CHN keeps only corn/soybeans/rice; AUS keeps only sugarcane/cotton)')

# ---- 2) full three-rule comparison on 5000 random predictions per target
_rng = np.random.default_rng(7)
for _t_ in ['ARG', 'CHN', 'AUS', 'FRA', 'USA']:
    yp = _rng.integers(0, 7, 5000); ref = call_authors_remap(yp, _t_); row = dict(target=_t_, n=5000)
    for name, fn in REMAPS.items(): row[f'{name}_identical'] = bool((fn(yp, _t_) == ref).all()); row[f'{name}_n_diff'] = int((fn(yp, _t_) != ref).sum())
    _mp = call_authors_remap(_allc, _t_); row['authors_class_mapping'] = ','.join(f'{CROP_LABELS[a]}->{CROP_LABELS[int(m)]}' for a, m in zip(_allc, _mp) if a != m) or '(none)'
    AUTH_REMAP_REPORT.append(row)
REP = pd.DataFrame(AUTH_REMAP_REPORT); print('\nTHREE-RULE COMPARISON vs the authors\' remap_prediction (identical on 5000 random predictions?):'); print(REP.to_string(index=False))
os.makedirs(os.path.join(OUT_DIR, 'tables'), exist_ok=True); atomic_write(os.path.join(OUT_DIR, 'tables', 'authors_remap_comparison.csv'), lambda f: REP.to_csv(f, index=False), mode='w')
_diff = lambda rule: sorted(r['target'] for r in AUTH_REMAP_REPORT if not r[f'{rule}_identical'])
print(f'\n  paper_text_v3 differs from the released code on: {_diff("paper_text_v3") or "(nowhere)"}   (expected: ARG only)')
print(f'  review_targetmap differs from the released code on: {_diff("review_targetmap") or "(nowhere)"}   (expected: FRA only; USA defines all 7 classes, so it never differs)')
print(f'  existing_release differs from the released code on: {_diff("existing_release") or "(nowhere)"}   (required: nowhere)')
_arg_map = [r for r in AUTH_REMAP_REPORT if r['target'] == 'ARG'][0]['authors_class_mapping']
PLAN_NOTE_ARG = ('paper sec.4.4 says no alignment for ARG; the released code maps wheat->other. Both are scored; no choice is made.' if 'wheat->other' in _arg_map
                 else f'paper sec.4.4 says no alignment for ARG; the released code maps: {_arg_map}. Both are scored; no choice is made.')
# ---- 3) the gate
_fail = [r['target'] for r in AUTH_REMAP_REPORT if not r['existing_release_identical']]
if _fail:
    raise RuntimeError(f'PART B FAILED -- existing_release differs from the authors\' released remap_prediction on {_fail} (see the table above / tables/authors_remap_comparison.csv). Training is BLOCKED. Report; nothing is resolved silently.')
TESTS_B_PASSED = True
TESTS_PASSED = bool(TESTS_A_PASSED and TESTS_B_PASSED)
print('\nTEST GATE: Part A', 'PASS' if TESTS_A_PASSED else 'FAIL', '| Part B', 'PASS' if TESTS_B_PASSED else 'FAIL', '->', 'training allowed after the plan approval' if TESTS_PASSED else 'TRAINING BLOCKED')

## 8b. USA row-order verification (soybeans first; exact match of every raw point to its packaged row; needs the raw USA files, otherwise prints NOT VERIFIED)

In [ ]:
# ======================================================================================
# USA ROW-ORDER VERIFICATION (instructor review, Oct 2026): "verify the USA soybean (and corn / other) ordering exactly".  Runs EARLY, before the plan.
# WHY: the Level 1 headline (Argentina soybeans) depends on every USA soybean sample being assigned to the right STATE.  The packaged USA npz has no state column; Level 1 inferred it.
# WHAT THIS CELL DOES (exact, no inference): every raw point of every raw USA file S2_USA_<crop>_<ST>.csv is matched to its packaged npz row by an exact comparison of its 5-day medians
# (the same method that verified USA wheat 23,226 / 23,226 and all of France).  It then checks, with pre-declared criteria:
#   (a) a declared binning rule reproduces the npz (>= 99% on a sample);  (b) >= 99.9% of the raw points of EVERY crop token match exactly one npz row;
#   (c) the npz label of every matched row equals the label of its crop token;  (d) each file is one contiguous, increasing block and the blocks follow (label, token A-Z, state);
#   (e) the state in each raw FILE NAME equals the state in the Level 1 USA state table (cache/usa_meta.csv from the frozen Level 1 run), row by row.
# The result is printed per crop (soybeans first) and saved to tables/usa_order_verification.json.  It is NOT a gate for Level 2 runs (Level 2 uses the USA test set whole, no USA state enters
# any Level 2 run) -- it validates the Level 1 state assignment and is reported in the closing cell.  If the raw files cannot be found the status says NOT VERIFIED; nothing is assumed.
# ======================================================================================
import os, re, json, glob, time
import numpy as np, pandas as pd

BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
N_BINS, N_BANDS = 73, 10
BIN_RULES = {                                    # doy -> bin index 0..72 (declared candidates; clipped into range)
    'floor(doy/5)':        lambda d: np.floor(d / 5.0).astype(int),
    'floor((doy-1)/5)':    lambda d: np.floor((d - 1.0) / 5.0).astype(int),
    'floor((doy+2.5)/5)':  lambda d: np.floor((d + 2.5) / 5.0).astype(int),
}
SCALES = {'raw units': 1.0, 'raw/10000': 1e-4}
LAYOUTS = {'band-major (10,73)': lambda a: a.reshape(len(a), N_BANDS, N_BINS), 'time-major (73,10)': lambda a: a.reshape(len(a), N_BINS, N_BANDS).transpose(0, 2, 1)}
MATCH_ACCEPT = 0.99
MATCH_TOL = 1e-3                                 # in raw units (reflectance integers); scaled with the value scale

def parse_name(path):
    b = re.sub(r'^[0-9a-f]{8}-\d+_', '', os.path.basename(path))            # Drive/upload hash prefix, if any
    m = re.match(r'^S2_([A-Z]{3})_(.+?)_LAMB93_R(\d+)\.csv$', b)             # France: S2_FRA_<crop>_LAMB93_R<region>.csv
    if m: return dict(country=m.group(1), crop_token=m.group(2), region_code=m.group(3))
    m = re.match(r'^S2_([A-Z]{3})_(.+?)(?:_([A-Z]{2}))?\.csv$', b)             # USA-style: S2_USA_<crop>[_<ST>].csv  (best effort: not verified on real USA files)
    return dict(country=m.group(1), crop_token=m.group(2), region_code=m.group(3)) if m else dict(country=None, crop_token=None, region_code=None)

def read_raw_points(path):
    cols = list(pd.read_csv(path, nrows=0).columns); miss = [c for c in ('system:index', 'raw_data', '.geo') if c not in cols]
    if miss: raise ValueError(f'{os.path.basename(path)}: raw layout differs from the one this matcher was written for (missing columns {miss}; columns are {cols[:12]}). Send me this list instead of forcing a match.')
    d = pd.read_csv(path, usecols=['system:index', 'raw_data', '.geo']); obs = [np.asarray(json.loads(x), dtype=float) for x in d['raw_data'].values]
    xy = np.array([json.loads(g)['coordinates'][:2] for g in d['.geo'].values], dtype=float)
    return dict(path=path, index=d['system:index'].values, obs=obs, lon=xy[:, 0], lat=xy[:, 1], **parse_name(path))

def bin_medians(obs, rule):
    # (10, 73) array of per-bin medians, NaN where the bin has no observation
    out = np.full((N_BANDS, N_BINS), np.nan)
    if len(obs) == 0: return out
    b = np.clip(BIN_RULES[rule](obs[:, 0]), 0, N_BINS - 1)
    for k in np.unique(b): out[:, k] = np.median(obs[b == k, 1:1 + N_BANDS], axis=0)
    return out

class NpzIndex:
    # sorted-column index of one (band, bin) pair per bin for fast candidate lookup
    def __init__(self, fea, layout, scale):
        a = LAYOUTS[layout](np.asarray(fea, dtype=np.float32)); self.a = a; self.scale = scale; self.order = {}; self.sorted = {}
        for b in range(N_BINS):
            col = a[:, 0, b]; o = np.argsort(col, kind='stable'); self.order[b] = o; self.sorted[b] = col[o]
    def candidates(self, med, tol):
        have = np.flatnonzero(~np.isnan(med[0])); 
        if len(have) == 0: return None, have
        b = int(have[0]); v = med[0, b] * self.scale; t = tol * self.scale + 1e-9
        lo, hi = np.searchsorted(self.sorted[b], v - t), np.searchsorted(self.sorted[b], v + t, side='right'); cand = self.order[b][lo:hi]
        if len(cand) == 0: return cand, have
        ok = np.abs(self.a[np.ix_(cand, np.arange(N_BANDS), have)] - (med[:, have] * self.scale)[None]).max(axis=(1, 2)) <= t
        return cand[ok], have

def match_points(points, fea, rule, layout, scale, tol=MATCH_TOL, max_points=None, progress=0):
    idx = NpzIndex(fea, layout, scale); rows = []; t0 = time.time(); total = sum(len(p['obs']) for p in points)
    for p in points:
        for j, obs in enumerate(p['obs'][:max_points] if max_points else p['obs']):
            med = bin_medians(obs, rule); c, have = idx.candidates(med, tol)
            if progress and rows and len(rows) % progress == 0: print(f'   ... {len(rows)}/{total} raw points matched ({time.time() - t0:.0f}s)')
            rows.append(dict(file=os.path.basename(p['path']), raw_index=p['index'][j], n_bins_used=len(have), n_match=0 if c is None else len(c), npz_row=int(c[0]) if (c is not None and len(c) == 1) else -1, lon=p['lon'][j], lat=p['lat'][j], region_code=p['region_code'], crop_token=p['crop_token']))
    return pd.DataFrame(rows)

def find_rule(points, fea, sample_per_file=200, seed=0):
    # try every declared (rule, layout, scale) on a small sample; the winner must reach MATCH_ACCEPT unique matches
    rng = np.random.default_rng(seed); sub = []
    for p in points:
        n = len(p['obs']); sel = np.sort(rng.choice(n, min(sample_per_file, n), replace=False)); sub.append(dict(p, obs=[p['obs'][i] for i in sel], index=p['index'][sel], lon=p['lon'][sel], lat=p['lat'][sel]))
    res = []
    for rule in BIN_RULES:
        for layout in LAYOUTS:
            for sc_name, sc in SCALES.items():
                m = match_points(sub, fea, rule, layout, sc); res.append(dict(rule=rule, layout=layout, scale=sc_name, unique=float((m.n_match == 1).mean()), none=float((m.n_match == 0).mean()), ambiguous=float((m.n_match > 1).mean()), n=len(m)))
    r = pd.DataFrame(res).sort_values('unique', ascending=False).reset_index(drop=True); r['accepted'] = r['unique'] >= MATCH_ACCEPT
    return r

def ordering_evidence(m):
    # for matched rows: do raw rows of one file occupy ONE contiguous, increasing block of npz rows?  (this is the join the Level 1/2 pipelines otherwise have to infer)
    out = []
    for f, g in m[m.npz_row >= 0].groupby('file'):
        g = g.sort_values('raw_index'); r = g.npz_row.values; out.append(dict(file=f, n_matched=len(g), npz_row_min=int(r.min()), npz_row_max=int(r.max()), increasing=bool((np.diff(r) > 0).all()) if len(r) > 1 else True, span=int(r.max() - r.min() + 1)))
    return pd.DataFrame(out)

def label_and_coverage(m, n_npz, lbl=None):
    # coverage of the npz by the matched raw points, and (if labels exist) which npz label each crop token received
    u = m[m.n_match == 1]; cov = dict(npz_rows=int(n_npz), matched_unique=int(len(u)), share=float(len(u) / n_npz))
    tab = None
    if lbl is not None:
        t = pd.DataFrame(dict(crop_token=u.crop_token.values, lbl=np.asarray(lbl)[u.npz_row.values])); tab = t.groupby(['crop_token', 'lbl']).size().rename('rows').reset_index()
        tab['tokens_with_this_label'] = tab.groupby('lbl').crop_token.transform('nunique'); tab['labels_for_this_token'] = tab.groupby('crop_token').lbl.transform('nunique')
    return cov, tab

def order_candidates(m, lbl=None):
    # Which sorting convention explains the order of the npz blocks?  Files are ordered by their observed first npz row and compared with each candidate sort.
    # keys: label id (from the npz labels of the matched rows), crop token, region/state code.  Works on any subset of files (relative order is compared).
    u = m[m.n_match == 1]; f = u.groupby('file').agg(first=('npz_row', 'min'), crop_token=('crop_token', 'first'), region=('region_code', 'first')).reset_index()
    f['lab'] = [int(pd.Series(np.asarray(lbl)[u[u.file == x].npz_row.values]).mode().iloc[0]) if lbl is not None else 0 for x in f.file]; f['region'] = f['region'].astype(str)
    obs = list(f.sort_values('first').file); res = []
    for nm, by, asc in (('label, token A-Z, region/state', ['lab', 'crop_token', 'region'], [True, True, True]), ('label, token Z-A, region/state', ['lab', 'crop_token', 'region'], [True, False, True]),
                        ('label, region/state, token A-Z', ['lab', 'region', 'crop_token'], [True, True, True]), ('token A-Z, region/state (no label)', ['crop_token', 'region'], [True, True])):
        cand = list(f.sort_values(by, ascending=asc).file); res.append(dict(candidate=nm, files=len(f), identical_order=cand == obs, files_in_same_position=int(sum(a == b for a, b in zip(cand, obs)))))
    return pd.DataFrame(res)

def run_match(raw_files, npz_path, out_dir, fea_key='fea', lbl_key='lbl'):
    os.makedirs(out_dir, exist_ok=True); z = np.load(npz_path); fea = z[fea_key]; print('npz', os.path.basename(npz_path), 'fea', fea.shape, '| labels', ('yes' if lbl_key in z.files else 'no'))
    points = [read_raw_points(p) for p in raw_files]; print('raw files:', [(os.path.basename(p['path'])[-22:], len(p['obs'])) for p in points])
    r = find_rule(points, fea); r.to_csv(os.path.join(out_dir, 'match_rule_search.csv'), index=False); print('\nrule search (sample of raw points):'); print(r.round(3).to_string(index=False))
    best = r.iloc[0]
    if not best.accepted:
        print(f"\n!! STOP: no declared rule reaches {MATCH_ACCEPT:.0%} unique matches (best {best['unique']:.1%}).  The raw files and this npz are not related by a simple 5-day median, or the raw files are not the ones this npz was built from.  Nothing is assumed; send me match_rule_search.csv."); return dict(accepted=False, search=r)
    print(f"\nACCEPTED rule: {best['rule']} | {best['layout']} | {best['scale']} -> {best['unique']:.1%} unique matches")
    m = match_points(points, fea, best['rule'], best['layout'], SCALES[best['scale']], progress=20000); m.to_csv(os.path.join(out_dir, 'row_map.csv'), index=False)
    ev = ordering_evidence(m); ev.to_csv(os.path.join(out_dir, 'ordering_evidence.csv'), index=False)
    print(f"full match: unique {(m.n_match == 1).mean():.2%} | none {(m.n_match == 0).mean():.2%} | ambiguous {(m.n_match > 1).mean():.2%}  (raw points {len(m)})"); print(ev.to_string(index=False))
    dup = int(m[m.npz_row >= 0].npz_row.duplicated().sum()); print('npz rows claimed by more than one raw point:', dup)
    oc = order_candidates(m, z[lbl_key] if lbl_key in z.files else None); oc.to_csv(os.path.join(out_dir, 'order_candidates.csv'), index=False); print('\nwhich sort order explains the npz block order?'); print(oc.to_string(index=False))
    cov, tab = label_and_coverage(m, len(fea), z[lbl_key] if lbl_key in z.files else None); print(f"coverage: {cov['matched_unique']} of {cov['npz_rows']} npz rows matched ({cov['share']:.1%})")
    if tab is not None:
        tab.to_csv(os.path.join(out_dir, 'label_by_crop_token.csv'), index=False); print('npz label received by each raw crop token (each token should map to ONE label):'); print(tab.to_string(index=False))
    return dict(accepted=True, search=r, map=m, evidence=ev, coverage=cov, labels=tab)


# ---------------------------------------------------------------- USA driver
USA_ORDER_STATUS = 'NOT RUN'; USA_ORDER_REPORT = {}
USA_RAW_RE = re.compile(r'^S2_USA_(?P<tok>[A-Za-z]+)_(?P<st>[A-Z]{2})\.csv$')
USA_UNIQUE_ACCEPT = 0.999

def find_usa_raw_files(roots):
    # raw files S2_USA_<token>_<ST>.csv under any of the roots (an upload / Drive hash prefix such as 'ab12cd34-17926..._' is ignored).  Two different files with the same name -> STOP.
    found = {}
    for root in roots:
        for p in glob.glob(os.path.join(root, '**', '*S2_USA_*.csv'), recursive=True):
            b = re.sub(r'^[0-9a-f]{8}-\d+_', '', os.path.basename(p)); m = USA_RAW_RE.match(b)
            if not m or m.group('tok') not in RAW_TO_CROP: continue
            if b in found and os.path.getsize(found[b]) != os.path.getsize(p): raise RuntimeError(f'STOP: two different files named {b}: {found[b]} and {p}')
            found.setdefault(b, p)
    return {b: found[b] for b in sorted(found, key=lambda x: (USA_RAW_RE.match(x).group('tok'), USA_RAW_RE.match(x).group('st')))}

def _per_crop_summary(m, lbl, n_npz):
    rows = []
    for tok, g in m.groupby('crop_token'):
        u = g[g.n_match == 1]; want = CROP_ID[RAW_TO_CROP[tok]]; bad = int((np.asarray(lbl)[u.npz_row.values] != want).sum()) if len(u) else 0
        rows.append(dict(crop_token=tok, crop=RAW_TO_CROP[tok], files=int(g.file.nunique()), raw_points=int(len(g)), unique=int(len(u)), unique_share=float(len(u) / len(g)), ambiguous=int((g.n_match > 1).sum()), unmatched=int((g.n_match == 0).sum()), label_mismatch=bad))
    return pd.DataFrame(rows).sort_values('crop_token').reset_index(drop=True)

def _blocks_ok(ev, m):
    # every file = ONE block of npz rows, increasing with the raw position; span may exceed the matched count only by rows lost to ambiguous matches in that file
    amb = m[m.n_match > 1].groupby('file').size().to_dict(); none = m[m.n_match == 0].groupby('file').size().to_dict(); bad = []
    for r in ev.itertuples():
        slack = amb.get(r.file, 0) + none.get(r.file, 0)
        if not r.increasing or (r.span - r.n_matched) > slack: bad.append(r.file)
    e = ev.sort_values('npz_row_min'); overlap = int((e.npz_row_min.values[1:] <= e.npz_row_max.values[:-1]).sum()) if len(e) > 1 else 0
    return bad, overlap

def compare_level1_table(m, table, name='level1'):
    # row-by-row: state in the raw FILE NAME vs state in the Level 1 table, for every uniquely matched raw point
    t = table[['npz_row', 'crop_id', 'state']].drop_duplicates('npz_row').set_index('npz_row'); u = m[m.n_match == 1].copy()
    u['l1_state'] = u.npz_row.map(t['state']); u['l1_crop_id'] = u.npz_row.map(t['crop_id']); u['file_state'] = u.region_code.astype(str)
    u['in_table'] = u.l1_state.notna(); u['agree'] = u.in_table & (u.l1_state.astype(str) == u.file_state)
    out = u.groupby('crop_token').agg(matched=('npz_row', 'size'), in_level1_table=('in_table', 'sum'), agree=('agree', 'sum')).reset_index(); out['disagree'] = out.in_level1_table - out.agree; out['table'] = name
    return out, u[u.in_table & ~u.agree][['file', 'raw_index', 'npz_row', 'crop_token', 'file_state', 'l1_state']]

def verify_usa_order(raw_files, npz_path, out_dir, level1_tables=None, log=print, rule_files_per_token=1):
    os.makedirs(out_dir, exist_ok=True); cache = os.path.join(out_dir, 'cache', 'usa_match'); os.makedirs(cache, exist_ok=True)
    z = np.load(npz_path); fea = z['fea']; lbl = z['lbl'].astype(int); n_npz = len(fea); rep = dict(npz=os.path.basename(npz_path), npz_rows=int(n_npz), raw_files=len(raw_files), status='FAILED', reasons=[])
    # (a) rule: tried on ONE file per crop token (declared candidates only; nothing is assumed)
    first = {}
    for b, p in raw_files.items(): first.setdefault(USA_RAW_RE.match(b).group('tok'), p)
    pts = [read_raw_points(p) for p in first.values()]; sr = find_rule(pts, fea); sr.to_csv(os.path.join(out_dir, 'usa_match_rule_search.csv'), index=False); best = sr.iloc[0]; del pts
    log('rule search on one file per crop token:'); log(sr.head(4).round(3).to_string(index=False))
    if not best.accepted: rep['reasons'].append(f"no declared rule reaches {MATCH_ACCEPT:.0%} (best {best['unique']:.1%}): the raw files and this npz are not related by a simple 5-day median"); return rep, None
    rule, layout, scale = best['rule'], best['layout'], SCALES[best['scale']]; rep['rule'] = f"{rule} | {best['layout']} | {best['scale']}"; log(f"ACCEPTED rule: {rep['rule']} ({best['unique']:.1%} unique on the sample)")
    # (b) full match, streamed file by file (memory-safe, resumable: a finished file is read back from cache/usa_match)
    parts = []; idx = None; t0 = time.time()
    for b, p in raw_files.items():
        cp = os.path.join(cache, b + '.csv')
        if os.path.exists(cp): parts.append(pd.read_csv(cp, dtype={'region_code': str})); continue
        if idx is None: idx = NpzIndex(fea, layout, scale)
        pt = read_raw_points(p); rows = []
        for j, obs in enumerate(pt['obs']):
            med = bin_medians(obs, rule); c, have = idx.candidates(med, MATCH_TOL)
            rows.append(dict(file=b, raw_index=pt['index'][j], n_match=0 if c is None else len(c), npz_row=int(c[0]) if (c is not None and len(c) == 1) else -1, crop_token=pt['crop_token'], region_code=pt['region_code']))
        df_ = pd.DataFrame(rows); df_.to_csv(cp, index=False); parts.append(df_); del pt
        log(f'   {b}: {len(df_)} raw points | unique {(df_.n_match == 1).mean():.2%} | {time.time() - t0:.0f}s')
    m = pd.concat(parts, ignore_index=True); m['region_code'] = m['region_code'].astype(str); m.to_csv(os.path.join(out_dir, 'usa_row_map.csv'), index=False)
    summ = _per_crop_summary(m, lbl, n_npz); summ.to_csv(os.path.join(out_dir, 'usa_order_by_crop.csv'), index=False); rep['by_crop'] = summ.to_dict('records')
    log('\nper crop token (soybeans is the Level 1 headline crop):'); log(summ.round(4).to_string(index=False))
    low = summ[summ.unique_share < USA_UNIQUE_ACCEPT]
    if len(low): rep['reasons'].append(f"crop tokens below {USA_UNIQUE_ACCEPT:.1%} exact matches: {low.crop_token.tolist()}")
    if int(summ.label_mismatch.sum()): rep['reasons'].append(f"{int(summ.label_mismatch.sum())} matched rows carry an npz label different from their crop token's label")
    dup = int(m[m.npz_row >= 0].npz_row.duplicated().sum()); rep['npz_rows_claimed_twice'] = dup
    if dup: rep['reasons'].append(f'{dup} npz rows are claimed by more than one raw point')
    ev = ordering_evidence(m); ev.to_csv(os.path.join(out_dir, 'usa_ordering_evidence.csv'), index=False); bad_blocks, overlap = _blocks_ok(ev, m); rep['blocks_not_contiguous'] = bad_blocks; rep['block_overlaps'] = overlap
    if bad_blocks or overlap: rep['reasons'].append(f'files that are not one contiguous increasing block: {bad_blocks}; overlapping blocks: {overlap}')
    mm = m.copy(); mm['crop_token'] = mm['crop_token'].astype(str); oc = order_candidates(mm, lbl); oc.to_csv(os.path.join(out_dir, 'usa_order_candidates.csv'), index=False); rep['order_candidates'] = oc.to_dict('records')
    conv = oc[oc.candidate == 'label, token A-Z, region/state']; rep['order_convention_holds'] = bool(len(conv) and conv.identical_order.iloc[0]); log('\nsort order of the npz blocks:'); log(oc.to_string(index=False))
    if not rep['order_convention_holds']: rep['reasons'].append('the npz block order is not (label, token A-Z, state)')
    cov = int((m.n_match == 1).sum()); rep['npz_rows_matched'] = cov; rep['npz_rows_matched_share'] = cov / n_npz
    # (e) the Level 1 state table, row by row
    rep['level1'] = []; l1_bad = 0
    for name, tab in (level1_tables or {}).items():
        cmp_, wrong = compare_level1_table(m, tab, name); cmp_.to_csv(os.path.join(out_dir, f'usa_vs_level1_{re.sub("[^A-Za-z0-9]+", "_", name)[-40:]}.csv'), index=False); rep['level1'].append(dict(table=name, by_crop=cmp_.to_dict('records'), disagreements=int(cmp_.disagree.sum())))
        wrong.to_csv(os.path.join(out_dir, 'usa_level1_disagreements.csv'), index=False); l1_bad += int(cmp_.disagree.sum())
        log(f'\nLevel 1 state table ({name}) vs the state in the raw file name:'); log(cmp_.to_string(index=False))
    rep['level1_disagreements'] = l1_bad
    if rep['reasons']: rep['status'] = 'FAILED'
    elif not rep['level1']: rep['status'] = 'ORDER VERIFIED (Level 1 state table not found: row-by-row state comparison NOT done)'
    elif l1_bad: rep['status'] = f'ORDER VERIFIED, BUT the Level 1 state table disagrees with the raw file names on {l1_bad} rows'
    else: rep['status'] = 'VERIFIED (order exact; Level 1 state table equals the raw file names on every matched row)'
    return rep, m

USA_RAW_DIR = ''      # OPTIONAL: if your raw USA files S2_USA_<crop>_<ST>.csv sit in a folder that is not under the project folder, put its full path here, e.g. '/content/drive/MyDrive/cropglobe_assignment/USA'.  Left empty, the whole project folder and the data folder are searched (any sub-folder name works, including 'USA').
def run_usa_order_cell(project_root, data_dir, out_dir, log=print):
    global USA_ORDER_STATUS, USA_ORDER_REPORT
    roots = [r for r in (USA_RAW_DIR, project_root, data_dir) if r]; print('searching for raw USA files in:', roots)
    raw = find_usa_raw_files(roots)
    if not raw:
        try:                                                                      # best effort only; the repository layout of the raw files is not documented here
            from huggingface_hub import snapshot_download
            snapshot_download(repo_id='x-ytong/CropGlobe', repo_type='dataset', local_dir=data_dir, allow_patterns=['**/S2_USA_*.csv']); raw = find_usa_raw_files(roots)
        except Exception as e: log(f'  (download attempt failed: {type(e).__name__})')
    npz = find_npz(data_dir, 'USA') if glob.glob(os.path.join(data_dir, '**', 'median_S2_USA_month1-12_day5.npz'), recursive=True) else None
    if npz is None:
        try:
            from huggingface_hub import snapshot_download; snapshot_download(repo_id='x-ytong/CropGlobe', repo_type='dataset', local_dir=data_dir, allow_patterns=['**/median_S2_USA_month1-12_day5.npz']); npz = find_npz(data_dir, 'USA')
        except Exception as e: log(f'  (USA npz not available: {type(e).__name__})')
    if not raw or npz is None:
        USA_ORDER_STATUS = 'NOT VERIFIED (' + ('raw USA files S2_USA_<crop>_<ST>.csv not found' if not raw else 'USA npz not found') + ': put them anywhere under the project folder and run this cell again)'
        USA_ORDER_REPORT = dict(status=USA_ORDER_STATUS); print('\n' + '=' * 100 + f'\nUSA ORDER VERIFICATION: {USA_ORDER_STATUS}\n' + '=' * 100); return USA_ORDER_REPORT
    print(f'{len(raw)} raw USA files: ' + ', '.join(f'{USA_RAW_RE.match(b).group("tok")}:{sum(1 for x in raw if USA_RAW_RE.match(x).group("tok") == USA_RAW_RE.match(b).group("tok"))}' for b in dict.fromkeys(raw)))
    tabs = {}
    for p in glob.glob(os.path.join(project_root, '**', 'cache', 'usa_meta.csv'), recursive=True):
        try:
            t = pd.read_csv(p)
            if {'npz_row', 'crop_id', 'state'} <= set(t.columns): tabs[os.path.relpath(p, project_root)] = t
        except Exception: pass
    print('Level 1 state tables found:', list(tabs) or 'none')
    rep, m = verify_usa_order(raw, npz, os.path.join(out_dir, 'tables'), tabs, log=log); USA_ORDER_REPORT = rep; USA_ORDER_STATUS = rep['status']
    atomic_write(os.path.join(out_dir, 'tables', 'usa_order_verification.json'), lambda f: f.write(json.dumps(rep, indent=1, default=str)), mode='w')
    sb = [r for r in rep.get('by_crop', []) if r['crop'] == 'soybeans']
    print('\n' + '=' * 100 + f'\nUSA ORDER VERIFICATION: {USA_ORDER_STATUS}')
    if sb: print('   soybeans: ' + ' | '.join(f"{r['crop_token']}: {r['unique']}/{r['raw_points']} exact matches, {r['label_mismatch']} label mismatches" for r in sb))
    if rep['reasons']: print('   reasons: ' + '; '.join(rep['reasons']))
    print('=' * 100)
    return rep


# ---- self-test of THIS verifier on a tiny synthetic world (runs every time; the verifier must say VERIFIED for correct data and FAILED for swapped blocks, wrong labels and unrelated data)
def _selftest_usa_order():
    import tempfile, io, contextlib
    ns = globals(); CID = CROP_ID; RTC = RAW_TO_CROP
    PLAN = [('corn', ['IA', 'IL']), ('soybeans', ['IA', 'IL', 'MN']), ('springwheat', ['ND']), ('winterwheat', ['KS', 'OK']), ('other', ['NY', 'TX']), ('rice', ['AR'])]
    def make_world(td, npts=60, seed=0, swap_soy_blocks=False, mislabel_soy=False, unrelated=False):
        rng = np.random.default_rng(seed); rawdir = os.path.join(td, 'raw'); os.makedirs(rawdir, exist_ok=True); order = sorted([(CID[RTC[t]], t, s) for t, sts in PLAN for s in sts]); blocks = []
        for lab, t, s in order:
            obs_l = []
            for j in range(npts):
                doy = np.sort(rng.choice(np.arange(1, 366), 40, replace=False)); vals = rng.integers(200, 4000, (40, 10)); obs_l.append(np.column_stack([doy, vals]).astype(float))
            blocks.append((lab, t, s, obs_l))
            rows = [dict(**{'system:index': f'{j:04d}', 'information': 'x', 'raw_data': json.dumps(o.tolist()), '.geo': json.dumps({'type': 'Point', 'coordinates': [-90.0 + j * .001, 40.0]})}) for j, o in enumerate(obs_l)]
            pd.DataFrame(rows).to_csv(os.path.join(rawdir, f'S2_USA_{t}_{s}.csv'), index=False)
        seq = list(blocks)
        if swap_soy_blocks:
            i = [k for k, b in enumerate(seq) if b[1] == 'soybeans']; seq[i[0]], seq[i[1]] = seq[i[1]], seq[i[0]]
        feas, labs, meta = [], [], []
        for lab, t, s, obs_l in seq:
            for o in obs_l:
                m = ns['bin_medians'](o, 'floor((doy-1)/5)')
                for b in range(10):
                    x = np.arange(73); ok = ~np.isnan(m[b]); m[b] = np.interp(x, x[ok], m[b][ok]) if ok.any() else 0
                feas.append(m.reshape(730)); labs.append(CID['corn'] if (mislabel_soy and t == 'soybeans') else lab); meta.append((CID[RTC[t]], s))
        A = np.array(feas, dtype=np.float32)
        if unrelated: A = rng.normal(2000, 500, A.shape).astype(np.float32)
        ddir = os.path.join(td, 'data'); os.makedirs(ddir, exist_ok=True); np.savez(os.path.join(ddir, 'median_S2_USA_month1-12_day5.npz'), fea=A, lbl=np.array(labs))
        # Level 1 style state table (npz_row, crop_id, state) for the packaged rows (correct unless altered by the caller)
        tab = pd.DataFrame({'npz_row': np.arange(len(meta)), 'crop_id': [m[0] for m in meta], 'state': [m[1] for m in meta]})
        return rawdir, os.path.join(ddir, 'median_S2_USA_month1-12_day5.npz'), tab
    def run(td, **kw):
        rawdir, npz, t = make_world(td, **kw); files = find_usa_raw_files([rawdir])
        with contextlib.redirect_stdout(io.StringIO()): rep, m = verify_usa_order(files, npz, os.path.join(td, 'out'), {'level1': t})
        return rep, m
    with tempfile.TemporaryDirectory() as td: rep, m = run(td); assert rep['status'].startswith('VERIFIED') and rep['level1_disagreements'] == 0 and (m.n_match == 1).all(), rep['status']
    with tempfile.TemporaryDirectory() as td: rep, m = run(td, swap_soy_blocks=True); assert rep['status'] == 'FAILED', rep['status']
    with tempfile.TemporaryDirectory() as td: rep, m = run(td, mislabel_soy=True); assert rep['status'] == 'FAILED'
    with tempfile.TemporaryDirectory() as td: rep, m = run(td, unrelated=True); assert rep['status'] == 'FAILED' and m is None
    print('ordering-verifier self-test passed: correct data VERIFIED; swapped state blocks, wrong labels and unrelated data FAILED')
_selftest_usa_order()

run_usa_order_cell(PROJECT_ROOT, DATA_DIR, OUT_DIR)

## 9. Load + verify the data (STOPs on any inconsistency)

In [ ]:
# ======================================================================================
# LOAD + VERIFY THE DATA (Level 2: source = France).  Nothing is trained here.
# ======================================================================================
FEATURES, NPZ_LBL, INFO, meta, FRA_META_REPORT = load_all_data(DATA_DIR, OUT_DIR, ROW_MAP_PATH)
os.makedirs(os.path.join(OUT_DIR, 'tables'), exist_ok=True)
atomic_write(os.path.join(OUT_DIR, 'tables', 'fra_region_assignment.json'), lambda f: f.write(json.dumps(FRA_META_REPORT, indent=1)), mode='w')
verify_data(FEATURES, NPZ_LBL, INFO, meta)
print('\n================ SOURCE REGION-TABLE GEOGRAPHY CHECK (STOP if violated) ================')
GEOGRAPHY_OBSERVED = check_geography(meta['state'].values, meta['crop'].values)
_bad = {c: int((meta['crop'] == c).sum()) for c in ('sugarcane', 'cotton') if (meta['crop'] == c).any()}; _check(not _bad, f'France cannot contain sugarcane / cotton rows: {_bad}')
print('GEOGRAPHY CHECK PASSED: rice rows fall only in regions 76 and 93; no sugarcane / cotton rows.')

## 10. Plan table (nothing trains) + design hash

In [ ]:
# ======================================================================================
# PLAN TABLE (nothing is trained here).  READ IT, then set APPROVED['plan'] = True in the config cell.
# ======================================================================================
assert TESTS_PASSED, 'the test gate (Part A + Part B) must pass first'
assert 'GEOGRAPHY_OBSERVED' in globals(), 'run the load cell (it contains the source-region geography check) first'
SRC_UNIT    = meta['state'].values.astype(str); SRC_CROP = meta['crop'].values.astype(str); SRC_CROP_ID = meta['crop_id'].values.astype(int)       # 'state' column = French REGION code
SRC_NPZ_ROW = meta['npz_row'].values.astype(np.int64)                                   # row index in the ORIGINAL (pre-dedup) France npz -> saved as sample ids
TARGET_IDS  = {t: INFO[t]['keep'] for t in TARGETS_TO_RUN}                              # original npz row of every kept target sample
POOLS = build_pools(SRC_UNIT, SRC_CROP); CAP = make_cap(SRC_UNIT, SRC_CROP)
EXCLUDED_SOURCE_UNITS = globals().get('EXCLUDED_SOURCE_UNITS', [])
if EXCLUDED_SOURCE_UNITS:                                                                   # FOLLOW-UP only: the removed source regions are taken out of the capacity table AND the row pools, so no plan cell can contain them and no row of them can be drawn
    _before = len(CAP); CAP = CAP.drop(index=[u_ for u_ in EXCLUDED_SOURCE_UNITS if u_ in CAP.index]); POOLS = {k_: v_ for k_, v_ in POOLS.items() if k_[0] not in EXCLUDED_SOURCE_UNITS}
    _check(len(CAP) == _before - len(EXCLUDED_SOURCE_UNITS) and not any(k_[0] in EXCLUDED_SOURCE_UNITS for k_ in POOLS), 'excluded source regions are still present in the capacity table / pools')
    print(f'FOLLOW-UP: source regions {EXCLUDED_SOURCE_UNITS} removed from the pool ({_before} -> {len(CAP)} regions); their rows can never be drawn.')
os.makedirs(os.path.join(OUT_DIR, 'tables'), exist_ok=True)
print('feature source check:', FEATURE_SOURCE)
print('\nREGION-ASSIGNMENT EVIDENCE (France; exact row-by-row match, see FRA_join_verification.md):'); print('  ', {k: v for k, v in FRA_META_REPORT.items() if k not in ('excluded_rows', 'rows_per_region')}); print('   rows per region (assigned):', FRA_META_REPORT['rows_per_region'])
CAP.sort_values('total', ascending=False).to_csv(os.path.join(OUT_DIR, 'tables', 'state_crop_table.csv'))
print(f'{SOURCE} regions: {len(CAP)} | regions with >= {MIN_STATE_ROWS} rows: {int((CAP.total >= MIN_STATE_ROWS).sum())}')
print(CAP.sort_values('total', ascending=False).head(25).to_string())

# ---- PHASE 4: phenology-spectral diversity of the USA source states (USA features only; no target data, labels or scores are used)
DIVERSITY_DIST = compute_diversity_distances(FEATURES[SOURCE], SRC_UNIT, SRC_CROP) if any(a in SELECTION_ARMS for a in ('diverse', 'clustered')) else None
if 'kcenter' in SELECTION_ARMS:
    KC_EMBED = build_kc_embedding(FEATURES[SOURCE]); _check(KC_EMBED.shape == (len(SRC_UNIT), KC_DIMS) and np.isfinite(KC_EMBED).all(), 'k-center embedding has the wrong shape / non-finite values')
    print(f'k-center embedding: USA window features (10 bands x 43 steps) -> standardised -> PCA {KC_DIMS} dims, shape {KC_EMBED.shape} (USA rows only; no target data is used)')
if DIVERSITY_DIST is not None:
    for _c, _D in DIVERSITY_DIST['dist'].items(): _D.to_csv(os.path.join(OUT_DIR, 'tables', f'diversity_distance_{_c}.csv'))
    print('Phase 4 diversity space:', DIVERSITY_SPACE); print('arms:', SELECTION_ARMS, '| pool rule:', POOL_RULE, '| N per arm:', N_ARM)
PLAN_CELLS, PLAN_GRID = build_plan(CAP)
PLAN_CELLS_RUN = [c for c in PLAN_CELLS if not c['pending']]
PLAN_GRID.to_csv(os.path.join(OUT_DIR, 'tables', 'feasibility_grid.csv'), index=False)
pd.DataFrame([dict(c, states='-'.join(c['states'])) for c in PLAN_CELLS]).to_csv(os.path.join(OUT_DIR, 'tables', 'plan_cells.csv'), index=False)

# ---- COMPLETE state-by-crop table is saved above; target-specific ELIGIBILITY tables (every state, with the reason for every exclusion) are saved per target/budget/mode
def _eligibility_report(mode):
    for t in TARGETS_TO_RUN:
        for B in BUDGETS:
            e = state_eligibility(t, B, CAP, mode); e.to_csv(os.path.join(OUT_DIR, 'tables', f'eligibility_{mode}_{t}_B{B}.csv'), index=False)
            bad = e[~e.eligible]; why = pd.Series([r.split(' ')[0] if r else '' for rs in bad.reason for r in rs.split('; ')]).value_counts().to_dict() if len(bad) else {}
            print(f'  [{mode}] {t} B={B}: {int(e.eligible.sum())}/{len(e)} states eligible; excluded {len(bad)} (reasons: {why or "-"}) -> tables/eligibility_{mode}_{t}_B{B}.csv')
print('\n================ ELIGIBILITY (every state is listed in the CSVs; nothing is excluded silently) ================')
for _m in COMPOSITION_MODES_TO_RUN:
    if _m == BASE_MODE or FIXED_COMPOSITION_CHOICE in FIXED_COMPOSITION_PROPOSALS: _eligibility_report(_m)
print('\n================ AUTHORS\' SCRIPT vs THIS NOTEBOOK (from run_CropNet_AUGM.py; deviations are deliberate and listed) ================')
print('  authors:', {k: v for k, v in AUTHORS_SCRIPT.items()}); print('  deviations:'); [print(f'    - {k}: {v}') for k, v in DEVIATIONS_FROM_AUTHORS.items()]
print(f'  this notebook: Adam lr={LR}, weight_decay={WEIGHT_DECAY}, batch {BATCH_SIZE}, no scheduler, TOTAL epochs {MAX_EPOCHS} with early stopping (patience {PATIENCE}, min {MIN_EPOCHS}, strictly greater, best weights restored), class weights from {CLASS_WEIGHT_SOURCE}')
print('  optimizer steps per epoch = ceil(train/256): ' + ' | '.join(f'B={B}: {steps_per_epoch(B - int(VAL_FRACTION * B))} (<= {MAX_EPOCHS * steps_per_epoch(B - int(VAL_FRACTION * B))} total)' for B in BUDGETS) + '  -> budget is confounded with training length; steps are reported per run')
print('\n================ REMAP RULES: where the three named rules differ (all three are scored, separately) ================')
print('  NOTE:', PLAN_NOTE_ARG)
print(remap_difference_table().to_string(index=False))
print('   existing_release = EXISTING (preserved) | paper_text_v3 = literal paper sec.4.4 text | review_targetmap = REVIEW-PROPOSED.  Part B compared all three with the authors\' released function (tables/authors_remap_comparison.csv). No rule is declared "the paper\'s rule".')

print('\n================ PLAN TABLE ================')
print(f'composition modes {COMPOSITION_MODES_TO_RUN} | state selector {STATE_SELECTOR} | epochs <= {MAX_EPOCHS} | class weights from {CLASS_WEIGHT_SOURCE} labels | feature source: {FEATURE_SOURCE}')
print(f'budgets {BUDGETS} | k {K_LIST} | seeds {SEEDS} | aug {AUG} | max {MAX_COMBOS} combos for k>=2 (k=1 census) | infeasible if <{MIN_COMBOS} combos or <{MIN_KPOINTS} k-points')
print('key-crop shares (fixed at every k; the rest sampled naturally):', {t: effective_key_shares(t) for t in TARGETS_TO_RUN})
for _m, t in [(m, t) for m in COMPOSITION_MODES_TO_RUN if (m == BASE_MODE or FIXED_COMPOSITION_CHOICE in FIXED_COMPOSITION_PROPOSALS) for t in TARGETS_TO_RUN]:
    print(f'\n--- [{_m}] {t} ({TARGET_ROLE[t]}; primary view: {PRIMARY_VIEW[t][0]}{"_" + PRIMARY_VIEW[t][1] if PRIMARY_VIEW[t][1] else " (existing release remap)"}; K_MAX={K_MAX[t]}) ---')
    g = PLAN_GRID[(PLAN_GRID.target == t) & (PLAN_GRID.composition == _m)]
    for B in BUDGETS:
        gb = g[g.budget == B]; kc = key_counts(t, B)
        print(f'  B={B}: key counts {kc if kc else "none (fully natural)"} + natural rest {B - sum(kc.values())} | pool = {int(gb.pool_size.iloc[0])} state(s): {gb.pool.iloc[0]}')
        print('      ' + ' | '.join(f'k={int(r.k)}: ' + (f'{int(r.n_combos)} combo(s) of {int(r.n_total_combos)}' + (' [census]' if r.k == 1 else '') if r.feasible else f'INFEASIBLE ({r.note.strip()})') for r in gb.itertuples()))
    _cm = [c for c in PLAN_CELLS if c['target'] == t and c['composition'] == _m]; n_runs = sum(len([c for c in plan_for_seed(s_) if c['target'] == t and c['composition'] == _m]) for s_ in SEEDS)
    print(f'  -> {len(_cm)} cells x {len(SEEDS)} seeds = {len(_cm) * len(SEEDS)} runs' + ('   *** EXCLUDED UNTIL APPROVED/CONFIRMED ***' if _cm and _cm[0]['pending'] else f'   ({n_runs} will run)'))
TOTAL_RUNS = sum(len(plan_for_seed(s_)) for s_ in SEEDS)
print(f'\nTOTAL training runs that will execute: {TOTAL_RUNS}  ({len(PLAN_CELLS_RUN)} combination-cells for seeds {KC_SEEDS}; seeds {[s_ for s_ in SEEDS if s_ not in KC_SEEDS]} skip the {sum(1 for c in PLAN_CELLS_RUN if c.get("sel") == "kcenter")} k-center cells; arms share combinations, so a combination is trained once); GPU time is estimated after the smoke test.')
_check(len(PLAN_CELLS_RUN) > 0, 'EMPTY PLAN: no cell is feasible -- see the tables above')
print('\n================ v3.7 ADDITIONS (listed in CHANGELOG_v3_7.md) ================')
print(f'  epochs: TOTAL cap {MAX_EPOCHS} (was 100: 38% of the 8-state runs hit the old cap) | results folder: {OUT_DIR}')
print(f'  MAIN metric: ARG/CHN = binary key-crop macro-F1 (primary_f1); USA (multi-class) = {"coverage-conditional macro-F1 (classes with >=" + str(COVERAGE_MIN_TRAIN) + " training samples; sugarcane and cotton do not exist in the French source)" if FRA_MAIN_RULE == "coverage_conditional" else "all-class macro-F1 (NO rule)"}.')
print(f'  SECONDARY metric: paper-style macro-F1 (read it next to the coverage columns: each absent crop costs ~11 points). All other score columns are stored but not summarised (REPORT_ALL_TRACKS={REPORT_ALL_TRACKS}).')
print('  input features recorded in every row:', INPUT_FEATURES)
_V = pd.DataFrame([dict(arm=a, target=c['target'], budget=c['budget'], pending=c['pending'], shared=('random' in c['arms'] and a != 'random')) for c in PLAN_CELLS for a in c['arms'] if a in ('common', 'kcenter', 'aug_on')])
if len(_V):
    print('  NEW TRACKS (cells; each cell x seeds = runs; "shared" = the run is also in the random arm and is trained once):')
    print(_V.groupby(['arm', 'target', 'budget']).agg(cells=('arm', 'size'), shared=('shared', 'sum'), pending=('pending', 'sum')).to_string())
    if 'common' in SELECTION_ARMS: print('  common = combinations re-drawn from the states eligible at EVERY budget of the target (fixed pool), so budgets can be compared on the same states; k above the common-pool size is not available.')
    if 'kcenter' in SELECTION_ARMS: print(f'  kcenter = same state combinations (k=1 census + up to {KC_COMBOS} per cell) and identical per-state/per-crop counts as the random arm; only WHICH rows are taken changes (greedy farthest-first in a {KC_DIMS}-dim PCA space, per state and crop).')
    if 'aug_on' in SELECTION_ARMS: print(f'  aug_on = authors\' augmentation (time-shift p=1, time-scale p=.5, magnitude-warp p=.25; original + {AUG_REPEAT} augmented copies) on the same combinations; targets {AUG_TARGETS}, budgets {AUG_BUDGETS}.  PENDING until APPROVED["aug_on"] = True.  It uses {1 + AUG_REPEAT}x the optimizer steps per epoch; compare with aug-OFF knowing this.')
_n_new = [c for c in PLAN_CELLS_RUN if c.get('sel', 'random') != 'random' or c.get('aug', 'off') != 'off' or c['arms'] == ('common',)]
print(f'  runs that exist only because of the new tracks: {sum(len(plan_for_seed(s_, _n_new)) for s_ in SEEDS)} ({len(_n_new)} cells; k-center only for seeds {KC_SEEDS}); drop an arm from SELECTION_ARMS in the config cell to remove it.')
if AUG_CONTROL and 'aug_on' in SELECTION_ARMS: print(f"  aug_copy (v3.8) = steps-matched control for aug_on: the SAME cells, the training rows copied {1 + AUG_REPEAT}x with NO transform (same rows per epoch and optimizer steps, no augmentation).  The decisive comparison is aug_on - aug_copy.")
DESIGN_HASH = design_hash()
atomic_write(os.path.join(OUT_DIR, 'tables', 'design_hash.txt'), lambda f: f.write(DESIGN_HASH + '\n'), mode='w')
print('\n================ INSTRUCTOR APPROVAL ================')
print('DESIGN HASH (locked settings + every planned run id of every seed):', DESIGN_HASH)
print('To run: after the instructor has signed the decisions, set INSTRUCTOR_APPROVED = True and APPROVED_DESIGN_SHA256 = this hash in the config cell, and re-run the config cell.  The run cell refuses any other plan.')
print('   Nothing has been trained.')
if PROFILE == 'extension':
    print('\n================ EXTENSION PLAN (appendix): CHN / FRA at budgets %s ================' % BUDGETS)
    print('  Allocation: each fixed crop and the natural remainder are split over the k states AS EQUALLY AS EACH STATE\'S CAPACITY ALLOWS (a short state gives all it has, the others make up the difference).')
    print(f'  A state joins a combination only if it can give at least {EXT_MIN_CONTRIB:.0%} of its equal share of every fixed quota and of the remainder; the combination as a whole must hold every quota.  k=1 is the census of states that can supply the whole budget alone.')
    _g = PLAN_GRID[PLAN_GRID.feasible].copy(); _g['max_share_worst'] = _g['max_share']
    print(_g[['target', 'budget', 'k', 'pool_size', 'n_total_combos', 'n_combos', 'max_share_worst']].to_string(index=False))
    print('  max_share_worst = the largest planned share of the budget supplied by ONE state in any combination of the cell (1/k would be perfectly equal).')
    _skipped = PLAN_GRID[~PLAN_GRID.feasible][['target', 'budget', 'k', 'note']]
    if len(_skipped): print('  NOT planned (reported, never silently dropped):'); print(_skipped.to_string(index=False))
    print(f'  runs: {sum(len(plan_for_seed(s_)) for s_ in SEEDS)} ({len(PLAN_CELLS_RUN)} cells x {len(SEEDS)} seeds)')

## 11. Model + training code + smoke test

In [ ]:
# ======================================================================================
# TORCH BACKEND -- the AUTHORS' CropNet and CropGlobeDataset (unchanged, pinned commit), official recipe:
#   CropNet(data_channel=1, classes=7), ALL 7 outputs trained | Adam lr 1e-4, weight_decay 0, no scheduler | batch 256
#   CrossEntropyLoss with balanced class weights over the drawn sample (= the authors' whole source set; CLASS_WEIGHT_SOURCE), zero-sample crops -> weight 0
#   official DOY 121-334 window (aug_mode='none') for validation AND test; training: aug_mode='none' (aug OFF) or the authors' 'augm' (aug ON, arm aug_on only) | validation/test are never augmented
#   TOTAL 200 epochs, early stopping inside them on VALIDATION macro-F1 (patience 15, min 20 epochs, strictly greater), best weights restored and re-verified
# ======================================================================================
assert TESTS_PASSED, 'test gate (Part A + Part B) has not passed -- no GPU time'
assert APPROVED['plan'], 'the plan table has not been approved: set APPROVED["plan"] = True in the config cell after reading it'
import torch
from torch.utils.data import TensorDataset, DataLoader
os.environ.setdefault('TQDM_DISABLE', '1')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'NO GPU (very slow) -- enable a GPU accelerator')
for _m in ('models', 'data'):
    if _m in sys.modules and REPO_DIR not in str(getattr(sys.modules[_m], '__file__', '') or ''): del sys.modules[_m]
if REPO_DIR in sys.path: sys.path.remove(REPO_DIR)
sys.path.insert(0, REPO_DIR)
from models.CropNet import CropNet
from data.augment import CropGlobeDataset
for _cls in (CropNet, CropGlobeDataset): assert os.path.abspath(inspect.getfile(_cls)).startswith(os.path.abspath(REPO_DIR)), f'{_cls} is not the authors\' class'
assert subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip() == EXPECTED_CROPNET_COMMIT
_probe = CropGlobeDataset()
assert (_probe.dayItv, _probe.staDOY, _probe.endDOY, _probe.target_len) == (5, 121, 334, 43), 'unexpected official DOY window'
print(f'official DOY window: day {_probe.staDOY}-{_probe.endDOY}, step {_probe.dayItv} -> {_probe.target_len} steps; model input = 10 bands x {_probe.target_len}')
assert AUG == 'off' and MAX_EPOCHS == 200 and CLASS_WEIGHT_SOURCE == 'train_val' and (LR, WEIGHT_DECAY, BATCH_SIZE) == (AUTHORS_SCRIPT['lr'], 0.0, AUTHORS_SCRIPT['batch_size'])

try:                                                                                   # informational: the authors' own trainer (NOT used here)
    from trainers.torch_trainer import trainer as _auth_trainer
    print('\n--- authors\' trainers/torch_trainer.py::trainer (read-only; compare with this notebook\'s early stopping) ---'); print(inspect.getsource(_auth_trainer)[:3500])
except Exception as _e: print('(could not display the authors\' trainer source:', repr(_e)[:100], ')')

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def official_window(X, y):
    ds = CropGlobeDataset(aug_mode='none')(TensorDataset(torch.as_tensor(X, dtype=torch.float32), torch.as_tensor(y).long()))
    return ds.tensors

@torch.no_grad()
def predict_tensor(model, X, bs=4096):
    model.eval(); out = []; X = torch.as_tensor(X) if not torch.is_tensor(X) else X
    for i in range(0, len(X), bs): out.append(model(X[i:i + bs].to(DEVICE, dtype=torch.float32)).argmax(1).cpu())
    return torch.cat(out).numpy()

def fit_model(Xtr, ytr, Xva, yva, seed, epochs_override=None, aug='off'):
    max_ep = int(epochs_override or MAX_EPOCHS)                                                    # TOTAL epochs (100); EarlyStopper refuses more
    set_seed(seed)
    ytr_np, yva_np = np.asarray(ytr).astype(int), np.asarray(yva).astype(int)
    w = training_class_weights(ytr_np, yva_np, NUM_CLASSES)                                         # balanced weights over the drawn sample (authors-equivalent); zero-sample crops -> 0
    assert aug in ('off', 'on', 'copy'), aug
    if aug == 'on':                                                                                 # the AUTHORS' augmentation (pinned commit): original + AUG_REPEAT augmented copies of every training row; their rng is unseeded, so we seed it (recorded deviation)
        _ds = CropGlobeDataset(aug_mode='augm', re__peat=AUG_REPEAT); _ds.rng = np.random.default_rng(stable_seed('aug', seed, len(ytr_np)))
        train_ds = _ds(TensorDataset(torch.as_tensor(Xtr, dtype=torch.float32), torch.as_tensor(ytr_np).long()))
        assert len(train_ds) == (1 + AUG_REPEAT) * len(ytr_np) and bool(torch.isfinite(train_ds.tensors[0]).all()), 'augmentation produced an unexpected training set'
    elif aug == 'copy':                                                                             # v3.8 steps-matched control: the SAME number of training rows as aug ON, all exact copies of the originals (no transform) -> same rows per epoch and optimizer steps, no augmentation
        _X = torch.as_tensor(Xtr, dtype=torch.float32); _y = torch.as_tensor(ytr_np).long()
        train_ds = CropGlobeDataset(aug_mode='none')(TensorDataset(torch.cat([_X] * (1 + AUG_REPEAT)), torch.cat([_y] * (1 + AUG_REPEAT))))
        assert len(train_ds) == (1 + AUG_REPEAT) * len(ytr_np), 'copy control produced an unexpected training set'
    else: train_ds = CropGlobeDataset(aug_mode='none')(TensorDataset(torch.as_tensor(Xtr, dtype=torch.float32), torch.as_tensor(ytr_np).long()))
    val_X, val_y = official_window(Xva, yva_np); val_X = val_X.to(DEVICE); val_y_np = val_y.numpy()
    g = torch.Generator(); g.manual_seed(seed)
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=g)                 # drop_last=False -> ceil(train/256) steps per epoch
    set_seed(seed)
    model = CropNet(data_channel=1, classes=NUM_CLASSES).to(DEVICE)                               # authors' architecture, 7-class head
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)                  # == authors' optim.Adam(lr); no scheduler
    crit = torch.nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=DEVICE))
    last = {}
    def train_one_epoch(ep):
        model.train()
        for bi, (xb, yb) in enumerate(loader):
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(set_to_none=True); loss = crit(model(xb), yb)
            if not torch.isfinite(loss): raise FloatingPointError(f'training loss non-finite at epoch {ep + 1}, batch {bi}')            # V33-07: EVERY batch, before the step (was: last batch only)
            loss.backward(); opt.step()
        last['loss'] = float(loss)
        if not math.isfinite(last['loss']): raise FloatingPointError(f'training loss became non-finite at epoch {ep + 1}')
        if not all(bool(torch.isfinite(p_).all()) for p_ in model.parameters()): raise FloatingPointError(f'model parameters non-finite after epoch {ep + 1}')
    def val_score(): return float(f1_score(val_y_np, predict_tensor(model, val_X), average='macro', zero_division=0))
    snapshot = lambda: {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    out = run_training_loop(train_one_epoch, val_score, snapshot, model.load_state_dict, max_ep)  # <= MAX_EPOCHS (200), early stopping, restore + verification
    return dict(model=model, info=out)

def prepare_eval_sets():
    EX, EY = {}, {}
    for t in TARGETS_TO_RUN:
        y = np.asarray(NPZ_LBL[t]).astype(int); Xw, _ = official_window(FEATURES[t], y); EX[t] = Xw.to(DEVICE); EY[t] = y
        print(f'  eval set {t}: {len(y)} samples (full, de-duplicated), window tensor {tuple(Xw.shape)}')
    return EX, EY

# ---------------------------------------------------------------- 2-EPOCH SMOKE TEST with the REAL model (spec 9: before any sweep)
print('\nSMOKE TEST'); SMOKE_OK = False
m = CropNet(data_channel=1, classes=NUM_CLASSES).to(DEVICE); out = m(torch.randn(8, 1, 10, 43, device=DEVICE)); assert out.shape == (8, 7); out.sum().backward()
print('  [PASS] official CropNet forward/backward (8,1,10,43) ->', tuple(out.shape))
wx, _ = official_window(np.random.rand(6, 1, 10, 73).astype(np.float32), np.zeros(6, dtype=int)); assert tuple(wx.shape) == (6, 1, 10, 43); print('  [PASS] official window (6,1,10,73) -> (6,1,10,43)')
_r = np.random.default_rng(0); _mu = _r.normal(0, 1, (3, 730)).astype(np.float32)
def _fake(n, labs): y = np.repeat(labs, n); return (_mu[y] + _r.normal(0, 1, (len(y), 730))).astype(np.float32).reshape(len(y), 1, 10, 73), y
Xa, ya = _fake(80, [0, 1, 2]); Xb, yb_ = _fake(20, [0, 1, 2]); t0 = time.time(); r = fit_model(Xa, ya, Xb, yb_, 0, epochs_override=3)
assert set(r['info']) >= {'val_f1', 'best_epoch', 'epochs_run', 'stop_reason', 'restore_delta', 'history'} and r['info']['restore_delta'] <= RESTORE_TOL and math.isfinite(r['info']['val_f1']) and r['info']['epochs_run'] == 3 and r['info']['stop_reason'] == 'max_epochs'
print(f"  [PASS] fit_model (3 epochs; 4 crops absent -> weight 0): val F1 {r['info']['val_f1']:.2f}, best epoch {r['info']['best_epoch']}, restore delta {r['info']['restore_delta']:.1e}, {time.time() - t0:.1f}s")
# R7: augmentation ON = authors' 'augm' (seeded here): 3x the training rows, deterministic under the same seed, validation untouched
r2 = fit_model(Xa, ya, Xb, yb_, 0, epochs_override=2, aug='on'); assert r2['info']['epochs_run'] == 2 and math.isfinite(r2['info']['val_f1'])
_d1 = CropGlobeDataset(aug_mode='augm', re__peat=AUG_REPEAT); _d1.rng = np.random.default_rng(1); _d2 = CropGlobeDataset(aug_mode='augm', re__peat=AUG_REPEAT); _d2.rng = np.random.default_rng(1)
_t1 = _d1(TensorDataset(torch.as_tensor(Xa[:30], dtype=torch.float32), torch.as_tensor(ya[:30]).long())); _t2 = _d2(TensorDataset(torch.as_tensor(Xa[:30], dtype=torch.float32), torch.as_tensor(ya[:30]).long()))
assert len(_t1) == 30 * (1 + AUG_REPEAT) and torch.equal(_t1.tensors[0], _t2.tensors[0]) and torch.equal(_t1.tensors[0][:30], official_window(Xa[:30], ya[:30])[0]), 'augmentation is not seed-deterministic / does not keep the unaugmented rows first'
print('  [PASS] augmentation ON (authors\' augm): 3x rows, first block = unaugmented official window, identical under the same seed, 2-epoch fit runs')
r3 = fit_model(Xa, ya, Xb, yb_, 0, epochs_override=2, aug='copy'); assert r3['info']['epochs_run'] == 2 and math.isfinite(r3['info']['val_f1'])
_dc = CropGlobeDataset(aug_mode='none')(TensorDataset(torch.cat([torch.as_tensor(Xa[:30], dtype=torch.float32)] * (1 + AUG_REPEAT)), torch.cat([torch.as_tensor(ya[:30]).long()] * (1 + AUG_REPEAT))))
assert len(_dc) == 30 * (1 + AUG_REPEAT) and all(torch.equal(_dc.tensors[0][i * 30:(i + 1) * 30], official_window(Xa[:30], ya[:30])[0]) for i in range(1 + AUG_REPEAT)), 'copy control is not (1+AUG_REPEAT) exact copies of the unaugmented official window'
print('  [PASS] copy control (v3.8): (1 + AUG_REPEAT)x rows, every block = the unaugmented official window, 2-epoch fit runs')
# V33-07: a non-finite value in ONE batch must stop the run even if the LAST batch of the epoch is finite (the shuffled loader puts the bad sample in a random batch)
_Xl, _yl = _fake(200, [0, 1, 2]); _Xl[[0, 100, 200, 300, 400, 500]] = np.nan; _caught = None          # 600 samples = 3 batches of <=256; NaN samples spread over the shuffled batches
try: fit_model(_Xl, _yl, Xb, yb_, 0, epochs_override=2)
except FloatingPointError as _e: _caught = str(_e)
assert _caught and ('non-finite' in _caught), f'a NaN training sample was not caught: {_caught}'; print(f'  [PASS] non-finite sample in an earlier batch is caught ({_caught})')
EVAL_X, EVAL_Y = prepare_eval_sets(); FIT_FN, PREDICT_FN = fit_model, predict_tensor
print('\n  one REAL cell per target, 2 epochs (full pipeline incl. both scorers + integrity checks; results NOT kept):')
for t in TARGETS_TO_RUN:
    cl = [c for c in PLAN_CELLS_RUN if c['target'] == t]
    if not cl: print(f'   {t}: no runnable cell -- skipped'); continue
    cell = cl[0]; t0 = time.time(); row, preds, y_true, _sc = run_one(cell, 0, 'SMOKE|' + make_run_id(cell, 0), tempfile.gettempdir(), epochs_override=2, save=False)
    assert len(preds) == len(y_true) == row['test_n'] and row['epochs_run'] <= 2 and row['integrity_pass'] is True, f'{t}: smoke run failed integrity: {row}'
    print(f"   [PASS] {t} B{cell['budget']} k{cell['k']} {row['states']}: {row['primary_view']} mF1 {_pct(row['primary_f1'])} | paper-rule mF1 {_pct(row['paper_f1'])} | integrity PASS | {time.time() - t0:.1f}s (2 epochs: skill is not meaningful)")
for _tag, _cl in (('k-center', [c for c in PLAN_CELLS_RUN if c.get('sel') == 'kcenter']), ('aug ON', [c for c in PLAN_CELLS if c.get('aug') == 'on']), ('aug COPY (steps-matched control)', [c for c in PLAN_CELLS if c.get('aug') == 'copy'])):
    if not _cl: continue
    cell = _cl[0]; t0 = time.time(); row, preds, y_true, _sc = run_one(cell, 0, 'SMOKE|' + make_run_id(cell, 0), tempfile.gettempdir(), epochs_override=2, save=False)
    assert row['integrity_pass'] is True and row['aug'] == cell.get('aug', 'off') and row['sample_sel'] == cell.get('sel', 'random'), f'{_tag}: smoke run failed: {row}'
    print(f"   [PASS] {_tag} {cell['target']} B{cell['budget']} k{cell['k']} {row['states']}: steps/epoch {row['steps_per_epoch']} (train rows seen per epoch {row['train_n_effective']}) | integrity PASS | {time.time() - t0:.1f}s")
SMOKE_OK = True; print('\nSMOKE TEST PASSED  -> run the seed-0 cell next.')

## 12. INSTRUCTOR APPROVAL

In [ ]:
# ======================================================================================
# INSTRUCTOR APPROVAL (fill this cell in ONLY after the instructor has signed the decisions).
#   1. Run every cell above this one.  The plan cell printed the DESIGN HASH below and saved it to tables/design_hash.txt.
#   2. Send the instructor: the plan table, tables/feasibility_grid.csv, and this hash.
#   3. After he/she signs, paste the hash here and set INSTRUCTOR_APPROVED = True.  Run this cell, then the run cell.
#   If anything in the design changes later (a setting, a budget, a seed, the data), the hash changes and the run cell refuses to start.
# ======================================================================================
INSTRUCTOR_APPROVED = False
APPROVED_DESIGN_SHA256 = ''                      # <- paste the 64-character hash that the instructor approved
CURRENT = design_hash()
print('DESIGN HASH of the plan that would run now:\n  ', CURRENT)
print('approval switch:', INSTRUCTOR_APPROVED, '| hash pasted:', bool(APPROVED_DESIGN_SHA256), '| matches the current plan:', APPROVED_DESIGN_SHA256 == CURRENT)
print('=> the run cell will', 'START' if (INSTRUCTOR_APPROVED and APPROVED_DESIGN_SHA256 == CURRENT) else 'REFUSE to start (this is intended until the instructor has signed)')

## 12b. Re-score ALL finished USA rows with the clean fixed-class rule (L2-D11; once, from saved predictions, no retraining)

In [ ]:
# ======================================================================================
# RE-SCORE ALL USA ROWS WITH THE CLEAN FIXED-CLASS RULE  -- L2-D11, requested by the instructor after seeds 0-3 were seen.  NO retraining: it reads the SAVED predictions of each run.
#   rule: score corn, soybeans, wheat, other.  TRUE rice / sugarcane / cotton samples are removed from the USA test set (the paper does the same for sugarcane and cotton in FRA->USA, Sec. 4.4;
#         extending it to rice is the instructor's decision); predictions of rice / sugarcane / cotton are counted as 'other'.  The score no longer depends on which regions were in the training mix.
#   * covers EVERY seed, rows still on the original coverage-conditional rule and rows already on L2-D9; run it ONCE before the run cell; safe to run again (finished rows are skipped)
#   * it first checks that the saved predictions reproduce the numbers already in results.csv (if not: STOP, nothing is written)
#   * the original results.csv is kept byte-for-byte as results.csv.before_rescore_<time> (SHA-256 in rescore_log.json); earlier scores stay in main_f1_before_fixed4 / main_f1_before_clean
#   * the run cell refuses to continue while any finished USA row has another rule
# ======================================================================================
_res_path = os.path.join(OUT_DIR, 'results.csv')
if not os.path.exists(_res_path): print('no results.csv yet -> nothing to re-score (new runs are scored with the clean rule from the start).')
else:
    RESCORE_REPORT = rescore_usa_fixed4(OUT_DIR)
    if len(RESCORE_REPORT):
        R_ = RESCORE_REPORT.copy(); pd.set_option('display.width', 200)
        print(f'\nUSA rows re-scored: {len(R_)} (seeds {sorted(R_.seed.unique().tolist())}) | mean main F1 before {100 * R_.old.mean():.2f} -> after {100 * R_.new.mean():.2f}')
        print('by the rule the row had before:'); print((100 * R_.groupby('old_rule')[['old', 'v1', 'new']].mean()).round(2).join(R_.groupby('old_rule').size().rename('n')).to_string())
        print(f'per run: true samples removed (rice/sugarcane/cotton) mean {R_.n_true_dropped.mean():.0f}, predictions moved to other mean {R_.n_pred_reassigned.mean():.0f}, samples scored mean {R_.n_scored.mean():.0f}')
        R_.drop(columns=['i']).to_csv(os.path.join(OUT_DIR, 'tables', 'rescore_fixed4_clean_rows.csv'), index=False)
    else: print('all finished USA rows already use the clean fixed-class rule.')

## 13. RUN (resumable: run again after a disconnect)

In [ ]:
# ======================================================================================
# LEVEL 2 RUN (refuses to start until INSTRUCTOR_APPROVED and the approved design hash match).  AUTO_MODE (default): runs EVERY seed and EVERY planned run (aug OFF and aug ON, all arms) in one go, resumable -- run this cell again after a disconnect or time limit.
#   Between seeds the gate report is auto-approved only if the seed is complete with no error / integrity failure; otherwise the run stops and tells you why.
#   AUTO_MODE = False: one seed per execution (set RUN_SEED, approve the previous gate report in the config cell), as before.
# ======================================================================================
assert TESTS_PASSED and SMOKE_OK and APPROVED['plan'], 'test gate, smoke test and plan approval are required'
if AUTO_MODE:
    AUTO_RESULT = run_all_seeds(AUTO_MAX_HOURS)
else:
    assert RUN_SEED in SEEDS, f'RUN_SEED must be one of {SEEDS}'
    assert run_guards(), 'seed-consistency guards failed (see above)'
    if run_seed_cell(RUN_SEED) == 'complete':
        print(f'\n>>> Gate report saved. To continue: review it, then set APPROVED["seed{RUN_SEED}_gate_report"] = True and RUN_SEED = {RUN_SEED + 1}.' if RUN_SEED < SEEDS[-1] else '\n>>> All seeds done: run the summary cell.')

## 14. Summary tables and figures (safe at any time)

In [ ]:
# ======================================================================================
# SUMMARY (safe to run at any time).  Unit = REGION combination (French regions); the seeds are averaged FIRST.  Binary (ARG/CHN) and multiclass (USA) F1 are never compared with each other.
# ======================================================================================
import matplotlib
if not IN_NOTEBOOK: matplotlib.use('Agg')
import matplotlib.pyplot as plt
if PROFILE == 'extension': print('=' * 110 + f'\n  EXTENSION APPENDIX ({TARGETS_TO_RUN} at budgets {BUDGETS}; random arm only; allocation = as equal as each state\'s capacity allows, floor {EXT_MIN_CONTRIB:.0%} of the equal share).\n  These results are NOT part of the main study: do not pool them with the main results.csv; the combinations here overlap heavily (few states), so intervals are descriptive.\n  Compare with main results only at the same target and k, and always look at max_state_share (how much of the budget the largest single state supplies).\n' + '=' * 110)
RESULTS_CSV = os.path.join(OUT_DIR, 'results.csv'); df = load_results(RESULTS_CSV); _check(len(df) > 0, 'no results yet')
print('runs by status:', df['status'].value_counts().to_dict()); ok = df[df['status'] == 'ok']
print(f'seeds present: {sorted(pd.to_numeric(ok.seed).unique().astype(int).tolist())} (final statistics need all of {SEEDS}); integrity-failed / errored runs are EXCLUDED from the statistics')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); os.makedirs(os.path.join(OUT_DIR, 'tables'), exist_ok=True)
SUMMARY = {}
def summarize(comp, metric, label, full=True):
    dfc = df[(df['composition'].fillna(BASE_MODE).replace('', BASE_MODE) == comp) & df['arms'].fillna('random').replace('', 'random').astype(str).str.split('|').apply(lambda x: 'random' in x)]; CS = cell_stats(dfc, metric)   # legacy tables = the RANDOM arm only (unchanged method)
    if not len(CS): return None
    GAIN = gain_kmin_kmax(CS); REG = regression_report(dfc, metric); tag = f'{comp}__{metric}'
    for name, t in [('cell_stats', CS), ('gain_kmin_kmax', GAIN), ('regression', REG)]: atomic_write(os.path.join(OUT_DIR, 'tables', f'{name}__{tag}.csv'), lambda f, t=t: t.to_csv(f, index=False), mode='w')
    SUMMARY[(comp, metric)] = (CS, GAIN, REG)
    if full:
        print(f'\n########## [{comp}] {label}  (column {metric}) ##########')
        show = CS.copy()
        for c in ['mean', 'ci_lo', 'ci_hi', 'jackknife_lo', 'jackknife_hi', 'seed_sd_within_combo', 'seed_sd_of_cell_means']: show[c] = (100 * show[c]).round(2)
        print('--- per cell (macro-F1 %; k>=2: 95% t-interval with finite-population correction over state combinations; k=1 census: no CI; jackknife = delete-one-state) ---')
        print(show.drop(columns=['census_values', 'se', 'jackknife_se', 'census', 'exhaustive']).to_string(index=False))
        if show.exhaustive.any(): print('   * rows marked "exhaustive" had EVERY combination enumerated (C(P,k) <= %d): the interval is a point BY CONSTRUCTION, not evidence of certainty about unseen states (see the jackknife columns).' % MAX_COMBOS)
        for r in CS[CS.census].itertuples(): print(f'  k=1 census {r.target} B{r.budget}: {r.census_values}')
        g2 = GAIN.copy()
        for c in ['gain', 'ci_lo', 'ci_hi']: g2[c] = (100 * g2[c]).round(2)
        print('--- gain k_min -> k_max (pp, with CI) ---'); print(g2.to_string(index=False))
        print('--- regression of combination-level F1 on k, n_target_classes_train_ge30, budget ---'); print(REG.round(5).to_string(index=False))
    return CS
comps = sorted(set(df['composition'].fillna(BASE_MODE).replace('', BASE_MODE)))
if 'arms' in df.columns and (df['arms'].fillna('').astype(str).str.contains('diverse|clustered|common|kcenter|aug_on')).any():
    print('\n########## ARMS: random = baseline | diverse / clustered = state-diversity selection (Phase 4) | common = fixed common state pool | kcenter = k-center SAMPLE selection | aug_on = authors\' augmentation ON ##########')
    _E = expand_arms(df)
    for _comp in comps:
        _Ec = _E[_E['composition'].fillna(BASE_MODE).replace('', BASE_MODE) == _comp]
        for _arm in ARM_ORDER:
            _Ea = _Ec[_Ec['arm'] == _arm]
            if not len(_Ea): continue
            _CS = cell_stats(_Ea, REPORT_PRIMARY); atomic_write(os.path.join(OUT_DIR, 'tables', f'cell_stats__{_comp}__arm_{_arm}.csv'), lambda f, t=_CS: t.to_csv(f, index=False), mode='w')
            _sh = _CS[['target', 'budget', 'k', 'n_combos', 'mean', 'ci_lo', 'ci_hi', 'ci_note']].copy()
            for c_ in ['mean', 'ci_lo', 'ci_hi']: _sh[c_] = (100 * _sh[c_]).round(2)
            print(f'--- [{_comp}] arm = {_arm} (MAIN metric, %) ---'); print(_sh.to_string(index=False))
        for _a in ('diverse', 'clustered', 'common', 'kcenter', 'aug_on', 'aug_copy'):
            if _a in ('kcenter', 'aug_on', 'aug_copy'):
                _C = arm_paired_contrast(_Ec.drop(columns=['arm']), REPORT_PRIMARY, _a, 'random')
                if len(_C):
                    atomic_write(os.path.join(OUT_DIR, 'tables', f'arm_paired__{_comp}__{_a}_vs_random.csv'), lambda f, t=_C: t.to_csv(f, index=False), mode='w'); _s = _C.copy()
                    for c_ in ['diff', 'ci_lo', 'ci_hi']: _s[c_] = (100 * _s[c_]).round(2)
                    print(f'--- [{_comp}] PAIRED contrast {_a} - random on the SAME state combinations (pp, MAIN metric; units overlap in states: DESCRIPTIVE) ---'); print(_s.round(4).to_string(index=False))
                continue
            if _a == 'common': continue                                  # different pool: reported as its own arm table + the budget table below, not as a contrast
            _C = arm_contrast(_Ec.drop(columns=['arm']), REPORT_PRIMARY, _a, 'random')
            if len(_C):
                atomic_write(os.path.join(OUT_DIR, 'tables', f'arm_contrast__{_comp}__{_a}_vs_random.csv'), lambda f, t=_C: t.to_csv(f, index=False), mode='w')
                _s = _C.copy()
                for c_ in ['diff', 'ci_lo', 'ci_hi']: _s[c_] = (100 * _s[c_]).round(2)
                print(f'--- [{_comp}] contrast {_a} - random (pp; Welch interval over combinations, which share states: DESCRIPTIVE) ---'); print(_s.round(4).to_string(index=False))
        _A = diversity_assoc(_Ec[_Ec['arm'].isin(['random', 'diverse', 'clustered'])].drop(columns=['arm']).drop_duplicates('run_id'), REPORT_PRIMARY)
        if len(_A): print(f'--- [{_comp}] Spearman(F1, diversity percentile) per cell (EXPLORATORY) ---'); print(_A.round(3).to_string(index=False))
    _Cm = _E[_E['arm'] == 'common']
    if len(_Cm):
        print('\n--- FIXED COMMON STATE POOL: MAIN metric (%) by budget and k, every budget drawn from the SAME states (no pool-shrinkage confound) ---')
        _tab = _Cm.assign(m=pd.to_numeric(_Cm[REPORT_PRIMARY], errors='coerce')).groupby(['target', 'k', 'budget'])['m'].mean().unstack('budget').mul(100).round(2); print(_tab.to_string())
        atomic_write(os.path.join(OUT_DIR, 'tables', 'common_pool_budget_by_k.csv'), lambda f: _tab.to_csv(f), mode='w')
    _Au = _E[_E['arm'] == 'aug_on']
    if len(_Au): print('\n   NOTE aug_on: the augmented runs see %dx the training rows per epoch (more optimizer steps); the aug_on - random contrast mixes augmentation with training length.  The steps-matched comparison is aug_on - aug_copy below.' % (1 + AUG_REPEAT))
    for _comp in comps:
        _Ec = _E[_E['composition'].fillna(BASE_MODE).replace('', BASE_MODE) == _comp]
        if (_Ec['arm'] == 'aug_copy').any() and (_Ec['arm'] == 'aug_on').any():
            _C = arm_paired_contrast(_Ec.drop(columns=['arm']), REPORT_PRIMARY, 'aug_on', 'aug_copy')
            if len(_C):
                atomic_write(os.path.join(OUT_DIR, 'tables', f'arm_paired__{_comp}__aug_on_vs_aug_copy.csv'), lambda f, t=_C: t.to_csv(f, index=False), mode='w'); _s = _C.copy()
                for c_ in ['diff', 'ci_lo', 'ci_hi']: _s[c_] = (100 * _s[c_]).round(2)
                print(f'--- [{_comp}] STEPS-MATCHED contrast aug_on - aug_copy on the SAME combinations (pp, MAIN metric): the part of the augmentation gain that is NOT just more steps ---'); print(_s.round(4).to_string(index=False))
    print('   NOTE: diverse / clustered combinations are CHOSEN by score, not sampled; their intervals describe the selected sets only.  A single target country per track: no claim beyond the countries tested.')
print('\n########## v3.8 GENERALISATION STATISTICS (random arm; MAIN metric, points) ##########')
_MM = mixed_model_report(df, REPORT_PRIMARY)
if len(_MM):
    atomic_write(os.path.join(OUT_DIR, 'tables', 'mixed_model.csv'), lambda f: _MM.to_csv(f, index=False), mode='w')
    print('--- mixed model: F1 ~ log2(k) + budget/1000 + seed, random intercept = region combination (REML; t-based p, df = combinations - parameters) ---'); print(_MM.round(4).to_string(index=False))
_SS = single_state_report(df, REPORT_PRIMARY)
if len(_SS):
    atomic_write(os.path.join(OUT_DIR, 'tables', 'single_state_vs_combinations.csv'), lambda f: _SS.to_csv(f, index=False), mode='w'); _x = _SS.copy()
    for c_ in ['single_mean', 'single_best', 'single_worst', 'combo_mean', 'combo_worst', 'combo_best', 'avg_gain', 'avg_gain_ci_lo', 'avg_gain_ci_hi', 'worst_case_gain', 'worst_vs_best_single']: _x[c_] = _x[c_].round(2)
    _x['share_beating_best_single'] = (100 * _x['share_beating_best_single']).round(0)
    print('--- single states (k=1 census) vs k>=2 combinations: average gain (bootstrap interval over combinations), worst-case gain, share of combinations beating the BEST single state (%) ---'); print(_x.to_string(index=False))
_RT = replication_table(df, REPORT_PRIMARY)
if len(_RT):
    atomic_write(os.path.join(OUT_DIR, 'tables', 'replication_exploratory_vs_confirmatory.csv'), lambda f: _RT.to_csv(f, index=False), mode='w'); _y = _RT.copy()
    for c_ in ['estimate', 'ci_lo', 'ci_hi']: _y[c_] = _y[c_].round(2)
    print('--- PRE-DECLARED CLAIMS (V38-P1): exploratory seeds 0-1 vs confirmatory seeds 2-4.  PASS = same sign in both groups and the confirmatory interval excludes 0.  A failed claim is reported as NOT REPLICATED; no seed is dropped or re-run. ---'); print(_y.to_string(index=False))
print('compositions present:', comps, '| tracks are reported SEPARATELY; binary (ARG/CHN) is never compared with multiclass (USA)')
for comp in comps:
    summarize(comp, REPORT_PRIMARY, 'MAIN metric (ARG/CHN: binary key-crop macro-F1; USA: macro-F1 over the FIXED classes corn, soybeans, wheat, other; true rice/sugarcane/cotton removed, their predictions -> other (L2-D11))'); summarize(comp, REPORT_SECONDARY, 'SECONDARY: paper-style macro-F1 (read with the coverage columns)')
    if REPORT_ALL_TRACKS:
        for col, lab in TRACKS:
            if col not in (REPORT_PRIMARY, REPORT_SECONDARY): summarize(comp, col, lab, full=False)
print('   NOTE: the target-crop COUNT equals fixed share x budget exactly (aliased with budget), so the regression uses n_target_classes_train_ge30 instead. Rows from different budgets reuse the same states (not independent).')
print('\n--- HEADLINE: MAIN metric and SECONDARY (paper-style) metric, each with its coverage-conditional version beside it (%%; cov-cond over classes with >=%d train samples) ---' % COVERAGE_MIN_TRAIN)
for comp in comps:
    dfc = df[(df['composition'].fillna(BASE_MODE).replace('', BASE_MODE) == comp) & df['arms'].fillna('random').replace('', 'random').astype(str).str.split('|').apply(lambda x: 'random' in x)]; frames = []
    for metric, cov, lab in ((REPORT_PRIMARY, 'cov_cond_primary_f1', 'MAIN'), (REPORT_SECONDARY, 'cov_cond_paper_f1', 'SECONDARY')):
        a = combo_level(dfc, metric).groupby(['target', 'budget', 'k'])['f1'].mean().rename(f'{lab} main'); b_ = combo_level(dfc, cov).groupby(['target', 'budget', 'k'])['f1'].mean().rename(f'{lab} cov-cond')
        frames += [a, b_]
    H = pd.concat(frames, axis=1); H = (100 * H).round(1); print(f'[{comp}]'); print(H.to_string())
print('\n--- optimizer steps (budget is confounded with training length) ---')
_st = ok.copy()
for c_ in ['budget', 'steps_per_epoch', 'total_optimizer_steps', 'epochs_run']: _st[c_] = pd.to_numeric(_st[c_], errors='coerce')
_st = _st[_st['aug'].fillna('off').astype(str) == 'off'] if 'aug' in _st.columns else _st
print('(aug-OFF runs only; aug-ON runs have %dx the steps per epoch)' % (1 + AUG_REPEAT)); print(_st.groupby('budget').agg(steps_per_epoch=('steps_per_epoch', 'first'), epochs_run_median=('epochs_run', 'median'), total_steps_min=('total_optimizer_steps', 'min'), total_steps_median=('total_optimizer_steps', 'median'), total_steps_max=('total_optimizer_steps', 'max')).to_string())
print('\n--- class coverage by target and k ---')
_cv = ok.copy()
for c_ in ['k', 'n_absent_target_classes_train', 'n_target_classes_train_ge30']: _cv[c_] = pd.to_numeric(_cv[c_], errors='coerce')
print(_cv.groupby(['target', 'k']).agg(runs=('k', 'size'), share_with_absent_class=('n_absent_target_classes_train', lambda x: round(float((x > 0).mean()), 3)), mean_absent=('n_absent_target_classes_train', 'mean'), mean_classes_ge30=('n_target_classes_train_ge30', 'mean')).round(2).to_string())
SEED_WARN = seed_direction_warnings(df); print('\n--- seed-direction check (WARNING only) ---'); print('\n'.join(SEED_WARN) if SEED_WARN else '  no disagreement')
if REPORT_ALL_TRACKS:
    print('\n--- ALL TRACK COLUMNS (mean over runs, %; each column has its OWN definition, see TRACKS in the config cell; no track is pooled or ranked) ---')
    b = ok.copy(); cols = [c for c, _ in TRACKS] + ['baseline_all_other_primary', 'baseline_all_other_review', 'cov_cond_primary_f1', 'cov_cond_review_f1']
    for c in cols: b[c] = pd.to_numeric(b[c], errors='coerce')
    print((100 * b.groupby(['composition', 'target'])[cols].mean()).round(2).to_string())
    for col, lab in TRACKS: print(f'   {col:14s} = {lab}')
fl = ok['skill_flags'].fillna('').astype(str).str.split('|').explode().replace('', np.nan).dropna(); print('skill flags raised:', fl.str.replace(r':.*', '', regex=True).value_counts().to_dict())
CS = SUMMARY.get((comps[0], REPORT_PRIMARY), (None,))[0]
tg = [t for t in TARGETS_ORDER if CS is not None and t in set(CS.target)]
if tg:
    fig, axes = plt.subplots(2, len(tg), figsize=(5.2 * len(tg), 8), squeeze=False)
    for r_, (metric, rowlab) in enumerate(((REPORT_PRIMARY, 'MAIN'), (REPORT_SECONDARY, 'SECONDARY (paper-style)'))):
        CSm = SUMMARY.get((comps[0], metric), (None,))[0]
        for ax, t in zip(axes[r_], tg):
            if CSm is None or t not in set(CSm.target): continue
            for B, g in CSm[CSm.target == t].groupby('budget'):
                samp = g[~g.census & ~g.exhaustive]
                line = ax.errorbar(g.k, g['mean'], yerr=None, marker='o', ls='-', label=f'B={B}')
                if len(samp): ax.errorbar(samp.k, samp['mean'], yerr=[(samp['mean'] - samp.ci_lo).fillna(0), (samp.ci_hi - samp['mean']).fillna(0)], fmt='none', ecolor=line[0].get_color(), capsize=3)
                ex = g[g.exhaustive]; ce = g[g.census]
                if len(ex): ax.scatter(ex.k, ex['mean'], marker='D', facecolors='none', edgecolors=line[0].get_color(), s=70, zorder=5)        # hollow diamond = exhaustive, CI degenerate
                if len(ce): ax.scatter(ce.k, ce['mean'], marker='s', facecolors='none', edgecolors=line[0].get_color(), s=70, zorder=5)         # hollow square = k=1 census, no CI
            ax.set_xscale('log', base=2); ks = sorted(CSm[CSm.target == t].k.unique()); ax.set_xticks(ks); ax.set_xticklabels(ks); ax.set_xlabel('k (states)'); ax.set_ylabel('macro-F1'); ax.set_title(f'{t} - {rowlab}\n(bars = 95% CI; hollow square = census, hollow diamond = exhaustive: no CI drawn)', fontsize=8); ax.legend(fontsize=7)
    fig.tight_layout(); atomic_write(os.path.join(OUT_DIR, 'tables', 'f1_vs_k_by_track.png'), lambda f: fig.savefig(f, dpi=130), mode='wb'); plt.show() if IN_NOTEBOOK else None

## 15. Closing analysis: freeze, pre-registered claims, exploratory additions, cross-checks (after ALL seeds)

In [ ]:
# ======================================================================================
# LEVEL 2 CLOSING ANALYSIS (read-only).  Run ONLY after all five seeds are finished.  Nothing here trains a model or writes to results.csv.
#   1. FREEZE: SHA-256 of results.csv is written to closing/FREEZE_level2.json (paste it into LEVEL2_FREEZE.md; after that nobody can change a number unnoticed)
#   2. CLAIMS (pre-registered): L2-C1 and L2-C2 on ARG and USA, evaluated separately on seeds 0-1 (exploratory) and 2-4 (confirmatory); PASS = same sign AND confirmatory 95% interval excludes 0;
#      anything else is printed as NOT REPLICATED.  CHN is descriptive only.  L2-C3 (aug_on - aug_copy) is reported with its interval, no direction assumed.
#   3. CROSS-CHECK of the mixed model with statsmodels (declared tolerance) and the training-quality robustness table.
#   The verdict words are produced by the code from the numbers; they are not edited by hand.
# ======================================================================================
import hashlib, json, os
import numpy as np, pandas as pd
from scipy import stats as scipy_stats
CLOSING_TOL = dict(coef=0.02, se=0.02)           # statsmodels vs own REML must agree this closely, declared before the cross-check was run
N_BOOT_CLOSING = 2000

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()

def load_final(path):
    d = load_results(path); d = d[d['status'] == 'ok'].drop_duplicates('run_id', keep='last').copy(); d['seed'] = pd.to_numeric(d['seed'])
    d['track'] = np.where(d['aug'].astype(str) == 'copy', 'aug_copy', np.where(d['aug'].astype(str) == 'on', 'aug_on', np.where(d['sample_sel'].astype(str) == 'kcenter', 'kcenter', 'base')))
    d['y'] = 100 * pd.to_numeric(d['main_f1'], errors='coerce'); return d

def _mixed_design(g):
    g = g.dropna(subset=['y']).copy(); g['lk'] = np.log2(g['k']); g['B'] = g['budget'] / 1000.0; g['combo'] = g['target'].astype(str) + '|' + g['states'].astype(str); seeds = sorted(g['seed'].dropna().unique())
    cols, names = [np.ones(len(g)), g['lk'].values], ['intercept', 'log2(k)']
    if g['B'].nunique() > 1: cols.append(g['B'].values); names.append('budget/1000')
    for s in seeds[1:]: cols.append((g['seed'].values == s).astype(float)); names.append(f'seed{int(s)}')
    return g, np.column_stack(cols), names

def own_vs_statsmodels(df, targets=('ARG', 'USA', 'CHN'), try_install=True):
    # fits the SAME model (random arm, F1 ~ log2(k) + budget/1000 + seed dummies, random intercept per state combination, REML) with the notebook's own code and with statsmodels MixedLM
    out = []; sm = None; err = ''
    try:
        import statsmodels.formula.api as smf; sm = smf
    except Exception:
        if try_install:
            try:
                import subprocess, sys; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'], check=True); import statsmodels.formula.api as smf; sm = smf
            except Exception as e: err = f'{type(e).__name__}: {str(e)[:80]}'
    base = _random_arm_runs(df); base['y'] = 100 * pd.to_numeric(base['main_f1'], errors='coerce')
    for t in targets:
        g, X, names = _mixed_design(base[base['target'] == t])
        if g['k'].nunique() < 2: continue
        r = reml_random_intercept(g['y'].values, X, g['combo'].values); row = dict(target=t, n=len(g), groups=int(r['G']), own_coef=float(r['beta'][1]), own_se=float(np.sqrt(r['cov'][1, 1])), sm_coef=np.nan, sm_se=np.nan, sm_p=np.nan, agree='NOT RUN (statsmodels unavailable' + (f': {err}' if err else '') + ')')
        if sm is not None:
            import warnings
            gg = g.copy(); gg['seedf'] = gg['seed'].astype(int).astype(str); terms = ['lk'] + (['B'] if g['B'].nunique() > 1 else []) + (['C(seedf)'] if g['seed'].nunique() > 1 else []); formula = 'y ~ ' + ' + '.join(terms)
            tried = []; fitted = None
            for scale in (1.0, 0.1):                              # second pass: y rescaled (coefficients and SEs are rescaled back) in case the optimiser failed on the scale of y
                for meth in ('lbfgs', 'bfgs', 'powell', 'nm', 'cg'):
                    try:
                        with warnings.catch_warnings():
                            warnings.simplefilter('ignore'); d2 = gg.assign(y=gg['y'] * scale); m = sm.mixedlm(formula, d2, groups=d2['combo']).fit(reml=True, method=meth)
                        if np.isfinite(m.params['lk']) and np.isfinite(m.bse['lk']) and m.bse['lk'] > 0: fitted = (m, meth, scale); break
                        tried.append(f'{meth}x{scale}: non-finite')
                    except Exception as e: tried.append(f'{meth}x{scale}: {type(e).__name__}')
                if fitted: break
            if fitted:
                m, meth, scale = fitted; row.update(sm_coef=float(m.params['lk']) / scale, sm_se=float(m.bse['lk']) / scale, sm_p=float(m.pvalues['lk']), sm_method=f'{meth}' + ('' if scale == 1.0 else ' (y x0.1)'))
                row['agree'] = 'YES' if (abs(row['sm_coef'] - row['own_coef']) <= CLOSING_TOL['coef'] and abs(row['sm_se'] - row['own_se']) <= CLOSING_TOL['se']) else 'NO -- report the statsmodels figures and investigate'
            else: row['agree'] = 'statsmodels fit failed with every optimiser: ' + '; '.join(tried[:4])
        out.append(row)
    return pd.DataFrame(out)

def robustness_table(df, target='ARG', budgets=(1000, 2000), k_lo=1, k_hi=8, n_boot=None, seed=0):
    # does the benefit of more states survive better training?  Gain k_lo -> k_hi on the SAME cells for plain training, the steps-matched copy control and augmentation.
    n_boot = n_boot or N_BOOT_CLOSING; rng = np.random.default_rng(seed); d = df[df['target'] == target].copy(); rows = []
    key = ['budget', 'k', 'states']
    cp = d[d['track'] == 'aug_copy'].groupby(key)['y'].mean(); on = d[d['track'] == 'aug_on'].groupby(key)['y'].mean(); pl = d[d['track'] == 'base'].groupby(key)['y'].mean()
    cells = cp.index   # like-for-like: plain training restricted to the cells that have a copy-control run
    pl = pl.reindex(cells)
    for B in budgets:
        out = dict(target=target, budget=B)
        for nm, s in (('plain', pl), ('copy_control', cp), ('augmentation', on.reindex(cells))):
            s = s[s.index.get_level_values('budget') == B]; lo = s[s.index.get_level_values('k') == k_lo].dropna(); hi = s[s.index.get_level_values('k') == k_hi].dropna()
            if len(lo) == 0 or len(hi) == 0: out[nm + '_gain'] = np.nan; continue
            out[nm + '_gain'] = float(hi.mean() - lo.mean())
            bs = [float(rng.choice(hi.values, len(hi)).mean() - rng.choice(lo.values, len(lo)).mean()) for _ in range(n_boot)]; out[nm + '_lo'], out[nm + '_hi'] = float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))
        rows.append(out)
    # does the improvement from better training depend on the number of states?  (combination-level regression of copy-control minus plain on log2 k)
    gdiff = (cp - pl).dropna().reset_index(); gdiff.columns = ['budget', 'k', 'states', 'g']
    sl = scipy_stats.linregress(np.log2(gdiff['k']), gdiff['g']) if len(gdiff) > 3 and gdiff['k'].nunique() > 1 else None
    dep = dict(slope=float(sl.slope), p=float(sl.pvalue), n_combinations=int(len(gdiff))) if sl else dict(slope=np.nan, p=np.nan, n_combinations=int(len(gdiff)))
    return pd.DataFrame(rows), dep


def freeze_results(path, out_dir):
    # records the hash and the shape of the finished results file.  Refuses an unfinished file (a hash of a partial file would freeze nothing).
    d = pd.read_csv(path); ok = d[d['status'] == 'ok']; planned = {s_: len(plan_for_seed(s_)) for s_ in SEEDS}
    per = {int(k): int(v) for k, v in ok.groupby(pd.to_numeric(ok['seed'], errors='coerce')).size().to_dict().items()}
    complete = (per == planned) and int(d['run_id'].duplicated().sum()) == 0 and int((d['status'] != 'ok').sum()) == 0
    rec = dict(results_file=os.path.basename(path), sha256=sha256_file(path), n_rows=int(len(d)), runs_per_seed=per, planned_per_seed=planned, complete=bool(complete), design_sha256=APPROVED_DESIGN_SHA256, level1_results_sha256='38d1876cde08e54cfc827aa6b2813355ef98a09dbd048885a67d58991534f891')
    if complete: atomic_write(os.path.join(out_dir, 'FREEZE_level2.json'), lambda f: f.write(json.dumps(rec, indent=1)), mode='w')
    return rec

def c3_aug_contrast(df, target='ARG', budgets=(1000, 2000), exploratory=(0, 1), confirmatory=(2, 3, 4), n_boot=None, seed=0):
    # L2-C3: aug_on minus aug_copy (same cells, same optimizer steps).  Per budget and per seed group: mean over cells of (mean over the group's seeds), with a bootstrap-over-cells 95% interval.  No direction is assumed.
    n_boot = n_boot or N_BOOT_CLOSING; rng = np.random.default_rng(seed); d = df[(df['target'] == target) & df['track'].isin(['aug_on', 'aug_copy'])].copy(); rows = []
    for B in budgets:
        for label, seeds in (('exploratory', exploratory), ('confirmatory', confirmatory)):
            g = d[(d['budget'] == B) & d['seed'].isin(seeds)]
            m = g.groupby(['k', 'states', 'track'])['y'].mean().unstack('track')
            if not {'aug_on', 'aug_copy'} <= set(m.columns): continue
            diff = (m['aug_on'] - m['aug_copy']).dropna().values
            if len(diff) < 2: continue
            bs = [float(rng.choice(diff, len(diff)).mean()) for _ in range(n_boot)]
            rows.append(dict(claim='C3 aug_on - aug_copy (points)', target=target, budget=B, group=label, seeds=sorted(int(s) for s in g['seed'].unique()), estimate=float(diff.mean()), ci_lo=float(np.percentile(bs, 2.5)), ci_hi=float(np.percentile(bs, 97.5)), n_cells=int(len(diff))))
    t = pd.DataFrame(rows)
    if not len(t): return t
    v = {}
    for B, g in t.groupby('budget'):
        ex, co = g[g['group'] == 'exploratory'], g[g['group'] == 'confirmatory']
        if not len(co) or not len(ex): v[B] = 'confirmatory seeds not finished'; continue
        e, c = ex.iloc[0], co.iloc[0]; same = np.sign(e['estimate']) == np.sign(c['estimate']); excl = c['ci_lo'] > 0 or c['ci_hi'] < 0
        v[B] = ('REPLICATED: augmentation beats its steps-matched copy' if c['estimate'] > 0 else 'REPLICATED: augmentation is worse than its steps-matched copy') if (same and excl) else ('NOT REPLICATED (sign differs)' if not same else 'NOT REPLICATED (confirmatory interval includes 0)')
    t['verdict'] = t['budget'].map(v); return t


# ---------------- EXPLORATORY additions requested in the instructor review (NOT pre-registered claims; every table is labelled exploratory) ----------------
HIGH_REGIONS_ARG = ('75', '76')                  # the two Argentine-like French regions that are far better than the rest as single regions (descriptive label only)

def _base_runs(df, target):
    return df[(df['track'] == 'base') & (df['target'] == target)].copy()

def mix_vs_own(df, target, n_boot=None, seed=0):
    """EXPLORATORY.  For every region mix (k>=2) at a budget: its seed-averaged F1 minus (a) the mean and (b) the best of its OWN single regions at the SAME budget.
    Mixes are not independent (they share regions and single-region runs), so the p-values are descriptive; the bootstrap resamples mixes."""
    n_boot = n_boot or N_BOOT_CLOSING; g = _base_runs(df, target); sc = g.groupby(['budget', 'k', 'states'])['y'].mean().reset_index(); sc['states'] = sc['states'].astype(str)
    single = sc[sc['k'] == 1].set_index(['budget', 'states'])['y']; rows = []
    for x in sc[sc['k'] >= 2].itertuples():
        mem = x.states.split('-')
        if not all((x.budget, m) in single.index for m in mem): continue
        v = [float(single[(x.budget, m)]) for m in mem]; rows.append(dict(budget=int(x.budget), k=int(x.k), states=x.states, y=float(x.y), own_mean=float(np.mean(v)), own_best=float(np.max(v)), has_high=bool(set(HIGH_REGIONS_ARG) & set(mem))))
    return pd.DataFrame(rows)

def _gap_summary(gap, n_boot, seed):
    gap = np.asarray(gap, float); gap = gap[np.isfinite(gap)]
    if len(gap) < 3: return dict(n=len(gap), mean_gap=np.nan, lo=np.nan, hi=np.nan, share_better=np.nan, p_t=np.nan, p_wilcoxon=np.nan)
    r = np.random.default_rng(seed); bs = np.array([gap[r.integers(0, len(gap), len(gap))].mean() for _ in range(n_boot)])
    try: pw = float(scipy_stats.wilcoxon(gap).pvalue)
    except Exception: pw = np.nan
    return dict(n=len(gap), mean_gap=float(gap.mean()), lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5)), share_better=float((gap > 0).mean()), p_t=float(scipy_stats.ttest_1samp(gap, 0).pvalue), p_wilcoxon=pw)

def mix_vs_own_table(df, targets=('ARG', 'USA'), n_boot=None):
    out = []
    for t in targets:
        m = mix_vs_own(df, t)
        if not len(m): continue
        for lab, sub in [('all mixes', m)] + ([('without 75/76', m[~m.has_high]), ('containing 75/76', m[m.has_high])] if t == 'ARG' else []):
            for ref, col in [('mean of own single regions', 'own_mean'), ('best own single region', 'own_best')]:
                r_ = _gap_summary(sub['y'] - sub[col], n_boot or N_BOOT_CLOSING, 0); r_.update(target=t, subset=lab, reference=ref, status='EXPLORATORY (not pre-registered)'); out.append(r_)
    cols = ['target', 'subset', 'reference', 'n', 'mean_gap', 'lo', 'hi', 'share_better', 'p_t', 'p_wilcoxon', 'status']
    return pd.DataFrame(out)[cols] if out else pd.DataFrame(columns=cols)

def fixed_pool_budget_effect(df, target='ARG'):
    """EXPLORATORY / correction.  Budget effect for single-region runs using ONLY regions that exist at every budget (the pool otherwise shrinks as the budget grows, which would change the average by itself)."""
    a = _base_runs(df, target); a = a[a['k'] == 1].copy(); a['r'] = a['states'].astype(str)
    if not len(a): return pd.DataFrame()
    nb = a['budget'].nunique(); av = a.groupby('r')['budget'].nunique(); common = sorted(av[av == nb].index.tolist())
    f = a[a['r'].isin(common)].groupby('budget')['y'].mean(); allr = a.groupby('budget')['y'].mean(); regs = a.groupby('budget')['r'].nunique()
    return pd.DataFrame(dict(target=target, budget=f.index.astype(int), fixed_pool_mean_f1=f.values, fixed_pool_regions=len(common), all_available_mean_f1=allr.reindex(f.index).values, regions_available=regs.reindex(f.index).values, fixed_pool_change_from_lowest_budget=f.values - f.values[0]))

def per_doubling_subset(df, target='ARG', exclude=HIGH_REGIONS_ARG):
    """EXPLORATORY.  Mixed-model log2(k) slope (same model as L2-C1, all seeds pooled) with and without runs whose region combination contains an excluded region."""
    rows = []
    for lab, drop in [('all combinations', False), ('without ' + '/'.join(exclude), True)]:
        b = _random_arm_runs(df); b = b[b['target'] == target]
        if drop: b = b[~b['states'].astype(str).apply(lambda s: bool(set(exclude) & set(s.split('-'))))]
        if b['k'].nunique() < 2: continue
        g, X, names = _mixed_design(b); r = reml_random_intercept(g['y'].values, X, g['combo'].values); se = float(np.sqrt(r['cov'][1, 1]))
        rows.append(dict(target=target, subset=lab, n=len(g), groups=int(r['G']), per_doubling=float(r['beta'][1]), se=se, lo=float(r['beta'][1]) - 1.96 * se, hi=float(r['beta'][1]) + 1.96 * se, status='EXPLORATORY'))
    return pd.DataFrame(rows)

def closing_report_l2(results_path, out_dir):
    os.makedirs(out_dir, exist_ok=True); fr = freeze_results(results_path, out_dir)
    print('FREEZE:', json.dumps({k: fr[k] for k in ('sha256', 'n_rows', 'runs_per_seed', 'planned_per_seed', 'complete')}))
    if not fr['complete']: print('STOPPED: the results file is not complete (see runs_per_seed vs planned_per_seed). Finish all seeds, then run this cell again. Nothing was frozen.'); return None
    df = load_final(results_path); out = dict(freeze=fr)
    print('\n--- pre-registered claims L2-C1 / L2-C2 (exploratory seeds 0-1 vs confirmatory seeds 2-4) ---'); rep = replication_table(df); print(rep.round(2).to_string(index=False)); rep.to_csv(os.path.join(out_dir, 'replication_exploratory_vs_confirmatory.csv'), index=False); out['replication'] = rep
    print('\n--- L2-C3 steps-matched augmentation contrast (Argentina; no direction assumed) ---'); c3 = c3_aug_contrast(df); print(c3.round(2).to_string(index=False) if len(c3) else '(no aug_on / aug_copy runs)'); c3.to_csv(os.path.join(out_dir, 'c3_aug_vs_copy.csv'), index=False); out['c3'] = c3
    print('\n--- mixed model: own REML code vs statsmodels (tolerance declared in advance: coef %.2f, SE %.2f) ---' % (CLOSING_TOL['coef'], CLOSING_TOL['se']))
    cmp_ = own_vs_statsmodels(df); print(cmp_.round(3).to_string(index=False)); cmp_.to_csv(os.path.join(out_dir, 'mixed_model_own_vs_statsmodels.csv'), index=False); out['cross'] = cmp_
    print('\n--- robustness: gain from 1 to 8 regions (points), Argentina, same cells: plain vs steps-matched copy vs augmentation ---')
    rb, dep = robustness_table(df); print(rb.round(2).to_string(index=False)); rb.to_csv(os.path.join(out_dir, 'robustness_more_regions_vs_training.csv'), index=False); out['robustness'] = rb; out['dep'] = dep
    print('copy-control improvement over plain training vs number of regions: slope %.2f points per doubling, p = %.3f (%d combinations)' % (dep['slope'], dep['p'], dep['n_combinations']))
    print('\nCHN is DESCRIPTIVE (rice exists in two regions only); its rows above carry the verdict "descriptive only (no claim)".')
    print('\n=== EXPLORATORY (requested in the instructor review; NOT pre-registered claims; not used for any verdict above) ===')
    mv = mix_vs_own_table(df); out['mix_vs_own'] = mv
    if len(mv): print('--- every region mix vs its OWN single regions at the same budget (mixes share regions: p-values are descriptive) ---'); print(mv.round(4).to_string(index=False)); mv.to_csv(os.path.join(out_dir, 'exploratory_mix_vs_own_regions.csv'), index=False)
    fp = pd.concat([fixed_pool_budget_effect(df, t) for t in ('ARG', 'USA')], ignore_index=True)
    out['fixed_pool'] = fp
    if len(fp): print('\n--- budget effect on single regions with the pool held FIXED (regions available at every budget); use this figure, not the change in the all-available average ---'); print(fp.round(2).to_string(index=False)); fp.to_csv(os.path.join(out_dir, 'exploratory_fixed_pool_budget_effect.csv'), index=False)
    pd_ = per_doubling_subset(df, 'ARG'); out['per_doubling_subset'] = pd_
    if len(pd_): print('\n--- per-doubling slope with and without regions 75/76 (exploratory; motivates the separate follow-up plan, which must be pre-registered and run on NEW seeds) ---'); print(pd_.round(3).to_string(index=False)); pd_.to_csv(os.path.join(out_dir, 'exploratory_per_doubling_with_without_75_76.csv'), index=False)
    print('\nUSA test-set ordering verification:', globals().get('USA_ORDER_STATUS', 'NOT RUN in this session (run the ordering-verification cell; the USA verdicts must not be submitted without it)'))
    print('USA scoring rule in force:', USA_SCORING, '| true classes removed:', USA_DROP_TRUE_CLASSES)
    print('Note: Argentina verdicts above are the pre-registered ones and stay as printed; the exploratory tables never change them.')
    return out

def _selftest_closing():
    # runs every time this cell is executed: the claim code must recover a KNOWN effect, and must say NOT REPLICATED for a sign flip and NOT TESTABLE without an interval
    r = np.random.default_rng(1); rows = []
    for seed in range(5):
        for ci in range(10):
            cell = ('11', '24', ci)
            base = 50 + r.normal(0, 1)
            rows.append(dict(target='ARG', budget=1000, k=2, states=str(cell), seed=seed, track='aug_copy', y=base))
            rows.append(dict(target='ARG', budget=1000, k=2, states=str(cell), seed=seed, track='aug_on', y=base + 2.0 + r.normal(0, .3)))                       # known effect +2
            rows.append(dict(target='ARG', budget=2000, k=2, states=str(cell), seed=seed, track='aug_copy', y=base))
            rows.append(dict(target='ARG', budget=2000, k=2, states=str(cell), seed=seed, track='aug_on', y=base + (1.0 if seed < 2 else -1.0) + r.normal(0, .1)))   # sign flips between groups
    t = c3_aug_contrast(pd.DataFrame(rows)); v = t.groupby('budget')['verdict'].first().to_dict()
    assert abs(t[(t.budget == 1000) & (t.group == 'confirmatory')]['estimate'].iloc[0] - 2.0) < 0.2, 'C3 contrast does not recover the known +2 effect'
    assert v[1000].startswith('REPLICATED') and v[2000].startswith('NOT REPLICATED (sign differs)'), v
    print('closing-analysis self-test passed: known +2 effect recovered (verdict:', v[1000] + '); sign flip ->', v[2000] + ')')
_selftest_closing()

def _selftest_exploratory():
    r = np.random.default_rng(5); rows = []
    for seed in range(3):
        for b in (1000, 2000):
            for reg, base in [('11', 50), ('24', 60), ('75', 70)]: rows.append(dict(target='ARG', budget=b, k=1, states=reg, seed=seed, track='base', y=base + (3 if b == 2000 else 0) + r.normal(0, .1)))
            for combo, v in [('11-24', 58), ('11-75', 63), ('24-75', 65)]: rows.append(dict(target='ARG', budget=b, k=2, states=combo, seed=seed, track='base', y=v + (3 if b == 2000 else 0) + r.normal(0, .1)))
    d = pd.DataFrame(rows); m = mix_vs_own(d, 'ARG'); assert len(m) == 6
    x = m[(m.states == '11-24') & (m.budget == 1000)].iloc[0]; assert abs(x.own_mean - 55) < .2 and abs(x.own_best - 60) < .2 and abs((x.y - x.own_mean) - 3) < .3
    t = mix_vs_own_table(d, ('ARG',), n_boot=200); a = t[(t.subset == 'all mixes') & (t.reference == 'mean of own single regions')].iloc[0]
    assert a.n == 6 and np.isfinite(a.mean_gap) and 0 <= a.share_better <= 1
    assert (t[t.subset == 'containing 75/76'].n == 4).all() and (t[t.subset == 'without 75/76'].n == 2).all()
    fp = fixed_pool_budget_effect(d); assert abs(fp.fixed_pool_change_from_lowest_budget.iloc[-1] - 3) < .2
    d2 = d[~((d.k == 1) & (d.states == '75') & (d.budget == 1000))]; fp2 = fixed_pool_budget_effect(d2); assert fp2.fixed_pool_regions.iloc[0] == 2           # a region missing at one budget is excluded from the fixed pool
    print('exploratory-analysis self-test passed (mix vs own mean/best, 75/76 split, fixed pool)')
_selftest_exploratory()

## 16. Appendix: Belgium / Netherlands / UK label check (descriptive only, no claims)

In [ ]:
# ======================================================================================
# APPENDIX (instructor item c): Belgium / Netherlands / United Kingdom -- label check ONLY.  Descriptive, NO claims, low priority, never gates anything, writes only tables/appendix_label_counts.csv.
# Before any of these countries could be SCORED, its real label set must be verified (a score needs the classes that exist in the test data).  This cell only counts labels; if a file is absent it says so.
# ======================================================================================
def appendix_label_counts(data_dir, countries=('BEL', 'NLD', 'GBR'), log=print):
    rows = []
    for c in countries:
        try: path = find_npz(data_dir, c, log=lambda *a, **k: None)
        except Exception as e: path = None; log(f'  {c}: {type(e).__name__}: {str(e)[:80]}')
        if not path: rows.append(dict(country=c, status='file not found (not downloaded; nothing assumed)', n=0)); continue
        lbl = np.load(path)['lbl'].astype(int); cnt = np.bincount(lbl, minlength=7)
        rows.append(dict(country=c, status='counted', n=int(len(lbl)), **{CROP_LABELS[i]: int(cnt[i]) for i in range(7)}, classes_present=', '.join(CROP_LABELS[i] for i in range(7) if cnt[i] > 0)))
    return pd.DataFrame(rows)
APPENDIX_LABELS = appendix_label_counts(DATA_DIR)
print(APPENDIX_LABELS.to_string(index=False)); os.makedirs(os.path.join(OUT_DIR, 'tables'), exist_ok=True); APPENDIX_LABELS.to_csv(os.path.join(OUT_DIR, 'tables', 'appendix_label_counts.csv'), index=False)
print('APPENDIX ONLY: no claim is made for these countries; a country may be scored only after its label set above has been checked against the paper.')

## 15b. Run the closing analysis
After every seed is finished: `closing_report_l2(os.path.join(OUT_DIR, 'results.csv'), os.path.join(OUT_DIR, 'closing'))`. Pre-registered claims come first and are never changed by the exploratory tables printed after them.

In [ ]:
# run ONLY after all five seeds are finished (it refuses an incomplete results file)
# CLOSING = closing_report_l2(os.path.join(OUT_DIR, 'results.csv'), os.path.join(OUT_DIR, 'closing'))
